In [ ]:
# =============================================================================
# CELL 0: ENVIRONMENT SETUP
# =============================================================================
# Install required packages for TRIVET Tier1 validation pipeline
# Run this cell first after opening the notebook
# =============================================================================

!pip install -q anthropic==0.42.0
!pip install -q python-Levenshtein
!pip install -q sentence-transformers
!pip install -q google-generativeai

import os
import json
import math
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# Create output directory
OUTPUT_DIR = "tier1_v3_3_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("CELL 0: ENVIRONMENT SETUP COMPLETE")
print("=" * 70)
print(f"✅ Packages installed")
print(f"✅ Output directory: {OUTPUT_DIR}/")
print(f"✅ Ready for Cell 1")

In [ ]:
# === CELL 1: Load Data ===
# Original: Cell 1
# Action: KEEP (cleaned up)
# ============================================================================

# =============================================================================
# CELL 1: LOAD DATA
# =============================================================================
# Load Tier1 candidates from CSV
# Expected: 2,770 rows with columns: ar, mt_output, Sub-Subtype, Keyword, Best_Match
# =============================================================================

import pandas as pd
import os
# =============================================================================
# Mount Google Drive
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')  # ✅ Mount at root

# Change to your working directory
WORKING_DIR = '/content/drive/MyDrive/25/Final/MT_Q/P3_F'
os.chdir(WORKING_DIR)

# Load Tier1 candidates
TIER1_INPUT = "candidates_TIER1.csv"  # Update path if needed

df_tier1 = pd.read_csv(TIER1_INPUT)

print("=" * 70)
print("CELL 1: LOAD DATA COMPLETE")
print("=" * 70)
print(f"✅ Loaded: {TIER1_INPUT}")
print(f"✅ Rows: {len(df_tier1):,}")
print(f"✅ Columns: {list(df_tier1.columns)}")

# Verify required columns
required_cols = ['ar', 'mt_output', 'Sub-Subtype', 'Keyword', 'Best_Match']
missing = [c for c in required_cols if c not in df_tier1.columns]
if missing:
    print(f"⚠️ Missing columns: {missing}")
else:
    print(f"✅ All required columns present")

# Show error type distribution
print(f"\n📊 Error Type Distribution:")
print(df_tier1['Sub-Subtype'].value_counts().head(10))


# === CELL 2: Load LaBSE Model ===
# Original: Cell 5 (lines 1-30)
# Action: EXTRACT (LaBSE loading only)
# ============================================================================


In [ ]:
# Action: EXTRACT (LaBSE loading only)
# ============================================================================

# =============================================================================
# CELL 2: LOAD LaBSE MODEL
# =============================================================================
# Load LaBSE model for semantic similarity (SFR calculation)
# This takes ~1-2 minutes on first run (downloads model)
# =============================================================================

import os
from sentence_transformers import SentenceTransformer

# =============================================================================
# FIX: Map 02_Final names to TAXONOMY_23 / Threshold Table names
# =============================================================================

NAME_MAPPING = {
    'Definiteness': 'Definiteness Shift',
    'Formality Level': 'Register Mismatch',
    'Terminological Substitution': 'Terminology Substitution',
    'Negation': 'Tense Shift Under Negation',
}

print("\n" + "=" * 80)
print("FIX: Mapping class names to threshold table names")
print("=" * 80)

for old, new in NAME_MAPPING.items():
    count = (df_tier1['Sub-Subtype'] == old).sum()
    if count > 0:
        df_tier1['Sub-Subtype'] = df_tier1['Sub-Subtype'].replace({old: new})
        print(f"   ✅ '{old}' → '{new}': {count} samples")

print(f"\nUnique classes after fix: {df_tier1['Sub-Subtype'].nunique()}")
print(f"Class distribution:")
print(df_tier1['Sub-Subtype'].value_counts())

# # Option 1: Load from local cache (if you have it saved)
LABSE_PATH = "/content/drive/MyDrive/LaBSE_model"

# Option 2: Load from HuggingFace (will download if not cached)
LABSE_HF = "sentence-transformers/LaBSE"

print("=" * 70)
print("CELL 2: LOADING LaBSE MODEL")
print("=" * 70)

if os.path.exists(LABSE_PATH):
    print(f"📂 Loading from local path: {LABSE_PATH}")
    labse_model = SentenceTransformer(LABSE_PATH)
else:
    print(f"🌐 Loading from HuggingFace: {LABSE_HF}")
    print("   (This may take 1-2 minutes on first run)")
    labse_model = SentenceTransformer(LABSE_HF)

print(f"\n✅ LaBSE model loaded: {type(labse_model).__name__}")
print(f"✅ Ready for Cell 3")
# === CELL 3: Load Rubric ===
# Original: Cell 5 (lines 31-152)
# Action: EXTRACT (Rubric loading only)
# ============================================================================

In [ ]:
# =============================================================================
# CELL 2.1: QUICK NAME VERIFICATION (Run after Cell 2)
# =============================================================================

print("=" * 80)
print("QUICK NAME VERIFICATION")
print("=" * 80)

# TAXONOMY_23 reference
TAXONOMY_23 = [
    "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
    "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
    "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
    "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
    "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
    "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
]

# Old names that should NOT exist anymore
OLD_NAMES = ['Definiteness', 'Formality Level', 'Terminological Substitution', 'Negation']

# New names that SHOULD exist now
NEW_NAMES = ['Definiteness Shift', 'Register Mismatch', 'Terminology Substitution', 'Tense Shift Under Negation']

# Get current classes
current_classes = set(df_tier1['Sub-Subtype'].unique())

# Check old names (should be GONE)
print("\n❌ OLD NAMES (should be 0):")
for old in OLD_NAMES:
    count = (df_tier1['Sub-Subtype'] == old).sum()
    status = "✅ GONE" if count == 0 else f"❌ STILL EXISTS: {count}"
    print(f"   {old}: {status}")

# Check new names (should EXIST)
print("\n✅ NEW NAMES (should have counts):")
for new in NEW_NAMES:
    count = (df_tier1['Sub-Subtype'] == new).sum()
    status = f"✅ {count} samples" if count > 0 else "❌ MISSING"
    print(f"   {new}: {status}")

# Overall alignment check
in_data_not_taxonomy = current_classes - set(TAXONOMY_23)
in_taxonomy_not_data = set(TAXONOMY_23) - current_classes

print("\n" + "=" * 80)
print("ALIGNMENT CHECK")
print("=" * 80)

if not in_data_not_taxonomy:
    print("✅ All class names in data match TAXONOMY_23")
else:
    print(f"❌ Classes in data but NOT in TAXONOMY_23:")
    for c in sorted(in_data_not_taxonomy):
        print(f"   - {c}")

print(f"\n⚠️ Classes in TAXONOMY_23 but not in data (may be OK - rare in real MT):")
for c in sorted(in_taxonomy_not_data):
    print(f"   - {c}")

print("\n" + "=" * 80)
if not in_data_not_taxonomy:
    print("✅ NAMES FIXED CORRECTLY - Proceed to Cell 3, 4, 5")
else:
    print("❌ NAME FIX INCOMPLETE - Check Cell 2 code")
print("=" * 80)

In [ ]:
# =============================================================================
# CELL 3: LOAD RUBRIC
# =============================================================================
# Load RUBRIC_23 from CSV (single source of truth for thresholds)
# Source: threshold_reference_table_v2_OCS.csv
# =============================================================================

import pandas as pd

RUBRIC_CSV = "threshold_reference_table_v2_OCS.csv"  # Update path if needed

print("=" * 70)
print("CELL 3: LOADING RUBRIC")
print("=" * 70)

# Load rubric CSV
rubric_df = pd.read_csv(RUBRIC_CSV)
print(f"✅ Loaded: {RUBRIC_CSV}")
print(f"✅ Error types: {len(rubric_df)}")

# Build RUBRIC_23 dictionary
RUBRIC_23 = {}

# Regime definitions
REGIME_F = {
    "Definiteness Shift", "Gender Disagreement", "Tanween Omission",
    "Spelling Error", "Register Mismatch", "Terminology Substitution",
    "Hypernym for Hyponym", "Hyponym for Hypernym", "Name Entity Error",
    "Total Omission", "Partial Translation", "Literal Translation",
    "Invalid Pattern", "Active to Passive Voice", "Passive to Active Voice",
    "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Adjective to Noun", "Noun to Adjective",
    "Wrong Word Order", "Wrong Structure",
}

REGIME_D = {"Meaning Shift"}

STRUCTURE_TOLERANT = {
    "Wrong Word Order", "Wrong Structure", "Active to Passive Voice",
    "Passive to Active Voice", "Perfective to Progressive",
    "Progressive to Perfective", "Tense Shift Under Negation",
}

for _, row in rubric_df.iterrows():
    et = str(row["Error_Type"]).strip()
    inverted = bool(int(row["Inverted"])) if str(row["Inverted"]).strip() else False

    regime = "D" if et in REGIME_D else "F"
    cps_floor = 0.45 if et in STRUCTURE_TOLERANT else 0.55

    RUBRIC_23[et] = {
        "class": str(row["Class"]).strip(),
        "min_csr": float(row["CSR_Min"]),
        "min_ocs": float(row["OCS_Min"]),
        "min_sfr": float(row["SFR_Min"]),
        "max_sfr": float(row["SFR_Max"]),
        "weight_csr": float(row["Weight_CSR"]),
        "weight_ocs": float(row["Weight_OCS"]),
        "weight_sfr": float(row["Weight_SFR"]),
        "inverted": inverted,
        "regime": regime,
        "cps_floor": cps_floor,
    }
    if inverted:
        RUBRIC_23[et]["max_csr"] = float(row["CSR_Min"])

RUBRIC_23["_rubric_source"] = RUBRIC_CSV
RUBRIC_23["_version"] = "3.3"

# Verify 23 classes
error_keys = [k for k in RUBRIC_23.keys() if not str(k).startswith("_")]
print(f"✅ RUBRIC_23 built: {len(error_keys)} error classes")

# Show regime breakdown
regime_f_count = sum(1 for k in error_keys if RUBRIC_23[k]["regime"] == "F")
regime_d_count = sum(1 for k in error_keys if RUBRIC_23[k]["regime"] == "D")
print(f"   - Regime F (Fidelity): {regime_f_count} classes")
print(f"   - Regime D (Divergence): {regime_d_count} classes")

if len(error_keys) != 23:
    print(f"⚠️ WARNING: Expected 23 classes, got {len(error_keys)}")
else:
    print(f"✅ All 23 classes loaded")

print(f"\n✅ Ready for Cell 4")


# === CELL 4: Load Validation Module v3.3 ===
# Original: Cell 2 + NEW
# Action: MERGE (module import + v3.3 verification)
# ============================================================================


In [ ]:
# =============================================================================
# CELL 4: LOAD VALIDATION MODULE v3.3
# =============================================================================
# Import the v3.3 validation module with clitic-aware matching
# IMPORTANT: Upload tier1_validation_implementation_v2_STAGE_A_V2.py first!
# =============================================================================

import importlib

print("=" * 70)
print("CELL 4: LOADING VALIDATION MODULE v3.3")
print("=" * 70)

# Import the module
try:
    import tier1_validation_implementation_v2_STAGE_A_V2 as validation_module
    importlib.reload(validation_module)
    print(f"✅ Module imported: tier1_validation_implementation_v2_STAGE_A_V2")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print(f"   Please upload: tier1_validation_implementation_v2_STAGE_A_V2.py")
    raise

# Verify v3.3 features
v33_checks = {
    "clitic_aware_match": hasattr(validation_module, 'clitic_aware_match'),
    "ARABIC_PREFIXES": hasattr(validation_module, 'ARABIC_PREFIXES'),
    "ARABIC_SUFFIXES": hasattr(validation_module, 'ARABIC_SUFFIXES'),
    "normalize_arabic_ocs": hasattr(validation_module, 'normalize_arabic_ocs'),
    "run_full_tier1_validation": hasattr(validation_module, 'run_full_tier1_validation'),
}

print(f"\n🔍 v3.3 Feature Verification:")
all_passed = True
for feature, present in v33_checks.items():
    status = "✅" if present else "❌"
    print(f"   {status} {feature}")
    if not present:
        all_passed = False

if not all_passed:
    print(f"\n⚠️ WARNING: Some v3.3 features missing!")
    print(f"   Make sure you uploaded the correct module version.")
else:
    print(f"\n✅ All v3.3 features confirmed")

# Show clitic patterns
if hasattr(validation_module, 'ARABIC_PREFIXES'):
    print(f"\n📋 Clitic Prefixes: {validation_module.ARABIC_PREFIXES[:5]}...")
if hasattr(validation_module, 'ARABIC_SUFFIXES'):
    print(f"📋 Clitic Suffixes: {validation_module.ARABIC_SUFFIXES[:5]}...")

# Quick functional test
if hasattr(validation_module, 'clitic_aware_match'):
    matched, rule = validation_module.clitic_aware_match('الحالة', 'حالة')
    print(f"\n🧪 Quick Test: 'الحالة' matches 'حالة'?")
    print(f"   Result: {matched}, Rule: {rule}")
    if matched and 'prefix' in rule:
        print(f"   ✅ Clitic matching working correctly")
    else:
        print(f"   ⚠️ Unexpected result - check module")

print(f"\n✅ Ready for Cell 5 (Stage A Validation)")


# === CELL 5: Stage A - Full Tier1 Validation v3.3 ===
# Original: Cell 6
# Action: REPLACE (v3.2 → v3.3)
# ============================================================================


In [ ]:
# =============================================================================
# CELL 5: STAGE A - FULL TIER1 VALIDATION v3.3
# =============================================================================
# Run full validation on all 2,770 Tier1 candidates
# Expected runtime: ~45-60 minutes
# Output: tier1_v3_3_final/ folder with clean/noisy/discarded CSVs
# =============================================================================

import os

print("=" * 70)
print("CELL 5: STAGE A - FULL TIER1 VALIDATION v3.3")
print("=" * 70)

# Verify prerequisites
prereq_checks = {
    "df_tier1": 'df_tier1' in dir(),
    "labse_model": 'labse_model' in dir(),
    "RUBRIC_23": 'RUBRIC_23' in dir(),
    "validation_module": 'validation_module' in dir(),
}

print(f"📋 Prerequisites Check:")
all_ready = True
for prereq, ready in prereq_checks.items():
    status = "✅" if ready else "❌"
    print(f"   {status} {prereq}")
    if not ready:
        all_ready = False

if not all_ready:
    raise RuntimeError("Prerequisites not met. Run Cells 1-4 first!")

print(f"\n🚀 Starting v3.3 validation...")
print(f"   Samples: {len(df_tier1):,}")
print(f"   Output: {OUTPUT_DIR}/")
print(f"   Expected time: ~45-60 minutes")
print(f"\n" + "=" * 70)

# Run validation
results = validation_module.run_full_tier1_validation(
    df_tier1,
    RUBRIC_23,
    labse_model=labse_model,
    show_progress=True
)

# Store results globally for later cells
validation_results = results

print(f"\n✅ Validation complete!")
print(f"✅ Ready for Cell 6 (Save Outputs)")


# === CELL 6: Save Validation Outputs ===
# Action: UPDATE (paths to v3.3 folder)
# ============================================================================


In [ ]:
# =============================================================================
# CELL 6: SAVE VALIDATION OUTPUTS
# =============================================================================
# Save all validation results to tier1_v3_3_final/ folder
# Creates: tier1_clean.csv, tier1_clean_A.csv, tier1_clean_B.csv,
#          tier1_noisy.csv, tier1_discarded.csv, validation_stats.json
# =============================================================================

import json
import os

print("=" * 70)
print("CELL 6: SAVE VALIDATION OUTPUTS")
print("=" * 70)

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save CSVs
results = validation_results  # From Cell 5

results['clean'].to_csv(f"{OUTPUT_DIR}/tier1_clean.csv", index=False, encoding='utf-8-sig')
results['clean_a'].to_csv(f"{OUTPUT_DIR}/tier1_clean_A.csv", index=False, encoding='utf-8-sig')
results['clean_b'].to_csv(f"{OUTPUT_DIR}/tier1_clean_B.csv", index=False, encoding='utf-8-sig')
results['noisy'].to_csv(f"{OUTPUT_DIR}/tier1_noisy.csv", index=False, encoding='utf-8-sig')
results['discarded'].to_csv(f"{OUTPUT_DIR}/tier1_discarded.csv", index=False, encoding='utf-8-sig')

print(f"✅ Saved CSVs to {OUTPUT_DIR}/:")
print(f"   - tier1_clean.csv ({len(results['clean']):,} rows)")
print(f"   - tier1_clean_A.csv ({len(results['clean_a']):,} rows)")
print(f"   - tier1_clean_B.csv ({len(results['clean_b']):,} rows)")
print(f"   - tier1_noisy.csv ({len(results['noisy']):,} rows)")
print(f"   - tier1_discarded.csv ({len(results['discarded']):,} rows)")

# Save stats JSON
with open(f"{OUTPUT_DIR}/validation_stats.json", "w") as f:
    json.dump(results['stats'], f, indent=2, default=str)
print(f"   - validation_stats.json")

# Save match rules JSON (v3.3 specific)
if 'match_rule_counts' in results['stats']:
    with open(f"{OUTPUT_DIR}/match_rules.json", "w") as f:
        json.dump(results['stats']['match_rule_counts'], f, indent=2)
    print(f"   - match_rules.json")

# Summary
print(f"\n📊 VALIDATION SUMMARY:")
print(f"   Total:     {results['stats']['total_candidates']:,}")
print(f"   Clean:     {results['stats']['clean']:,} ({results['stats']['clean_rate']:.1%})")
print(f"   Clean_A:   {results['stats']['clean_a']:,} ({results['stats']['clean_a_rate']:.1%})")
print(f"   Clean_B:   {results['stats']['clean_b']:,} ({results['stats']['clean_b_rate']:.1%})")
print(f"   Noisy:     {results['stats']['noisy']:,} ({results['stats']['noisy_rate']:.1%})")
print(f"   Discarded: {results['stats']['discarded']:,} ({results['stats']['discard_rate']:.1%})")

print(f"\n✅ All outputs saved to {OUTPUT_DIR}/")
print(f"✅ Ready for Cell 7 (Diagnostics)")
# === CELL 7: Validation Diagnostics ===
# Original: Cells 8 + 9 + 10 + 11 + 19
# Action: MERGE (all diagnostic cells into one comprehensive cell)
# ============================================================================

In [ ]:
# =============================================================================
# CELL 7: VALIDATION DIAGNOSTICS
# =============================================================================
# Comprehensive diagnostics for validation results
# Merged from old Cells: 8 (rejection reasons), 9 (samples), 10 (CSR analysis),
#                        11 (Tier2 seed check), 19 (pilot diagnostics)
# =============================================================================

import pandas as pd

print("=" * 70)
print("CELL 7: VALIDATION DIAGNOSTICS")
print("=" * 70)

# Load results (in case running separately)
if 'validation_results' not in dir():
    print("📂 Loading from saved files...")
    clean_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean.csv")
    noisy_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_noisy.csv")
else:
    clean_df = validation_results['clean']
    noisy_df = validation_results['noisy']

# --- DIAGNOSTIC 1: Rejection Reasons by Error Type ---
print(f"\n" + "=" * 50)
print("DIAGNOSTIC 1: REJECTION REASONS BY ERROR TYPE")
print("=" * 50)

if 'reject_reason' in noisy_df.columns and 'Sub-Subtype' in noisy_df.columns:
    rejection_pivot = pd.crosstab(
        noisy_df['Sub-Subtype'],
        noisy_df['reject_reason'],
        margins=True
    )
    print(f"\nTop rejection reasons overall:")
    print(noisy_df['reject_reason'].value_counts().head(10))

    # OCS failure breakdown
    ocs_failed = noisy_df[noisy_df['reject_reason'] == 'ocs_failed']
    if len(ocs_failed) > 0 and 'ocs_failed_constraints' in ocs_failed.columns:
        print(f"\nOCS constraint failures:")
        print(ocs_failed['ocs_failed_constraints'].value_counts().head(10))

# --- DIAGNOSTIC 2: Clean Rate by Error Type ---
print(f"\n" + "=" * 50)
print("DIAGNOSTIC 2: CLEAN RATE BY ERROR TYPE")
print("=" * 50)

error_col = 'Sub-Subtype'
if error_col in clean_df.columns:
    # Calculate per-type rates
    all_types = set(clean_df[error_col].unique()) | set(noisy_df[error_col].unique())

    type_stats = []
    for et in sorted(all_types):
        clean_count = len(clean_df[clean_df[error_col] == et])
        noisy_count = len(noisy_df[noisy_df[error_col] == et])
        total = clean_count + noisy_count
        rate = clean_count / total if total > 0 else 0

        # Get regime
        regime = RUBRIC_23.get(et, {}).get('regime', 'F')

        type_stats.append({
            'error_type': et,
            'regime': regime,
            'clean': clean_count,
            'noisy': noisy_count,
            'total': total,
            'clean_rate': rate
        })

    type_df = pd.DataFrame(type_stats).sort_values('clean_rate', ascending=False)

    print(f"\n{'Error Type':<35} {'Regime':<8} {'Clean':<8} {'Total':<8} {'Rate':<8}")
    print("-" * 70)
    for _, row in type_df.iterrows():
        status = "✓" if row['clean_rate'] >= 0.10 else "⚠" if row['clean_rate'] > 0 else "✗"
        print(f"{status} {row['error_type']:<33} {row['regime']:<8} {row['clean']:<8} {row['total']:<8} {row['clean_rate']:.1%}")

# --- DIAGNOSTIC 3: v3.3 Clitic Match Rules ---
print(f"\n" + "=" * 50)
print("DIAGNOSTIC 3: v3.3 CLITIC MATCH RULES")
print("=" * 50)

if 'ocs_match_rule_kw' in clean_df.columns:
    print(f"\nKeyword match rules (clean samples):")
    print(clean_df['ocs_match_rule_kw'].value_counts().head(10))

if 'ocs_match_rule_bm' in clean_df.columns:
    print(f"\nBest_match match rules (clean samples):")
    print(clean_df['ocs_match_rule_bm'].value_counts().head(10))

# Count non-exact matches (rescued by clitic stripping)
if 'ocs_match_rule_kw' in clean_df.columns:
    exact_kw = (clean_df['ocs_match_rule_kw'] == 'exact').sum()
    clitic_kw = (clean_df['ocs_match_rule_kw'] != 'exact').sum()
    print(f"\nKeyword matches: {exact_kw} exact, {clitic_kw} via clitic rules")

# --- DIAGNOSTIC 4: Quality Tier Distribution ---
print(f"\n" + "=" * 50)
print("DIAGNOSTIC 4: QUALITY TIER DISTRIBUTION")
print("=" * 50)

if 'quality_tier' in clean_df.columns:
    print(f"\nQuality tiers:")
    print(clean_df['quality_tier'].value_counts())

# --- DIAGNOSTIC 5: Metric Distributions ---
print(f"\n" + "=" * 50)
print("DIAGNOSTIC 5: METRIC DISTRIBUTIONS (Clean)")
print("=" * 50)

for metric in ['csr', 'ocs', 'sfr', 'cps', 'mpqs']:
    if metric in clean_df.columns:
        vals = clean_df[metric].dropna()
        if len(vals) > 0:
            print(f"{metric.upper()}: min={vals.min():.3f}, mean={vals.mean():.3f}, max={vals.max():.3f}")

# --- DIAGNOSTIC 6: Sample Examples ---
print(f"\n" + "=" * 50)
print("DIAGNOSTIC 6: SAMPLE CLEAN EXAMPLES (5 per top type)")
print("=" * 50)

top_types = type_df.head(3)['error_type'].tolist() if 'type_df' in dir() else []
for et in top_types:
    samples = clean_df[clean_df[error_col] == et].head(2)
    print(f"\n[{et}] - {len(samples)} samples shown:")
    for idx, row in samples.iterrows():
        ar = str(row.get('ar', ''))[:60]
        mt = str(row.get('mt_output', ''))[:60]
        print(f"   ar: {ar}...")
        print(f"   mt: {mt}...")

print(f"\n" + "=" * 70)
print("✅ DIAGNOSTICS COMPLETE")
print("=" * 70)
print(f"\n✅ Phase 1 Complete! Ready for Phase 2 (Calibration + Selection)")
# === CELL 8: MPQS Calibration ===
# Original: Cell 25
# Action: UPDATE (adapted for v3.3 outputs)
# ============================================================================

In [ ]:
# DIAGNOSTIC: Rejection reason by error type
import pandas as pd

noisy_df = pd.read_csv('tier1_outputs_stage_a_v2/tier1_noisy.csv')

# Cross-tabulation: Error Type × Rejection Reason
error_col = 'Sub-Subtype'
reason_col = 'reject_reason'

# Create pivot table
pivot = pd.crosstab(
    noisy_df[error_col],
    noisy_df[reason_col],
    normalize='index'  # Row percentages
) * 100

print("="*80)
print("REJECTION REASON BY ERROR TYPE (% of noisy samples per class)")
print("="*80)
print(pivot.round(1).to_string())

# Highlight the worst classes
print("\n" + "="*80)
print("CLASSES WITH 100% REJECTION (0% clean rate)")
print("="*80)
zero_clean = ['Meaning Shift', 'Total Omission', 'Active to Passive Voice',
              'Passive to Active Voice', 'Wrong Structure', 'Literal Translation',
              'Partial Translation', 'Spelling Error', 'Name Entity Error',
              'Noun to Adjective', 'Adjective to Noun']

for et in zero_clean:
    if et in pivot.index:
        row = pivot.loc[et]
        dominant = row.idxmax()
        pct = row.max()
        print(f"  {et:35s}: {pct:5.1f}% due to {dominant}")
        ####
        # See how far off your CSR values are from thresholds
import pandas as pd

noisy_df = pd.read_csv('tier1_outputs_stage_a_v2/tier1_noisy.csv')

# Focus on csr_below_floor failures
csr_failures = noisy_df[noisy_df['reject_reason'] == 'csr_below_floor']

print("CSR DISTRIBUTION FOR csr_below_floor FAILURES")
print("="*60)

for et in ['Definiteness Shift', 'Gender Disagreement', 'Active to Passive Voice', 'Spelling Error']:
    subset = csr_failures[csr_failures['Sub-Subtype'] == et]
    if len(subset) > 0:
        threshold = subset['csr_threshold'].iloc[0] if 'csr_threshold' in subset.columns else 'N/A'
        print(f"\n{et} (n={len(subset)}, threshold={threshold}):")
        print(f"  CSR mean:   {subset['csr'].mean():.3f}")
        print(f"  CSR median: {subset['csr'].median():.3f}")
        print(f"  CSR min:    {subset['csr'].min():.3f}")
        print(f"  CSR max:    {subset['csr'].max():.3f}")
        print(f"  % within 0.05 of threshold: {(subset['csr'] >= threshold - 0.05).mean()*100:.1f}%")
        ####
        # Check your Tier2 seed distribution
tier2_seed = pd.read_csv('tier2_seed_from_clean.csv')
print("TIER2 SEED DISTRIBUTION")
print("="*40)
print(tier2_seed['Sub-Subtype'].value_counts().to_string())

In [ ]:
# === CELL 8: MPQS Calibration ===
# Original: Cell 25
# Action: UPDATE (adapted for v3.3 outputs)
# ============================================================================

# =============================================================================
# CELL 8: MPQS CALIBRATION ANALYSIS
# =============================================================================
# Analyze MPQS distribution and thresholds per error type
# Uses v3.3 validation outputs from tier1_v3_3_final/
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("CELL 8: MPQS CALIBRATION ANALYSIS")
print("=" * 80)

# Load v3.3 outputs
OUTPUT_DIR = "tier1_v3_3_final"

clean_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean.csv")
noisy_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_noisy.csv")

print(f"✅ Loaded v3.3 outputs from {OUTPUT_DIR}/")
print(f"   Clean: {len(clean_df):,} rows")
print(f"   Noisy: {len(noisy_df):,} rows")

# Combine for full analysis
all_df = pd.concat([clean_df, noisy_df], ignore_index=True)
all_df['track'] = ['clean'] * len(clean_df) + ['noisy'] * len(noisy_df)

# ============================================================================
# MPQS Distribution by Error Type
# ============================================================================

print("\n" + "=" * 80)
print("MPQS DISTRIBUTION BY ERROR TYPE")
print("=" * 80)

error_col = 'Sub-Subtype'

mpqs_stats = []
for et in sorted(all_df[error_col].unique()):
    et_df = all_df[all_df[error_col] == et]
    clean_et = et_df[et_df['track'] == 'clean']
    noisy_et = et_df[et_df['track'] == 'noisy']

    # Get regime from RUBRIC_23 if available
    regime = RUBRIC_23.get(et, {}).get('regime', 'F') if 'RUBRIC_23' in dir() else 'F'

    stats = {
        'error_type': et,
        'regime': regime,
        'total': len(et_df),
        'clean': len(clean_et),
        'noisy': len(noisy_et),
        'clean_rate': len(clean_et) / len(et_df) if len(et_df) > 0 else 0,
        'mpqs_mean_all': et_df['mpqs'].mean() if 'mpqs' in et_df.columns else np.nan,
        'mpqs_mean_clean': clean_et['mpqs'].mean() if len(clean_et) > 0 and 'mpqs' in clean_et.columns else np.nan,
        'mpqs_p5_all': et_df['mpqs'].quantile(0.05) if 'mpqs' in et_df.columns else np.nan,
        'mpqs_p5_clean': clean_et['mpqs'].quantile(0.05) if len(clean_et) > 0 and 'mpqs' in clean_et.columns else np.nan,
    }
    mpqs_stats.append(stats)

mpqs_df = pd.DataFrame(mpqs_stats).sort_values('clean_rate', ascending=False)

# Display
print(f"\n{'Error Type':<35} {'Regime':<6} {'Clean':<8} {'Rate':<8} {'MPQS(clean)':<12} {'P5(clean)':<10}")
print("-" * 85)
for _, row in mpqs_df.iterrows():
    mpqs_clean = f"{row['mpqs_mean_clean']:.3f}" if pd.notna(row['mpqs_mean_clean']) else "N/A"
    p5_clean = f"{row['mpqs_p5_clean']:.3f}" if pd.notna(row['mpqs_p5_clean']) else "N/A"
    print(f"{row['error_type']:<35} {row['regime']:<6} {row['clean']:<8} {row['clean_rate']:.1%}    {mpqs_clean:<12} {p5_clean:<10}")

# Save calibration analysis
mpqs_df.to_csv(f"{OUTPUT_DIR}/mpqs_calibration_analysis.csv", index=False)
print(f"\n✅ Saved: {OUTPUT_DIR}/mpqs_calibration_analysis.csv")

# ============================================================================
# Threshold Recommendations
# ============================================================================

print("\n" + "=" * 80)
print("THRESHOLD ANALYSIS")
print("=" * 80)

# Linguistic floors (reference values)
LINGUISTIC_FLOORS = {
    "Gender Disagreement": {"csr": 0.87, "sfr": 0.85},
    "Definiteness Shift": {"csr": 0.90, "sfr": 0.88},
    "Tanween Omission": {"csr": 0.92, "sfr": 0.90},
    "Meaning Shift": {"csr_max": 0.88, "sfr_min": 0.45, "sfr_max": 0.78},
    "Total Omission": {"csr_max": 0.85, "sfr_min": 0.50, "sfr_max": 0.80},
}

print("\nLinguistic Floor Reference (from documentation):")
for et, floors in LINGUISTIC_FLOORS.items():
    floor_str = ", ".join(f"{k}={v}" for k, v in floors.items())
    print(f"  {et}: {floor_str}")

# Empirical vs Floor comparison
print("\n" + "-" * 80)
print("Empirical 5th Percentile vs Linguistic Floors (Clean samples)")
print("-" * 80)

for et in LINGUISTIC_FLOORS.keys():
    et_clean = clean_df[clean_df[error_col] == et] if error_col in clean_df.columns else pd.DataFrame()
    if len(et_clean) > 0:
        emp_csr = et_clean['csr'].quantile(0.05) if 'csr' in et_clean.columns else np.nan
        emp_sfr = et_clean['sfr'].quantile(0.05) if 'sfr' in et_clean.columns else np.nan
        floor_csr = LINGUISTIC_FLOORS[et].get('csr', LINGUISTIC_FLOORS[et].get('csr_max', np.nan))
        floor_sfr = LINGUISTIC_FLOORS[et].get('sfr', LINGUISTIC_FLOORS[et].get('sfr_min', np.nan))

        csr_status = "✓" if pd.notna(emp_csr) and emp_csr >= floor_csr else "⚠"
        sfr_status = "✓" if pd.notna(emp_sfr) and emp_sfr >= floor_sfr else "⚠"

        print(f"\n{et} (n={len(et_clean)}):")
        print(f"  CSR: empirical={emp_csr:.3f}, floor={floor_csr:.3f} {csr_status}")
        print(f"  SFR: empirical={emp_sfr:.3f}, floor={floor_sfr:.3f} {sfr_status}")
    else:
        print(f"\n{et}: No clean samples (n=0)")

print("\n" + "=" * 80)
print("✅ MPQS Calibration Analysis Complete")
print("=" * 80)
print(f"\n✅ Ready for Cell 9 (Tier1-Clean Selection)")


# === CELL 9: Tier1-Clean Selection ===
# Original: Cells 26 + 30
# Action: MERGE (combined selection logic for v3.3)
# ============================================================================

In [ ]:
# =============================================================================
# CELL 9: TIER1-CLEAN SELECTION
# =============================================================================
# Apply final selection criteria to produce Tier1-clean
# v3.3 already provides: passed, quality_tier, reject_reason
# This cell verifies and documents the selection
# =============================================================================

import pandas as pd

print("=" * 80)
print("CELL 9: TIER1-CLEAN SELECTION")
print("=" * 80)

# Load v3.3 outputs
OUTPUT_DIR = "tier1_v3_3_final"

clean_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean.csv")
clean_a_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean_A.csv")
clean_b_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean_B.csv")
noisy_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_noisy.csv")

print(f"✅ Loaded v3.3 outputs:")
print(f"   Clean:   {len(clean_df):,} rows")
print(f"   Clean_A: {len(clean_a_df):,} rows (CPS ≥ 0.65)")
print(f"   Clean_B: {len(clean_b_df):,} rows (CPS ≥ 0.55)")
print(f"   Noisy:   {len(noisy_df):,} rows")

# ============================================================================
# Selection Criteria (v3.3)
# ============================================================================

print("\n" + "=" * 80)
print("TIER1-CLEAN SELECTION CRITERIA (v3.3)")
print("=" * 80)

print("""
A sample is Tier1-clean IF AND ONLY IF:

1) OCS = 1.0 (Option-2A Constraint Satisfied)
   - Keyword found in clean sentence (clitic-aware matching)
   - Best_match found in error sentence (for non-omission types)
   - Keyword NOT in error sentence (for substitution types)

2) CPS ≥ Floor (Context Preservation Score)
   - Default floor: 0.55
   - Structure-tolerant classes: 0.45
   - Clean_A requires: CPS ≥ 0.65

3) SFR ≥ Min (Semantic Fluency Ratio)
   - Class-specific minimum from rubric
   - Regime D also has SFR maximum

4) Regime-specific rules:
   - Regime F: No SFR ceiling
   - Regime D: SFR must be within [min, max] range
""")

# ============================================================================
# Verify Selection
# ============================================================================

print("\n" + "=" * 80)
print("SELECTION VERIFICATION")
print("=" * 80)

# Check that clean samples have OCS = 1.0
if 'ocs' in clean_df.columns:
    ocs_check = (clean_df['ocs'] == 1.0).all()
    print(f"✓ All clean samples have OCS = 1.0: {ocs_check}")
    if not ocs_check:
        bad_ocs = clean_df[clean_df['ocs'] != 1.0]
        print(f"  ⚠ Found {len(bad_ocs)} samples with OCS ≠ 1.0")

# Check quality tier distribution
if 'quality_tier' in clean_df.columns:
    tier_dist = clean_df['quality_tier'].value_counts()
    print(f"\n✓ Quality Tier Distribution:")
    for tier, count in tier_dist.items():
        print(f"    {tier}: {count:,} ({count/len(clean_df):.1%})")

# Check regime distribution
if 'regime' in clean_df.columns:
    regime_dist = clean_df['regime'].value_counts()
    print(f"\n✓ Regime Distribution:")
    for regime, count in regime_dist.items():
        print(f"    Regime {regime}: {count:,} ({count/len(clean_df):.1%})")

# ============================================================================
# Class Coverage
# ============================================================================

print("\n" + "=" * 80)
print("CLASS COVERAGE")
print("=" * 80)

error_col = 'Sub-Subtype'
if error_col in clean_df.columns:
    clean_classes = set(clean_df[error_col].unique())
    noisy_classes = set(noisy_df[error_col].unique()) if error_col in noisy_df.columns else set()
    all_classes = clean_classes | noisy_classes

    print(f"\nTotal classes in data: {len(all_classes)}")
    print(f"Classes with clean samples: {len(clean_classes)}")

    missing_classes = all_classes - clean_classes
    if missing_classes:
        print(f"\n⚠ Classes with NO clean samples ({len(missing_classes)}):")
        for mc in sorted(missing_classes):
            noisy_count = len(noisy_df[noisy_df[error_col] == mc]) if error_col in noisy_df.columns else 0
            print(f"    - {mc} (noisy: {noisy_count})")
    else:
        print(f"✓ All classes have at least one clean sample")

# ============================================================================
# Summary Statistics
# ============================================================================

print("\n" + "=" * 80)
print("SELECTION SUMMARY")
print("=" * 80)

total = len(clean_df) + len(noisy_df)
print(f"""
Total validated:     {total:,}
─────────────────────────────
Tier1-Clean:         {len(clean_df):,} ({len(clean_df)/total:.1%})
  ├─ Clean_A:        {len(clean_a_df):,} ({len(clean_a_df)/total:.1%})
  └─ Clean_B:        {len(clean_b_df):,} ({len(clean_b_df)/total:.1%})
Tier1-Noisy:         {len(noisy_df):,} ({len(noisy_df)/total:.1%})
─────────────────────────────
Classes covered:     {len(clean_classes) if error_col in clean_df.columns else 'N/A'} / 23
""")

print("✅ Tier1-Clean Selection Complete")
print(f"\n✅ Ready for Cell 10 (Per-Class Statistics)")


# === CELL 10: Per-Class Acceptance Statistics ===
# Original: Cells 27 + 32
# Action: MERGE (combined statistics)
# ============================================================================


In [ ]:
# =============================================================================
# CELL 10: PER-CLASS ACCEPTANCE STATISTICS
# =============================================================================
# Comprehensive per-class analysis:
# - Acceptance rates
# - Metric distributions
# - Rejection reason breakdown
# - Tier2 support needs assessment
# =============================================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("CELL 10: PER-CLASS ACCEPTANCE STATISTICS")
print("=" * 80)

# Load v3.3 outputs
OUTPUT_DIR = "tier1_v3_3_final"

clean_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean.csv")
noisy_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_noisy.csv")

error_col = 'Sub-Subtype'

# ============================================================================
# Per-Class Acceptance Rates
# ============================================================================

print("\n" + "=" * 80)
print("PER-CLASS ACCEPTANCE RATES")
print("=" * 80)

class_stats = []

all_types = set()
if error_col in clean_df.columns:
    all_types |= set(clean_df[error_col].unique())
if error_col in noisy_df.columns:
    all_types |= set(noisy_df[error_col].unique())

for et in sorted(all_types):
    clean_et = clean_df[clean_df[error_col] == et] if error_col in clean_df.columns else pd.DataFrame()
    noisy_et = noisy_df[noisy_df[error_col] == et] if error_col in noisy_df.columns else pd.DataFrame()

    total = len(clean_et) + len(noisy_et)
    clean_count = len(clean_et)

    # Quality tier breakdown
    clean_a = len(clean_et[clean_et['quality_tier'] == 'A']) if 'quality_tier' in clean_et.columns else 0
    clean_b = len(clean_et[clean_et['quality_tier'] == 'B']) if 'quality_tier' in clean_et.columns else 0

    # Regime
    regime = RUBRIC_23.get(et, {}).get('regime', 'F') if 'RUBRIC_23' in dir() else 'F'

    # Metrics (clean samples)
    mpqs_mean = clean_et['mpqs'].mean() if len(clean_et) > 0 and 'mpqs' in clean_et.columns else np.nan
    cps_mean = clean_et['cps'].mean() if len(clean_et) > 0 and 'cps' in clean_et.columns else np.nan
    sfr_mean = clean_et['sfr'].mean() if len(clean_et) > 0 and 'sfr' in clean_et.columns else np.nan

    # Top rejection reason (noisy samples)
    top_reason = noisy_et['reject_reason'].mode().iloc[0] if len(noisy_et) > 0 and 'reject_reason' in noisy_et.columns and len(noisy_et['reject_reason'].mode()) > 0 else 'N/A'

    class_stats.append({
        'error_type': et,
        'regime': regime,
        'total': total,
        'clean': clean_count,
        'clean_a': clean_a,
        'clean_b': clean_b,
        'noisy': len(noisy_et),
        'clean_rate': clean_count / total if total > 0 else 0,
        'mpqs_mean': mpqs_mean,
        'cps_mean': cps_mean,
        'sfr_mean': sfr_mean,
        'top_rejection': top_reason,
    })

stats_df = pd.DataFrame(class_stats).sort_values('clean_rate', ascending=False)

# Display table
print(f"\n{'Error Type':<30} {'Rgm':<4} {'Total':<7} {'Clean':<7} {'A':<5} {'B':<5} {'Rate':<7} {'MPQS':<7} {'Top Rejection':<25}")
print("-" * 115)
for _, row in stats_df.iterrows():
    mpqs_str = f"{row['mpqs_mean']:.3f}" if pd.notna(row['mpqs_mean']) else "N/A"
    print(f"{row['error_type']:<30} {row['regime']:<4} {row['total']:<7} {row['clean']:<7} {row['clean_a']:<5} {row['clean_b']:<5} {row['clean_rate']:.1%}   {mpqs_str:<7} {row['top_rejection']:<25}")

# Save statistics
stats_df.to_csv(f"{OUTPUT_DIR}/per_class_acceptance_stats.csv", index=False)
print(f"\n✅ Saved: {OUTPUT_DIR}/per_class_acceptance_stats.csv")

# ============================================================================
# Rejection Reason Analysis
# ============================================================================

print("\n" + "=" * 80)
print("REJECTION REASON ANALYSIS")
print("=" * 80)

if 'reject_reason' in noisy_df.columns:
    rejection_counts = noisy_df['reject_reason'].value_counts()
    print(f"\nOverall rejection reasons:")
    for reason, count in rejection_counts.head(10).items():
        print(f"  {reason}: {count:,} ({count/len(noisy_df):.1%})")

# OCS constraint breakdown
if 'ocs_failed_constraints' in noisy_df.columns:
    ocs_failed = noisy_df[noisy_df['reject_reason'] == 'ocs_failed']
    if len(ocs_failed) > 0:
        print(f"\nOCS constraint failures ({len(ocs_failed):,} total):")
        constraint_counts = ocs_failed['ocs_failed_constraints'].value_counts()
        for constraint, count in constraint_counts.head(8).items():
            if pd.notna(constraint) and constraint != '':
                print(f"  {constraint}: {count:,}")

# ============================================================================
# Tier2 Support Assessment
# ============================================================================

print("\n" + "=" * 80)
print("TIER2 SUPPORT ASSESSMENT")
print("=" * 80)

MIN_CLEAN_FOR_TIER2 = 3  # Minimum clean samples needed

low_coverage = stats_df[stats_df['clean'] < MIN_CLEAN_FOR_TIER2]
if len(low_coverage) > 0:
    print(f"\n⚠ Classes needing Tier2 support (clean < {MIN_CLEAN_FOR_TIER2}):")
    for _, row in low_coverage.iterrows():
        print(f"  - {row['error_type']}: {row['clean']} clean samples")
    print(f"\nRecommendation: Supplement these classes from Tier0 or generate Tier2 samples")
else:
    print(f"\n✓ All classes have ≥ {MIN_CLEAN_FOR_TIER2} clean samples")

# Zero clean classes
zero_clean = stats_df[stats_df['clean'] == 0]
if len(zero_clean) > 0:
    print(f"\n⚠ Classes with ZERO clean samples ({len(zero_clean)}):")
    for _, row in zero_clean.iterrows():
        print(f"  - {row['error_type']} (total: {row['total']}, top rejection: {row['top_rejection']})")

print("\n" + "=" * 80)
print("✅ Per-Class Statistics Complete")
print("=" * 80)
print(f"\n✅ Ready for Cell 11 (Deduplication)")


# === CELL 11: Deduplication ===
# Original: Cell 33
# Action: UPDATE (for v3.3 outputs)
# ============================================================================


In [ ]:
# =============================================================================
# CELL 11: DEDUPLICATION CHECK
# =============================================================================
# Verify no duplicate samples in Tier1-clean
# Deduplication key: Sentence_ID + MT Tool + Sub-Subtype
# =============================================================================

import pandas as pd

print("=" * 80)
print("CELL 11: DEDUPLICATION CHECK")
print("=" * 80)

# Load v3.3 clean output
OUTPUT_DIR = "tier1_v3_3_final"
clean_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean.csv")

print(f"✅ Loaded: {OUTPUT_DIR}/tier1_clean.csv")
print(f"   Rows: {len(clean_df):,}")

# ============================================================================
# Define Deduplication Key
# ============================================================================

# Primary key (strict)
DEDUP_KEY_STRICT = ['Sentence_ID', 'MT Tool', 'Sub-Subtype']

# Extended key (if you want to allow same sentence with different keywords)
DEDUP_KEY_EXTENDED = ['Sentence_ID', 'MT Tool', 'Sub-Subtype', 'Keyword', 'Best_Match']

# Check which columns exist
available_strict = [c for c in DEDUP_KEY_STRICT if c in clean_df.columns]
available_extended = [c for c in DEDUP_KEY_EXTENDED if c in clean_df.columns]

print(f"\nDeduplication key columns:")
print(f"  Strict key: {available_strict}")
print(f"  Extended key: {available_extended}")

# ============================================================================
# Check for Duplicates
# ============================================================================

print("\n" + "=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

# Check strict duplicates
if len(available_strict) > 0:
    strict_dupes = clean_df.duplicated(subset=available_strict, keep=False)
    strict_dupe_count = strict_dupes.sum()

    if strict_dupe_count > 0:
        print(f"\n⚠ Found {strict_dupe_count} rows with duplicate strict key")
        print(f"  Key: {available_strict}")

        # Show examples
        dupe_examples = clean_df[strict_dupes].head(10)
        print(f"\n  Example duplicates:")
        for idx, row in dupe_examples.iterrows():
            key_vals = [str(row.get(c, 'N/A')) for c in available_strict]
            print(f"    {' | '.join(key_vals)}")
    else:
        print(f"✓ No duplicates found (strict key: {available_strict})")

# Check extended duplicates
if len(available_extended) > 0:
    extended_dupes = clean_df.duplicated(subset=available_extended, keep=False)
    extended_dupe_count = extended_dupes.sum()

    if extended_dupe_count > 0:
        print(f"\n⚠ Found {extended_dupe_count} rows with duplicate extended key")
    else:
        print(f"✓ No duplicates found (extended key: {available_extended})")

# ============================================================================
# Deduplicate if Needed
# ============================================================================

if strict_dupe_count > 0 if 'strict_dupe_count' in dir() else False:
    print("\n" + "=" * 80)
    print("APPLYING DEDUPLICATION")
    print("=" * 80)

    before_count = len(clean_df)
    clean_df_deduped = clean_df.drop_duplicates(subset=available_strict, keep='first')
    after_count = len(clean_df_deduped)

    print(f"Before deduplication: {before_count:,}")
    print(f"After deduplication:  {after_count:,}")
    print(f"Removed: {before_count - after_count:,}")

    # Save deduplicated version
    clean_df_deduped.to_csv(f"{OUTPUT_DIR}/tier1_clean_deduped.csv", index=False, encoding='utf-8-sig')
    print(f"\n✅ Saved: {OUTPUT_DIR}/tier1_clean_deduped.csv")
else:
    print("\n✓ No deduplication needed")

# ============================================================================
# Verify Against Original
# ============================================================================

print("\n" + "=" * 80)
print("VERIFICATION AGAINST ORIGINAL")
print("=" * 80)

# Load original candidates to compare
try:
    original_df = pd.read_csv("candidates_TIER1.csv")
    original_unique = len(original_df.drop_duplicates(subset=[c for c in available_strict if c in original_df.columns]))

    print(f"Original candidates unique samples: {original_unique:,}")
    print(f"Current Tier1-clean samples: {len(clean_df):,}")
    print(f"Clean rate: {len(clean_df)/original_unique:.1%}")
except FileNotFoundError:
    print("⚠ Could not load original candidates_TIER1.csv for comparison")

print("\n" + "=" * 80)
print("✅ Deduplication Check Complete")
print("=" * 80)
print(f"\n✅ Ready for Cell 12 (Final Freeze)")


# === CELL 12: Final Tier1-Clean Freeze ===
# Original: NEW
# Action: NEW (create final frozen output)
# ============================================================================


In [ ]:
# =============================================================================
# CELL 12: FINAL TIER1-CLEAN FREEZE
# =============================================================================
# Create final frozen Tier1-clean dataset with all metadata
# This is the authoritative output for downstream use
# =============================================================================

import pandas as pd
import json
from datetime import datetime

print("=" * 80)
print("CELL 12: FINAL TIER1-CLEAN FREEZE")
print("=" * 80)

# Load v3.3 outputs
OUTPUT_DIR = "tier1_v3_3_final"

clean_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean.csv")
clean_a_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean_A.csv")
clean_b_df = pd.read_csv(f"{OUTPUT_DIR}/tier1_clean_B.csv")

print(f"✅ Loaded Tier1-clean: {len(clean_df):,} rows")

# ============================================================================
# Add Freeze Metadata
# ============================================================================

freeze_timestamp = datetime.now().isoformat()
freeze_version = "v3.3"

# Add metadata columns
clean_df['freeze_version'] = freeze_version
clean_df['freeze_timestamp'] = freeze_timestamp

clean_a_df['freeze_version'] = freeze_version
clean_a_df['freeze_timestamp'] = freeze_timestamp

clean_b_df['freeze_version'] = freeze_version
clean_b_df['freeze_timestamp'] = freeze_timestamp

print(f"\n✓ Added freeze metadata:")
print(f"  Version: {freeze_version}")
print(f"  Timestamp: {freeze_timestamp}")

# ============================================================================
# Create Freeze Summary
# ============================================================================

error_col = 'Sub-Subtype'

freeze_summary = {
    'version': freeze_version,
    'timestamp': freeze_timestamp,
    'total_clean': len(clean_df),
    'clean_a': len(clean_a_df),
    'clean_b': len(clean_b_df),
    'classes_covered': int(clean_df[error_col].nunique()) if error_col in clean_df.columns else 0,
    'classes_total': 23,
    'per_class_counts': clean_df[error_col].value_counts().to_dict() if error_col in clean_df.columns else {},
    'metrics': {
        'mpqs_mean': float(clean_df['mpqs'].mean()) if 'mpqs' in clean_df.columns else None,
        'cps_mean': float(clean_df['cps'].mean()) if 'cps' in clean_df.columns else None,
        'sfr_mean': float(clean_df['sfr'].mean()) if 'sfr' in clean_df.columns else None,
        'csr_mean': float(clean_df['csr'].mean()) if 'csr' in clean_df.columns else None,
    },
    'validation_module': 'tier1_validation_implementation_v3_3',
    'clitic_aware_matching': True,
}

# ============================================================================
# Save Frozen Outputs
# ============================================================================

print("\n" + "=" * 80)
print("SAVING FROZEN OUTPUTS")
print("=" * 80)

# Save frozen CSVs
clean_df.to_csv(f"{OUTPUT_DIR}/tier1_clean_FROZEN.csv", index=False, encoding='utf-8-sig')
clean_a_df.to_csv(f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv", index=False, encoding='utf-8-sig')
clean_b_df.to_csv(f"{OUTPUT_DIR}/tier1_clean_B_FROZEN.csv", index=False, encoding='utf-8-sig')

print(f"✅ Saved frozen CSVs:")
print(f"   - tier1_clean_FROZEN.csv ({len(clean_df):,} rows)")
print(f"   - tier1_clean_A_FROZEN.csv ({len(clean_a_df):,} rows)")
print(f"   - tier1_clean_B_FROZEN.csv ({len(clean_b_df):,} rows)")

# Save freeze summary
with open(f"{OUTPUT_DIR}/freeze_summary.json", "w") as f:
    json.dump(freeze_summary, f, indent=2, default=str)
print(f"   - freeze_summary.json")

# ============================================================================
# Final Summary
# ============================================================================

print("\n" + "=" * 80)
print("TIER1-CLEAN FREEZE SUMMARY")
print("=" * 80)

print(f"""
Version:             {freeze_version}
Timestamp:           {freeze_timestamp}

Total Clean:         {freeze_summary['total_clean']:,}
  ├─ Clean_A:        {freeze_summary['clean_a']:,}
  └─ Clean_B:        {freeze_summary['clean_b']:,}

Classes Covered:     {freeze_summary['classes_covered']} / {freeze_summary['classes_total']}

Mean Metrics:
  MPQS:              {freeze_summary['metrics']['mpqs_mean']:.3f}
  CPS:               {freeze_summary['metrics']['cps_mean']:.3f}
  SFR:               {freeze_summary['metrics']['sfr_mean']:.3f}
  CSR:               {freeze_summary['metrics']['csr_mean']:.3f}

Output Directory:    {OUTPUT_DIR}/
""")

# Classes with low coverage
if error_col in clean_df.columns:
    class_counts = clean_df[error_col].value_counts()
    low_classes = class_counts[class_counts < 3]
    if len(low_classes) > 0:
        print(f"⚠ Classes with < 3 samples:")
        for cls, cnt in low_classes.items():
            print(f"    - {cls}: {cnt}")

print("\n" + "=" * 80)
print("🎉 PHASE 2 COMPLETE")
print("=" * 80)
print(f"\n✅ Tier1-clean is frozen and ready for Tier2 preparation")
print(f"✅ Ready for Phase 3 (Tier2 Preparation + Generation)")
# === CELL 13: Tier2 Few-Shot Bank Builder ===
# Original: Cell 37
# Action: UPDATE (paths for v3.3, improved logic)
# ============================================================================

In [ ]:
# =============================================================================
# CELL 13: TIER2 FEW-SHOT BANK BUILDER
# =============================================================================
# Build a few-shot bank for Tier2 generation
# Priority: Tier1-clean_A > Tier1-clean_B > Tier0 canonical
# Output: few_shot_bank_tier2.csv, few_shot_bank_tier2.json
# =============================================================================

import os
import json
import pandas as pd
from pathlib import Path

print("=" * 80)
print("CELL 13: TIER2 FEW-SHOT BANK BUILDER")
print("=" * 80)

# ============================================================================
# Configuration
# ============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# Input files
TIER1_CLEAN_PATH = f"{OUTPUT_DIR}/tier1_clean_FROZEN.csv"
TIER1_CLEAN_A_PATH = f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv"
TIER0_PATH = "tier0_canonical_examples.csv"  # Update if different

# Output files
OUT_CSV = f"{OUTPUT_DIR}/few_shot_bank_tier2.csv"
OUT_JSON = f"{OUTPUT_DIR}/few_shot_bank_tier2.json"
OUT_COVERAGE = f"{OUTPUT_DIR}/few_shot_bank_coverage_report.csv"

MAX_PER_CLASS = 3  # Maximum examples per error type

# Taxonomy (single source of truth)
TAXONOMY_23 = [
    "Meaning Shift",
    "Total Omission",
    "Partial Translation",
    "Name Entity Error",
    "Terminology Substitution",
    "Literal Translation",
    "Hypernym for Hyponym",
    "Hyponym for Hypernym",
    "Tanween Omission",
    "Gender Disagreement",
    "Definiteness Shift",
    "Perfective to Progressive",
    "Progressive to Perfective",
    "Tense Shift Under Negation",
    "Wrong Structure",
    "Wrong Word Order",
    "Active to Passive Voice",
    "Passive to Active Voice",
    "Noun to Adjective",
    "Adjective to Noun",
    "Register Mismatch",
    "Spelling Error",
    "Invalid Pattern",
]

print(f"Configuration:")
print(f"  Max per class: {MAX_PER_CLASS}")
print(f"  Total classes: {len(TAXONOMY_23)}")

# ============================================================================
# Load Data
# ============================================================================

print("\n" + "=" * 80)
print("LOADING DATA")
print("=" * 80)

# Load Tier1-clean (prefer Clean_A)
tier1_clean = None
tier1_clean_a = None

if Path(TIER1_CLEAN_A_PATH).exists():
    tier1_clean_a = pd.read_csv(TIER1_CLEAN_A_PATH)
    print(f"✅ Loaded Tier1-clean_A: {len(tier1_clean_a):,} rows")

if Path(TIER1_CLEAN_PATH).exists():
    tier1_clean = pd.read_csv(TIER1_CLEAN_PATH)
    print(f"✅ Loaded Tier1-clean: {len(tier1_clean):,} rows")
else:
    # Try alternate path
    alt_path = f"{OUTPUT_DIR}/tier1_clean.csv"
    if Path(alt_path).exists():
        tier1_clean = pd.read_csv(alt_path)
        print(f"✅ Loaded Tier1-clean (alt): {len(tier1_clean):,} rows")

# Load Tier0 (for filling gaps)
tier0 = None
if Path(TIER0_PATH).exists():
    tier0 = pd.read_csv(TIER0_PATH)
    print(f"✅ Loaded Tier0: {len(tier0):,} rows")
else:
    print(f"⚠ Tier0 not found: {TIER0_PATH}")
    print(f"   Will use Tier1-clean only (some classes may have gaps)")

# ============================================================================
# Build Few-Shot Bank
# ============================================================================

print("\n" + "=" * 80)
print("BUILDING FEW-SHOT BANK")
print("=" * 80)

error_col = 'Sub-Subtype'
few_shot_rows = []
coverage_report = []

for error_type in TAXONOMY_23:
    selected = []
    source_info = []

    # Priority 1: Tier1-clean_A (highest quality)
    if tier1_clean_a is not None and error_col in tier1_clean_a.columns:
        candidates = tier1_clean_a[tier1_clean_a[error_col] == error_type]
        if len(candidates) > 0:
            n_take = min(MAX_PER_CLASS - len(selected), len(candidates))
            selected.extend(candidates.head(n_take).to_dict('records'))
            source_info.extend(['Tier1_A'] * n_take)

    # Priority 2: Tier1-clean (if not enough from A)
    if len(selected) < MAX_PER_CLASS and tier1_clean is not None and error_col in tier1_clean.columns:
        # Exclude already selected (by avoiding Clean_A rows)
        candidates = tier1_clean[tier1_clean[error_col] == error_type]
        if tier1_clean_a is not None and 'quality_tier' in candidates.columns:
            candidates = candidates[candidates['quality_tier'] != 'A']
        if len(candidates) > 0:
            n_take = min(MAX_PER_CLASS - len(selected), len(candidates))
            selected.extend(candidates.head(n_take).to_dict('records'))
            source_info.extend(['Tier1_B'] * n_take)

    # Priority 3: Tier0 (fill gaps)
    if len(selected) < MAX_PER_CLASS and tier0 is not None:
        # Try different column names for error type in Tier0
        tier0_col = None
        for col in ['Sub-Subtype', 'error_type', 'Error_Type', 'SubSubtype']:
            if col in tier0.columns:
                tier0_col = col
                break

        if tier0_col:
            candidates = tier0[tier0[tier0_col] == error_type]
            if len(candidates) > 0:
                n_take = min(MAX_PER_CLASS - len(selected), len(candidates))
                for _, row in candidates.head(n_take).iterrows():
                    # Convert Tier0 schema to Tier1 schema
                    row_dict = row.to_dict()
                    # Rename columns if needed
                    if 'clean_ar' in row_dict and 'ar' not in row_dict:
                        row_dict['ar'] = row_dict['clean_ar']
                    if 'error_ar' in row_dict and 'mt_output' not in row_dict:
                        row_dict['mt_output'] = row_dict['error_ar']
                    if tier0_col != error_col:
                        row_dict[error_col] = row_dict.get(tier0_col, error_type)
                    selected.append(row_dict)
                source_info.extend(['Tier0'] * n_take)

    # Add source column
    for i, row in enumerate(selected):
        row['few_shot_source'] = source_info[i] if i < len(source_info) else 'Unknown'
        few_shot_rows.append(row)

    # Coverage report
    coverage_report.append({
        'error_type': error_type,
        'total_examples': len(selected),
        'from_tier1_a': source_info.count('Tier1_A'),
        'from_tier1_b': source_info.count('Tier1_B'),
        'from_tier0': source_info.count('Tier0'),
        'gap': MAX_PER_CLASS - len(selected),
    })

# Create DataFrames
few_shot_df = pd.DataFrame(few_shot_rows)
coverage_df = pd.DataFrame(coverage_report)

# ============================================================================
# Save Outputs
# ============================================================================

print("\n" + "=" * 80)
print("SAVING OUTPUTS")
print("=" * 80)

# Save CSV
few_shot_df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUT_CSV} ({len(few_shot_df)} rows)")

# Save JSON (grouped by error type)
few_shot_json = {}
for error_type in TAXONOMY_23:
    et_rows = few_shot_df[few_shot_df[error_col] == error_type] if error_col in few_shot_df.columns else pd.DataFrame()
    if len(et_rows) > 0:
        few_shot_json[error_type] = et_rows.to_dict('records')
    else:
        few_shot_json[error_type] = []

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(few_shot_json, f, ensure_ascii=False, indent=2)
print(f"✅ Saved: {OUT_JSON}")

# Save coverage report
coverage_df.to_csv(OUT_COVERAGE, index=False)
print(f"✅ Saved: {OUT_COVERAGE}")

# ============================================================================
# Summary
# ============================================================================

print("\n" + "=" * 80)
print("FEW-SHOT BANK SUMMARY")
print("=" * 80)

total_examples = len(few_shot_df)
classes_covered = coverage_df[coverage_df['total_examples'] > 0]['error_type'].nunique()
classes_with_gap = coverage_df[coverage_df['gap'] > 0]['error_type'].nunique()

print(f"""
Total examples:      {total_examples}
Classes covered:     {classes_covered} / {len(TAXONOMY_23)}
Classes with gaps:   {classes_with_gap}

Source breakdown:
  From Tier1_A:      {coverage_df['from_tier1_a'].sum()}
  From Tier1_B:      {coverage_df['from_tier1_b'].sum()}
  From Tier0:        {coverage_df['from_tier0'].sum()}
""")

# Show classes with gaps
gaps = coverage_df[coverage_df['gap'] > 0]
if len(gaps) > 0:
    print("Classes with fewer than 3 examples:")
    for _, row in gaps.iterrows():
        print(f"  - {row['error_type']}: {row['total_examples']} examples (gap: {row['gap']})")

print("\n✅ Few-Shot Bank Builder Complete")
print(f"✅ Ready for Cell 14 (Ensure All 23 Classes)")
###
# View the few-shot bank
import pandas as pd

few_shot_df = pd.read_csv("tier1_v3_3_final/few_shot_bank_tier2.csv")
print(f"Total examples: {len(few_shot_df)}")
print(f"\nPer class:")
print(few_shot_df['Sub-Subtype'].value_counts())

# View specific class examples
print("\n--- Example: Meaning Shift ---")
print(few_shot_df[few_shot_df['Sub-Subtype'] == 'Meaning Shift'][['ar', 'mt_output', 'Keyword', 'Best_Match', 'few_shot_source']])

# View coverage report
coverage = pd.read_csv("tier1_v3_3_final/few_shot_bank_coverage_report.csv")
print(coverage)

# === CELL 14: Ensure All 23 Classes ===
# Original: Cell 38
# Action: UPDATE (paths for v3.3)
# ============================================================================

In [ ]:
# =============================================================================
# CELL 14: PREPARE CLEAN_A FOR ETCA AUDIT
# =============================================================================
"""
CELL 14: Prepare Clean_A for ETCA Audit

PURPOSE:
Prepare the Clean_A dataset for ETCA-Lite audit in Cell 15.
NO Tier0 supplements here — Tier0 is added AFTER ETCA in Cell 16.

INPUT:
- tier1_v3_3_final/tier1_clean_A_FROZEN.csv (~920 rows from Cell 12)

OUTPUT:
- tier1_v3_3_final/tier1_clean_A_for_etca.csv (same data, verified)

WHY NO TIER0 HERE:
- Cell 15 (ETCA) audits only MT-mined Tier1 samples
- Tier0 is gold/canonical — doesn't need ETCA validation
- Tier0 is added in Cell 16 as FIRST priority for few-shot bank
"""

import pandas as pd
import os

print("=" * 80)
print("CELL 14: PREPARE CLEAN_A FOR ETCA AUDIT")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# Input (from Cell 12: Final Freeze)
INPUT_CLEAN_A = f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv"

# Output (for Cell 15: ETCA)
OUTPUT_FOR_ETCA = f"{OUTPUT_DIR}/tier1_clean_A_for_etca.csv"

# Taxonomy (23 classes)
TAXONOMY_23 = [
    "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
    "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
    "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
    "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
    "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
    "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
]

# =============================================================================
# STEP 1: LOAD CLEAN_A
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD CLEAN_A")
print("=" * 80)

if not os.path.exists(INPUT_CLEAN_A):
    raise FileNotFoundError(f"Clean_A not found: {INPUT_CLEAN_A}\nRun Cells 0-12 first!")

clean_a = pd.read_csv(INPUT_CLEAN_A)
print(f"✅ Loaded Clean_A: {len(clean_a):,} rows")

# =============================================================================
# STEP 2: VERIFY CLASS COVERAGE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: VERIFY CLASS COVERAGE")
print("=" * 80)

error_col = 'Sub-Subtype'

if error_col not in clean_a.columns:
    raise ValueError(f"Missing column: {error_col}")

# Count per class
class_counts = clean_a[error_col].value_counts()
classes_present = set(class_counts.index)
classes_missing = set(TAXONOMY_23) - classes_present

print(f"\n📊 CLASS COVERAGE:")
print(f"   Total classes in taxonomy: {len(TAXONOMY_23)}")
print(f"   Classes with samples: {len(classes_present)}")
print(f"   Classes missing: {len(classes_missing)}")

if classes_missing:
    print(f"\n⚠️ MISSING CLASSES (will need Tier0 in Cell 16):")
    for mc in sorted(classes_missing):
        print(f"   - {mc}")

# Show distribution
print(f"\n📊 SAMPLES PER CLASS:")
print(f"{'Class':<35} {'Count':<8} {'Status'}")
print("-" * 55)

for et in TAXONOMY_23:
    count = class_counts.get(et, 0)
    if count == 0:
        status = "❌ MISSING (Tier0 needed)"
    elif count < 3:
        status = f"⚠️ LOW (may need Tier0)"
    else:
        status = "✅"
    print(f"{et:<35} {count:<8} {status}")

# =============================================================================
# STEP 3: ADD SEED SOURCE MARKER
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: ADD SEED SOURCE MARKER")
print("=" * 80)

# Mark all as Tier1 (MT-mined)
clean_a['seed_source'] = 'TIER1_CLEAN_A'
print(f"✅ Added seed_source = 'TIER1_CLEAN_A' to all rows")

# =============================================================================
# STEP 4: SAVE FOR ETCA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: SAVE FOR ETCA")
print("=" * 80)

clean_a.to_csv(OUTPUT_FOR_ETCA, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUTPUT_FOR_ETCA} ({len(clean_a):,} rows)")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 14 COMPLETE")
print("=" * 80)

# Classes that will need Tier0 supplement
low_classes = [et for et in TAXONOMY_23 if class_counts.get(et, 0) < 3]

print(f"""
📁 OUTPUT:
   - {OUTPUT_FOR_ETCA} ({len(clean_a):,} rows)

📊 COVERAGE SUMMARY:
   Total samples:          {len(clean_a):,}
   Classes with samples:   {len(classes_present)} / {len(TAXONOMY_23)}
   Classes missing:        {len(classes_missing)}
   Classes with < 3:       {len(low_classes)}

⚠️ NOTE:
   - Classes missing or with < 3 samples will be supplemented from Tier0
   - But that happens in Cell 16 AFTER ETCA validation
   - Tier0 = gold examples, don't need ETCA validation

✅ Ready for Cell 15: ETCA-Lite Audit
""")

# List classes needing Tier0
if low_classes:
    print(f"📋 CLASSES THAT MAY NEED TIER0 SUPPLEMENT:")
    for et in sorted(low_classes):
        count = class_counts.get(et, 0)
        needed = 3 - count
        print(f"   - {et}: has {count}, needs {needed} from Tier0")

In [ ]:
# =============================================================================
# CELL: SUPPLEMENT CLEAN_A WITH CLEAN_B (For Manual Evaluation)
# =============================================================================
"""
PURPOSE:
Extract samples from Clean_B to supplement classes that are:
1. Missing entirely from Clean_A
2. Have fewer than 3 samples in Clean_A

This creates a small manual evaluation set to assess Clean_B quality
before deciding whether to include these samples in the pipeline.

INPUTS:
- tier1_v3_3_final/tier1_clean_A_FROZEN.csv
- tier1_v3_3_final/tier1_clean_B_FROZEN.csv

OUTPUTS:
- tier1_v3_3_final/clean_B_supplement_for_eval.csv (samples to evaluate)
- tier1_v3_3_final/clean_B_supplement_summary.json (coverage report)
"""

import pandas as pd
import json
import os
from pathlib import Path

print("=" * 80)
print("SUPPLEMENT CLEAN_A WITH CLEAN_B (Manual Evaluation Set)")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# Inputs
CLEAN_A_PATH = f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv"
CLEAN_B_PATH = f"{OUTPUT_DIR}/tier1_clean_B_FROZEN.csv"

# Outputs
OUT_SUPPLEMENT = f"{OUTPUT_DIR}/clean_B_supplement_for_eval.csv"
OUT_SUMMARY = f"{OUTPUT_DIR}/clean_B_supplement_summary.json"

# Settings
MAX_PER_CLASS = 3  # Target per class
MIN_THRESHOLD = 3  # Classes with fewer than this need supplement

# Taxonomy (23 classes)
TAXONOMY_23 = [
    "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
    "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
    "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
    "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
    "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
    "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
]

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

# Load Clean_A
if not Path(CLEAN_A_PATH).exists():
    raise FileNotFoundError(f"Clean_A not found: {CLEAN_A_PATH}")

clean_a = pd.read_csv(CLEAN_A_PATH)
print(f"✅ Loaded Clean_A: {len(clean_a):,} rows")

# Load Clean_B
if not Path(CLEAN_B_PATH).exists():
    raise FileNotFoundError(f"Clean_B not found: {CLEAN_B_PATH}")

clean_b = pd.read_csv(CLEAN_B_PATH)
print(f"✅ Loaded Clean_B: {len(clean_b):,} rows")

# =============================================================================
# STEP 2: ANALYZE CLEAN_A COVERAGE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: ANALYZE CLEAN_A COVERAGE")
print("=" * 80)

error_col = 'Sub-Subtype'

# Count per class in Clean_A
clean_a_counts = clean_a[error_col].value_counts().to_dict()

# Identify classes needing supplement
classes_missing = []      # 0 samples in Clean_A
classes_low = []          # 1-2 samples in Clean_A
classes_sufficient = []   # 3+ samples in Clean_A

for et in TAXONOMY_23:
    count_a = clean_a_counts.get(et, 0)

    if count_a == 0:
        classes_missing.append(et)
    elif count_a < MIN_THRESHOLD:
        classes_low.append(et)
    else:
        classes_sufficient.append(et)

print(f"\n📊 CLEAN_A COVERAGE ANALYSIS:")
print(f"   Classes with 0 samples:    {len(classes_missing)}")
print(f"   Classes with 1-2 samples:  {len(classes_low)}")
print(f"   Classes with 3+ samples:   {len(classes_sufficient)}")
print(f"   Total taxonomy classes:    {len(TAXONOMY_23)}")

if classes_missing:
    print(f"\n❌ MISSING CLASSES ({len(classes_missing)}):")
    for et in classes_missing:
        print(f"   - {et}")

if classes_low:
    print(f"\n⚠️ LOW CLASSES ({len(classes_low)}):")
    for et in classes_low:
        count = clean_a_counts.get(et, 0)
        print(f"   - {et}: {count} samples (need {MIN_THRESHOLD - count} more)")

# =============================================================================
# STEP 3: CHECK CLEAN_B AVAILABILITY
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: CHECK CLEAN_B AVAILABILITY")
print("=" * 80)

# Count per class in Clean_B
clean_b_counts = clean_b[error_col].value_counts().to_dict()

print(f"\n📊 CLEAN_B CLASS DISTRIBUTION:")
print(f"{'Class':<35} {'Clean_A':<10} {'Clean_B':<10} {'Status'}")
print("-" * 65)

classes_needing_supplement = classes_missing + classes_low
supplement_available = {}

for et in classes_needing_supplement:
    count_a = clean_a_counts.get(et, 0)
    count_b = clean_b_counts.get(et, 0)
    needed = MIN_THRESHOLD - count_a
    can_take = min(needed, count_b, MAX_PER_CLASS)

    supplement_available[et] = {
        'clean_a_count': count_a,
        'clean_b_count': count_b,
        'needed': needed,
        'can_take': can_take
    }

    if count_b > 0:
        status = f"✅ Can take {can_take}"
    else:
        status = "❌ None available"

    print(f"{et:<35} {count_a:<10} {count_b:<10} {status}")

# =============================================================================
# STEP 4: EXTRACT SUPPLEMENT SAMPLES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: EXTRACT SUPPLEMENT SAMPLES FROM CLEAN_B")
print("=" * 80)

supplement_rows = []
extraction_report = []

for et in classes_needing_supplement:
    info = supplement_available[et]
    can_take = info['can_take']

    if can_take > 0:
        # Get samples from Clean_B for this class
        et_samples = clean_b[clean_b[error_col] == et].head(can_take).copy()

        # Add source marker
        et_samples['supplement_source'] = 'CLEAN_B'
        et_samples['supplement_reason'] = 'missing' if info['clean_a_count'] == 0 else 'low_count'

        supplement_rows.append(et_samples)

        extraction_report.append({
            'error_type': et,
            'clean_a_count': info['clean_a_count'],
            'samples_extracted': len(et_samples),
            'reason': 'missing' if info['clean_a_count'] == 0 else 'low_count'
        })

        print(f"✅ {et}: Extracted {len(et_samples)} samples")
    else:
        extraction_report.append({
            'error_type': et,
            'clean_a_count': info['clean_a_count'],
            'samples_extracted': 0,
            'reason': 'no_clean_b_available'
        })
        print(f"⚠️ {et}: No samples available in Clean_B")

# Combine all supplement samples
if supplement_rows:
    supplement_df = pd.concat(supplement_rows, ignore_index=True)
else:
    supplement_df = pd.DataFrame()

print(f"\n📊 TOTAL SUPPLEMENT SAMPLES: {len(supplement_df)}")

# =============================================================================
# STEP 5: SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: SAVE OUTPUTS")
print("=" * 80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save supplement CSV
if len(supplement_df) > 0:
    supplement_df.to_csv(OUT_SUPPLEMENT, index=False, encoding='utf-8-sig')
    print(f"✅ Saved: {OUT_SUPPLEMENT} ({len(supplement_df)} rows)")
else:
    print(f"⚠️ No supplement samples to save")

# Save summary JSON
summary = {
    'clean_a_total': len(clean_a),
    'clean_b_total': len(clean_b),
    'taxonomy_classes': len(TAXONOMY_23),
    'classes_missing_in_a': len(classes_missing),
    'classes_low_in_a': len(classes_low),
    'classes_sufficient_in_a': len(classes_sufficient),
    'supplement_samples_extracted': len(supplement_df),
    'min_threshold': MIN_THRESHOLD,
    'max_per_class': MAX_PER_CLASS,
    'extraction_report': extraction_report,
    'classes_missing': classes_missing,
    'classes_low': classes_low,
}

with open(OUT_SUMMARY, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {OUT_SUMMARY}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 SUPPLEMENT EXTRACTION COMPLETE")
print("=" * 80)

# Count by reason
missing_filled = sum(1 for r in extraction_report if r['reason'] == 'missing' and r['samples_extracted'] > 0)
low_filled = sum(1 for r in extraction_report if r['reason'] == 'low_count' and r['samples_extracted'] > 0)
not_filled = sum(1 for r in extraction_report if r['samples_extracted'] == 0)

print(f"""
📁 OUTPUT FILES:
   - {OUT_SUPPLEMENT} ({len(supplement_df)} samples for manual evaluation)
   - {OUT_SUMMARY}

📊 EXTRACTION SUMMARY:
   Missing classes filled:       {missing_filled} / {len(classes_missing)}
   Low-count classes filled:     {low_filled} / {len(classes_low)}
   Classes not filled:           {not_filled}

📊 SAMPLES BY CLASS:
""")

for r in extraction_report:
    if r['samples_extracted'] > 0:
        print(f"   {r['error_type']}: {r['samples_extracted']} samples ({r['reason']})")

print(f"""
📋 NEXT STEPS:
   1. Open {OUT_SUPPLEMENT}
   2. Manually evaluate each sample for:
      - Error presence and correctness
      - Keyword/Best_Match accuracy
      - Arabic quality
   3. Decide which samples to include in final dataset

✅ Ready for manual evaluation!
""")

In [ ]:
# =============================================================================
# CELL 15 — ETCA-Lite Audit Gate (Label-aware, anchor-based; Arabic-only)
# =============================================================================
"""
CELL 15: ETCA-Lite Audit Gate

PURPOSE:
Audit Clean_A + Tier0 samples as potential Tier2 generation seeds.
This is a label-aware, anchor-based verification — NOT reclassification.

INPUT:
- tier1_v3_3_final/tier1_complete_23_classes.csv (~919 rows from Cell 14)
- Composition: Clean_A (~920) + Tier0 supplements for missing classes

OUTPUTS:
- tier2_seed_audit_cell15.csv (all audited samples with 5 scores)
- tier2_seed_audit_trace_cell15.csv (failures / parse issues)
- tier2_seeds_ready_cell15.csv (samples with tier2_seed_recommendation=1)

EVALUATION AXES (1-5 scale, except last which is 0/1):
1. anchor_validity: 1=contradicts label, 5=perfect anchor alignment
2. phenomenon_clarity: 1=barely observable, 5=unambiguous
3. arabic_naturalness: 1=broken, 5=fluent plausible MT Arabic
4. collateral_severity: 1=no unrelated distortion, 5=severe unrelated distortion
5. tier2_seed_recommendation: 0=do NOT use, 1=safe to use

KEY DESIGN DECISIONS:
- Arabic-only: No src_en, no bt_en (this is Arabic MT research)
- Label-respecting: Does NOT reclassify, only verifies anchors
- Uses ocs_match_rule_kw from v3.3 (clitic-aware matching)
- Keeps seed_source to track Tier0 vs Tier1 origins

CHECKPOINTING:
- Saves progress to JSONL checkpoint file
- Resumes from last checkpoint if interrupted
"""

from google.colab import userdata
import os
import re
import time
import json
from tqdm import tqdm
from datetime import datetime
from typing import Dict, Any, Optional, Tuple

import pandas as pd
import anthropic

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# Input (from Cell 14: Clean_A only, no Tier0)
IN_COMPLETE = f"{OUTPUT_DIR}/tier1_clean_A_for_etca.csv"

# Output files
OUT_AUDIT = f"{OUTPUT_DIR}/tier2_seed_audit_cell15.csv"
OUT_TRACE = f"{OUTPUT_DIR}/tier2_seed_audit_trace_cell15.csv"
OUT_SEEDS = f"{OUTPUT_DIR}/tier2_seeds_ready_cell15.csv"
CKPT_PATH = f"{OUTPUT_DIR}/cell15_checkpoint.jsonl"
STATS_PATH = f"{OUTPUT_DIR}/cell15_audit_stats.json"

# Audit settings
MAX_ROWS = None  # None = all rows

# LLM settings
JUDGE_MODEL = "claude-sonnet-4-20250514"
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS = 64
JUDGE_RETRIES = 3
SLEEP_BETWEEN_CALLS_SEC = 0.2
CHECKPOINT_INTERVAL = 50

# Optional pipeline-side gating (set to None to disable)
FORCE_REJECT_NATURALNESS_LEQ = None  # e.g., 2 to reject if naturalness <= 2

print("=" * 80)
print("CELL 15: ETCA-Lite Audit Gate")
print("=" * 80)
print(f"Input: {IN_COMPLETE}")
print(f"Model: {JUDGE_MODEL}")

# =============================================================================
# PROMPT (EXACT TEXT - DO NOT MODIFY)
# =============================================================================

CELL15_SYSTEM_PROMPT = """CELL 15 — Label-Aware Anchor Verification (ETCA-Lite, Pre-Tier2 Audit Gate)

Role
You are an expert Arabic linguistics auditor evaluating whether a specific annotated linguistic phenomenon is correctly instantiated in a machine-translated Arabic sentence pair.

You are NOT judging overall translation adequacy.
You are NOT reclassifying labels.
You are NOT fixing or rewriting text.

Your task is to audit anchor-level correctness and usability of this sample as a Tier2 generation seed.

You will receive
- error_label: annotated Sub-Subtype (fixed; do not change)
- keyword: lexical anchor expected in the clean sentence
- best_match: expected transformation in the MT output (may be a lexical form or the sentinel [OMITTED])
- clean_ar: original Arabic sentence
- error_ar: machine-translated Arabic sentence
- ocs_match_rule: deterministic rule used to match the keyword (e.g., exact, prefix_ال, suffix_ة, both_ال_ة)

Your task
Evaluate only whether the annotated phenomenon (error_label) is:
1) correctly instantiated via keyword / best_match,
2) clearly observable in the clean vs. error Arabic pair,
3) linguistically plausible as MT output,
4) suitable for use as a Tier2 generation seed.

Judge with respect to the given label only.
Do NOT consider alternative labels.
Do NOT infer meaning from any other language.

Evaluation axes (return five values)
1) anchor_validity: 1 = contradicts label, 5 = perfect anchor alignment
2) phenomenon_clarity: 1 = barely observable, 5 = unambiguous
3) arabic_naturalness: 1 = broken, 5 = fluent plausible MT Arabic
4) collateral_severity: 1 = no unrelated distortion, 5 = severe unrelated distortion
5) tier2_seed_recommendation: 0 = do NOT use, 1 = safe to use

Decision guidance
- Collateral changes are acceptable if the target phenomenon is intact.
- Rare classes are valid if correct.
- If anchors are ambiguous or weak, set tier2_seed_recommendation = 0.
- Never rescue by imagining intent — judge only what is present.

Output format (STRICT)
Return exactly five values separated by TAB characters in this order:
anchor_validity<TAB>phenomenon_clarity<TAB>arabic_naturalness<TAB>collateral_severity<TAB>tier2_seed_recommendation

Valid example:
5	4	4	2	1

Hard constraints
- No explanations
- No text
- No JSON
- No additional numbers
- No class reassignment
Any deviation invalidates the output.

Interpretation note (for the pipeline, not the evaluator)
This audit is used only to verify Tier1 Clean_A + Tier0 supplementation candidates and filter Tier2 seeds.
It does not modify Tier1 labels, thresholds, or validation logic."""

# =============================================================================
# API SETUP
# =============================================================================

print("\n" + "=" * 80)
print("STEP 0: API SETUP")
print("=" * 80)

try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab secrets")
except Exception as e:
    print(f"⚠ Could not load from secrets: {e}")
    if 'ANTHROPIC_API_KEY' not in os.environ:
        raise ValueError("ANTHROPIC_API_KEY not found!")

key = os.environ.get('ANTHROPIC_API_KEY', '')
if key:
    print(f"✅ API key found: {key[:10]}...{key[-4:]}")
else:
    raise ValueError("ANTHROPIC_API_KEY is empty!")

client = anthropic.Anthropic()
print(f"✅ Claude client initialized")

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def build_user_message(row: Dict[str, Any]) -> str:
    """Build the user message with Arabic-only fields (no src_en, no bt_en)."""
    return (
        f"error_label: {row.get('Sub-Subtype', '')}\n"
        f"keyword: {row.get('Keyword', '')}\n"
        f"best_match: {row.get('Best_Match', '')}\n"
        f"clean_ar: {row.get('ar', '')}\n"
        f"error_ar: {row.get('mt_output', '')}\n"
        f"ocs_match_rule: {row.get('ocs_match_rule_kw', '')}\n"
    )


# Strict regex for TAB-separated 5 values
TAB5_RE = re.compile(r"^\s*([1-5])\t([1-5])\t([1-5])\t([1-5])\t([01])\s*$")

def parse_tab5(raw: str) -> Tuple[Optional[Dict[str, int]], str]:
    """Parse the strict TAB-separated output format."""
    if not isinstance(raw, str) or not raw.strip():
        return None, "empty_response"

    m = TAB5_RE.match(raw.strip())
    if not m:
        return None, "format_mismatch"

    a, b, c, d, e = map(int, m.groups())
    return {
        "anchor_validity": a,
        "phenomenon_clarity": b,
        "arabic_naturalness": c,
        "collateral_severity": d,
        "tier2_seed_recommendation": e,
    }, ""


def call_judge_llm(user_message: str) -> str:
    """Call Claude API for audit judgment."""
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=JUDGE_MAX_TOKENS,
        temperature=JUDGE_TEMPERATURE,
        system=CELL15_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text


def load_checkpoint(path: str) -> Dict[str, Any]:
    """Load checkpoint from JSONL file."""
    if not os.path.exists(path):
        return {}
    seen = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                sid = obj.get("Sentence_ID")
                if sid:
                    seen[str(sid)] = obj
            except json.JSONDecodeError:
                continue
    return seen


def append_checkpoint(path: str, record: Dict[str, Any]) -> None:
    """Append a record to the checkpoint file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

if not os.path.exists(IN_COMPLETE):
    raise FileNotFoundError(f"Input file not found: {IN_COMPLETE}\nRun Cell 14 first!")

df = pd.read_csv(IN_COMPLETE)

if MAX_ROWS is not None:
    df = df.head(MAX_ROWS)
    print(f"⚠ Limited to {MAX_ROWS} rows for testing")

print(f"✅ Loaded: {len(df):,} rows")

# Show composition
error_col = 'Sub-Subtype'
if error_col in df.columns:
    print(f"✅ Classes: {df[error_col].nunique()} unique")

# Ensure seed_source column exists
if "seed_source" not in df.columns:
    # Infer from supplemented_from column (from Cell 14) or default
    if "supplemented_from" in df.columns:
        df["seed_source"] = df["supplemented_from"].apply(
            lambda x: "TIER0_CANONICAL" if str(x).upper() == "TIER0" else "TIER1_CLEAN_A"
        )
    else:
        df["seed_source"] = "TIER1_CLEAN_A"

# Show seed source breakdown
print(f"\n📊 Seed source breakdown:")
for src, count in df["seed_source"].value_counts().items():
    print(f"   {src}: {count:,}")

# =============================================================================
# STEP 2: CHECK CHECKPOINT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: CHECK CHECKPOINT")
print("=" * 80)

ckpt = load_checkpoint(CKPT_PATH)
print(f"✅ Checkpoint loaded: {len(ckpt):,} rows already audited")

remaining = len(df) - len(ckpt)
print(f"📊 Remaining to audit: {remaining:,} rows")

if remaining == 0:
    print("✅ All rows already audited! Loading from checkpoint...")

# =============================================================================
# STEP 3: RUN AUDIT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: RUN ETCA-LITE AUDIT")
print("=" * 80)

print(f"\n📊 AUDIT PLAN:")
print(f"   Total rows: {len(df):,}")
print(f"   Already done: {len(ckpt):,}")
print(f"   Remaining: {remaining:,}")
print(f"   Estimated time: {remaining * 0.5 / 60:.1f} minutes")
print(f"   Estimated cost: ${remaining * 0.003:.2f}")

if remaining > 0:
    input(f"\nPress ENTER to start audit of {remaining:,} samples...")

start_time = time.time()
audit_rows = []
trace_rows = []
api_errors = 0

for idx, row in df.iterrows():
    sid = str(row.get("Sentence_ID", "")).strip()

    if not sid:
        trace_rows.append({"row_index": idx, "error": "missing_sentence_id"})
        continue

    # Resume from checkpoint
    if sid in ckpt:
        audit_rows.append(ckpt[sid])
        continue

    row_dict = row.to_dict()
    user_msg = build_user_message(row_dict)

    # Call judge with retries
    raw = ""
    parsed = None
    err = ""

    for attempt in range(1, JUDGE_RETRIES + 1):
        try:
            raw = call_judge_llm(user_msg)
            parsed, err = parse_tab5(raw)
            if parsed is not None:
                break
        except anthropic.RateLimitError:
            print(f"\n⚠ Rate limit hit, waiting 60s...")
            time.sleep(60)
        except Exception as e:
            err = f"exception_{type(e).__name__}"
            api_errors += 1
        time.sleep(SLEEP_BETWEEN_CALLS_SEC)

    if parsed is None:
        trace_rows.append({
            "Sentence_ID": sid,
            "row_index": idx,
            "Sub-Subtype": row_dict.get("Sub-Subtype", ""),
            "Keyword": row_dict.get("Keyword", ""),
            "Best_Match": row_dict.get("Best_Match", ""),
            "ocs_match_rule_kw": row_dict.get("ocs_match_rule_kw", ""),
            "error": err,
            "raw_response": raw[:500] if isinstance(raw, str) else str(raw)[:500],
        })
        # Checkpoint as failed
        rec = {"Sentence_ID": sid, "cell15_status": "failed", "error": err}
        append_checkpoint(CKPT_PATH, rec)
        continue

    # Pipeline-side gating (optional)
    rec_reco = int(parsed["tier2_seed_recommendation"])
    if FORCE_REJECT_NATURALNESS_LEQ is not None:
        if int(parsed["arabic_naturalness"]) <= int(FORCE_REJECT_NATURALNESS_LEQ):
            rec_reco = 0

    # Build output record
    out_rec = {
        # Identity
        "Sentence_ID": sid,
        "MT_Tool": row_dict.get("MT Tool", ""),
        "seed_source": row_dict.get("seed_source", ""),
        # Label and anchors
        "error_label": row_dict.get("Sub-Subtype", ""),
        "keyword": row_dict.get("Keyword", ""),
        "best_match": row_dict.get("Best_Match", ""),
        "ocs_match_rule": row_dict.get("ocs_match_rule_kw", ""),
        # Arabic text
        "clean_ar": row_dict.get("ar", ""),
        "error_ar": row_dict.get("mt_output", ""),
        # Tier1 metrics (for analysis)
        "ocs": row_dict.get("ocs", None),
        "cps": row_dict.get("cps", None),
        "sfr": row_dict.get("sfr", None),
        "csr": row_dict.get("csr", None),
        # Judge outputs
        **parsed,
        # Final recommendation (after pipeline-side enforcement)
        "tier2_seed_recommendation_final": rec_reco,
        # Status
        "cell15_status": "success",
    }

    audit_rows.append(out_rec)
    append_checkpoint(CKPT_PATH, out_rec)

    # Progress update
    if len(audit_rows) % CHECKPOINT_INTERVAL == 0:
        elapsed = time.time() - start_time
        rate = len(audit_rows) / elapsed if elapsed > 0 else 0
        eta = (len(df) - len(audit_rows)) / rate / 60 if rate > 0 else 0
        print(f"   Audited {len(audit_rows):,} rows... (ETA: {eta:.1f}m)")

# =============================================================================
# STEP 4: SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: SAVE OUTPUTS")
print("=" * 80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save audit results
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT_AUDIT, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {OUT_AUDIT} ({len(audit_df):,} rows)")

# Save trace (failures)
trace_df = pd.DataFrame(trace_rows)
trace_df.to_csv(OUT_TRACE, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {OUT_TRACE} ({len(trace_df):,} rows)")

# Save seeds ready (final gate)
if not audit_df.empty and "tier2_seed_recommendation_final" in audit_df.columns:
    seeds_df = audit_df[audit_df["tier2_seed_recommendation_final"] == 1].copy()
else:
    seeds_df = pd.DataFrame()

seeds_df.to_csv(OUT_SEEDS, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {OUT_SEEDS} ({len(seeds_df):,} rows)")

# =============================================================================
# STEP 5: STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: AUDIT STATISTICS")
print("=" * 80)

elapsed_time = time.time() - start_time

# Calculate statistics
if not audit_df.empty:
    total_audited = len(audit_df)
    recommended = len(seeds_df)
    rejected = total_audited - recommended

    print(f"\n📊 AUDIT SUMMARY:")
    print(f"   Total audited:     {total_audited:,}")
    print(f"   Recommended (=1):  {recommended:,} ({recommended/total_audited*100:.1f}%)")
    print(f"   Rejected (=0):     {rejected:,} ({rejected/total_audited*100:.1f}%)")
    print(f"   Parse failures:    {len(trace_df):,}")

    # Score distributions
    print(f"\n📊 SCORE DISTRIBUTIONS (audited samples):")
    for col in ['anchor_validity', 'phenomenon_clarity', 'arabic_naturalness', 'collateral_severity']:
        if col in audit_df.columns:
            vals = audit_df[col].dropna()
            if len(vals) > 0:
                print(f"   {col:25s}: mean={vals.mean():.2f}, min={vals.min()}, max={vals.max()}")

    # By seed source
    print(f"\n📊 RECOMMENDATION BY SEED SOURCE:")
    if 'seed_source' in audit_df.columns and 'tier2_seed_recommendation_final' in audit_df.columns:
        for src in audit_df['seed_source'].unique():
            src_df = audit_df[audit_df['seed_source'] == src]
            src_reco = src_df['tier2_seed_recommendation_final'].sum()
            print(f"   {src}: {src_reco:,} / {len(src_df):,} recommended ({src_reco/len(src_df)*100:.1f}%)")

    # By error type
    print(f"\n📊 RECOMMENDATION BY ERROR TYPE (bottom 5):")
    if 'error_label' in audit_df.columns and 'tier2_seed_recommendation_final' in audit_df.columns:
        type_stats = audit_df.groupby('error_label').agg({
            'tier2_seed_recommendation_final': ['sum', 'count']
        })
        type_stats.columns = ['recommended', 'total']
        type_stats['rate'] = type_stats['recommended'] / type_stats['total']
        type_stats = type_stats.sort_values('rate')
        for et, row in type_stats.head(5).iterrows():
            print(f"   {et}: {int(row['recommended']):,} / {int(row['total']):,} ({row['rate']*100:.1f}%)")

    # Save statistics
    stats = {
        'timestamp': datetime.now().isoformat(),
        'total_input': len(df),
        'total_audited': total_audited,
        'recommended': recommended,
        'rejected': rejected,
        'parse_failures': len(trace_df),
        'api_errors': api_errors,
        'runtime_minutes': elapsed_time / 60,
        'recommendation_rate': recommended / total_audited if total_audited > 0 else 0,
        'score_means': {
            col: float(audit_df[col].mean())
            for col in ['anchor_validity', 'phenomenon_clarity', 'arabic_naturalness', 'collateral_severity']
            if col in audit_df.columns
        },
    }

    with open(STATS_PATH, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"\n✅ Saved: {STATS_PATH}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 15 COMPLETE")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   - {OUT_AUDIT} ({len(audit_df):,} audited samples)
   - {OUT_TRACE} ({len(trace_df):,} parse failures)
   - {OUT_SEEDS} ({len(seeds_df):,} recommended seeds)
   - {STATS_PATH}

📊 FINAL COUNTS:
   Input (Cell 14):        {len(df):,} rows (Clean_A)
   Audited successfully:   {len(audit_df):,} rows
   Recommended for Tier2:  {len(seeds_df):,} rows
   Rejected:               {len(audit_df) - len(seeds_df):,} rows

⏱️ Runtime: {elapsed_time/60:.1f} minutes
💰 Estimated cost: ${len(audit_df) * 0.003:.2f}

✅ Ready for Cell 16: Tier2 Generation (using {len(seeds_df):,} validated seeds)
""")

In [ ]:
# # =============================================================================
# # CELL 15.1: TIER1 MANUAL EVALUATION SAMPLING (v2.0 - Research-Grade)
# # =============================================================================
# """
# CELL 15.1: Manual Evaluation Sampling for Tier1 (Clean_A vs Clean_B)

# PURPOSE:
# Prepare stratified samples from FROZEN Clean_A and Clean_B artifacts for
# human evaluation. This cell does NOT recompute tiers - it loads final
# frozen outputs from the pipeline.

# FORMAT: v2.0 Research-Grade (Matches Cell 18 for Tier2)
# - Unified evaluation columns across Tier1 and Tier2
# - Error type validation (B1, B2, B3)
# - Annotator confidence and review flagging
# - Inter-annotator agreement support

# RESEARCH QUESTIONS:
# 1. Are both Clean_A and Clean_B truly "clean" or only Clean_A?
# 2. What is the correlation between automatic scores and human judgments?
# 3. What percentage of each tier is acceptable for downstream use?
# 4. Is the error type label correct for each sample?

# OUTPUTS:
# - TIER1_HUMANEVAL_SAMPLE.csv (evaluation sheet for annotators)
# - TIER1_HUMANEVAL_METRICS_HIDDEN.csv (metrics for post-analysis)
# - TIER1_EVALUATION_STATS.json (comprehensive statistics)
# - TIER1_CLASS_BREAKDOWN.csv (per-class breakdown)
# """

# import pandas as pd
# import numpy as np
# import json
# from datetime import datetime
# from pathlib import Path

# print("=" * 80)
# print("CELL 15.1: TIER1 MANUAL EVALUATION SAMPLING (v2.0 Research-Grade)")
# print("=" * 80)
# print("Format: Unified with Cell 18 (Tier2) for consistent evaluation")

# # =============================================================================
# # CONFIGURATION
# # =============================================================================

# OUTPUT_DIR = "tier1_v3_3_final"
# Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# # INPUT FILES (FROZEN - DO NOT RECOMPUTE)
# CLEAN_A_PATHS = [
#     f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv",
#     f"{OUTPUT_DIR}/tier1_clean_A.csv",
#     "tier1_clean_A.csv",
# ]

# CLEAN_B_PATHS = [
#     f"{OUTPUT_DIR}/tier1_clean_B_FROZEN.csv",
#     f"{OUTPUT_DIR}/tier1_clean_B.csv",
#     "tier1_clean_B.csv",
# ]

# # OUTPUT FILES (v2.0 naming convention)
# OUT_EVAL_SAMPLE = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_SAMPLE.csv"
# OUT_METRICS_HIDDEN = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_METRICS_HIDDEN.csv"
# OUT_STATS = f"{OUTPUT_DIR}/TIER1_EVALUATION_STATS.json"
# OUT_CLASS_BREAKDOWN = f"{OUTPUT_DIR}/TIER1_CLASS_BREAKDOWN.csv"

# # SAMPLING CONFIGURATION
# TARGET_PER_TIER = 100    # Target samples per tier
# MAX_PER_CLASS = 10       # Maximum samples per class per tier
# MIN_PER_CLASS = 1        # Minimum samples per class (include rare classes)
# RANDOM_SEED = 42

# np.random.seed(RANDOM_SEED)

# # Taxonomy (23 classes) - FIXED: Use "Name Entity Error" (no 'd')
# TAXONOMY_23 = [
#     "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
#     "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
#     "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
#     "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
#     "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
#     "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
#     "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
# ]

# # Omission classes (special OCS handling)
# OMISSION_CLASSES = {"Total Omission", "Partial Translation"}

# # v2.0 Evaluation columns (MUST MATCH Cell 18)
# EVAL_COLUMNS = [
#     # Identification
#     'eval_id', 'annotator_id', 'source_tier',
#     # Content
#     'en', 'ar', 'mt_output', 'Keyword', 'Best_Match', 'Sub_Subtype',
#     # Section A: Anchor/OCS Validation
#     'A1_kw_in_ar', 'A2_kw_count', 'A3_bm_in_mt', 'A4_bm_count', 'A5_anchor_aligned',
#     # Section B: Error Type Validation
#     'B1_error_type_correct', 'B2_suggested_type', 'B3_severity_appropriate',
#     # Section C: Quality Assessment
#     'C1_phenomenon_clear', 'C2_ar_natural', 'C3_mt_plausible', 'C4_no_collateral', 'C5_comprehension_impact',
#     # Section D: Decision
#     'D1_accept', 'D2_confidence', 'D3_needs_review', 'D4_notes',
# ]

# print(f"\n📊 CONFIGURATION:")
# print(f"   Target per tier: {TARGET_PER_TIER}")
# print(f"   Max per class: {MAX_PER_CLASS}")
# print(f"   Min per class: {MIN_PER_CLASS}")
# print(f"   Random seed: {RANDOM_SEED}")
# print(f"   Evaluation columns: {len(EVAL_COLUMNS)} (v2.0 format)")

# # =============================================================================
# # STEP 1: LOAD FROZEN DATA (NO RECOMPUTATION)
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 1: LOAD FROZEN DATA")
# print("=" * 80)
# print("⚠️  Loading from frozen files - NOT recomputing from CSR thresholds")

# def load_frozen_file(paths: list, name: str) -> pd.DataFrame:
#     """Load frozen file with fallback to alternative paths."""
#     for path in paths:
#         if Path(path).exists():
#             df = pd.read_csv(path)
#             print(f"✅ Loaded {name}: {path} ({len(df):,} rows)")
#             return df
#     print(f"❌ {name} not found. Tried: {paths}")
#     return pd.DataFrame()

# df_clean_A = load_frozen_file(CLEAN_A_PATHS, "Clean_A")
# df_clean_B = load_frozen_file(CLEAN_B_PATHS, "Clean_B")

# # Identify error column
# error_col = 'Sub-Subtype'
# if error_col not in df_clean_A.columns and not df_clean_A.empty:
#     # Try alternative column names
#     for alt in ['Sub_Subtype', 'error_type', 'Error_Type']:
#         if alt in df_clean_A.columns:
#             error_col = alt
#             break

# print(f"   Error column: {error_col}")

# # =============================================================================
# # STEP 2: GENERATE COMPREHENSIVE STATISTICS
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 2: COMPREHENSIVE STATISTICS")
# print("=" * 80)

# def get_class_statistics(df: pd.DataFrame, tier_name: str) -> dict:
#     """Get detailed statistics for a tier."""
#     if df.empty or error_col not in df.columns:
#         return {
#             'total_samples': 0,
#             'unique_classes': 0,
#             'classes_present': [],
#             'classes_missing': TAXONOMY_23.copy(),
#             'per_class_counts': {},
#             'classes_with_1': 0,
#             'classes_with_2': 0,
#             'classes_with_3_plus': 0,
#         }

#     class_counts = df[error_col].value_counts().to_dict()
#     classes_present = list(class_counts.keys())
#     classes_missing = [c for c in TAXONOMY_23 if c not in classes_present]

#     classes_with_1 = sum(1 for c in class_counts.values() if c == 1)
#     classes_with_2 = sum(1 for c in class_counts.values() if c == 2)
#     classes_with_3_plus = sum(1 for c in class_counts.values() if c >= 3)

#     return {
#         'total_samples': len(df),
#         'unique_classes': len(classes_present),
#         'classes_present': classes_present,
#         'classes_missing': classes_missing,
#         'per_class_counts': class_counts,
#         'classes_with_1': classes_with_1,
#         'classes_with_2': classes_with_2,
#         'classes_with_3_plus': classes_with_3_plus,
#     }

# stats_A = get_class_statistics(df_clean_A, "Clean_A")
# stats_B = get_class_statistics(df_clean_B, "Clean_B")

# print(f"\n📊 CLEAN_A STATISTICS:")
# print(f"   Total samples: {stats_A['total_samples']:,}")
# print(f"   Unique classes: {stats_A['unique_classes']}/23")
# print(f"   Missing classes: {len(stats_A['classes_missing'])}")

# print(f"\n📊 CLEAN_B STATISTICS:")
# print(f"   Total samples: {stats_B['total_samples']:,}")
# print(f"   Unique classes: {stats_B['unique_classes']}/23")
# print(f"   Missing classes: {len(stats_B['classes_missing'])}")

# # =============================================================================
# # STEP 3: STRATIFIED SAMPLING
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 3: STRATIFIED SAMPLING")
# print("=" * 80)

# def stratified_sample(df: pd.DataFrame, tier_name: str, target: int, max_per_class: int, min_per_class: int) -> pd.DataFrame:
#     """Perform stratified sampling ensuring all classes are represented."""
#     if df.empty or error_col not in df.columns:
#         return pd.DataFrame()

#     sampled_dfs = []
#     sample_counts = {}

#     classes_present = df[error_col].unique()
#     n_classes = len(classes_present)

#     if n_classes == 0:
#         return pd.DataFrame()

#     # Calculate samples per class
#     base_per_class = min(max_per_class, max(min_per_class, target // n_classes))

#     print(f"\n   {tier_name}: Sampling ~{base_per_class} per class from {n_classes} classes")

#     for cls in classes_present:
#         cls_df = df[df[error_col] == cls]
#         n_available = len(cls_df)

#         # Sample: min of (available, base_per_class)
#         n_sample = min(n_available, base_per_class)

#         # For rare classes, include all
#         if n_available <= min_per_class:
#             n_sample = n_available

#         if n_sample > 0:
#             sampled = cls_df.sample(n=n_sample, random_state=RANDOM_SEED)
#             sampled_dfs.append(sampled)
#             sample_counts[cls] = n_sample

#     if sampled_dfs:
#         result = pd.concat(sampled_dfs, ignore_index=True)
#         # Shuffle
#         result = result.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
#         return result, sample_counts

#     return pd.DataFrame(), {}

# sample_A, sample_counts_A = stratified_sample(df_clean_A, "Clean_A", TARGET_PER_TIER, MAX_PER_CLASS, MIN_PER_CLASS) if not df_clean_A.empty else (pd.DataFrame(), {})
# sample_B, sample_counts_B = stratified_sample(df_clean_B, "Clean_B", TARGET_PER_TIER, MAX_PER_CLASS, MIN_PER_CLASS) if not df_clean_B.empty else (pd.DataFrame(), {})

# print(f"\n📊 SAMPLING RESULTS:")
# print(f"   Clean_A: {len(sample_A)} samples from {len(sample_counts_A)} classes")
# print(f"   Clean_B: {len(sample_B)} samples from {len(sample_counts_B)} classes")

# # =============================================================================
# # STEP 4: BUILD v2.0 EVALUATION DATAFRAME
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 4: BUILD v2.0 EVALUATION DATAFRAME")
# print("=" * 80)
# print("Format: Unified with Cell 18 (Tier2) for consistent annotation")

# def build_eval_rows(df: pd.DataFrame, tier_label: str, id_prefix: str, start_idx: int = 0) -> list:
#     """Build evaluation rows in v2.0 format."""
#     eval_rows = []

#     for idx, row in df.iterrows():
#         eval_row = {
#             # === IDENTIFICATION ===
#             'eval_id': f"{id_prefix}_{start_idx + len(eval_rows) + 1:03d}",
#             'annotator_id': '',             # Annotator fills this
#             'source_tier': tier_label,      # "Tier1_Clean_A" or "Tier1_Clean_B"

#             # === CONTENT (for evaluation) ===
#             'en': row.get('en', ''),
#             'ar': row.get('ar', ''),
#             'mt_output': row.get('mt_output', ''),
#             'Keyword': row.get('Keyword', ''),
#             'Best_Match': row.get('Best_Match', ''),
#             'Sub_Subtype': row.get(error_col, ''),

#             # === SECTION A: ANCHOR VALIDATION (OCS) ===
#             'A1_kw_in_ar': '',           # Is Keyword in ar? (1/0)
#             'A2_kw_count': '',           # How many times? (0/1/2+)
#             'A3_bm_in_mt': '',           # Is Best_Match in mt_output? (1/0/OMIT)
#             'A4_bm_count': '',           # How many times? (0/1/2+/NA)
#             'A5_anchor_aligned': '',     # Same semantic slot? (1/0)

#             # === SECTION B: ERROR TYPE VALIDATION ===
#             'B1_error_type_correct': '', # Is labeled Sub_Subtype correct? (1/0)
#             'B2_suggested_type': '',     # If wrong, what should it be?
#             'B3_severity_appropriate': '',# Is severity rating appropriate? (1/0)

#             # === SECTION C: QUALITY ASSESSMENT ===
#             'C1_phenomenon_clear': '',   # Is error clearly visible? (1/0)
#             'C2_ar_natural': '',         # Is ar natural Arabic? (1/0)
#             'C3_mt_plausible': '',       # Is mt_output plausible Arabic? (1/0)
#             'C4_no_collateral': '',      # No extra errors? (1/0)
#             'C5_comprehension_impact': '',# Does error affect understanding? (1/0)

#             # === SECTION D: FINAL DECISION ===
#             'D1_accept': '',             # Accept into dataset? (1/0)
#             'D2_confidence': '',         # Confidence level (1=low/2=med/3=high)
#             'D3_needs_review': '',       # Flag for adjudication? (1/0)
#             'D4_notes': '',              # Free text comments
#         }
#         eval_rows.append(eval_row)

#     return eval_rows

# # Build evaluation rows for both tiers
# eval_rows_A = build_eval_rows(sample_A, "Tier1_Clean_A", "T1A", 0) if not sample_A.empty else []
# eval_rows_B = build_eval_rows(sample_B, "Tier1_Clean_B", "T1B", 0) if not sample_B.empty else []

# # Combine all rows
# all_eval_rows = eval_rows_A + eval_rows_B

# # Create DataFrame with proper column order
# df_eval = pd.DataFrame(all_eval_rows, columns=EVAL_COLUMNS)

# # Shuffle combined dataset
# df_eval = df_eval.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# # Re-assign sequential eval_ids after shuffling
# df_eval['eval_id'] = [f"T1_{i+1:03d}" for i in range(len(df_eval))]

# print(f"✅ Built evaluation dataframe: {len(df_eval)} samples")
# print(f"   Clean_A samples: {len(eval_rows_A)}")
# print(f"   Clean_B samples: {len(eval_rows_B)}")
# print(f"   Columns: {len(EVAL_COLUMNS)}")

# # =============================================================================
# # STEP 5: SAVE EVALUATION SAMPLE (FOR ANNOTATORS)
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 5: SAVE EVALUATION SAMPLE")
# print("=" * 80)

# df_eval.to_csv(OUT_EVAL_SAMPLE, index=False, encoding='utf-8-sig')
# print(f"✅ Saved: {OUT_EVAL_SAMPLE}")
# print(f"   {len(df_eval)} samples for annotation")
# print(f"   ℹ️  Share this file with annotators")

# # =============================================================================
# # STEP 6: SAVE HIDDEN METRICS (FOR POST-ANALYSIS)
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 6: SAVE HIDDEN METRICS")
# print("=" * 80)

# def build_metrics_rows(df: pd.DataFrame, tier_label: str, id_prefix: str) -> list:
#     """Build metrics rows (hidden from annotators)."""
#     metrics_rows = []

#     metric_cols = ['Sentence_ID', 'csr', 'ocs', 'sfr', 'mpqs',
#                    'Severity_Level', 'Severity_Score', 'TQA_Category',
#                    'Class', 'Subtype', 'rejection_reason']

#     for idx, row in df.iterrows():
#         metrics_row = {
#             'eval_id': f"{id_prefix}_{len(metrics_rows) + 1:03d}",
#             'source_tier': tier_label,
#         }

#         # Add available metric columns
#         for col in metric_cols:
#             if col in row.index:
#                 metrics_row[col] = row[col]
#             else:
#                 metrics_row[col] = ''

#         # Add Sub-Subtype
#         metrics_row['Sub-Subtype'] = row.get(error_col, '')

#         metrics_rows.append(metrics_row)

#     return metrics_rows

# metrics_A = build_metrics_rows(sample_A, "Tier1_Clean_A", "T1A") if not sample_A.empty else []
# metrics_B = build_metrics_rows(sample_B, "Tier1_Clean_B", "T1B") if not sample_B.empty else []

# all_metrics = metrics_A + metrics_B
# df_metrics = pd.DataFrame(all_metrics)

# # Match eval_id order with evaluation sample
# if not df_eval.empty and not df_metrics.empty:
#     # Create mapping from original eval_id to new shuffled eval_id
#     # This is complex since we shuffled - for now, just save separately
#     pass

# df_metrics.to_csv(OUT_METRICS_HIDDEN, index=False)
# print(f"✅ Saved: {OUT_METRICS_HIDDEN}")
# print(f"   ⚠️ DO NOT share with annotators!")

# # =============================================================================
# # STEP 7: SAVE CLASS BREAKDOWN
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 7: SAVE CLASS BREAKDOWN")
# print("=" * 80)

# breakdown_rows = []
# for cls in TAXONOMY_23:
#     breakdown_rows.append({
#         'Sub_Subtype': cls,
#         'Clean_A_available': stats_A['per_class_counts'].get(cls, 0),
#         'Clean_A_sampled': sample_counts_A.get(cls, 0),
#         'Clean_B_available': stats_B['per_class_counts'].get(cls, 0),
#         'Clean_B_sampled': sample_counts_B.get(cls, 0),
#         'Total_sampled': sample_counts_A.get(cls, 0) + sample_counts_B.get(cls, 0),
#     })

# df_breakdown = pd.DataFrame(breakdown_rows)
# df_breakdown.to_csv(OUT_CLASS_BREAKDOWN, index=False)
# print(f"✅ Saved: {OUT_CLASS_BREAKDOWN}")

# # Show breakdown
# print(f"\n{'Sub-Subtype':<35} | {'A_avail':>7} | {'A_samp':>6} | {'B_avail':>7} | {'B_samp':>6} | {'Total':>5}")
# print("-" * 85)
# for _, row in df_breakdown.iterrows():
#     print(f"{row['Sub_Subtype']:<35} | {row['Clean_A_available']:>7} | {row['Clean_A_sampled']:>6} | {row['Clean_B_available']:>7} | {row['Clean_B_sampled']:>6} | {row['Total_sampled']:>5}")

# # =============================================================================
# # STEP 8: SAVE COMPREHENSIVE STATISTICS JSON
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 8: SAVE STATISTICS JSON")
# print("=" * 80)

# reviewer_stats = {
#     "metadata": {
#         "generated_at": datetime.now().isoformat(),
#         "random_seed": RANDOM_SEED,
#         "taxonomy_classes": 23,
#         "format_version": "v2.0_research_grade",
#         "note": "Clean_A and Clean_B loaded from FROZEN files - no threshold recomputation"
#     },
#     "configuration": {
#         "target_per_tier": TARGET_PER_TIER,
#         "max_per_class": MAX_PER_CLASS,
#         "min_per_class": MIN_PER_CLASS,
#     },
#     "source_data": {
#         "clean_A": {
#             "total_samples": stats_A['total_samples'],
#             "unique_classes": stats_A['unique_classes'],
#             "classes_missing": stats_A['classes_missing'],
#             "classes_with_1_sample": stats_A['classes_with_1'],
#             "classes_with_2_samples": stats_A['classes_with_2'],
#             "classes_with_3plus_samples": stats_A['classes_with_3_plus'],
#         },
#         "clean_B": {
#             "total_samples": stats_B['total_samples'],
#             "unique_classes": stats_B['unique_classes'],
#             "classes_missing": stats_B['classes_missing'],
#             "classes_with_1_sample": stats_B['classes_with_1'],
#             "classes_with_2_samples": stats_B['classes_with_2'],
#             "classes_with_3plus_samples": stats_B['classes_with_3_plus'],
#         },
#         "combined_total": stats_A['total_samples'] + stats_B['total_samples']
#     },
#     "sampled_data": {
#         "clean_A": {
#             "samples_drawn": len(sample_A) if not sample_A.empty else 0,
#             "classes_covered": sample_A[error_col].nunique() if not sample_A.empty and error_col in sample_A.columns else 0,
#             "per_class_counts": sample_counts_A,
#         },
#         "clean_B": {
#             "samples_drawn": len(sample_B) if not sample_B.empty else 0,
#             "classes_covered": sample_B[error_col].nunique() if not sample_B.empty and error_col in sample_B.columns else 0,
#             "per_class_counts": sample_counts_B,
#         },
#         "combined_total": len(df_eval)
#     },
#     "evaluation_schema": {
#         "format_version": "v2.0",
#         "total_columns": len(EVAL_COLUMNS),
#         "section_a_anchor": ['A1_kw_in_ar', 'A2_kw_count', 'A3_bm_in_mt', 'A4_bm_count', 'A5_anchor_aligned'],
#         "section_b_error_type": ['B1_error_type_correct', 'B2_suggested_type', 'B3_severity_appropriate'],
#         "section_c_quality": ['C1_phenomenon_clear', 'C2_ar_natural', 'C3_mt_plausible', 'C4_no_collateral', 'C5_comprehension_impact'],
#         "section_d_decision": ['D1_accept', 'D2_confidence', 'D3_needs_review', 'D4_notes'],
#     },
#     "improvements_v2": [
#         "Added B1_error_type_correct for taxonomy validation",
#         "Added B2_suggested_type for label correction",
#         "Added B3_severity_appropriate for severity validation",
#         "Added C3_mt_plausible for MT quality assessment",
#         "Added C5_comprehension_impact for error significance",
#         "Added D2_confidence for annotator certainty",
#         "Added D3_needs_review for adjudication flagging",
#         "Added annotator_id for inter-annotator agreement",
#         "Added source_tier for tier-specific analysis",
#         "Column prefixes (A1_, B1_, etc.) for clarity",
#         "Unified format with Cell 18 (Tier2)",
#     ]
# }

# with open(OUT_STATS, 'w', encoding='utf-8') as f:
#     json.dump(reviewer_stats, f, indent=2, ensure_ascii=False)
# print(f"✅ Saved: {OUT_STATS}")

# # =============================================================================
# # SUMMARY
# # =============================================================================

# print("\n" + "=" * 80)
# print("🎉 CELL 15.1 COMPLETE (v2.0 Research-Grade)")
# print("=" * 80)

# clean_a_sampled = len(sample_A) if not sample_A.empty else 0
# clean_b_sampled = len(sample_B) if not sample_B.empty else 0
# clean_a_classes = sample_A[error_col].nunique() if not sample_A.empty and error_col in sample_A.columns else 0
# clean_b_classes = sample_B[error_col].nunique() if not sample_B.empty and error_col in sample_B.columns else 0

# print(f"""
# {'='*80}
# 📊 FINAL SUMMARY
# {'='*80}

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │                        SOURCE DATA (FROZEN TIER1)                           │
# ├─────────────────────────────┬───────────────────┬───────────────────────────┤
# │ Metric                      │     Clean_A       │      Clean_B              │
# ├─────────────────────────────┼───────────────────┼───────────────────────────┤
# │ Total Samples               │ {stats_A['total_samples']:>13,}     │ {stats_B['total_samples']:>17,}     │
# │ Unique Classes (of 23)      │ {stats_A['unique_classes']:>13}     │ {stats_B['unique_classes']:>17}     │
# │ Missing Classes             │ {len(stats_A['classes_missing']):>13}     │ {len(stats_B['classes_missing']):>17}     │
# └─────────────────────────────┴───────────────────┴───────────────────────────┘

# ┌─────────────────────────────────────────────────────────────────────────────┐
# │                        EVALUATION SAMPLE (v2.0)                             │
# ├─────────────────────────────┬───────────────────┬───────────────────────────┤
# │ Metric                      │     Clean_A       │      Clean_B              │
# ├─────────────────────────────┼───────────────────┼───────────────────────────┤
# │ Samples Drawn               │ {clean_a_sampled:>13,}     │ {clean_b_sampled:>17,}     │
# │ Classes Covered             │ {clean_a_classes:>13}     │ {clean_b_classes:>17}     │
# └─────────────────────────────┴───────────────────┴───────────────────────────┘

# COMBINED: {len(df_eval)} total samples for manual evaluation

# {'='*80}
# 📋 v2.0 EVALUATION FORMAT (17 columns)
# {'='*80}

# Section A - Anchor Validation (5 columns):
#    A1_kw_in_ar, A2_kw_count, A3_bm_in_mt, A4_bm_count, A5_anchor_aligned

# Section B - Error Type Validation (3 columns): ⭐ NEW
#    B1_error_type_correct, B2_suggested_type, B3_severity_appropriate

# Section C - Quality Assessment (5 columns):
#    C1_phenomenon_clear, C2_ar_natural, C3_mt_plausible, C4_no_collateral, C5_comprehension_impact

# Section D - Decision (4 columns):
#    D1_accept, D2_confidence, D3_needs_review, D4_notes

# {'='*80}
# 📁 OUTPUT FILES
# {'='*80}

# FOR ANNOTATORS:
#    📄 {OUT_EVAL_SAMPLE}

# FOR RESEARCH TEAM (DO NOT SHARE):
#    🔒 {OUT_METRICS_HIDDEN}
#    📊 {OUT_STATS}
#    📊 {OUT_CLASS_BREAKDOWN}

# {'='*80}
# 📋 NEXT STEPS
# {'='*80}

# 1. Send to annotators:
#    - {OUT_EVAL_SAMPLE}
#    - UNIFIED_ANNOTATOR_MANUAL_v2.md (same manual as Tier2)

# 2. Annotators fill ALL evaluation columns (A1-D4)

# 3. Collect completed files: TIER1_HUMANEVAL_SAMPLE_[ANNOTATOR_ID].csv

# 4. Calculate inter-annotator agreement (Cohen's Kappa)

# 5. Adjudicate disagreements (D3_needs_review=1 samples)

# 6. Run Cell 15.6 for post-evaluation analysis

# ✅ Format unified with Cell 18 (Tier2) - annotators use ONE manual for both!
# """)

In [ ]:
# # =============================================================================
# # CELL 15.5: MANUAL EVALUATION OF CLEAN_A vs CLEAN_B QUALITY
# # =============================================================================
# """
# CELL 15.5: Manual Evaluation of Clean_A vs Clean_B Quality

# PURPOSE:
# Validate the effectiveness of our quality thresholds (Clean_A > 0.65, Clean_B > 0.55)
# through manual evaluation of stratified samples.

# RESEARCH QUESTIONS:
# 1. Are both Clean_A and Clean_B truly "clean" or only Clean_A?
# 2. What is the correlation between automatic csr scores and human judgments?
# 3. Are our thresholds optimally separating quality tiers?

# METHODOLOGY:
# 1. Stratified random sampling from Clean_A and Clean_B
# 2. Balanced sampling across error classes where possible
# 3. Manual annotation with 5-point scales + binary recommendation
# 4. Statistical analysis of results

# OUTPUTS:
# - tier1_v3_3_final/manual_evaluation_samples.csv (samples for annotators)
# - tier1_v3_3_final/manual_evaluation_analysis.csv (analysis results)
# - tier1_v3_3_final/quality_threshold_validation.json (validation metrics)
# """

# import pandas as pd
# import numpy as np
# import random
# import json
# import os
# from datetime import datetime
# from typing import List, Dict, Tuple
# import matplotlib.pyplot as plt
# import seaborn as sns
# from scipy import stats

# print("=" * 80)
# print("CELL 15.5: MANUAL EVALUATION OF CLEAN_A vs CLEAN_B")
# print("=" * 80)

# # =============================================================================
# # CONFIGURATION
# # =============================================================================

# OUTPUT_DIR = "tier1_v3_3_final"

# # Input files
# CLEAN_A_AUDIT = f"{OUTPUT_DIR}/tier2_seed_audit_cell15.csv"  # From Cell 15
# CLEAN_A_CSR = f"{OUTPUT_DIR}/tier1_clean_A_for_etca.csv"     # From Cell 14
# COMPLETE_DATA = f"{OUTPUT_DIR}/tier1_complete_23_classes.csv" # Full dataset with A/B classification

# # Output files
# SAMPLES_CSV = f"{OUTPUT_DIR}/manual_evaluation_samples.csv"
# ANALYSIS_CSV = f"{OUTPUT_DIR}/manual_evaluation_analysis.csv"
# VALIDATION_JSON = f"{OUTPUT_DIR}/quality_threshold_validation.json"
# PLOT_PATH = f"{OUTPUT_DIR}/quality_threshold_validation.png"

# # Sampling settings
# SAMPLING_RATE = 0.10  # 10% overall
# MAX_PER_CLASS = 10    # Cap per error class per tier
# MIN_PER_CLASS = 2     # Minimum per error class per tier (if available)

# # Random seed for reproducibility
# RANDOM_SEED = 42
# random.seed(RANDOM_SEED)
# np.random.seed(RANDOM_SEED)

# print(f"\n📊 CONFIGURATION:")
# print(f"   Sampling rate: {SAMPLING_RATE*100:.0f}%")
# print(f"   Max per class: {MAX_PER_CLASS}")
# print(f"   Min per class: {MIN_PER_CLASS}")
# print(f"   Random seed: {RANDOM_SEED}")

# # =============================================================================
# # STEP 1: LOAD AND PREPARE DATA
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 1: LOAD AND PREPARE DATA")
# print("=" * 80)

# # Load complete dataset to get Clean_B samples
# if os.path.exists(COMPLETE_DATA):
#     df_complete = pd.read_csv(COMPLETE_DATA)

#     # Identify Clean_A and Clean_B based on csr thresholds
#     # (Assuming 'csr' column exists and thresholds are as defined)
#     if 'csr' in df_complete.columns:
#         df_clean_A = df_complete[df_complete['csr'] > 0.65].copy()
#         df_clean_B = df_complete[(df_complete['csr'] >= 0.55) & (df_complete['csr'] <= 0.65)].copy()

#         print(f"✅ Loaded complete dataset: {len(df_complete):,} rows")
#         print(f"   Clean_A (csr > 0.65): {len(df_clean_A):,} rows")
#         print(f"   Clean_B (0.55 ≤ csr ≤ 0.65): {len(df_clean_B):,} rows")
#     else:
#         print("⚠️ 'csr' column not found in complete dataset")
#         # Try to load Clean_A from Cell 15 output and infer Clean_B
#         df_clean_A = pd.DataFrame()
#         df_clean_B = pd.DataFrame()
# else:
#     print(f"⚠️ Complete dataset not found: {COMPLETE_DATA}")
#     df_clean_A = pd.DataFrame()
#     df_clean_B = pd.DataFrame()

# # Load Cell 15 audit results for Clean_A
# if os.path.exists(CLEAN_A_AUDIT):
#     df_audit_A = pd.read_csv(CLEAN_A_AUDIT)
#     print(f"✅ Loaded Clean_A audit: {len(df_audit_A):,} rows")

#     # Merge audit scores with Clean_A data
#     if not df_clean_A.empty and 'Sentence_ID' in df_clean_A.columns:
#         df_clean_A = df_clean_A.merge(
#             df_audit_A[['Sentence_ID', 'anchor_validity', 'phenomenon_clarity',
#                        'arabic_naturalness', 'collateral_severity',
#                        'tier2_seed_recommendation_final']],
#             on='Sentence_ID', how='left'
#         )
# else:
#     print(f"⚠️ Clean_A audit not found: {CLEAN_A_AUDIT}")
#     df_audit_A = pd.DataFrame()

# # If Clean_B is empty, try to load from alternative source
# if df_clean_B.empty:
#     # Check if there's a separate Clean_B file
#     clean_b_file = f"{OUTPUT_DIR}/tier1_clean_B.csv"
#     if os.path.exists(clean_b_file):
#         df_clean_B = pd.read_csv(clean_b_file)
#         print(f"✅ Loaded Clean_B from separate file: {len(df_clean_B):,} rows")
#     else:
#         print("⚠️ Clean_B data not found - using empty DataFrame")

# # =============================================================================
# # STEP 2: STRATIFIED SAMPLING FUNCTION
# # =============================================================================

# def stratified_sample_with_limits(
#     df: pd.DataFrame,
#     error_col: str,
#     sampling_rate: float,
#     max_per_class: int,
#     min_per_class: int,
#     tier_name: str
# ) -> pd.DataFrame:
#     """
#     Perform stratified sampling with limits per error class.

#     Args:
#         df: DataFrame to sample from
#         error_col: Column name for error classes
#         sampling_rate: Overall sampling rate (e.g., 0.1 for 10%)
#         max_per_class: Maximum samples per class
#         min_per_class: Minimum samples per class (if class has enough samples)
#         tier_name: Name of the tier (for logging)

#     Returns:
#         Sampled DataFrame
#     """

#     if df.empty:
#         return pd.DataFrame()

#     sampled_rows = []

#     # Get unique error classes
#     unique_classes = df[error_col].unique()

#     print(f"\n📊 Sampling from {tier_name}:")
#     print(f"   Total rows: {len(df):,}")
#     print(f"   Unique classes: {len(unique_classes)}")

#     for error_class in unique_classes:
#         class_df = df[df[error_col] == error_class]
#         class_size = len(class_df)

#         # Calculate base sample size
#         base_sample = int(np.ceil(class_size * sampling_rate))

#         # Apply limits
#         if class_size >= min_per_class:
#             sample_size = max(min_per_class, min(base_sample, max_per_class))
#         else:
#             # If class has fewer than min_per_class, take all
#             sample_size = min(class_size, max_per_class)

#         # Sample from this class
#         if sample_size > 0 and sample_size <= class_size:
#             class_sample = class_df.sample(n=sample_size, random_state=RANDOM_SEED)
#             sampled_rows.append(class_sample)

#             print(f"   {error_class}: {sample_size}/{class_size}")

#     if sampled_rows:
#         return pd.concat(sampled_rows, ignore_index=True)
#     else:
#         return pd.DataFrame()

# # =============================================================================
# # STEP 3: PERFORM SAMPLING
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 3: PERFORM STRATIFIED SAMPLING")
# print("=" * 80)

# # Sample from Clean_A
# df_sample_A = stratified_sample_with_limits(
#     df=df_clean_A,
#     error_col='Sub-Subtype',
#     sampling_rate=SAMPLING_RATE,
#     max_per_class=MAX_PER_CLASS,
#     min_per_class=MIN_PER_CLASS,
#     tier_name="Clean_A"
# )

# # Sample from Clean_B
# df_sample_B = stratified_sample_with_limits(
#     df=df_clean_B,
#     error_col='Sub-Subtype',
#     sampling_rate=SAMPLING_RATE,
#     max_per_class=MAX_PER_CLASS,
#     min_per_class=MIN_PER_CLASS,
#     tier_name="Clean_B"
# )

# # Combine samples
# df_samples = pd.concat([df_sample_A, df_sample_B], ignore_index=True)

# print(f"\n📊 SAMPLING RESULTS:")
# print(f"   Clean_A samples: {len(df_sample_A):,}")
# print(f"   Clean_B samples: {len(df_sample_B):,}")
# print(f"   Total samples: {len(df_samples):,}")

# # Add tier identifier
# df_sample_A['tier'] = 'Clean_A'
# df_sample_B['tier'] = 'Clean_B'
# df_samples['tier'] = np.where(
#     df_samples['Sentence_ID'].isin(df_sample_A['Sentence_ID']),
#     'Clean_A',
#     'Clean_B'
# )

# # =============================================================================
# # STEP 4: CREATE MANUAL EVALUATION SHEET
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 4: CREATE MANUAL EVALUATION SHEET")
# print("=" * 80)

# # Define evaluation columns
# EVALUATION_COLUMNS = [
#     # Sample identifier
#     "Sentence_ID", "tier", "csr", "cps", "ocs", "sfr",

#     # Error information
#     "Sub-Subtype", "Keyword", "Best_Match",

#     # Arabic sentences
#     "ar", "mt_output", "en",

#     # Cell 15 audit scores (if available)
#     "anchor_validity_auto", "phenomenon_clarity_auto",
#     "arabic_naturalness_auto", "collateral_severity_auto",
#     "tier2_seed_recommendation_auto",

#     # Manual evaluation columns (TO BE FILLED)
#     "manual_anchor_validity", "manual_phenomenon_clarity",
#     "manual_arabic_naturalness", "manual_collateral_distortion",
#     "manual_overall_agreement", "manual_notes"
# ]

# # Create evaluation DataFrame
# eval_df = pd.DataFrame(columns=EVALUATION_COLUMNS)

# # Populate with sample data
# for col in ['Sentence_ID', 'tier', 'csr', 'cps', 'ocs', 'sfr',
#             'Sub-Subtype', 'Keyword', 'Best_Match', 'ar', 'mt_output', 'en']:
#     if col in df_samples.columns:
#         eval_df[col] = df_samples[col]
#     else:
#         eval_df[col] = ""

# # Add audit scores for Clean_A samples
# if not df_audit_A.empty:
#     for idx, row in eval_df.iterrows():
#         if row['tier'] == 'Clean_A':
#             audit_row = df_audit_A[df_audit_A['Sentence_ID'] == row['Sentence_ID']]
#             if not audit_row.empty:
#                 eval_df.at[idx, 'anchor_validity_auto'] = audit_row.iloc[0].get('anchor_validity', '')
#                 eval_df.at[idx, 'phenomenon_clarity_auto'] = audit_row.iloc[0].get('phenomenon_clarity', '')
#                 eval_df.at[idx, 'arabic_naturalness_auto'] = audit_row.iloc[0].get('arabic_naturalness', '')
#                 eval_df.at[idx, 'collateral_severity_auto'] = audit_row.iloc[0].get('collateral_severity', '')
#                 eval_df.at[idx, 'tier2_seed_recommendation_auto'] = audit_row.iloc[0].get('tier2_seed_recommendation_final', '')

# # Initialize manual evaluation columns
# eval_df['manual_anchor_validity'] = ""
# eval_df['manual_phenomenon_clarity'] = ""
# eval_df['manual_arabic_naturalness'] = ""
# eval_df['manual_collateral_distortion'] = ""
# eval_df['manual_overall_agreement'] = ""  # Y/N
# eval_df['manual_notes'] = ""

# # Add evaluation instructions as the first row
# instructions = {
#     'Sentence_ID': 'INSTRUCTIONS',
#     'tier': 'Please evaluate each sample on a scale of 1-5 for the first four metrics:',
#     'csr': '1. Anchor Validity: 1=contradicts label, 5=perfect anchor alignment',
#     'cps': '2. Phenomenon Clarity: 1=barely observable, 5=unambiguous',
#     'ocs': '3. Arabic Naturalness: 1=broken Arabic, 5=fluent native-like Arabic',
#     'sfr': '4. Collateral Distortion: 1=no unrelated changes, 5=severe unrelated distortion',
#     'Sub-Subtype': '5. Overall Agreement: Y=valid example, N=invalid example',
#     'Keyword': 'Notes: Explain any issues or special considerations',
#     'Best_Match': '',
#     'ar': '',
#     'mt_output': '',
#     'en': '',
#     'anchor_validity_auto': '',
#     'phenomenon_clarity_auto': '',
#     'arabic_naturalness_auto': '',
#     'collateral_severity_auto': '',
#     'tier2_seed_recommendation_auto': '',
#     'manual_anchor_validity': '',
#     'manual_phenomenon_clarity': '',
#     'manual_arabic_naturalness': '',
#     'manual_collateral_distortion': '',
#     'manual_overall_agreement': '',
#     'manual_notes': ''
# }

# # Insert instructions as first row
# instructions_df = pd.DataFrame([instructions])
# eval_df = pd.concat([instructions_df, eval_df], ignore_index=True)

# # Save evaluation sheet
# eval_df.to_csv(SAMPLES_CSV, index=False, encoding='utf-8-sig')
# print(f"✅ Saved evaluation sheet: {SAMPLES_CSV}")
# print(f"   Total rows: {len(eval_df)-1} samples + 1 instruction row")

# # =============================================================================
# # STEP 5: CREATE ANALYSIS TEMPLATE
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 5: CREATE ANALYSIS TEMPLATE")
# print("=" * 80)

# # Create analysis DataFrame template
# analysis_columns = [
#     'Sentence_ID', 'tier', 'Sub-Subtype',

#     # Automatic scores
#     'csr', 'anchor_validity_auto', 'phenomenon_clarity_auto',
#     'arabic_naturalness_auto', 'collateral_severity_auto',
#     'tier2_seed_recommendation_auto',

#     # Manual scores (to be filled after evaluation)
#     'manual_anchor_validity', 'manual_phenomenon_clarity',
#     'manual_arabic_naturalness', 'manual_collateral_distortion',
#     'manual_overall_agreement_binary',

#     # Derived metrics
#     'manual_score_avg', 'auto_score_avg',
#     'agreement_binary', 'score_difference',

#     # Analysis flags
#     'is_true_positive', 'is_false_positive',
#     'is_true_negative', 'is_false_negative'
# ]

# analysis_df = pd.DataFrame(columns=analysis_columns)

# # Populate with sample data
# for col in ['Sentence_ID', 'tier', 'Sub-Subtype', 'csr',
#             'anchor_validity_auto', 'phenomenon_clarity_auto',
#             'arabic_naturalness_auto', 'collateral_severity_auto',
#             'tier2_seed_recommendation_auto']:
#     if col in eval_df.columns:
#         # Skip instruction row
#         analysis_df[col] = eval_df.iloc[1:][col].reset_index(drop=True)

# # Save analysis template
# analysis_df.to_csv(ANALYSIS_CSV, index=False, encoding='utf-8-sig')
# print(f"✅ Saved analysis template: {ANALYSIS_CSV}")

# # =============================================================================
# # STEP 6: STATISTICAL ANALYSIS FUNCTIONS (FOR LATER USE)
# # =============================================================================

# def analyze_quality_thresholds(analysis_df: pd.DataFrame) -> Dict:
#     """
#     Analyze correlation between automatic and manual scores.

#     Returns:
#         Dictionary with analysis results
#     """

#     if analysis_df.empty:
#         return {}

#     results = {
#         'timestamp': datetime.now().isoformat(),
#         'n_samples': len(analysis_df),
#         'n_clean_a': len(analysis_df[analysis_df['tier'] == 'Clean_A']),
#         'n_clean_b': len(analysis_df[analysis_df['tier'] == 'Clean_B'])
#     }

#     # Only proceed if manual scores are available
#     required_cols = ['manual_anchor_validity', 'manual_overall_agreement_binary']
#     if all(col in analysis_df.columns for col in required_cols):
#         # Convert manual scores to numeric
#         numeric_cols = ['manual_anchor_validity', 'manual_phenomenon_clarity',
#                        'manual_arabic_naturalness', 'manual_collateral_distortion']

#         for col in numeric_cols:
#             if col in analysis_df.columns:
#                 analysis_df[col] = pd.to_numeric(analysis_df[col], errors='coerce')

#         # Calculate manual average score
#         analysis_df['manual_score_avg'] = analysis_df[numeric_cols].mean(axis=1)

#         # Calculate automatic average score (if available)
#         auto_cols = ['anchor_validity_auto', 'phenomenon_clarity_auto',
#                     'arabic_naturalness_auto', 'collateral_severity_auto']
#         auto_cols = [c for c in auto_cols if c in analysis_df.columns]

#         if auto_cols:
#             for col in auto_cols:
#                 analysis_df[col] = pd.to_numeric(analysis_df[col], errors='coerce')
#             analysis_df['auto_score_avg'] = analysis_df[auto_cols].mean(axis=1)

#         # Compare Clean_A vs Clean_B
#         clean_a_scores = analysis_df[analysis_df['tier'] == 'Clean_A']['manual_score_avg'].dropna()
#         clean_b_scores = analysis_df[analysis_df['tier'] == 'Clean_B']['manual_score_avg'].dropna()

#         if len(clean_a_scores) > 1 and len(clean_b_scores) > 1:
#             t_stat, p_value = stats.ttest_ind(clean_a_scores, clean_b_scores)
#             results['clean_a_vs_b'] = {
#                 'mean_a': float(clean_a_scores.mean()),
#                 'mean_b': float(clean_b_scores.mean()),
#                 'std_a': float(clean_a_scores.std()),
#                 'std_b': float(clean_b_scores.std()),
#                 't_statistic': float(t_stat),
#                 'p_value': float(p_value),
#                 'significant': p_value < 0.05
#             }

#         # Calculate correlation between csr and manual scores
#         if 'csr' in analysis_df.columns and 'manual_score_avg' in analysis_df.columns:
#             analysis_df['csr'] = pd.to_numeric(analysis_df['csr'], errors='coerce')
#             valid_mask = analysis_df['csr'].notna() & analysis_df['manual_score_avg'].notna()

#             if valid_mask.sum() > 2:
#                 r_csr, p_csr = stats.pearsonr(
#                     analysis_df.loc[valid_mask, 'csr'],
#                     analysis_df.loc[valid_mask, 'manual_score_avg']
#                 )
#                 results['correlation_csr_manual'] = {
#                     'pearson_r': float(r_csr),
#                     'p_value': float(p_csr),
#                     'significant': p_csr < 0.05
#                 }

#         # Calculate classification metrics
#         if 'manual_overall_agreement_binary' in analysis_df.columns:
#             # Convert Y/N to 1/0
#             analysis_df['manual_binary'] = analysis_df['manual_overall_agreement_binary'].map({'Y': 1, 'N': 0, 'y': 1, 'n': 0})

#             # For Clean_A: we expect mostly Y (1)
#             clean_a_manual = analysis_df[analysis_df['tier'] == 'Clean_A']['manual_binary'].dropna()
#             if len(clean_a_manual) > 0:
#                 results['clean_a_accuracy'] = {
#                     'n_samples': int(len(clean_a_manual)),
#                     'n_positive': int(clean_a_manual.sum()),
#                     'n_negative': int(len(clean_a_manual) - clean_a_manual.sum()),
#                     'accuracy': float(clean_a_manual.mean()),
#                     'precision': None  # Will calculate later
#                 }

#             # For Clean_B
#             clean_b_manual = analysis_df[analysis_df['tier'] == 'Clean_B']['manual_binary'].dropna()
#             if len(clean_b_manual) > 0:
#                 results['clean_b_accuracy'] = {
#                     'n_samples': int(len(clean_b_manual)),
#                     'n_positive': int(clean_b_manual.sum()),
#                     'n_negative': int(len(clean_b_manual) - clean_b_manual.sum()),
#                     'accuracy': float(clean_b_manual.mean()),
#                     'precision': None
#                 }

#     return results

# def plot_quality_distributions(df: pd.DataFrame, save_path: str):
#     """
#     Create visualization of quality distributions.
#     """
#     if df.empty or 'tier' not in df.columns:
#         return

#     # Create figure
#     fig, axes = plt.subplots(2, 2, figsize=(12, 10))

#     # Plot 1: csr distribution by tier
#     if 'csr' in df.columns:
#         axes[0, 0].hist([df[df['tier']=='Clean_A']['csr'].dropna(),
#                         df[df['tier']=='Clean_B']['csr'].dropna()],
#                        bins=20, alpha=0.7, label=['Clean_A', 'Clean_B'])
#         axes[0, 0].axvline(x=0.65, color='r', linestyle='--', label='A threshold')
#         axes[0, 0].axvline(x=0.55, color='g', linestyle='--', label='B threshold')
#         axes[0, 0].set_xlabel('CSR Score')
#         axes[0, 0].set_ylabel('Frequency')
#         axes[0, 0].set_title('CSR Distribution by Tier')
#         axes[0, 0].legend()

#     # Plot 2: Manual scores by tier (if available)
#     manual_cols = ['manual_anchor_validity', 'manual_phenomenon_clarity',
#                    'manual_arabic_naturalness', 'manual_collateral_distortion']
#     manual_cols = [c for c in manual_cols if c in df.columns]

#     if manual_cols and any(df[c].notna().any() for c in manual_cols):
#         # Convert to numeric
#         for col in manual_cols:
#             df[col] = pd.to_numeric(df[col], errors='coerce')

#         # Calculate average manual score
#         df['manual_avg'] = df[manual_cols].mean(axis=1)

#         # Boxplot
#         tier_data = []
#         tier_labels = []
#         for tier in ['Clean_A', 'Clean_B']:
#             tier_scores = df[df['tier'] == tier]['manual_avg'].dropna()
#             if len(tier_scores) > 0:
#                 tier_data.append(tier_scores)
#                 tier_labels.append(tier)

#         if tier_data:
#             axes[0, 1].boxplot(tier_data, labels=tier_labels)
#             axes[0, 1].set_ylabel('Average Manual Score (1-5)')
#             axes[0, 1].set_title('Manual Evaluation Scores by Tier')

#     # Plot 3: Error class distribution
#     if 'Sub-Subtype' in df.columns:
#         class_counts = df['Sub-Subtype'].value_counts()
#         axes[1, 0].barh(range(len(class_counts.head(10))),
#                        class_counts.head(10).values)
#         axes[1, 0].set_yticks(range(len(class_counts.head(10))))
#         axes[1, 0].set_yticklabels(class_counts.head(10).index)
#         axes[1, 0].set_xlabel('Count')
#         axes[1, 0].set_title('Top 10 Error Classes in Sample')

#     # Plot 4: Agreement rate by tier (if manual_binary exists)
#     if 'manual_binary' in df.columns:
#         agreement_rates = df.groupby('tier')['manual_binary'].mean()
#         axes[1, 1].bar(range(len(agreement_rates)), agreement_rates.values)
#         axes[1, 1].set_xticks(range(len(agreement_rates)))
#         axes[1, 1].set_xticklabels(agreement_rates.index)
#         axes[1, 1].set_ylabel('Agreement Rate')
#         axes[1, 1].set_title('Manual Agreement Rate by Tier')

#     plt.tight_layout()
#     plt.savefig(save_path, dpi=300, bbox_inches='tight')
#     print(f"✅ Saved plot: {save_path}")

# # =============================================================================
# # STEP 7: SAVE VALIDATION METRICS
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 7: SAVE VALIDATION METRICS")
# print("=" * 80)

# # Create initial validation metrics
# validation_metrics = {
#     'sampling_config': {
#         'sampling_rate': SAMPLING_RATE,
#         'max_per_class': MAX_PER_CLASS,
#         'min_per_class': MIN_PER_CLASS,
#         'random_seed': RANDOM_SEED
#     },
#     'sample_counts': {
#         'total_samples': len(df_samples),
#         'clean_a_samples': len(df_sample_A),
#         'clean_b_samples': len(df_sample_B),
#         'unique_classes': df_samples['Sub-Subtype'].nunique() if 'Sub-Subtype' in df_samples.columns else 0
#     },
#     'thresholds': {
#         'clean_a_threshold': 0.65,
#         'clean_b_threshold': 0.55
#     },
#     'analysis_plan': {
#         'metrics_to_calculate': [
#             'correlation_csr_manual',
#             'clean_a_vs_b_t_test',
#             'classification_accuracy',
#             'precision_recall_f1'
#         ],
#         'required_manual_columns': [
#             'manual_anchor_validity',
#             'manual_phenomenon_clarity',
#             'manual_arabic_naturalness',
#             'manual_collateral_distortion',
#             'manual_overall_agreement'
#         ]
#     }
# }

# with open(VALIDATION_JSON, 'w') as f:
#     json.dump(validation_metrics, f, indent=2)
# print(f"✅ Saved validation metrics: {VALIDATION_JSON}")

# # =============================================================================
# # STEP 8: CREATE POST-EVALUATION ANALYSIS CELL (FOR LATER USE)
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 8: CREATE POST-EVALUATION ANALYSIS SCRIPT")
# print("=" * 80)

# post_eval_script = """
# # =============================================================================
# # CELL 15.6: POST-MANUAL EVALUATION ANALYSIS
# # =============================================================================
# '''
# Run this after manual evaluation is complete to analyze results.
# '''

# import pandas as pd
# import numpy as np
# import json
# import matplotlib.pyplot as plt
# import seaborn as sns
# from scipy import stats
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# # Load manually evaluated data
# MANUAL_RESULTS = "tier1_v3_3_final/manual_evaluation_samples_FILLED.csv"
# ANALYSIS_OUTPUT = "tier1_v3_3_final/final_quality_analysis.json"

# # Load data
# df = pd.read_csv(MANUAL_RESULTS)

# # Remove instruction row
# df = df[df['Sentence_ID'] != 'INSTRUCTIONS']

# # Convert manual scores to numeric
# score_cols = ['manual_anchor_validity', 'manual_phenomenon_clarity',
#               'manual_arabic_naturalness', 'manual_collateral_distortion']
# for col in score_cols:
#     df[col] = pd.to_numeric(df[col], errors='coerce')

# # Convert overall agreement to binary
# df['manual_binary'] = df['manual_overall_agreement'].str.upper().map({'Y': 1, 'N': 0, 'YES': 1, 'NO': 0})

# # Calculate average manual score
# df['manual_avg'] = df[score_cols].mean(axis=1)

# # Separate Clean_A and Clean_B
# clean_a = df[df['tier'] == 'Clean_A']
# clean_b = df[df['tier'] == 'Clean_B']

# # Statistical tests
# results = {
#     'sample_counts': {
#         'total': len(df),
#         'clean_a': len(clean_a),
#         'clean_b': len(clean_b)
#     },
#     'manual_score_summary': {
#         'clean_a_mean': float(clean_a['manual_avg'].mean()),
#         'clean_a_std': float(clean_a['manual_avg'].std()),
#         'clean_b_mean': float(clean_b['manual_avg'].mean()),
#         'clean_b_std': float(clean_b['manual_avg'].std())
#     },
#     'agreement_rates': {
#         'clean_a_agreement': float(clean_a['manual_binary'].mean()),
#         'clean_b_agreement': float(clean_b['manual_binary'].mean()),
#         'overall_agreement': float(df['manual_binary'].mean())
#     }
# }

# # T-test between Clean_A and Clean_B
# if len(clean_a) > 1 and len(clean_b) > 1:
#     t_stat, p_value = stats.ttest_ind(clean_a['manual_avg'].dropna(),
#                                       clean_b['manual_avg'].dropna())
#     results['ttest'] = {
#         't_statistic': float(t_stat),
#         'p_value': float(p_value),
#         'significant': p_value < 0.05
#     }

# # Correlation between csr and manual scores
# if 'csr' in df.columns:
#     valid_mask = df['csr'].notna() & df['manual_avg'].notna()
#     if valid_mask.sum() > 2:
#         r, p = stats.pearsonr(df.loc[valid_mask, 'csr'],
#                              df.loc[valid_mask, 'manual_avg'])
#         results['correlation'] = {
#             'pearson_r': float(r),
#             'p_value': float(p),
#             'significant': p < 0.05
#         }

# # Calculate classification metrics
# # Define "good" as manual_binary = 1
# if 'manual_binary' in df.columns and 'tier' in df.columns:
#     # For Clean_A: precision = TP/(TP+FP) where TP = correctly identified as good
#     # We consider all Clean_A should be good
#     clean_a_precision = clean_a['manual_binary'].mean()

#     # For Clean_B: we expect some to be good, some not
#     # We can't calculate precision without ground truth, but we can report agreement rate

#     results['classification'] = {
#         'clean_a_precision': float(clean_a_precision),
#         'clean_b_agreement_rate': float(clean_b['manual_binary'].mean())
#     }

# # Save results
# with open(ANALYSIS_OUTPUT, 'w') as f:
#     json.dump(results, f, indent=2)

# print("✅ Analysis complete!")
# print(f"Clean_A agreement rate: {results['agreement_rates']['clean_a_agreement']:.2%}")
# print(f"Clean_B agreement rate: {results['agreement_rates']['clean_b_agreement']:.2%}")
# print(f"T-test p-value: {results.get('ttest', {}).get('p_value', 'N/A')}")
# """

# # Save the post-evaluation script
# post_eval_path = f"{OUTPUT_DIR}/cell15_6_post_evaluation_analysis.py"
# with open(post_eval_path, 'w') as f:
#     f.write(post_eval_script)

# print(f"✅ Saved post-evaluation analysis script: {post_eval_path}")

# # =============================================================================
# # SUMMARY
# # =============================================================================

# print("\n" + "=" * 80)
# print("🎉 CELL 15.5 COMPLETE")
# print("=" * 80)

# print(f"""
# 📁 OUTPUT FILES:
#    - {SAMPLES_CSV} (Evaluation sheet for annotators)
#    - {ANALYSIS_CSV} (Analysis template)
#    - {VALIDATION_JSON} (Validation metrics)
#    - {post_eval_path} (Post-evaluation analysis script)

# 📊 SAMPLING SUMMARY:
#    Clean_A samples: {len(df_sample_A):,} of {len(df_clean_A):,} total
#    Clean_B samples: {len(df_sample_B):,} of {len(df_clean_B):,} total
#    Total samples: {len(df_samples):,}
#    Unique error classes: {df_samples['Sub-Subtype'].nunique() if not df_samples.empty else 0}

# 📋 NEXT STEPS:
#    1. Distribute {SAMPLES_CSV} to annotators
#    2. Annotators fill in manual evaluation columns (1-5 scales + Y/N)
#    3. Save filled file as: tier1_v3_3_final/manual_evaluation_samples_FILLED.csv
#    4. Run Cell 15.6 to analyze results

# 📈 RESEARCH QUESTIONS TO ANSWER:
#    1. What is the correlation between csr and manual quality scores?
#    2. Are Clean_A samples significantly better than Clean_B?
#    3. What percentage of Clean_A samples are truly "clean"?
#    4. What percentage of Clean_B samples are acceptable for use?

# ✅ Ready for manual evaluation!
# """)

# # Show sample of the evaluation sheet
# if not eval_df.empty:
#     print("\n📋 SAMPLE OF EVALUATION SHEET (first 3 rows after instructions):")
#     print(eval_df.head(4).to_string())

In [ ]:
# import os

# # Delete the old checkpoint with failed records
# checkpoint_path = "tier1_v3_3_final/cell15_checkpoint.jsonl"
# if os.path.exists(checkpoint_path):
#     os.remove(checkpoint_path)
#     print(f"✅ Deleted checkpoint: {checkpoint_path}")
# else:
#     print("Checkpoint not found")

# # Also delete the output files to start fresh
# for f in [
#     "tier1_v3_3_final/tier2_seed_audit_cell15.csv",
#     "tier1_v3_3_final/tier2_seed_audit_trace_cell15.csv",
#     "tier1_v3_3_final/tier2_seeds_ready_cell15.csv",
#     "tier1_v3_3_final/cell15_audit_stats.json"
# ]:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f"✅ Deleted: {f}")

# print("\n✅ Ready to re-run Cell 15 from scratch!")

In [ ]:
# =============================================================================
# RESET: Delete few-shot bank files, then re-run Cell 16
# =============================================================================

import os

files_to_delete = [
    "tier1_v3_3_final/few_shot_bank_tier2.csv",
    "tier1_v3_3_final/few_shot_bank_tier2.json",
    "tier1_v3_3_final/few_shot_bank_coverage_report.csv",
]

print("=" * 60)
print("RESETTING FEW-SHOT BANK")
print("=" * 60)

for f in files_to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"✅ Deleted: {f}")
    else:
        print(f"⚠️ Not found: {f}")

print("\n✅ Now re-run Cell 16 to rebuild clean few-shot bank")
print("✅ Then run the STRICT addition script")

In [ ]:
# =============================================================================
# CELL 16: BUILD FEW-SHOT BANK (ENHANCED v2.0)
# =============================================================================
"""
CELL 16 v2.0: Build Few-Shot Bank for Tier2 Generation (Enhanced)

ENHANCEMENTS:
1. Fixed naming inconsistency: Handles both "Name Entity Error" and "Named Entity Error"
2. Adds name normalization for all common variants
3. Includes MANUAL FALLBACK examples for classes with <3 examples
4. Better coverage reporting and diagnostics

PRIORITY ORDER (per class):
1. Tier0 (gold/canonical examples) — FIRST PRIORITY
2. Tier1 Clean_A from ESV CPS-tier output (or ETCA-validated subset
   if tier2_seeds_ready_cell15.csv is present in the run directory) — FALLBACK
3. MANUAL_FALLBACK examples (Name Entity Error, Spelling Error,
   Noun-to-Adjective, Adjective-to-Noun) — LAST RESORT

OUTPUTS:
- tier1_v3_3_final/few_shot_bank_tier2.csv
- tier1_v3_3_final/few_shot_bank_tier2.json
- tier1_v3_3_final/few_shot_bank_coverage_report.csv
"""

import os
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

print("=" * 80)
print("CELL 16: BUILD FEW-SHOT BANK (ENHANCED v2.0)")
print("=" * 80)
print("✅ Name normalization for all variants")
print("✅ Manual fallback examples for missing classes")

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# Inputs
TIER0_PATH = "tier0_canonical_examples.csv"
TIER1_VALIDATED_PATH = f"{OUTPUT_DIR}/tier2_seeds_ready_cell15.csv"

# Alternative Tier1 paths
TIER1_ALT_PATHS = [
    f"{OUTPUT_DIR}/tier1_clean_A.csv",
    "tier1_clean_A.csv",
    f"{OUTPUT_DIR}/tier1_clean.csv",
    "tier1_clean.csv",
]

# Outputs
OUT_CSV = f"{OUTPUT_DIR}/few_shot_bank_tier2.csv"
OUT_JSON = f"{OUTPUT_DIR}/few_shot_bank_tier2.json"
OUT_COVERAGE = f"{OUTPUT_DIR}/few_shot_bank_coverage_report.csv"

# Settings
MAX_PER_CLASS = 3

# =============================================================================
# CANONICAL TAXONOMY (23 classes) - USE THIS AS SOURCE OF TRUTH
# =============================================================================

TAXONOMY_23 = [
    "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
    "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
    "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
    "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
    "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
    "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
]

# =============================================================================
# NAME NORMALIZATION MAP (Handle all known variants)
# =============================================================================

NAME_NORMALIZATION = {
    # Named Entity variations
    "Named Entity Error": "Name Entity Error",
    "NamedEntityError": "Name Entity Error",
    "Named_Entity_Error": "Name Entity Error",
    "NER Error": "Name Entity Error",

    # Terminology variations
    "Terminological Substitution": "Terminology Substitution",
    "Term Substitution": "Terminology Substitution",

    # Definiteness variations
    "Definiteness": "Definiteness Shift",
    "Article Shift": "Definiteness Shift",

    # Register variations
    "Formality Level": "Register Mismatch",
    "Formality Shift": "Register Mismatch",
    "Formality Mismatch": "Register Mismatch",

    # Negation variations
    "Negation": "Tense Shift Under Negation",
    "Negation Tense Shift": "Tense Shift Under Negation",
}

def normalize_class_name(name: str) -> str:
    """Normalize class name to canonical form."""
    if pd.isna(name):
        return ""
    name = str(name).strip()
    return NAME_NORMALIZATION.get(name, name)

# =============================================================================
# MANUAL FALLBACK EXAMPLES v2.0 (ENHANCED)
# =============================================================================
# High-quality examples with clear linguistic distinctions between error types
# Each example includes a linguistic note explaining why it's valid

MANUAL_FALLBACK_EXAMPLES = {

    # =========================================================================
    # NAME ENTITY ERROR - TRUE NER ERRORS (not spelling)
    # Definition: Incorrect, distorted, or substituted rendering of proper names
    # KEY: Organization substitution, transliteration variants, name distortion
    # =========================================================================

    "Name Entity Error": [
        {
            # Organization substitution: UNICEF → UNESCO (semantic error)
            "ar": "أكد التقرير أن اليونيسف تواصل تقديم المساعدات للأطفال في مناطق النزاع.",
            "mt_output": "أكد التقرير أن اليونسكو تواصل تقديم المساعدات للأطفال في مناطق النزاع.",
            "Keyword": "اليونيسف",
            "Best_Match": "اليونسكو",
            "en": "The report confirmed that UNICEF continues to provide assistance to children in conflict zones.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Organization substitution: UNICEF→UNESCO (wrong UN body)",
        },
        {
            # Transliteration variant: Kosovo rendering
            "ar": "دعا مجلس الأمن جميع الأطراف في كوسوفو إلى الالتزام بالقرارات الدولية.",
            "mt_output": "دعا مجلس الأمن جميع الأطراف في كوصوفو إلى الالتزام بالقرارات الدولية.",
            "Keyword": "كوسوفو",
            "Best_Match": "كوصوفو",
            "en": "The Security Council called on all parties in Kosovo to comply with international resolutions.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Transliteration variant: Standard vs non-standard rendering",
        },
        {
            # Treaty name distortion: Minsk Agreement
            "ar": "أشار التقرير إلى أهمية تنفيذ اتفاق مينسك لتحقيق السلام في المنطقة.",
            "mt_output": "أشار التقرير إلى أهمية تنفيذ اتفاق منسك لتحقيق السلام في المنطقة.",
            "Keyword": "مينسك",
            "Best_Match": "منسك",
            "en": "The report noted the importance of implementing the Minsk Agreement to achieve peace in the region.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Place name distortion in treaty name",
        },
    ],

    # =========================================================================
    # SPELLING ERROR - ORTHOGRAPHIC ERRORS IN COMMON WORDS (not proper nouns)
    # Definition: Incorrect spelling of Arabic common nouns/verbs
    # KEY: Distinguishes from NER by targeting common vocabulary
    # =========================================================================

    "Spelling Error": [
        {
            # Missing letter in common noun
            "ar": "أكد الأمين العام أن المفاوضات الجارية ضرورية لتحقيق السلام الدائم.",
            "mt_output": "أكد الأمين العام أن المفوضات الجارية ضرورية لتحقيق السلام الدائم.",
            "Keyword": "المفاوضات",
            "Best_Match": "المفوضات",
            "en": "The Secretary-General affirmed that the ongoing negotiations are essential for achieving lasting peace.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Missing alef in common noun (not proper name)",
        },
        {
            # Ta marbuta → Ha (common MSA error)
            "ar": "طالب المجلس الحكومة باتخاذ إجراءات فورية لحماية المدنيين.",
            "mt_output": "طالب المجلس الحكومه باتخاذ إجراءات فورية لحماية المدنيين.",
            "Keyword": "الحكومة",
            "Best_Match": "الحكومه",
            "en": "The Council demanded that the government take immediate measures to protect civilians.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Ta marbuta error: ة → ه (common in informal MT)",
        },
    ],

    # =========================================================================
    # NOUN TO ADJECTIVE
    # Definition: A noun is incorrectly converted to an adjective form
    # =========================================================================

    "Noun to Adjective": [
        {
            # Abstract noun → Adjective
            "ar": "أعرب المجلس عن قلقه البالغ إزاء تصاعد العنف في المنطقة.",
            "mt_output": "أعرب المجلس عن قلقه البالغ إزاء تصاعد العنيف في المنطقة.",
            "Keyword": "العنف",
            "Best_Match": "العنيف",
            "en": "The Council expressed its deep concern over the escalation of violence in the region.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Noun→Adjective: العنف (violence) → العنيف (violent)",
        },
        {
            # Concrete noun → Adjective
            "ar": "شدد التقرير على ضرورة تعزيز الأمن في المناطق الحدودية.",
            "mt_output": "شدد التقرير على ضرورة تعزيز الآمن في المناطق الحدودية.",
            "Keyword": "الأمن",
            "Best_Match": "الآمن",
            "en": "The report emphasized the need to strengthen security in border areas.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Noun→Adjective: الأمن (security) → الآمن (safe)",
        },
    ],

    # =========================================================================
    # ADJECTIVE TO NOUN
    # Definition: An adjective is incorrectly converted to a noun form
    # =========================================================================

    "Adjective to Noun": [
        {
            # Attributive adjective → Abstract noun
            "ar": "أكد المجلس أهمية الحوار السلمي بين جميع الأطراف المعنية.",
            "mt_output": "أكد المجلس أهمية الحوار السلام بين جميع الأطراف المعنية.",
            "Keyword": "السلمي",
            "Best_Match": "السلام",
            "en": "The Council affirmed the importance of peaceful dialogue among all concerned parties.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Adjective→Noun: السلمي (peaceful) → السلام (peace)",
        },
        {
            # Descriptive adjective → Abstract noun
            "ar": "دعا الأمين العام إلى تكثيف العمل الإنساني في المناطق المتضررة.",
            "mt_output": "دعا الأمين العام إلى تكثيف العمل الإنسانية في المناطق المتضررة.",
            "Keyword": "الإنساني",
            "Best_Match": "الإنسانية",
            "en": "The Secretary-General called for intensifying humanitarian work in affected areas.",
            "seed_source": "MANUAL_FALLBACK",
            "linguistic_note": "Adjective→Noun: الإنساني (humanitarian) → الإنسانية (humanity)",
        },
    ],
}

# =============================================================================
# DELETE OLD FILES
# =============================================================================

files_to_delete = [OUT_CSV, OUT_JSON, OUT_COVERAGE]
for f in files_to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"✅ Deleted: {f}")

# =============================================================================
# STEP 1: LOAD TIER0 (GOLD EXAMPLES)
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD TIER0 (GOLD EXAMPLES)")
print("=" * 80)

tier0 = None
tier0_col = None

if Path(TIER0_PATH).exists():
    tier0 = pd.read_csv(TIER0_PATH)
    print(f"✅ Loaded Tier0: {len(tier0):,} rows")

    # Find error type column
    for col in ['Sub-Subtype', 'error_type', 'Error_Type', 'SubSubtype', 'error_label']:
        if col in tier0.columns:
            tier0_col = col
            break

    if tier0_col:
        # Apply name normalization
        tier0[tier0_col] = tier0[tier0_col].apply(normalize_class_name)
        print(f"✅ Error type column: {tier0_col}")
        print(f"✅ Classes in Tier0 (after normalization): {tier0[tier0_col].nunique()}")

        # Show class distribution
        print(f"\n   Tier0 class counts:")
        for cls in TAXONOMY_23:
            count = len(tier0[tier0[tier0_col] == cls])
            if count > 0:
                print(f"      {cls}: {count}")
    else:
        print(f"⚠️ Could not find error type column in Tier0")
        print(f"   Columns: {list(tier0.columns)}")
else:
    print(f"⚠️ Tier0 not found: {TIER0_PATH}")

# =============================================================================
# STEP 2: LOAD TIER1 (VALIDATED OR CLEAN)
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: LOAD TIER1 DATA")
print("=" * 80)

tier1 = None
tier1_col = None

# Try primary path first
if Path(TIER1_VALIDATED_PATH).exists():
    tier1 = pd.read_csv(TIER1_VALIDATED_PATH)
    print(f"✅ Loaded ETCA-validated Tier1: {len(tier1):,} rows")
else:
    # Try alternative paths
    for alt_path in TIER1_ALT_PATHS:
        if Path(alt_path).exists():
            tier1 = pd.read_csv(alt_path)
            print(f"✅ Loaded Tier1 (fallback): {alt_path} ({len(tier1):,} rows)")
            break

if tier1 is not None:
    # Find error type column
    for col in ['error_label', 'Sub-Subtype', 'Sub_Subtype', 'error_type']:
        if col in tier1.columns:
            tier1_col = col
            break

    if tier1_col:
        # Apply name normalization
        tier1[tier1_col] = tier1[tier1_col].apply(normalize_class_name)
        print(f"✅ Error type column: {tier1_col}")
        print(f"✅ Classes in Tier1 (after normalization): {tier1[tier1_col].nunique()}")
else:
    print(f"⚠️ No Tier1 data found")

# Check we have at least one source
if tier0 is None and tier1 is None:
    raise FileNotFoundError("Neither Tier0 nor Tier1 found! Cannot build few-shot bank.")

# =============================================================================
# STEP 3: BUILD FEW-SHOT BANK
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: BUILD FEW-SHOT BANK")
print("=" * 80)
print(f"Target: {MAX_PER_CLASS} examples per class")
print(f"Priority: Tier0 → Tier1 → Manual Fallback")

few_shot_rows = []
coverage_report = []

for error_type in tqdm(TAXONOMY_23, desc="Building few-shot bank"):
    selected = []
    source_breakdown = {'tier0': 0, 'tier1': 0, 'manual': 0}

    # --------------------------------------------------
    # PRIORITY 1: Tier0 (gold canonical examples)
    # --------------------------------------------------
    if tier0 is not None and tier0_col is not None:
        tier0_candidates = tier0[tier0[tier0_col] == error_type]

        if len(tier0_candidates) > 0:
            n_take = min(MAX_PER_CLASS, len(tier0_candidates))

            for _, row in tier0_candidates.head(n_take).iterrows():
                row_dict = row.to_dict()

                normalized = {
                    'error_type': error_type,
                    'clean_ar': row_dict.get('ar', row_dict.get('clean_ar', '')),
                    'error_ar': row_dict.get('mt_output', row_dict.get('error_ar', '')),
                    'keyword': row_dict.get('Keyword', row_dict.get('keyword', '')),
                    'best_match': row_dict.get('Best_Match', row_dict.get('best_match', '')),
                    'Sentence_ID': row_dict.get('Sentence_ID', ''),
                    'MT Tool': row_dict.get('MT Tool', ''),
                    'source_tier': 'TIER0_CANONICAL',
                    'confidence_tier': 'HIGH',
                    'validators_pass_count': 3,
                }

                selected.append(normalized)
                source_breakdown['tier0'] += 1

    # --------------------------------------------------
    # PRIORITY 2: Tier1 (validated or clean)
    # --------------------------------------------------
    if len(selected) < MAX_PER_CLASS and tier1 is not None and tier1_col is not None:
        needed = MAX_PER_CLASS - len(selected)
        tier1_candidates = tier1[tier1[tier1_col] == error_type]

        if len(tier1_candidates) > 0:
            n_take = min(needed, len(tier1_candidates))

            for _, row in tier1_candidates.head(n_take).iterrows():
                row_dict = row.to_dict()

                normalized = {
                    'error_type': error_type,
                    'clean_ar': row_dict.get('ar', row_dict.get('clean_ar', '')),
                    'error_ar': row_dict.get('mt_output', row_dict.get('error_ar', '')),
                    'keyword': row_dict.get('Keyword', row_dict.get('keyword', '')),
                    'best_match': row_dict.get('Best_Match', row_dict.get('best_match', '')),
                    'Sentence_ID': row_dict.get('Sentence_ID', ''),
                    'MT Tool': row_dict.get('MT Tool', row_dict.get('MT_Tool', '')),
                    'source_tier': 'TIER1_VALIDATED',
                    'confidence_tier': 'MEDIUM',
                    'validators_pass_count': row_dict.get('validators_pass_count', 2),
                }

                selected.append(normalized)
                source_breakdown['tier1'] += 1

    # --------------------------------------------------
    # PRIORITY 3: Manual Fallback (if still empty or < 3)
    # --------------------------------------------------
    if len(selected) < MAX_PER_CLASS and error_type in MANUAL_FALLBACK_EXAMPLES:
        needed = MAX_PER_CLASS - len(selected)
        fallback_examples = MANUAL_FALLBACK_EXAMPLES[error_type]

        for ex in fallback_examples[:needed]:
            normalized = {
                'error_type': error_type,
                'clean_ar': ex.get('ar', ''),
                'error_ar': ex.get('mt_output', ''),
                'keyword': ex.get('Keyword', ''),
                'best_match': ex.get('Best_Match', ''),
                'Sentence_ID': f"MANUAL_{error_type.replace(' ', '_')}_{len(selected)+1}",
                'MT Tool': 'MANUAL',
                'source_tier': 'MANUAL_FALLBACK',
                'confidence_tier': 'HIGH',
                'validators_pass_count': 3,
            }

            selected.append(normalized)
            source_breakdown['manual'] += 1

    # --------------------------------------------------
    # Add to few-shot bank
    # --------------------------------------------------
    few_shot_rows.extend(selected)

    # Coverage report
    status = '✅ FULL' if len(selected) >= MAX_PER_CLASS else ('⚠️ PARTIAL' if len(selected) > 0 else '❌ EMPTY')
    coverage_report.append({
        'error_type': error_type,
        'total_examples': len(selected),
        'from_tier0': source_breakdown['tier0'],
        'from_tier1': source_breakdown['tier1'],
        'from_manual': source_breakdown['manual'],
        'gap': MAX_PER_CLASS - len(selected),
        'status': status,
    })

# Create DataFrames
few_shot_df = pd.DataFrame(few_shot_rows)
coverage_df = pd.DataFrame(coverage_report)

# =============================================================================
# STEP 4: SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: SAVE OUTPUTS")
print("=" * 80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save CSV
few_shot_df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUT_CSV} ({len(few_shot_df):,} rows)")

# Save JSON (grouped by error type)
few_shot_json = {}
for error_type in TAXONOMY_23:
    et_rows = few_shot_df[few_shot_df['error_type'] == error_type]
    if len(et_rows) > 0:
        few_shot_json[error_type] = et_rows.to_dict('records')
    else:
        few_shot_json[error_type] = []

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(few_shot_json, f, ensure_ascii=False, indent=2)
print(f"✅ Saved: {OUT_JSON}")

# Save coverage report
coverage_df.to_csv(OUT_COVERAGE, index=False)
print(f"✅ Saved: {OUT_COVERAGE}")

# =============================================================================
# STEP 5: SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: FEW-SHOT BANK SUMMARY")
print("=" * 80)

total_examples = len(few_shot_df)
from_tier0 = coverage_df['from_tier0'].sum()
from_tier1 = coverage_df['from_tier1'].sum()
from_manual = coverage_df['from_manual'].sum()
classes_full = len(coverage_df[coverage_df['gap'] == 0])
classes_partial = len(coverage_df[(coverage_df['gap'] > 0) & (coverage_df['total_examples'] > 0)])
classes_empty = len(coverage_df[coverage_df['total_examples'] == 0])

print(f"""
📊 FEW-SHOT BANK STATISTICS:
   Total examples:        {total_examples}
   From Tier0 (gold):     {from_tier0} ({from_tier0/total_examples*100:.1f}%)
   From Tier1 (validated):{from_tier1} ({from_tier1/total_examples*100:.1f}%)
   From Manual fallback:  {from_manual} ({from_manual/total_examples*100:.1f}%)

📊 CLASS COVERAGE:
   Full (3/3):            {classes_full} classes
   Partial (1-2/3):       {classes_partial} classes
   Empty (0/3):           {classes_empty} classes

📊 TARGET: {MAX_PER_CLASS} per class × {len(TAXONOMY_23)} classes = {MAX_PER_CLASS * len(TAXONOMY_23)} max
   Achieved:              {total_examples} ({total_examples/(MAX_PER_CLASS * len(TAXONOMY_23))*100:.1f}%)
""")

# Show per-class breakdown
print(f"\n📋 PER-CLASS BREAKDOWN:")
print(f"{'Class':<35} {'Total':<7} {'T0':<5} {'T1':<5} {'Man':<5} {'Status'}")
print("-" * 75)

for _, row in coverage_df.iterrows():
    print(f"{row['error_type']:<35} {row['total_examples']:<7} {row['from_tier0']:<5} {row['from_tier1']:<5} {row['from_manual']:<5} {row['status']}")

# Warnings for empty classes
empty_classes = coverage_df[coverage_df['total_examples'] == 0]
if len(empty_classes) > 0:
    print(f"\n❌ EMPTY CLASSES ({len(empty_classes)}):")
    for _, row in empty_classes.iterrows():
        print(f"   - {row['error_type']}: NO EXAMPLES AVAILABLE")
        print(f"     → Add manual examples to MANUAL_FALLBACK_EXAMPLES")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 16 COMPLETE")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   - {OUT_CSV} ({len(few_shot_df):,} examples)
   - {OUT_JSON} (grouped by class)
   - {OUT_COVERAGE} (coverage report)

📊 FINAL STATUS:
   Classes with examples: {23 - classes_empty}/23
   {'✅ ALL 23 CLASSES COVERED' if classes_empty == 0 else f'⚠️ {classes_empty} CLASSES NEED MANUAL EXAMPLES'}

📋 NEXT STEP:
   Run Cell 17A to generate Tier2 samples using this few-shot bank
""")

In [ ]:
# # =============================================================================
# # CELL: SUPPLEMENT CLEAN_A WITH CLEAN_B (For Manual Evaluation)
# # =============================================================================
# """
# PURPOSE:
# Extract samples from Clean_B to supplement classes that are:
# 1. Missing entirely from Clean_A
# 2. Have fewer than 3 samples in Clean_A

# This creates a small manual evaluation set to assess Clean_B quality
# before deciding whether to include these samples in the pipeline.

# INPUTS:
# - tier1_v3_3_final/tier1_clean_A_FROZEN.csv
# - tier1_v3_3_final/tier1_clean_B_FROZEN.csv

# OUTPUTS:
# - tier1_v3_3_final/clean_B_supplement_for_eval.csv (samples to evaluate)
# - tier1_v3_3_final/clean_B_supplement_summary.json (coverage report)
# """

# import pandas as pd
# import json
# import os
# from pathlib import Path

# print("=" * 80)
# print("SUPPLEMENT CLEAN_A WITH CLEAN_B (Manual Evaluation Set)")
# print("=" * 80)

# # =============================================================================
# # CONFIGURATION
# # =============================================================================

# OUTPUT_DIR = "tier1_v3_3_final"

# # Inputs
# CLEAN_A_PATH = f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv"
# CLEAN_B_PATH = f"{OUTPUT_DIR}/tier1_clean_B_FROZEN.csv"

# # Outputs
# OUT_SUPPLEMENT = f"{OUTPUT_DIR}/clean_B_supplement_for_eval.csv"
# OUT_SUMMARY = f"{OUTPUT_DIR}/clean_B_supplement_summary.json"

# # Settings
# MAX_PER_CLASS = 3  # Target per class
# MIN_THRESHOLD = 3  # Classes with fewer than this need supplement

# # Taxonomy (23 classes)
# TAXONOMY_23 = [
#     "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
#     "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
#     "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
#     "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
#     "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
#     "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
#     "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
# ]

# # =============================================================================
# # STEP 1: LOAD DATA
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 1: LOAD DATA")
# print("=" * 80)

# # Load Clean_A
# if not Path(CLEAN_A_PATH).exists():
#     raise FileNotFoundError(f"Clean_A not found: {CLEAN_A_PATH}")

# clean_a = pd.read_csv(CLEAN_A_PATH)
# print(f"✅ Loaded Clean_A: {len(clean_a):,} rows")

# # Load Clean_B
# if not Path(CLEAN_B_PATH).exists():
#     raise FileNotFoundError(f"Clean_B not found: {CLEAN_B_PATH}")

# clean_b = pd.read_csv(CLEAN_B_PATH)
# print(f"✅ Loaded Clean_B: {len(clean_b):,} rows")

# # =============================================================================
# # STEP 2: ANALYZE CLEAN_A COVERAGE
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 2: ANALYZE CLEAN_A COVERAGE")
# print("=" * 80)

# error_col = 'Sub-Subtype'

# # Count per class in Clean_A
# clean_a_counts = clean_a[error_col].value_counts().to_dict()

# # Identify classes needing supplement
# classes_missing = []      # 0 samples in Clean_A
# classes_low = []          # 1-2 samples in Clean_A
# classes_sufficient = []   # 3+ samples in Clean_A

# for et in TAXONOMY_23:
#     count_a = clean_a_counts.get(et, 0)

#     if count_a == 0:
#         classes_missing.append(et)
#     elif count_a < MIN_THRESHOLD:
#         classes_low.append(et)
#     else:
#         classes_sufficient.append(et)

# print(f"\n📊 CLEAN_A COVERAGE ANALYSIS:")
# print(f"   Classes with 0 samples:    {len(classes_missing)}")
# print(f"   Classes with 1-2 samples:  {len(classes_low)}")
# print(f"   Classes with 3+ samples:   {len(classes_sufficient)}")
# print(f"   Total taxonomy classes:    {len(TAXONOMY_23)}")

# if classes_missing:
#     print(f"\n❌ MISSING CLASSES ({len(classes_missing)}):")
#     for et in classes_missing:
#         print(f"   - {et}")

# if classes_low:
#     print(f"\n⚠️ LOW CLASSES ({len(classes_low)}):")
#     for et in classes_low:
#         count = clean_a_counts.get(et, 0)
#         print(f"   - {et}: {count} samples (need {MIN_THRESHOLD - count} more)")

# # =============================================================================
# # STEP 3: CHECK CLEAN_B AVAILABILITY
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 3: CHECK CLEAN_B AVAILABILITY")
# print("=" * 80)

# # Count per class in Clean_B
# clean_b_counts = clean_b[error_col].value_counts().to_dict()

# print(f"\n📊 CLEAN_B CLASS DISTRIBUTION:")
# print(f"{'Class':<35} {'Clean_A':<10} {'Clean_B':<10} {'Status'}")
# print("-" * 65)

# classes_needing_supplement = classes_missing + classes_low
# supplement_available = {}

# for et in classes_needing_supplement:
#     count_a = clean_a_counts.get(et, 0)
#     count_b = clean_b_counts.get(et, 0)
#     needed = MIN_THRESHOLD - count_a
#     can_take = min(needed, count_b, MAX_PER_CLASS)

#     supplement_available[et] = {
#         'clean_a_count': count_a,
#         'clean_b_count': count_b,
#         'needed': needed,
#         'can_take': can_take
#     }

#     if count_b > 0:
#         status = f"✅ Can take {can_take}"
#     else:
#         status = "❌ None available"

#     print(f"{et:<35} {count_a:<10} {count_b:<10} {status}")

# # =============================================================================
# # STEP 4: EXTRACT SUPPLEMENT SAMPLES
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 4: EXTRACT SUPPLEMENT SAMPLES FROM CLEAN_B")
# print("=" * 80)

# supplement_rows = []
# extraction_report = []

# for et in classes_needing_supplement:
#     info = supplement_available[et]
#     can_take = info['can_take']

#     if can_take > 0:
#         # Get samples from Clean_B for this class
#         et_samples = clean_b[clean_b[error_col] == et].head(can_take).copy()

#         # Add source marker
#         et_samples['supplement_source'] = 'CLEAN_B'
#         et_samples['supplement_reason'] = 'missing' if info['clean_a_count'] == 0 else 'low_count'

#         supplement_rows.append(et_samples)

#         extraction_report.append({
#             'error_type': et,
#             'clean_a_count': info['clean_a_count'],
#             'samples_extracted': len(et_samples),
#             'reason': 'missing' if info['clean_a_count'] == 0 else 'low_count'
#         })

#         print(f"✅ {et}: Extracted {len(et_samples)} samples")
#     else:
#         extraction_report.append({
#             'error_type': et,
#             'clean_a_count': info['clean_a_count'],
#             'samples_extracted': 0,
#             'reason': 'no_clean_b_available'
#         })
#         print(f"⚠️ {et}: No samples available in Clean_B")

# # Combine all supplement samples
# if supplement_rows:
#     supplement_df = pd.concat(supplement_rows, ignore_index=True)
# else:
#     supplement_df = pd.DataFrame()

# print(f"\n📊 TOTAL SUPPLEMENT SAMPLES: {len(supplement_df)}")

# # =============================================================================
# # STEP 5: SAVE OUTPUTS
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 5: SAVE OUTPUTS")
# print("=" * 80)

# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Save supplement CSV
# if len(supplement_df) > 0:
#     supplement_df.to_csv(OUT_SUPPLEMENT, index=False, encoding='utf-8-sig')
#     print(f"✅ Saved: {OUT_SUPPLEMENT} ({len(supplement_df)} rows)")
# else:
#     print(f"⚠️ No supplement samples to save")

# # Save summary JSON
# summary = {
#     'clean_a_total': len(clean_a),
#     'clean_b_total': len(clean_b),
#     'taxonomy_classes': len(TAXONOMY_23),
#     'classes_missing_in_a': len(classes_missing),
#     'classes_low_in_a': len(classes_low),
#     'classes_sufficient_in_a': len(classes_sufficient),
#     'supplement_samples_extracted': len(supplement_df),
#     'min_threshold': MIN_THRESHOLD,
#     'max_per_class': MAX_PER_CLASS,
#     'extraction_report': extraction_report,
#     'classes_missing': classes_missing,
#     'classes_low': classes_low,
# }

# with open(OUT_SUMMARY, 'w', encoding='utf-8') as f:
#     json.dump(summary, f, indent=2, ensure_ascii=False)
# print(f"✅ Saved: {OUT_SUMMARY}")

# # =============================================================================
# # SUMMARY
# # =============================================================================

# print("\n" + "=" * 80)
# print("🎉 SUPPLEMENT EXTRACTION COMPLETE")
# print("=" * 80)

# # Count by reason
# missing_filled = sum(1 for r in extraction_report if r['reason'] == 'missing' and r['samples_extracted'] > 0)
# low_filled = sum(1 for r in extraction_report if r['reason'] == 'low_count' and r['samples_extracted'] > 0)
# not_filled = sum(1 for r in extraction_report if r['samples_extracted'] == 0)

# print(f"""
# 📁 OUTPUT FILES:
#    - {OUT_SUPPLEMENT} ({len(supplement_df)} samples for manual evaluation)
#    - {OUT_SUMMARY}

# 📊 EXTRACTION SUMMARY:
#    Missing classes filled:       {missing_filled} / {len(classes_missing)}
#    Low-count classes filled:     {low_filled} / {len(classes_low)}
#    Classes not filled:           {not_filled}

# 📊 SAMPLES BY CLASS:
# """)

# for r in extraction_report:
#     if r['samples_extracted'] > 0:
#         print(f"   {r['error_type']}: {r['samples_extracted']} samples ({r['reason']})")

# print(f"""
# 📋 NEXT STEPS:
#    1. Open {OUT_SUPPLEMENT}
#    2. Manually evaluate each sample for:
#       - Error presence and correctness
#       - Keyword/Best_Match accuracy
#       - Arabic quality
#    3. Decide which samples to include in final dataset

# ✅ Ready for manual evaluation!
# """)

In [ ]:
# # =============================================================================
# # ADD MANUALLY EVALUATED CLEAN_B SUPPLEMENTS TO FEW-SHOT BANK
# # =============================================================================

# import pandas as pd

# # File paths
# FEW_SHOT_BANK = "tier1_v3_3_final/few_shot_bank_tier2.csv"
# SUPPLEMENT_FILE = "tier1_v3_3_final/clean_B_supplement_for_eval.csv"
# OUTPUT_FILE = "tier1_v3_3_final/few_shot_bank_tier2.csv"  # Overwrite original

# # =============================================================================
# # STEP 1: LOAD CURRENT FEW-SHOT BANK (BEFORE)
# # =============================================================================

# print("=" * 70)
# print("STEP 1: CURRENT FEW-SHOT BANK (BEFORE)")
# print("=" * 70)

# few_shot_df = pd.read_csv(FEW_SHOT_BANK)
# print(f"Total samples: {len(few_shot_df)}")
# print(f"Classes covered: {few_shot_df['Sub-Subtype'].nunique()}")

# print("\nPer-class counts (BEFORE):")
# before_counts = few_shot_df['Sub-Subtype'].value_counts()
# for cls, count in sorted(before_counts.items()):
#     print(f"   {cls}: {count}")

# # =============================================================================
# # STEP 2: LOAD SUPPLEMENT SAMPLES
# # =============================================================================

# print("\n" + "=" * 70)
# print("STEP 2: SUPPLEMENT SAMPLES TO ADD")
# print("=" * 70)

# supplement_df = pd.read_csv(SUPPLEMENT_FILE)
# print(f"Supplement samples: {len(supplement_df)}")

# print("\nSupplement breakdown:")
# for cls, count in supplement_df['Sub-Subtype'].value_counts().items():
#     print(f"   {cls}: {count}")

# # =============================================================================
# # STEP 3: NORMALIZE SUPPLEMENT COLUMNS TO MATCH FEW-SHOT BANK
# # =============================================================================

# print("\n" + "=" * 70)
# print("STEP 3: NORMALIZE AND MERGE")
# print("=" * 70)

# # Ensure supplement has required columns
# required_cols = ['ar', 'mt_output', 'Sub-Subtype', 'Keyword', 'Best_Match']

# # Add seed_source to identify these as manually evaluated Clean_B
# supplement_df['seed_source'] = 'CLEAN_B_MANUAL_EVAL'

# # Keep only columns that exist in few-shot bank + required
# few_shot_cols = few_shot_df.columns.tolist()
# supplement_cols_to_keep = [c for c in supplement_df.columns if c in few_shot_cols]
# supplement_cols_to_keep.append('seed_source') if 'seed_source' not in supplement_cols_to_keep else None

# # Align columns
# for col in few_shot_cols:
#     if col not in supplement_df.columns:
#         supplement_df[col] = ''

# # Select same columns as few-shot bank
# supplement_aligned = supplement_df[few_shot_cols].copy()

# print(f"Columns aligned: {len(few_shot_cols)}")

# # =============================================================================
# # STEP 4: MERGE
# # =============================================================================

# combined_df = pd.concat([few_shot_df, supplement_aligned], ignore_index=True)

# print(f"\nMerged: {len(few_shot_df)} + {len(supplement_df)} = {len(combined_df)}")

# # =============================================================================
# # STEP 5: SHOW AFTER STATE
# # =============================================================================

# print("\n" + "=" * 70)
# print("STEP 5: FEW-SHOT BANK (AFTER)")
# print("=" * 70)

# print(f"Total samples: {len(combined_df)}")
# print(f"Classes covered: {combined_df['Sub-Subtype'].nunique()}")

# print("\nPer-class counts (AFTER):")
# after_counts = combined_df['Sub-Subtype'].value_counts()
# for cls, count in sorted(after_counts.items()):
#     before = before_counts.get(cls, 0)
#     diff = count - before
#     diff_str = f" (+{diff})" if diff > 0 else ""
#     print(f"   {cls}: {count}{diff_str}")

# # =============================================================================
# # STEP 6: COMPARISON TABLE
# # =============================================================================

# print("\n" + "=" * 70)
# print("STEP 6: BEFORE vs AFTER COMPARISON")
# print("=" * 70)

# TAXONOMY_23 = [
#     "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
#     "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
#     "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
#     "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
#     "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
#     "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
#     "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
# ]

# print(f"\n{'Class':<35} {'Before':>8} {'After':>8} {'Change':>8}")
# print("-" * 65)

# for cls in TAXONOMY_23:
#     before = before_counts.get(cls, 0)
#     after = after_counts.get(cls, 0)
#     change = after - before
#     change_str = f"+{change}" if change > 0 else str(change) if change < 0 else "-"
#     print(f"{cls:<35} {before:>8} {after:>8} {change_str:>8}")

# print("-" * 65)
# print(f"{'TOTAL':<35} {len(few_shot_df):>8} {len(combined_df):>8} {'+' + str(len(supplement_df)):>8}")

# # =============================================================================
# # STEP 7: SAVE UPDATED FEW-SHOT BANK
# # =============================================================================

# print("\n" + "=" * 70)
# print("STEP 7: SAVE UPDATED FEW-SHOT BANK")
# print("=" * 70)

# combined_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
# print(f"✅ Saved: {OUTPUT_FILE} ({len(combined_df)} rows)")

# # Also update JSON
# import json
# few_shot_json = {}
# for cls in TAXONOMY_23:
#     cls_df = combined_df[combined_df['Sub-Subtype'] == cls]
#     few_shot_json[cls] = cls_df.to_dict('records')

# json_path = "tier1_v3_3_final/few_shot_bank_tier2.json"
# with open(json_path, 'w', encoding='utf-8') as f:
#     json.dump(few_shot_json, f, ensure_ascii=False, indent=2)
# print(f"✅ Saved: {json_path}")

# # =============================================================================
# # SUMMARY
# # =============================================================================

# print("\n" + "=" * 70)
# print("🎉 FEW-SHOT BANK UPDATED")
# print("=" * 70)

# print(f"""
# 📊 SUMMARY:
#    Before: {len(few_shot_df)} samples, {few_shot_df['Sub-Subtype'].nunique()} classes
#    After:  {len(combined_df)} samples, {combined_df['Sub-Subtype'].nunique()} classes
#    Added:  {len(supplement_df)} manually evaluated Clean_B samples

# 📋 SAMPLES ADDED:
# """)

# for _, row in supplement_df.iterrows():
#     print(f"   - {row['Sub-Subtype']}: {row.get('Sentence_ID', 'N/A')}")

# print(f"""
# ✅ Ready to re-run Cell 17 for Tier2 generation!

# ⚠️  REMINDER: Delete the Cell 17 checkpoint before re-running:
#    rm tier1_v3_3_final/tier2_generation_checkpoint.jsonl
# """)

In [ ]:
import pandas as pd
import json
from datetime import datetime

print("=" * 70)
print("CELL 16B: ADD SUPPLEMENTS TO FEW-SHOT BANK (v2.0)")
print("=" * 70)
print("Priority: Clean_A (ETCA-validated) → Approved Manual/Clean_B")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Classes that generated <40 samples in Tier2 - need 5 examples in few-shot bank
CLASSES_BELOW_40 = [
    "Tense Shift Under Negation",  # 17
    "Noun to Adjective",           # 25
    "Spelling Error",              # 27
    "Total Omission",              # 30
    "Definiteness Shift",          # 30
    "Adjective to Noun",           # 33
    "Wrong Structure",             # 33
    "Hyponym for Hypernym",        # 34
    "Invalid Pattern",             # 34
    "Register Mismatch",           # 36
]

TARGET_FOR_HARD_CLASSES = 5
TARGET_FOR_NORMAL_CLASSES = 3

# File paths
CLEAN_A_PATH = f"{OUTPUT_DIR}/tier1_clean_A.csv" # Use OUTPUT_DIR to construct path
FEW_SHOT_BANK_PATH = "tier1_v3_3_final/few_shot_bank_tier2.csv"
# CLEAN_A_PATH = "tier1_clean_A.csv"  # This line was causing the error, removed it
APPROVED_SUPPLEMENTS_PATH = "pilot_accepted_samples_FULL_19.csv"

# =============================================================================
# STEP 1: LOAD CURRENT FEW-SHOT BANK
# =============================================================================
print("\n" + "=" * 70)
print("STEP 1: LOAD CURRENT FEW-SHOT BANK")
print("=" * 70)

few_shot_df = pd.read_csv(FEW_SHOT_BANK_PATH)
print(f"✅ Loaded: {FEW_SHOT_BANK_PATH} ({len(few_shot_df)} samples)")

# Find error type column
error_col = None
for col in ['Sub-Subtype', 'error_type', 'Sub_Subtype']:
    if col in few_shot_df.columns:
        error_col = col
        break
print(f"   Error type column: {error_col}")

# Current counts per class
current_counts = few_shot_df[error_col].value_counts().to_dict()

# =============================================================================
# STEP 2: LOAD SOURCES
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2: LOAD SUPPLEMENT SOURCES")
print("=" * 70)

# Load Clean_A
clean_a_df = pd.read_csv(CLEAN_A_PATH)
print(f"✅ Loaded Clean_A: {CLEAN_A_PATH} ({len(clean_a_df)} samples)")

# Load Approved supplements
approved_df = pd.read_csv(APPROVED_SUPPLEMENTS_PATH)
if 'Decision' in approved_df.columns:
    approved_df = approved_df[approved_df['Decision'] == 'ACCEPT']
print(f"✅ Loaded Approved: {APPROVED_SUPPLEMENTS_PATH} ({len(approved_df)} samples)")

# =============================================================================
# STEP 3: ADD SUPPLEMENTS (Clean_A first, then Approved)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3: ADD SUPPLEMENTS (Target: 5 per class for <40 classes)")
print("=" * 70)
print(f"{'Class':<35} {'Has':<5} {'Need':<5} {'From A':<8} {'From Appr':<10} {'Final'}")
print("-" * 70)

samples_to_add = []
summary = []

for cls in CLASSES_BELOW_40:
    current = current_counts.get(cls, 0)
    target = TARGET_FOR_HARD_CLASSES
    needed = target - current

    if needed <= 0:
        print(f"{cls:<35} {current:<5} {0:<5} {'-':<8} {'-':<10} {current} ✅")
        continue

    from_a = 0
    from_approved = 0

    # --- PRIORITY 1: Clean_A ---
    clean_a_available = clean_a_df[clean_a_df['Sub-Subtype'] == cls]

    if len(clean_a_available) > 0 and needed > 0:
        to_take = min(needed, len(clean_a_available))
        samples = clean_a_available.head(to_take).copy()
        samples['seed_source'] = 'TIER1_CLEAN_A'
        samples_to_add.append(samples)
        from_a = to_take
        needed -= to_take

    # --- PRIORITY 2: Approved (Manual + Clean_B) ---
    if needed > 0:
        approved_available = approved_df[approved_df['Sub-Subtype'] == cls]

        if len(approved_available) > 0:
            to_take = min(needed, len(approved_available))
            samples = approved_available.head(to_take).copy()
            samples['seed_source'] = samples['Source'].apply(
                lambda x: 'MANUAL_APPROVED' if x == 'Manual' else 'TIER1_CLEAN_B_APPROVED'
            )
            samples_to_add.append(samples)
            from_approved = to_take
            needed -= to_take

    final = current + from_a + from_approved
    status = "✅" if final >= target else f"⚠️ short {target - final}"
    print(f"{cls:<35} {current:<5} {target - current:<5} {from_a:<8} {from_approved:<10} {final} {status}")

    summary.append({
        'class': cls,
        'original': current,
        'from_clean_a': from_a,
        'from_approved': from_approved,
        'final': final,
        'target': target,
    })

# =============================================================================
# STEP 4: MERGE AND SAVE
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: MERGE AND SAVE")
print("=" * 70)

if samples_to_add:
    # Combine all supplements
    supplements_combined = pd.concat(samples_to_add, ignore_index=True)

    # Standardize column names
    if 'Sub-Subtype' in supplements_combined.columns and error_col == 'error_type':
        supplements_combined = supplements_combined.rename(columns={'Sub-Subtype': 'error_type'})

    # Select columns to keep (match few-shot bank schema)
    cols_to_keep = [error_col, 'Keyword', 'Best_Match', 'ar', 'mt_output', 'en', 'seed_source']
    cols_available = [c for c in cols_to_keep if c in supplements_combined.columns]
    supplements_final = supplements_combined[cols_available].copy()

    # Add missing columns from few-shot bank schema
    for col in few_shot_df.columns:
        if col not in supplements_final.columns:
            supplements_final[col] = ''

    # Reorder to match few-shot bank
    supplements_final = supplements_final[few_shot_df.columns]

    # Merge
    updated_df = pd.concat([few_shot_df, supplements_final], ignore_index=True)

    # Count by source
    total_from_a = sum(s['from_clean_a'] for s in summary)
    total_from_approved = sum(s['from_approved'] for s in summary)

    print(f"\n   Original few-shot bank: {len(few_shot_df)} samples")
    print(f"   Added from Clean_A: {total_from_a} samples")
    print(f"   Added from Approved: {total_from_approved} samples")
    print(f"   Updated few-shot bank: {len(updated_df)} samples")

    # Save CSV
    updated_df.to_csv(FEW_SHOT_BANK_PATH, index=False, encoding='utf-8-sig')
    print(f"\n✅ Saved: {FEW_SHOT_BANK_PATH}")

    # Save JSON version
    json_path = FEW_SHOT_BANK_PATH.replace('.csv', '.json')
    grouped = {}
    for cls in updated_df[error_col].unique():
        cls_data = updated_df[updated_df[error_col] == cls]
        grouped[cls] = cls_data.to_dict('records')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(grouped, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved: {json_path}")

else:
    print("   No supplements to add.")

# =============================================================================
# STEP 5: FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: FINAL SUMMARY")
print("=" * 70)

# Reload and show final counts
final_df = pd.read_csv(FEW_SHOT_BANK_PATH)
final_counts = final_df[error_col].value_counts().to_dict()

print(f"\n📊 FINAL FEW-SHOT BANK: {len(final_df)} samples")
print(f"\n{'Class':<35} {'Count':<8} {'Target':<8} {'Status'}")
print("-" * 70)

all_good = True
for cls in sorted(final_counts.keys()):
    count = final_counts.get(cls, 0)
    target = TARGET_FOR_HARD_CLASSES if cls in CLASSES_BELOW_40 else TARGET_FOR_NORMAL_CLASSES
    status = "✅" if count >= target else f"⚠️ need {target - count} more"
    if count < target:
        all_good = False
    print(f"{cls:<35} {count:<8} {target:<8} {status}")

print("\n" + "=" * 70)
if all_good:
    print("🎉 ALL CLASSES HAVE REACHED TARGET!")
else:
    print("⚠️ Some classes still below target (may need manual examples)")
print("=" * 70)

print(f"\n📑 NEXT STEPS:")
print(f"   1. Delete checkpoint: rm tier1_v3_3_final/tier2_generation_checkpoint*.jsonl")
print(f"   2. Run Cell 17A to regenerate Tier2 with enhanced few-shot bank")

In [ ]:
# Run this in a cell BEFORE Cell 17A
import os

checkpoint_files = [
    "tier1_v3_3_final/tier2_generation_checkpoint.jsonl",
    "tier1_v3_3_final/tier2_generation_checkpoint_v2.jsonl",
]

for f in checkpoint_files:
    if os.path.exists(f):
        os.remove(f)
        print(f"✅ Deleted: {f}")
    else:
        print(f"   Not found: {f}")

print("\n🎉 Now run Cell 17A to regenerate fresh")

In [ ]:
import glob
files = glob.glob("tier1_v3_3_final/*checkpoint*")
print(files if files else "No checkpoint files found ✅")

In [ ]:
import glob, os

patterns = [
    "tier1_v3_3_final/*checkpoint*",
    "*checkpoint*.jsonl",
    "*checkpoint*.json",
]

found = []
for p in patterns:
    found.extend(glob.glob(p))

# Exclude Cell 15
found = [f for f in found if "cell15" not in f.lower()]

if found:
    for f in found:
        print(f"Found: {f}")
    confirm = input("\nDelete all? (yes/no): ")
    if confirm.lower() == "yes":
        for f in found:
            os.remove(f)
            print(f"✅ Deleted: {f}")
else:
    print("✅ All clean — no Cell 17A checkpoint files found")

In [ ]:
# =============================================================================
# CELL 17A: TIER2 LLM GENERATION (ENHANCED FOR CLASSIFIER TRAINING)
# =============================================================================
"""
CELL 17A v2.0: Enhanced LLM-Based Tier2 Generation for Model Training

PURPOSE:
Generate synthetic translation error examples with HIGH DIVERSITY for training
a robust 23-class linguistic shift classifier.

ENHANCEMENTS FOR MODEL TRAINING:
1. Keyword tracking per class - prevents model memorizing keyword→class
2. Sentence uniqueness check - prevents data leakage in train/test splits
3. Near-duplicate detection - catches similar sentences
4. Domain rotation - ensures vocabulary variety across UN subdomains
5. Sentence opening tracking - prevents structural overfitting
6. Used keywords shown in prompt - guides LLM to create diversity
7. Enhanced validation with diversity checks
8. Post-generation quality analysis

OUTPUT:
- tier2_generated_raw.csv (LLM output only, no metadata)
- tier2_generation_stats.json (includes diversity metrics)
"""

import pandas as pd
import json
import re
import os
import uuid
import hashlib
import math
from datetime import datetime, timezone # Import timezone
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from openai import OpenAI

print("=" * 80)
print("CELL 17A: TIER2 LLM GENERATION v2.0 (ENHANCED FOR TRAINING)")
print("=" * 80)
print("✅ Diversity enforcement for classifier training")
print("✅ Keyword tracking, duplicate prevention, domain rotation")

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"
FEW_SHOT_BANK_PATH = f"{OUTPUT_DIR}/few_shot_bank_tier2.csv"
CHECKPOINT_PATH = f"{OUTPUT_DIR}/tier2_generation_checkpoint_v2.jsonl"

# Output files
OUTPUT_RAW_CSV = f"{OUTPUT_DIR}/tier2_generated_raw.csv"
OUTPUT_RAW_JSON = f"{OUTPUT_DIR}/tier2_generated_raw.json"
OUTPUT_STATS = f"{OUTPUT_DIR}/tier2_generation_stats.json"
OUTPUT_QUALITY_REPORT = f"{OUTPUT_DIR}/tier2_quality_report.json"

# Generation settings
SAMPLES_PER_CLASS = 369  # <-- set this (e.g., 50/200/300/600)

# -------------------------------------------------------------------------
# CONTROLLED KEYWORD DUPLICATION (for classes with small keyword banks)
# - Duplicates are allowed ONLY for keywords (not sentence templates).
# - Sentence-level near-duplicate filtering remains STRICT.
# -------------------------------------------------------------------------
ALLOWED_DUPLICATION_RATES = {
    # 0.30 — very constrained (pilot ceilings / small bank / morphology)
    "Tense Shift Under Negation": 0.30,
    "Total Omission": 0.30,
    "Definiteness Shift": 0.30,
    "Spelling Error": 0.30,

    # 0.25 — constrained (structure/agreement/semantic control)
    "Noun to Adjective": 0.25,
    "Adjective to Noun": 0.25,
    "Wrong Structure": 0.25,
    "Invalid Pattern": 0.25,
    "Tanween Omission": 0.25,
    "Gender Disagreement": 0.25,
    "Meaning Shift": 0.25,

    # 0.15 — default (easier / larger space)
    "Default": 0.15
}

# Global cap enforcing the "15/25/30" policy
MAX_DUPLICATION_RATE_CAP = 0.30

# Optional rationale for documentation/export (paper-friendly)
DUP_CAP_RATIONALE = {
    "Tense Shift Under Negation": "Pilot ceiling (17/50); morpho-syntactic constraint",
    "Total Omission": "Pilot ceiling (30/50); limited plausible omission contexts",
    "Definiteness Shift": "Pilot ceiling (30/50); constrained by definite article patterns",
    "Spelling Error": "Pilot ceiling (27/50); limited realistic typo patterns",
    "Noun to Adjective": "Pilot ceiling (25/50); structure-sensitive transformation",
    "Adjective to Noun": "Structure-sensitive transformation",
    "Wrong Structure": "Structure-sensitive; high validation rejection risk",
    "Invalid Pattern": "High schema/constraint rejection risk",
    "Tanween Omission": "Morphological constraint; limited surface realizations",
    "Gender Disagreement": "Agreement constraint; limited valid disagreement contexts",
    "Meaning Shift": "Hard to control without drift; allow modest keyword reuse",
    "Default": "Default cap for broader classes"
}

# Export caps table for documentation (reviewer-friendly)
CAPS_TABLE_CSV = os.path.join(OUTPUT_DIR, "keyword_duplication_caps_table.csv")
caps_rows = []
for cls in TAXONOMY_23:
    cap = ALLOWED_DUPLICATION_RATES.get(cls, ALLOWED_DUPLICATION_RATES["Default"])
    cap = min(cap, MAX_DUPLICATION_RATE_CAP)
    caps_rows.append({
        "Sub-Subtype": cls,
        "keyword_dup_cap": cap,
        "rationale": DUP_CAP_RATIONALE.get(cls, DUP_CAP_RATIONALE["Default"])
    })
pd.DataFrame(caps_rows).to_csv(CAPS_TABLE_CSV, index=False, encoding="utf-8")

# Hard cap: maximum times a single keyword may repeat within a class.
# (Prevents one keyword from dominating even if duplication rate allows it.)
def max_keyword_repeats_cap(target_per_class: int) -> int:
    return max(3, int(round(0.08 * target_per_class)))  # e.g., 300 -> 24, 600 -> 48
     # Adjust as needed (10 for pilot, 50+ for training)
MAX_RETRIES = 5             # Increased for diversity validation failures
MODEL = "gpt-4o"  # GPT-4o for cost-effective high-quality generation

# Temperature map (slightly higher for more diversity)
TEMPERATURE_MAP = {
    "Meaning Shift": 0.8,
    "Total Omission": 0.6,
    "Partial Translation": 0.7,
    "Name Entity Error": 0.9,
    "Terminology Substitution": 0.8,
    "Literal Translation": 0.9,
    "Hypernym for Hyponym": 0.7,
    "Hyponym for Hypernym": 0.7, # Corrected
    "Tanween Omission": 0.6,
    "Gender Disagreement": 0.7,
    "Definiteness Shift": 0.7,
    "Perfective to Progressive": 0.7,
    "Progressive to Perfective": 0.7,
    "Tense Shift Under Negation": 0.7,
    "Wrong Structure": 0.7,
    "Wrong Word Order": 0.7,
    "Active to Passive Voice": 0.7,
    "Passive to Active Voice": 0.7,
    "Noun to Adjective": 0.7,
    "Adjective to Noun": 0.7,
    "Register Mismatch": 0.8,
    "Spelling Error": 0.6,
    "Invalid Pattern": 0.7,
}
DEFAULT_TEMPERATURE = 0.7

# Taxonomy (23 classes)
TAXONOMY_23 = [
    "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
    "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
    "Hyponym for Hypernym", # Corrected
    "Tanween Omission", "Gender Disagreement",
    "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
    "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
    "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
]

# Error definitions
ARABIC_ERROR_DEFINITIONS = {
    "Meaning Shift": "The translation conveys an incorrect or distorted meaning compared to the source.",
    "Total Omission": "A source text element is completely missing from the translation.",
    "Partial Translation": "Part of a word or phrase is translated while another part is omitted or left untranslated.",
    "Name Entity Error": "Incorrect, distorted, or substituted rendering of a proper name (person, place, organization, treaty).",
    "Terminology Substitution": "A domain-specific term is replaced with an incorrect or less precise term.",
    "Literal Translation": "Word-for-word translation that sounds unnatural or changes the intended meaning.",
    "Hypernym for Hyponym": "A specific term is replaced with a more general term (e.g., 'treaty' → 'agreement').",
    "Hyponym for Hypernym": "A general term is replaced with a more specific term (e.g., 'vehicle' → 'truck').",
    "Tanween Omission": "Arabic nunation (tanween) is incorrectly omitted.",
    "Gender Disagreement": "Mismatch in grammatical gender between related words.",
    "Definiteness Shift": "Incorrect use of the definite article 'ال' (added or removed incorrectly).",
    "Perfective to Progressive": "A completed action (perfective) is incorrectly rendered as ongoing (progressive).",
    "Progressive to Perfective": "An ongoing action (progressive) is incorrectly rendered as completed (perfective).",
    "Tense Shift Under Negation": "Tense changes incorrectly when negation is applied.",
    "Wrong Structure": "Incorrect grammatical structure or word form.",
    "Wrong Word Order": "Words are arranged in an incorrect or unnatural order.",
    "Active to Passive Voice": "Active voice is incorrectly changed to passive voice.",
    "Passive to Active Voice": "Passive voice is incorrectly changed to active voice.",
    "Noun to Adjective": "A noun is incorrectly converted to an adjective form.",
    "Adjective to Noun": "An adjective is incorrectly converted to a noun form.",
    "Register Mismatch": "Inappropriate level of formality for the context.",
    "Spelling Error": "Incorrect spelling of an Arabic word.",
    "Invalid Pattern": "Use of an incorrect Arabic morphological pattern.",
}

# =============================================================================
# DIVERSITY ENFORCEMENT: UN DOMAIN CATEGORIES
# =============================================================================

UN_DOMAIN_KEYWORDS = {
    "political": [
        "السلام", "الأمن", "الاستقرار", "الانتخابات", "السيادة", "الحكومة",
        "الدولة", "النزاع", "الصراع", "المصالحة", "الديمقراطية", "الحوكمة"
    ],
    "humanitarian": [
        "اللاجئين", "المساعدات", "الحماية", "حقوق الإنسان", "الإغاثة",
        "النازحين", "المدنيين", "الضحايا", "الأزمة", "الطوارئ"
    ],
    "legal": [
        "القانون", "المعاهدة", "الاتفاقية", "القرار", "الولاية", "المادة",
        "الميثاق", "البروتوكول", "التصديق", "الامتثال", "الإلزام"
    ],
    "diplomatic": [
        "المفاوضات", "العلاقات", "التعاون", "الحوار", "الوساطة",
        "السفير", "البعثة", "الممثل", "المندوب", "المشاورات"
    ],
    "security": [
        "القوات", "العسكرية", "الأسلحة", "الإرهاب", "العقوبات",
        "الحظر", "المراقبة", "التفتيش", "نزع السلاح", "حفظ السلام"
    ]
}

# Sentence opening verbs for variety
SENTENCE_OPENINGS = [
    "وأكد", "وشدد", "أعرب", "طالب", "يرحب", "قرر", "دعا", "أشار",
    "ناشد", "حث", "رحب", "أعلن", "أوصى", "لاحظ", "اعتمد", "أيد",
    "وأعاد", "واستذكر", "وإذ يؤكد", "تؤكد اللجنة", "يؤكد المجلس",
    "يدعو الأمين العام", "تحث الجمعية", "يطلب إلى", "يقرر أن"
]

print(f"✅ Loaded taxonomy: {len(TAXONOMY_23)} classes")
print(f"✅ Domain categories: {len(UN_DOMAIN_KEYWORDS)}")
print(f"✅ Sentence openings: {len(SENTENCE_OPENINGS)}")

# =============================================================================
# DIVERSITY TRACKING (Global state for entire generation run)
# =============================================================================

# Track used keywords per class (for prompt guidance)
used_keywords_per_class = defaultdict(set)
# Track keyword usage counts per class (for controlled duplication)
keyword_counts_per_class = defaultdict(lambda: defaultdict(int))

# Track used sentences (exact duplicates)
generated_sentences = set()

# Track sentence hashes for near-duplicate detection
generated_ar_hashes = set()

# Track sentence openings per class
used_openings_per_class = defaultdict(list)

# Current domain index for rotation
domain_rotation_index = defaultdict(int)

def get_sentence_hash(text: str) -> str:
    """Create a normalized hash for near-duplicate detection."""
    # Normalize: remove spaces, diacritics, take first 60 chars
    normalized = re.sub(r'[\s\u064B-\u065F]', '', text)[:60]
    return hashlib.md5(normalized.encode()).hexdigest()[:12]

def get_current_domain(error_type: str) -> str:
    """Rotate through domains for each class."""
    domains = list(UN_DOMAIN_KEYWORDS.keys())
    idx = domain_rotation_index[error_type] % len(domains)
    domain_rotation_index[error_type] += 1
    return domains[idx]

def get_recent_openings(error_type: str, n: int = 3) -> list:
    """Get recent sentence openings used for this class."""
    return used_openings_per_class[error_type][-n:]

def extract_opening(sentence: str) -> str:
    """Extract first word/phrase of sentence."""
    words = sentence.split()[:2]
    return ' '.join(words) if words else ''


# =============================================================================
# DOMAIN VOCABULARY TRACKING (Progressive Diversity + Reviewer Documentation)
# =============================================================================
from collections import defaultdict
import re

# Track token vocabulary used per domain (tokens)
used_domain_tokens = defaultdict(set)

# Track which predefined UN_DOMAIN_KEYWORDS terms (tokens + phrases) have been actually USED
used_predefined_terms = defaultdict(set)  # domain -> set(term strings)

# Track suggested vs. used recent terms separately
recent_suggested_terms = defaultdict(list)
recent_used_terms = defaultdict(list)
MAX_RECENT_TERMS = 20  # keep last N items for rotation

def _arabic_tokens(text: str):
    """Extract Arabic tokens (letters only)."""
    if not isinstance(text, str) or not text.strip():
        return []
    return re.findall(r'[\u0600-\u06FF]+', text)

def update_vocabulary_tracking(ar_sentence: str, domain: str):
    """
    Update domain-level vocabulary tracking based on the ACTUAL generated Arabic sentence.
    - Tracks token vocabulary
    - Tracks coverage of predefined UN_DOMAIN_KEYWORDS terms (tokens + phrases)
    """
    if not ar_sentence or not domain:
        return

    tokens = _arabic_tokens(ar_sentence)
    if tokens:
        used_domain_tokens[domain].update(tokens)

    # Phrase-aware term coverage
    for term in UN_DOMAIN_KEYWORDS.get(domain, []):
        if not term:
            continue
        # Phrase term: substring match
        if " " in term:
            if term in ar_sentence:
                used_predefined_terms[domain].add(term)
        else:
            # Single token term: token match
            if term in tokens:
                used_predefined_terms[domain].add(term)

def get_progressive_keywords(domain: str, k: int = 3):
    """
    Return up to k suggested UN-domain terms for prompting.
    Strategy:
    - Prioritize terms NOT recently SUGGESTED
    - Mix in some older terms if list becomes exhausted
    This is prompt guidance only; actual usage is tracked separately.
    """
    terms = UN_DOMAIN_KEYWORDS.get(domain, [])
    if not terms:
        return []

    recent = recent_suggested_terms.get(domain, [])
    candidates = [t for t in terms if t not in recent]

    if len(candidates) < k:
        # fall back: bring back older items (not the most recent)
        if len(recent) > 2*k:
            candidates += recent[:k]
        else:
            candidates += terms

    selected = candidates[:k]
    # rotate memory
    recent_suggested_terms[domain].extend(selected)
    recent_suggested_terms[domain] = recent_suggested_terms[domain][-MAX_RECENT_TERMS:]
    return selected


# =============================================================================
# DOMAIN VOCABULARY ANALYSIS (Reviewer-facing documentation)
# =============================================================================
print("\n📊 DOMAIN VOCABULARY USAGE STATISTICS:")
print("-" * 70)

vocab_report = {}
# Initialize quality_report before its usage
quality_report = {}
for domain in UN_DOMAIN_KEYWORDS:
    predefined_terms = UN_DOMAIN_KEYWORDS[domain]
    total_terms = len(predefined_terms)

    used_terms = len(used_predefined_terms.get(domain, set()))
    total_unique_tokens = len(used_domain_tokens.get(domain, set()))
    coverage_ratio = (used_terms / total_terms) if total_terms else 0.0

    vocab_report[domain] = {
        "predefined_terms": total_terms,
        "used_predefined_terms": used_terms,
        "coverage_ratio_terms": coverage_ratio,
        "total_unique_tokens": total_unique_tokens,
        "vocabulary_expansion_tokens": max(0, total_unique_tokens - total_terms),
    }

    print(f"{domain.upper():12s}: terms_used={used_terms:2d}/{total_terms:2d} "
          f"({coverage_ratio:.0%}) | unique_tokens={total_unique_tokens:4d}")

# Attach to quality report and save standalone CSV
try:
    quality_report['domain_vocabulary'] = vocab_report
except Exception:
    pass

try:
    import pandas as pd
    # PLOTS_DIR needs to be defined for this to work. Define it here if not global.
    PLOTS_DIR = Path(OUTPUT_DIR) / "plots"
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    vocab_df = pd.DataFrame([{"domain": d, **vocab_report[d]} for d in vocab_report]).sort_values("domain")
    vocab_csv = os.path.join(PLOTS_DIR if 'PLOTS_DIR' in globals() else ".", "domain_vocabulary_report.csv")
    vocab_df.to_csv(vocab_csv, index=False, encoding="utf-8-sig")
    print(f"✅ Saved domain vocabulary report CSV: {vocab_csv}")
except Exception as e:
    print(f"⚠️ Could not write domain vocabulary CSV: {str(e)[:120]}")

try:
    with open(OUTPUT_QUALITY_REPORT, 'w', encoding='utf-8') as f:
        json.dump(quality_report, f, indent=2, ensure_ascii=False)
    print("✅ Updated quality report with domain vocabulary analysis")
except Exception as e:
    print(f"⚠️ Could not update quality report JSON: {str(e)[:120]}")

# =============================================================================
# STEP 0: API SETUP
# =============================================================================

from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

# =============================================================================
# ENHANCED SYSTEM PROMPT (With Diversity Requirements)
# =============================================================================

SYSTEM_PROMPT = """You are an expert Arabic linguist generating synthetic MT error examples for a UN-domain benchmark.

YOUR ROLE:
Generate NEW examples of specific translation error types. You will see a few real examples of each error type, then create NEW instances with NEW Keyword/Best_Match pairs that follow the same pattern of linguistic shift.

DOMAIN GUIDANCE:
Generate content that resembles authentic United Nations texts (e.g., Security Council resolutions, Secretary-General reports, official UN statements). Focus on political, diplomatic, legal, humanitarian, or conflict-related topics with formal, neutral, institutional tone.

MINIMAL PAIR REQUIREMENT:
The 'ar' (clean) and 'mt_output' (error) sentences MUST be identical except for the specified anchor edit (Keyword→Best_Match, or Keyword omitted). Do not introduce additional changes to vocabulary, structure, or meaning beyond the specified error.

=== CRITICAL: DIVERSITY REQUIREMENTS FOR MODEL TRAINING ===

1. LEXICAL DIVERSITY: Use a COMPLETELY DIFFERENT keyword than any shown in examples or listed as "already used"
2. SYNTACTIC VARIATION: Use a DIFFERENT sentence opening than recent examples
3. DOMAIN ROTATION: Draw vocabulary from the specified UN subdomain
4. NO REPETITION: Never reuse keywords, sentence structures, or templates

=== HARD CONSTRAINTS ===

- Generate NEW Keyword DIFFERENT from seed examples AND "already used" list
- Keyword MUST appear exactly ONCE in 'ar'
- Best_Match MUST appear in 'mt_output' (unless Best_Match is [OMITTED])
- 'ar' and 'mt_output' must be IDENTICAL except the anchor edit
- Formal Arabic, institutional tone, 15–30 words
- Start sentence with a DIFFERENT opening than recent examples
- Output ONLY valid JSON with exactly 5 keys:
  {"Keyword": "...", "Best_Match": "...", "ar": "...", "mt_output": "...", "en": "..."}"""

# =============================================================================
# STEP 1: LOAD FEW-SHOT BANK
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD FEW-SHOT BANK")
print("=" * 80)

# Try multiple paths
few_shot_paths = [
    FEW_SHOT_BANK_PATH,
    "few_shot_bank_tier2.csv",
    f"{OUTPUT_DIR}/few_shot_bank_tier21.csv",
]

few_shot_df = None
for path in few_shot_paths:
    if Path(path).exists():
        few_shot_df = pd.read_csv(path)
        print(f"✅ Loaded few-shot bank: {path} ({len(few_shot_df)} samples)")
        break

if few_shot_df is None:
    raise FileNotFoundError(f"Few-shot bank not found in: {few_shot_paths}")

# Rename 'error_type' column to 'Sub-Subtype' for consistency with generated samples
# few_shot_df contains the 'error_type' column (from PHOJoDkRLJZl)
# But the rest of the code expects 'Sub-Subtype'
# Rename columns to match expected schema
column_mapping = {
    'error_type': 'Sub-Subtype',
    'clean_ar': 'ar',
    'error_ar': 'mt_output',
    'keyword': 'Keyword',
    'best_match': 'Best_Match',
}

rename_dict = {k: v for k, v in column_mapping.items() if k in few_shot_df.columns and v not in few_shot_df.columns}
if rename_dict:
    few_shot_df = few_shot_df.rename(columns=rename_dict)
    print(f"✅ Renamed columns: {rename_dict}")


# Group by class
few_shot_by_class = {}
for cls in TAXONOMY_23:
    cls_df = few_shot_df[few_shot_df['Sub-Subtype'] == cls]
    if len(cls_df) > 0:
        few_shot_by_class[cls] = cls_df.to_dict('records')
        # Pre-populate used keywords from few-shot examples
        for ex in cls_df.to_dict('records'):
            keyword = ex.get('Keyword')
            if pd.notna(keyword):
                used_keywords_per_class[cls].add(str(keyword))

print(f"✅ Classes with examples: {len(few_shot_by_class)} / {len(TAXONOMY_23)}")


# =============================================================================
# CONTROLLED DUPLICATION: PER-CLASS KEYWORD CAPS (computed from keyword banks)
# =============================================================================
# We compute a per-class maximum repeat count for any single keyword so that:
# - classes with small keyword banks can still reach SAMPLES_PER_CLASS
# - no single keyword dominates (prevents memorization)
# Keyword bank is derived from the few-shot bank keywords per class.
_bank_sizes = (
    few_shot_df.dropna(subset=['Keyword', 'Sub-Subtype'])
              .copy()
              .assign(Keyword=lambda x: x['Keyword'].astype(str))
              .groupby('Sub-Subtype')['Keyword']
              .nunique()
              .to_dict()
)

PER_CLASS_KEYWORD_REPEAT_CAP = {}
EFFECTIVE_DUPLICATION_RATE = {}

_default_rate = ALLOWED_DUPLICATION_RATES.get("Default", 0.15)

for _cls in TAXONOMY_23:
    k = int(_bank_sizes.get(_cls, 0) or 0)

    # If bank is missing, assume a modest effective bank; LLM can invent keywords,
    # but we keep a conservative cap so training won't memorize one keyword.
    if k <= 0:
        k = max(20, int(round(0.2 * SAMPLES_PER_CLASS)))

    # Minimal duplication needed IF the model largely reuses known keywords.
    needed_dup_rate = max(0, SAMPLES_PER_CLASS - k) / max(SAMPLES_PER_CLASS, 1)

    base_rate = ALLOWED_DUPLICATION_RATES.get(_cls, _default_rate);
    # Ensure feasibility: add small buffer to avoid stalling late.
    eff_rate = min(MAX_DUPLICATION_RATE_CAP, max(base_rate, needed_dup_rate + 0.02))
    EFFECTIVE_DUPLICATION_RATE[_cls] = eff_rate

    # Per-keyword cap: at least ceil(target/bank)+2; also bounded by global heuristic.
    cap_from_bank = int(math.ceil(SAMPLES_PER_CLASS / max(1, k)) + 2)
    cap_global = max_keyword_repeats_cap(SAMPLES_PER_CLASS)
    PER_CLASS_KEYWORD_REPEAT_CAP[_cls] = max(3, min(max(cap_from_bank, 3), cap_global))

print("\n" + "=" * 80)
print("CONTROLLED DUPLICATION POLICY (computed)")
print("=" * 80)
# Show the most constrained classes first (smallest keyword banks)
for _cls, _k in sorted(_bank_sizes.items(), key=lambda x: x[1])[:8]:
    print(f"  {_cls:30s} bank={_k:>4}  dup_rate<= {EFFECTIVE_DUPLICATION_RATE[_cls]:.0%}  kw_cap<= {PER_CLASS_KEYWORD_REPEAT_CAP[_cls]}")
print("  ...")


# =============================================================================
# STEP 2: LOAD CHECKPOINT (if exists)
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: LOAD CHECKPOINT")
print("=" * 80)

generated_samples = []
checkpoint_classes = set()

if Path(CHECKPOINT_PATH).exists():
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                sample = json.loads(line)
                generated_samples.append(sample)
                checkpoint_classes.add(sample.get('Sub-Subtype'))

                # Restore tracking state
                cls = sample.get('Sub-Subtype')
                kw = sample.get('Keyword')
                ar = sample.get('ar', '')

                if pd.notna(kw):
                    used_keywords_per_class[cls].add(str(kw))
                if ar:
                    generated_sentences.add(ar)
                    generated_ar_hashes.add(get_sentence_hash(ar))
                    used_openings_per_class[cls].append(extract_opening(ar))

    print(f"✅ Checkpoint loaded: {len(generated_samples)} samples from {len(checkpoint_classes)} classes")
    print(f"   Tracking restored: {sum(len(v) for v in used_keywords_per_class.values())} keywords")
else:
    print("✅ No checkpoint found - starting fresh")

# =============================================================================
# ENHANCED PROMPT BUILDER (With Diversity Guidance)
# =============================================================================


def build_prompt(error_type: str, examples: list, sample_index: int) -> str:
    """
    Build prompt with diversity guidance for classifier training.
    Notes:
    - Domain terms/openings are SOFT guidance (not hard constraints).
    - Keep prompt compact to preserve JSON compliance and control cost.
    """
    definition = ARABIC_ERROR_DEFINITIONS.get(error_type, "")

    # Choose domain for this prompt (store domain on returned sample later)
    current_domain = get_current_domain(error_type)
    domain_terms = get_progressive_keywords(current_domain, k=3)

    # Recently used per-class anchors/openings (short lists only)
    used_kw = list(used_keywords_per_class[error_type]) if error_type in used_keywords_per_class else []
    recent_openings = get_recent_openings(error_type)

    # Error-type specific guidance (concise)
    if error_type == "Name Entity Error":
        keyword_guidance = "Keyword MUST be a proper name (ORG/LOC/PER/UN body)."
    elif error_type == "Literal Translation":
        keyword_guidance = "Keyword should be an idiom/collocation that can be mistranslated literally."
    elif error_type in ["Hypernym for Hyponym", "Hyponym for Hypernym"]:
        keyword_guidance = "Choose words with clear hypernym–hyponym relationship."
    elif error_type in ["Total Omission", "Partial Translation"]:
        keyword_guidance = "For omission: Best_Match must be '[OMITTED]' and Keyword must be missing in mt_output."
    else:
        keyword_guidance = "Keyword should be UN-domain; keep tone formal and UN-style."

    # Lightweight info for diversity (avoid long lists)
    vocab_info = ""
    if current_domain in used_domain_tokens:
        vocab_info = f"used_vocab_tokens={len(used_domain_tokens[current_domain])}"

    prompt = f"""ERROR TYPE: {error_type}
DEFINITION: {definition}
KEYWORD GUIDANCE: {keyword_guidance}

DIVERSITY (SOFT):
- Domain: {current_domain} {vocab_info}
- Prefer including 1 of these domain terms: {', '.join(domain_terms) if domain_terms else '(none)'}
- Avoid reusing these recent Keywords: {', '.join(used_kw[-5:]) if used_kw else '(none)'}
- Avoid recent openings: {', '.join(recent_openings[:3]) if recent_openings else '(none)'}
- Suggested openings (pick 1): {', '.join(SENTENCE_OPENINGS[sample_index % len(SENTENCE_OPENINGS):(sample_index % len(SENTENCE_OPENINGS))+3])}

SEED EXAMPLES (learn pattern, generate DIFFERENT):
"""
    for i, ex in enumerate(examples[:3], 1):
        prompt += f"""
Example {i}:
- Keyword: {ex.get('Keyword','')}
- Best_Match: {ex.get('Best_Match','')}
- ar: {ex.get('ar','')}
- mt_output: {ex.get('mt_output','')}
"""

    prompt += """
TASK:
Generate ONE new sample as strict JSON ONLY:
{"Keyword":"...","Best_Match":"...","ar":"...","mt_output":"...","en":"..."}

Rules:
- ar and mt_output must be identical except the Keyword→Best_Match transformation (or omission rule).
- Formal Arabic, institutional tone, 15–30 words.
- No extra text outside JSON.
"""
    return prompt


# =============================================================================
# ENHANCED VALIDATION (For Training Quality)
# =============================================================================

def validate_generation(sample: dict, error_type: str) -> tuple:
    """Validate LLM output meets training quality requirements."""

    keyword = sample.get('Keyword', '')
    best_match = sample.get('Best_Match', '')
    ar = sample.get('ar', '')
    mt_output = sample.get('mt_output', '')

    # === BASIC VALIDATION ===

    # Check required fields
    if not all([keyword, ar, mt_output]):
        return False, "Missing required fields"

    # Keyword must appear exactly once in ar
    keyword_count = ar.count(keyword)
    if keyword_count == 0:
        return False, f"Keyword '{keyword}' not found in ar"
    if keyword_count > 1:
        return False, f"Keyword '{keyword}' appears {keyword_count} times in ar (should be 1)"

    # Best_Match validation
    omission_types = ["Total Omission", "Partial Translation"]
    if error_type in omission_types:
        if best_match not in ['[OMITTED]', '[omitted]', 'OMITTED', '']:
            if best_match and best_match in mt_output:
                pass  # Partial omission might have partial match
    else:
        if best_match and best_match not in mt_output:
            return False, f"Best_Match '{best_match}' not found in mt_output"

    # Keyword should NOT appear in mt_output (it should be replaced)
    if error_type not in omission_types:
        if keyword in mt_output:
            return False, f"Keyword '{keyword}' should not appear in mt_output"

    # Minimal pair check
    ar_words = set(ar.split())
    mt_words = set(mt_output.split())
    overlap = len(ar_words & mt_words) / max(len(ar_words), len(mt_words), 1)
    if overlap < 0.5:
        return False, f"Sentences too different (overlap={overlap:.2f})"

    # === TRAINING QUALITY VALIDATION ===

    # 1. Controlled keyword duplication within class (balance vs diversity)
    if keyword:
        already_used = keyword in used_keywords_per_class[error_type]
        if already_used:
            # Current per-class counts
            current_total = int(samples_per_class_done.get(error_type, 0))
            current_unique = len(used_keywords_per_class[error_type])
            current_dup = max(0, current_total - current_unique)

            # Allow per-class duplicate budget
            allowed_rate = EFFECTIVE_DUPLICATION_RATE.get(error_type, ALLOWED_DUPLICATION_RATES.get("Default", 0.15))
            # If we accept this sample as a duplicate keyword, dup count would increase by 1
            projected_total = current_total + 1
            projected_dup = current_dup + 1

            # Per-keyword repeat cap
            cap = PER_CLASS_KEYWORD_REPEAT_CAP.get(error_type, max_keyword_repeats_cap(SAMPLES_PER_CLASS))
            if keyword_counts_per_class[error_type].get(keyword, 0) >= cap:
                return False, f"Keyword '{keyword}' exceeded repeat cap ({cap}) for this class"

            # Global duplicate-rate cap for the class
            if projected_dup / max(projected_total, 1) > allowed_rate:
                return False, (
                    f"Keyword duplication budget exceeded for class "
                    f"({projected_dup}/{projected_total} > {allowed_rate:.0%})"
                )

    # 2. Exact sentence duplicate
    if ar in generated_sentences:
        return False, "Exact duplicate sentence"

    # 3. Near-duplicate check
    ar_hash = get_sentence_hash(ar)
    if ar_hash in generated_ar_hashes:
        return False, "Near-duplicate sentence (too similar to existing)"

    # 4. Minimum sentence length
    if len(ar.split()) < 10:
        return False, "Sentence too short (<10 words)"

    return True, "OK"

# =============================================================================
# GENERATION FUNCTION
# =============================================================================

def generate_one_sample(prompt: str, temperature: float) -> dict:
    """Generate one sample using OpenAI."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=600,
        response_format={"type": "json_object"}
    )

    raw = response.choices[0].message.content.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        json_match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        raise ValueError(f"Could not parse JSON from: {raw[:100]}")

def save_checkpoint(sample: dict):
    """Append sample to checkpoint file."""
    with open(CHECKPOINT_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')

def update_tracking(sample: dict, error_type: str):
    """Update diversity tracking after successful generation."""
    keyword = sample.get('Keyword', '')
    ar = sample.get('ar', '')

    if pd.notna(keyword): # Check if keyword is not NaN
        str_keyword = str(keyword) # Convert to string
        used_keywords_per_class[error_type].add(str_keyword)
        keyword_counts_per_class[error_type][str_keyword] += 1
    if ar:
        generated_sentences.add(ar)
        generated_ar_hashes.add(get_sentence_hash(ar))
        used_openings_per_class[error_type].append(extract_opening(ar))

# =============================================================================
# STEP 3: GENERATE SAMPLES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: GENERATE TIER2 SAMPLES (Enhanced for Training)")
print("=" * 80)

# Count existing samples per class from checkpoint
samples_per_class_done = defaultdict(int)
for s in generated_samples:
    samples_per_class_done[s.get('Sub-Subtype')] += 1

print(f"""
📊 GENERATION PLAN:
   Classes to generate: {len(few_shot_by_class)}
   Samples per class: {SAMPLES_PER_CLASS}
   Total target: {len(few_shot_by_class) * SAMPLES_PER_CLASS}
   Model: {MODEL}

😡 DIVERSITY ENFORCEMENT:
   ✓ Keyword tracking per class
   ✓ Sentence uniqueness check
   ✓ Near-duplicate detection
   ✓ Domain rotation
   ✓ Sentence opening variety
"""
)

stats = {
    'api_errors': 0,
    'validation_failures': 0,
    'diversity_rejections': 0,
    'successful': 0,
    'per_class': {}
}

input("Press ENTER to start generation...")

for error_type in tqdm(TAXONOMY_23, desc="Generating Tier2"):

    # Skip if no seeds
    if error_type not in few_shot_by_class:
        print(f"   ⚠️ Skipping {error_type}: no seed examples")
        continue

    examples = few_shot_by_class[error_type]
    temperature = TEMPERATURE_MAP.get(error_type, DEFAULT_TEMPERATURE)

    # How many more do we need?
    already_done = samples_per_class_done.get(error_type, 0)
    needed = SAMPLES_PER_CLASS - already_done

    if needed <= 0:
        continue

    class_stats = {
        'attempts': 0,
        'success': 0,
        'api_errors': 0,
        'validation_failures': 0,
        'diversity_rejections': 0
    }

    for i in tqdm(range(needed), desc=f"  {error_type[:20]}", leave=False):

        sample_index = already_done + i
        prompt = build_prompt(error_type, examples, sample_index)

        for retry in range(MAX_RETRIES):
            try:
                # Generate
                sample = generate_one_sample(prompt, temperature)
                class_stats['attempts'] += 1

                # Validate (includes diversity checks)
                valid, reason = validate_generation(sample, error_type)

                if valid:
                    # Build raw output row
                    row = {
                        'Sentence_ID': f"TIER2_{error_type.replace(' ', '_')}_{uuid.uuid4().hex[:8]}",
                        'Sub-Subtype': error_type,
                        'Keyword': sample.get('Keyword', ''),
                        'Best_Match': sample.get('Best_Match', ''),
                        'ar': sample.get('ar', ''),
                        'mt_output': sample.get('mt_output', ''),
                        'en': sample.get('en', ''),
                        'generation_timestamp': datetime.now().isoformat(),
                        'seed_source': 'LLM_GENERATED',
                    }

                    # Update tracking BEFORE saving
                    update_tracking(row, error_type)

                    generated_samples.append(row)
                    save_checkpoint(row)
                    class_stats['success'] += 1
                    stats['successful'] += 1
                    break
                else:
                    # Track diversity vs other validation failures
                    if 'diversity' in reason.lower() or 'duplicate' in reason.lower() or 'already used' in reason.lower():
                        class_stats['diversity_rejections'] += 1
                        stats['diversity_rejections'] += 1
                    else:
                        class_stats['validation_failures'] += 1
                        stats['validation_failures'] += 1

                    # If diversity failure, increase temperature slightly for next retry
                    if retry < MAX_RETRIES - 1 and 'diversity' in reason.lower():
                        temperature = min(1.0, temperature + 0.1)

            except Exception as e:
                error_msg = str(e).lower()

                # Check for billing/quota errors
                if any(term in error_msg for term in ['billing', 'quota', 'insufficient', 'rate_limit', 'exceeded']):
                    print(f"\n" + "=" * 60)
                    print("⚠️ BILLING/QUOTA ERROR DETECTED - STOPPING EARLY")
                    print("=" * 60)
                    print(f"   Error: {str(e)[:100]}")
                    print(f"   Samples saved: {len(generated_samples)}")
                    print(f"   Checkpoint: {CHECKPOINT_PATH}")
                    raise SystemExit("Billing error - checkpoint saved")

                class_stats['api_errors'] += 1
                stats['api_errors'] += 1
                if retry == MAX_RETRIES - 1:
                    print(f"   ⚠️ Failed after {MAX_RETRIES} retries: {str(e)[:50]}")

    stats['per_class'][error_type] = class_stats

# =============================================================================
# STEP 4: SAVE RAW OUTPUT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: SAVE RAW OUTPUT")
print("=" * 80)

raw_df = pd.DataFrame(generated_samples)

RAW_COLUMNS = [
    'Sentence_ID', 'Sub-Subtype', 'Keyword', 'Best_Match',
    'ar', 'mt_output', 'en', 'generation_timestamp', 'seed_source',
]

for col in RAW_COLUMNS:
    if col not in raw_df.columns:
        raw_df[col] = ''

raw_df = raw_df[RAW_COLUMNS]

raw_df.to_csv(OUTPUT_RAW_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUTPUT_RAW_CSV} ({len(raw_df)} rows)")

# Save JSON
raw_json = {}
for cls in TAXONOMY_23:
    cls_df = raw_df[raw_df['Sub-Subtype'] == cls]
    raw_json[cls] = cls_df.to_dict('records')

with open(OUTPUT_RAW_JSON, 'w', encoding='utf-8') as f:
    json.dump(raw_json, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {OUTPUT_RAW_JSON}")

# =============================================================================
# STEP 5: QUALITY ANALYSIS (For Training Data)
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: TRAINING DATA QUALITY ANALYSIS")
print("=" * 80)

quality_report = {
    'summary': {
        'total_samples': len(raw_df),
        'classes_covered': raw_df['Sub-Subtype'].nunique(),
        'target_per_class': SAMPLES_PER_CLASS,
    },
    'diversity_metrics': {},
    'issues': [],
    'per_class': {}
}

print("\n📊 KEYWORD DIVERSITY / DUPLICATION PER CLASS:")
print("-" * 60)

for cls in TAXONOMY_23:
    cls_df = raw_df[raw_df['Sub-Subtype'] == cls]
    if cls_df.empty:
        continue

    keywords = cls_df['Keyword'].tolist()
    total = len(keywords)
    unique_keywords = len(set(keywords))
    diversity_ratio = unique_keywords / total if total else 0
    dup_rate = 1.0 - diversity_ratio if total else 0.0

    allowed_rate = ALLOWED_DUPLICATION_RATES.get(cls, ALLOWED_DUPLICATION_RATES["Default"])

    # Status compares observed dup_rate to allowed_rate (not to a fixed 0.8 rule)
    if dup_rate <= allowed_rate + 1e-9:
        status = "✅"
    elif dup_rate <= allowed_rate + 0.10:
        status = "⚠️"
    else:
        status = "❌"

    print(f"   {status} {cls:<35}: {unique_keywords}/{total} unique "
          f"({diversity_ratio:.0%}) | dup={dup_rate:.0%} | allowed≤{allowed_rate:.0%}")

    quality_report['per_class'][cls] = {
        'samples': int(total),
        'unique_keywords': int(unique_keywords),
        'diversity_ratio': float(diversity_ratio),
        'keyword_dup_rate': float(dup_rate),
        'allowed_keyword_dup_rate': float(allowed_rate),
    }

    if dup_rate > allowed_rate + 0.10:
        quality_report['issues'].append(
            f"{cls}: Keyword duplication too high ({dup_rate:.0%} > allowed {allowed_rate:.0%})"
        )
# Sentence uniqueness
duplicate_count = raw_df['ar'].duplicated().sum()
quality_report['diversity_metrics']['sentence_duplicates'] = int(duplicate_count) # Convert to int
print(f"\n   Duplicate sentences: {duplicate_count}")

if duplicate_count > 0:
    quality_report['issues'].append(f"{duplicate_count} duplicate sentences found")

# Overall diversity score
avg_diversity = sum(
    q['diversity_ratio'] for q in quality_report['per_class'].values()
) / len(quality_report['per_class']) if quality_report['per_class'] else 0

quality_report['diversity_metrics']['average_keyword_diversity'] = float(avg_diversity) # Convert to float
quality_report['diversity_metrics']['sentence_uniqueness'] = float((len(raw_df) - duplicate_count) / len(raw_df)) if len(raw_df) > 0 else 0 # Convert to float

print(f"\n📊 OVERALL QUALITY SCORE:")
print(f"   Average keyword diversity: {avg_diversity:.0%}")
print(f"   Sentence uniqueness: {quality_report['diversity_metrics']['sentence_uniqueness']:.0%}")

# Save quality report
with open(OUTPUT_QUALITY_REPORT, 'w', encoding='utf-8') as f:
    json.dump(quality_report, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {OUTPUT_QUALITY_REPORT}")


# =============================================================================
# STEP 6: LOGGING + PLOTS (Prove keyword reuse never dominates)
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6: LOGGING + PLOTS (KEYWORD REUSE DIAGNOSTICS)")
print("=" * 80)

import matplotlib.pyplot as plt
from collections import Counter

PLOTS_DIR = Path(OUTPUT_DIR) / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for cls in TAXONOMY_23:
    cls_df = raw_df[raw_df["Sub-Subtype"] == cls]
    if cls_df.empty:
        continue

    kws = [k for k in cls_df["Keyword"].tolist() if isinstance(k, str)]
    total = len(kws)
    cnt = Counter(kws)
    unique = len(cnt)

    max_kw = None
    max_freq = 0
    if cnt:
        max_kw, max_freq = cnt.most_common(1)[0]

    max_share = (max_freq / total) if total else 0.0
    # "Dominance" heuristic: if a single keyword exceeds 15% of class samples,
    # the class may be overly concentrated. This should stay low with repeat caps.
    dominance_flag = int(max_share > 0.15)

    rows.append({
        "Sub-Subtype": cls,
        "samples": int(total),
        "unique_keywords": int(unique),
        "keyword_dup_rate": float(quality_report["per_class"][cls]["keyword_dup_rate"]),
        "allowed_keyword_dup_rate": float(quality_report["per_class"][cls]["allowed_keyword_dup_rate"]),
        "max_keyword": max_kw or "",
        "max_keyword_freq": int(max_freq),
        "max_keyword_share": float(max_share),
        "dominance_flag_gt_15pct": dominance_flag,
        "unique_sentences_hash": int(cls_df["ar"].apply(get_sentence_hash).nunique()),
        "sentence_dup_rate_hash": float(1.0 - (cls_df["ar"].apply(get_sentence_hash).nunique() / max(1, len(cls_df)))),
        "per_keyword_repeat_cap": int(PER_CLASS_KEYWORD_REPEAT_CAP.get(cls, -1)),
        "effective_duplication_rate": float(EFFECTIVE_DUPLICATION_RATE.get(cls, -1.0)),
    })

diag_df = pd.DataFrame(rows).sort_values(["samples", "Sub-Subtype"], ascending=[False, True])
diag_csv = str(PLOTS_DIR / "keyword_reuse_diagnostics.csv")
diag_df.to_csv(diag_csv, index=False, encoding="utf-8-sig")
print(f"✅ Saved diagnostics CSV: {diag_csv}")

# ---- Plot 1: Samples per class vs target ----
plt.figure()
plt.bar(diag_df["Sub-Subtype"], diag_df["samples"])
plt.xticks(rotation=90)
plt.axhline(SAMPLES_PER_CLASS, linestyle="--")
plt.title("Tier2 Samples per Class (Target Line)")
plt.tight_layout()
p1 = str(PLOTS_DIR / "samples_per_class.png")
plt.savefig(p1, dpi=200)
plt.close()
print(f"✅ Saved plot: {p1}")

# ---- Plot 2: Keyword duplication rate vs allowed ----
plt.figure()
plt.bar(diag_df["Sub-Subtype"], diag_df["keyword_dup_rate"])
plt.xticks(rotation=90)
plt.title("Keyword Duplication Rate per Class")
plt.tight_layout()
p2 = str(PLOTS_DIR / "keyword_dup_rate_per_class.png")
plt.savefig(p2, dpi=200)
plt.close()
print(f"✅ Saved plot: {p2}")

plt.figure()
plt.bar(diag_df["Sub-Subtype"], diag_df["allowed_keyword_dup_rate"])
plt.xticks(rotation=90)
plt.title("Allowed Keyword Duplication Rate per Class")
plt.tight_layout()
p2b = str(PLOTS_DIR / "allowed_keyword_dup_rate_per_class.png")
plt.savefig(p2b, dpi=200)
plt.close()
print(f"✅ Saved plot: {p2b}")


# ---- Plot 2C: Configured keyword duplication caps (policy) ----
try:
    caps_df = pd.read_csv(CAPS_TABLE_CSV)
    plt.figure()
    plt.bar(caps_df["Sub-Subtype"], caps_df["keyword_dup_cap"])
    plt.xticks(rotation=90)
    plt.title("Configured Keyword Duplication Caps per Class (Policy)")
    plt.tight_layout()
    p2c = str(PLOTS_DIR / "configured_keyword_dup_caps_per_class.png")
    plt.savefig(p2c, dpi=200)
    plt.close()
    print(f"✅ Saved plot: {p2c}")
except Exception as _e:
    print(f"⚠️ Could not plot caps table: {_e}")


# ---- Plot 3: Max keyword share per class (dominance check) ----
plt.figure()
plt.bar(diag_df["Sub-Subtype"], diag_df["max_keyword_share"])
plt.xticks(rotation=90)
plt.axhline(0.15, linestyle="--")
plt.title("Max Keyword Share per Class (Dominance Check)")
plt.tight_layout()
p3 = str(PLOTS_DIR / "max_keyword_share_per_class.png")
plt.savefig(p3, dpi=200)
plt.close()
print(f"✅ Saved plot: {p3}")

# Add diagnostics summary to quality_report
quality_report["keyword_reuse_diagnostics"] = {
    "diagnostics_csv": diag_csv,
    "plots_dir": str(PLOTS_DIR),
    "dominance_threshold": 0.15,
    "num_classes_flagged": int(diag_df["dominance_flag_gt_15pct"].sum()),
    "classes_flagged": diag_df.loc[diag_df["dominance_flag_gt_15pct"] == 1, "Sub-Subtype"].tolist(),
    "notes": (
        "max_keyword_share measures concentration: max(count(keyword))/class_samples. "
        "With per-keyword repeat caps, this should remain low. "
        "Sentence-level near-duplicate blocking remains the primary diversity guardrail."
    )
}

# Re-save quality report with diagnostics included
with open(OUTPUT_QUALITY_REPORT, 'w', encoding='utf-8') as f:
    json.dump(quality_report, f, indent=2, ensure_ascii=False)
print(f"✅ Updated quality report with diagnostics: {OUTPUT_QUALITY_REPORT}")



# =============================================================================
# DATASET DATASHEET (REPRODUCIBLE, REVIEWER-FRIENDLY)
# =============================================================================

DATASET_DATASHEET_JSON = os.path.join(OUTPUT_DIR, "dataset_datasheet.json")

# Collect plot artifacts (if they exist)
plot_files = []
try:
    if os.path.isdir(PLOTS_DIR):
        plot_files = sorted([str(p) for p in Path(PLOTS_DIR).glob("*.png")])
except Exception:
    plot_files = []

# Load cap policy table (exported above)
cap_policy = []
try:
    cap_df = pd.read_csv(CAPS_TABLE_CSV)
    cap_policy = cap_df.to_dict(orient="records")
except Exception:
    pass

# Build per-class summary from diagnostics (diag_df)
per_class_summary = []
try:
    per_class_summary = diag_df.to_dict(orient="records")
except Exception:
    pass

# Pull key run config if available
run_config = {
    "model": globals().get("MODEL", None),
    "temperature": globals().get("TEMPERATURE", globals().get("temperature", None)),
    "max_retries": globals().get("MAX_RETRIES", None),
    "samples_per_class_target": globals().get("SAMPLES_PER_CLASS", None),
    "max_duplication_rate_cap": globals().get("MAX_DUPLICATION_RATE_CAP", None),
    "dominance_threshold": 0.15,
    "strict_sentence_dedup": True,
    "ceilings_allowed": True
}

datasheet = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat() + "Z", # Updated to use timezone.utc
    "tool_version": "CELL17A_v2.1_rebuilt_300_caps_uniqueness",
    "run_config": run_config,
    "artifacts": {
        "raw_csv": str(OUTPUT_RAW_CSV),
        "raw_json": str(OUTPUT_RAW_JSON),
        "quality_report_json": str(OUTPUT_QUALITY_REPORT),
        "cap_policy_csv": str(CAPS_TABLE_CSV),
        "diagnostics_csv": str(diag_csv),
        "plots": plot_files
    },
    "summary": {
        "total_samples": int(len(raw_df)) if "raw_df" in globals() else None,
        "num_classes": int(len(TAXONOMY_23)) if "TAXONOMY_23" in globals() else None,
        "avg_keyword_diversity": float(avg_diversity) if "avg_diversity" in globals() else None,
        "sentence_uniqueness": float(quality_report.get("diversity_metrics", {}).get("sentence_uniqueness", None)) if "quality_report" in globals() else None,
        "dominance_flagged_classes": diag_df.loc[diag_df["dominance_flag_gt_15pct"] == 1, "Sub-Subtype"].tolist() if "diag_df" in globals() and "dominance_flag_gt_15pct" in diag_df.columns else []
    },
    "cap_policy": cap_policy,
    "per_class_diagnostics": per_class_summary
}

with open(DATASET_DATASHEET_JSON, "w", encoding="utf-8") as f:
    json.dump(datasheet, f, indent=2, ensure_ascii=False)

print(f"📄 Datasheet saved: {DATASET_DATASHEET_JSON}")


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 17A v2.0 COMPLETE (Enhanced for Training)")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   - {OUTPUT_RAW_CSV} ({len(raw_df)} samples)
   - {OUTPUT_RAW_JSON}
   - {OUTPUT_QUALITY_REPORT}

📊 GENERATION STATS:
   Total generated: {stats['successful']}
   API errors: {stats['api_errors']}
   Validation failures: {stats['validation_failures']}
   Diversity rejections: {stats['diversity_rejections']}

📊 TRAINING QUALITY:
   Average keyword diversity: {avg_diversity:.0%} {'✅' if avg_diversity >= 0.8 else '⚠️'}
   Sentence uniqueness: {quality_report['diversity_metrics']['sentence_uniqueness']:.0%} {'✅' if duplicate_count == 0 else '⚠️'}
   Issues found: {len(quality_report['issues'])}

{'='*60}
{'✅ READY FOR CLASSIFIER TRAINING' if avg_diversity >= 0.8 and duplicate_count == 0 else '⚠️ REVIEW ISSUES BEFORE TRAINING'}
{'='*60}

📑 NEXT STEP:
   Run Cell 17B to add taxonomy metadata (deterministic enrichment)
""")

In [ ]:
# =============================================================================
# CELL 17B: DETERMINISTIC METADATA ENRICHMENT
# =============================================================================
"""
CELL 17B: Add Taxonomy Metadata to Generated Samples

PURPOSE:
Enrich raw LLM output with deterministic metadata from taxonomy table.
This cell adds NO LLM-generated content - only static lookups.

INPUT:
- tier2_generated_raw.csv (from Cell 17A)

METADATA ADDED (via left-join on Sub-Subtype):
- Class (parent category)
- Subtype (parent category)
- Severity_Level
- Severity_Score
- TQA_Category
- Definition
- Weight percentages

OUTPUT:
- tier2_generated_enriched.csv (full schema, ready for use)

VALIDATION:
- Hard-fail if any Sub-Subtype not in taxonomy
- Hard-fail if any metadata is missing after join
"""

import pandas as pd
import json
from datetime import datetime
from pathlib import Path

print("=" * 80)
print("CELL 17B: DETERMINISTIC METADATA ENRICHMENT")
print("=" * 80)
print("Adding taxonomy metadata to raw LLM output (no LLM calls)")

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# Input (from Cell 17A)
INPUT_RAW_CSV = f"{OUTPUT_DIR}/tier2_generated_raw.csv"

# Output (enriched)
OUTPUT_ENRICHED_CSV = f"{OUTPUT_DIR}/tier2_generated_enriched.csv"
OUTPUT_ENRICHED_JSON = f"{OUTPUT_DIR}/tier2_generated_enriched.json"
OUTPUT_ENRICHMENT_LOG = f"{OUTPUT_DIR}/tier2_enrichment_log.json"

# =============================================================================
# TAXONOMY METADATA TABLE (V3)
# =============================================================================
TAXONOMY_V3 = {
    # =========================================================================
    # 1. SEMANTIC DIVERGENCE (40% of total)
    # =========================================================================
    "Meaning Shift": {
        "Class": "Semantic Divergence",
        "Subtype": "Semantic Shift",
        "Definition": "The translation conveys an incorrect or distorted meaning compared to the source.",
        "Severity_Level": "Critical",
        "Severity_Score": 5,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.30,  # 30% within Semantic Divergence
        "Category_Weight": 0.40,  # Semantic Divergence = 40%
    },
    "Total Omission": {
        "Class": "Semantic Divergence",
        "Subtype": "Translation Gap",
        "Definition": "A source text element is completely missing from the translation.",
        "Severity_Level": "Critical",
        "Severity_Score": 5,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.30,  # 30% within Semantic Divergence
        "Category_Weight": 0.40,
    },
    "Partial Translation": {
        "Class": "Semantic Divergence",
        "Subtype": "Translation Gap",
        "Definition": "A named entity or key term is only partially translated.",
        "Severity_Level": "Severe",
        "Severity_Score": 4,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.20,  # 20% within Semantic Divergence (includes Named Entity Error context)
        "Category_Weight": 0.40,
    },
    "Name Entity Error": {
        "Class": "Semantic Divergence",
        "Subtype": "Translation Gap",
        "Definition": "Incorrect, distorted, or substituted rendering of a proper name (person, place, organization, treaty).",
        "Severity_Level": "Severe",
        "Severity_Score": 4,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.20,  # Grouped with Partial Translation in new taxonomy
        "Category_Weight": 0.40,
    },
    "Literal Translation": {
        "Class": "Semantic Divergence",
        "Subtype": "Idiomatic Shift",
        "Definition": "An idiom is translated word-for-word, losing its figurative meaning.",
        "Severity_Level": "Major",
        "Severity_Score": 3,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.10,  # 10% within Semantic Divergence
        "Category_Weight": 0.40,
    },
    "Hypernym for Hyponym": {
        "Class": "Semantic Divergence",
        "Subtype": "Granularity Shift",
        "Definition": "Using a general term for a specific one.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.05,  # 5% within Semantic Divergence
        "Category_Weight": 0.40,
    },
    "Hyponym for Hypernym": {
        "Class": "Semantic Divergence",
        "Subtype": "Granularity Shift",
        "Definition": "Using a specific term for a general one.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.05,  # 5% within Semantic Divergence
        "Category_Weight": 0.40,
    },

    # =========================================================================
    # 2. MORPHOSYNTACTIC SHIFTS (10% of total)
    # =========================================================================
    "Invalid Pattern": {
        "Class": "Morphosyntactic Shifts",
        "Subtype": "Morphological Error",
        "Definition": "Use of an incorrect form of a word.",
        "Severity_Level": "Major",
        "Severity_Score": 3,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.30,  # 30% within Morphosyntactic
        "Category_Weight": 0.10,
    },
    "Tanween Omission": {
        "Class": "Morphosyntactic Shifts",
        "Subtype": "Morphological Error",
        "Definition": "Omission of the Arabic nunation where required.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.20,  # 20% within Morphosyntactic
        "Category_Weight": 0.10,
    },
    "Gender Disagreement": {
        "Class": "Morphosyntactic Shifts",
        "Subtype": "Agreement Error",
        "Definition": "Mismatch in grammatical gender.",
        "Severity_Level": "Major",
        "Severity_Score": 3,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.30,  # 30% within Morphosyntactic
        "Category_Weight": 0.10,
    },
    "Definiteness Shift": {
        "Class": "Morphosyntactic Shifts",
        "Subtype": "Definiteness Shift",
        "Definition": "Incorrect use of the definite article 'al-' or its absence.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.20,  # 20% within Morphosyntactic
        "Category_Weight": 0.10,
    },

    # =========================================================================
    # 3. SYNTACTIC SHIFT (10% of total)
    # =========================================================================
    "Perfective to Progressive": {
        "Class": "Syntactic Shift",
        "Subtype": "Aspect Shift",
        "Definition": "Completed to Ongoing event.",
        "Severity_Level": "Severe",
        "Severity_Score": 4,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.15,  # 15% within Syntactic
        "Category_Weight": 0.10,
    },
    "Progressive to Perfective": {
        "Class": "Syntactic Shift",
        "Subtype": "Aspect Shift",
        "Definition": "Ongoing event to Completed.",
        "Severity_Level": "Severe",
        "Severity_Score": 4,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.15,  # 15% within Syntactic
        "Category_Weight": 0.10,
    },
    "Tense Shift Under Negation": {
        "Class": "Syntactic Shift",
        "Subtype": "Tense Shift",
        "Definition": "Tense shift under negation.",
        "Severity_Level": "Major",
        "Severity_Score": 3,
        "TQA_Category": "Accuracy",
        "Weight_WGT": 0.20,  # 20% within Syntactic
        "Category_Weight": 0.10,
    },
    "Wrong Structure": {
        "Class": "Syntactic Shift",
        "Subtype": "Structural Error",
        "Definition": "Use of a grammatically incorrect sentence construction.",
        "Severity_Level": "Major",
        "Severity_Score": 3,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.20,  # 20% within Syntactic
        "Category_Weight": 0.10,
    },
    "Wrong Word Order": {
        "Class": "Syntactic Shift",
        "Subtype": "Structural Error",
        "Definition": "Words are arranged in an incorrect or unnatural order.",
        "Severity_Level": "Major",
        "Severity_Score": 3,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.20,  # 20% within Syntactic
        "Category_Weight": 0.10,
    },
    "Noun to Adjective": {
        "Class": "Syntactic Shift",
        "Subtype": "Structural Error",
        "Definition": "A noun is translated as an adjective.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.025,  # 2.5% within Syntactic
        "Category_Weight": 0.10,
    },
    "Adjective to Noun": {
        "Class": "Syntactic Shift",
        "Subtype": "Structural Error",
        "Definition": "An adjective is translated as a noun.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Fluency",
        "Weight_WGT": 0.025,  # 2.5% within Syntactic
        "Category_Weight": 0.10,
    },
    "Active to Passive Voice": {
        "Class": "Syntactic Shift",
        "Subtype": "Voice Shift",
        "Definition": "A change in voice that may alter focus or style.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Style & Register",
        "Weight_WGT": 0.025,  # 2.5% within Syntactic
        "Category_Weight": 0.10,
    },
    "Passive to Active Voice": {
        "Class": "Syntactic Shift",
        "Subtype": "Voice Shift",
        "Definition": "A change in voice that may alter focus or style.",
        "Severity_Level": "Moderate",
        "Severity_Score": 2,
        "TQA_Category": "Style & Register",
        "Weight_WGT": 0.025,  # 2.5% within Syntactic
        "Category_Weight": 0.10,
    },

    # =========================================================================
    # 4. PRAGMATIC DIVERGENCE (35% of total)
    # =========================================================================
    "Register Mismatch": {
        "Class": "Pragmatic Divergence",
        "Subtype": "Register Mismatch",
        "Definition": "Using an informal expression in a formal text, or vice-versa.",
        "Severity_Level": "Major",
        "Severity_Score": 3,
        "TQA_Category": "Style & Register",
        "Weight_WGT": 0.20,  # 20% within Pragmatic
        "Category_Weight": 0.35,
    },
    "Terminology Substitution": {
        "Class": "Pragmatic Divergence",
        "Subtype": "Register Mismatch",
        "Definition": "Wrong domain-specific term.",
        "Severity_Level": "Severe",
        "Severity_Score": 4,
        "TQA_Category": "Terminology",
        "Weight_WGT": 0.80,  # 80% within Pragmatic
        "Category_Weight": 0.35,
    },

    # =========================================================================
    # 5. ORTHOGRAPHIC SHIFTS (5% of total)
    # =========================================================================
    "Spelling Error": {
        "Class": "Orthographic Shifts",
        "Subtype": "Orthographic Issues",
        "Definition": "A plain misspelling.",
        "Severity_Level": "Minor",
        "Severity_Score": 1,
        "TQA_Category": "Fluency",
        "Weight_WGT": 1.00,  # 100% within Orthographic (only class)
        "Category_Weight": 0.05,
    },
}

# Validation
TAXONOMY_23 = list(TAXONOMY_V3.keys())
assert len(TAXONOMY_23) == 23, f"Expected 23 classes, got {len(TAXONOMY_23)}"

print(f"✅ TAXONOMY_V3 loaded: {len(TAXONOMY_V3)} classes")

# =============================================================================
# STEP 1: LOAD RAW LLM OUTPUT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD RAW LLM OUTPUT")
print("=" * 80)

if not Path(INPUT_RAW_CSV).exists():
    raise FileNotFoundError(f"Raw output not found: {INPUT_RAW_CSV}\nRun Cell 17A first!")

raw_df = pd.read_csv(INPUT_RAW_CSV)
print(f"✅ Loaded raw output: {len(raw_df)} rows")
print(f"   Classes in raw: {raw_df['Sub-Subtype'].nunique()}")

# =============================================================================
# STEP 2: VALIDATE SUB-SUBTYPES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: VALIDATE SUB-SUBTYPES")
print("=" * 80)

# Check all Sub-Subtypes are in taxonomy
unknown_classes = set(raw_df['Sub-Subtype'].unique()) - set(TAXONOMY_23)

if unknown_classes:
    print(f"❌ HARD FAIL: Unknown Sub-Subtypes found:")
    for cls in unknown_classes:
        count = len(raw_df[raw_df['Sub-Subtype'] == cls])
        print(f"   - '{cls}' ({count} rows)")
    raise ValueError(f"Unknown Sub-Subtypes: {unknown_classes}")
else:
    print(f"✅ All {raw_df['Sub-Subtype'].nunique()} Sub-Subtypes are valid")

# =============================================================================
# STEP 3: ENRICH WITH TAXONOMY METADATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: ENRICH WITH TAXONOMY METADATA")
print("=" * 80)

# ✅ Safety net: LLM might generate "Named Entity Error" despite prompts
# (It's a natural NLP term, so the model may "correct" it)
NAME_FIXES = {
    'Named Entity Error': 'Name Entity Error',
}

for old_name, new_name in NAME_FIXES.items():
    count = (raw_df['Sub-Subtype'] == old_name).sum()
    if count > 0:
        raw_df['Sub-Subtype'] = raw_df['Sub-Subtype'].replace({old_name: new_name})
        print(f"   ✅ Fixed LLM output: '{old_name}' → '{new_name}' ({count} rows)")

def enrich_row(row: pd.Series) -> pd.Series:
    """Add taxonomy metadata to a row."""
    sub_subtype = row['Sub-Subtype']

    if sub_subtype not in TAXONOMY_V3:
        raise ValueError(f"Unknown Sub-Subtype: {sub_subtype}")

    metadata = TAXONOMY_V3[sub_subtype]

    # Add metadata columns
    row['Class'] = metadata['Class']
    row['Subtype'] = metadata['Subtype']
    row['Definition'] = metadata['Definition']
    row['Severity_Level'] = metadata['Severity_Level']
    row['Severity_Score'] = metadata['Severity_Score']
    row['TQA_Category'] = metadata['TQA_Category']
    row['Weight_WGT'] = metadata['Weight_WGT']

    return row

# Apply enrichment
enriched_df = raw_df.apply(enrich_row, axis=1)
print(f"✅ Enriched {len(enriched_df)} rows with metadata")

# =============================================================================
# STEP 4: VALIDATE ENRICHMENT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: VALIDATE ENRICHMENT")
print("=" * 80)

# Check no missing metadata
metadata_cols = ['Class', 'Subtype', 'Severity_Level', 'Severity_Score', 'TQA_Category']
missing_metadata = []

for col in metadata_cols:
    missing = enriched_df[col].isna().sum()
    if missing > 0:
        missing_metadata.append((col, missing))
        print(f"❌ Missing {col}: {missing} rows")

if missing_metadata:
    raise ValueError(f"Missing metadata after enrichment: {missing_metadata}")
else:
    print(f"✅ All metadata columns populated")

# Show sample
print(f"\n📋 SAMPLE ENRICHED ROW:")
sample_row = enriched_df.iloc[0] if len(enriched_df) > 0 else None
if sample_row is not None:
    print(f"   Sub-Subtype: {sample_row['Sub-Subtype']}")
    print(f"   Class: {sample_row['Class']}")
    print(f"   Subtype: {sample_row['Subtype']}")
    print(f"   Severity: {sample_row['Severity_Level']} ({sample_row['Severity_Score']})")
    print(f"   TQA_Category: {sample_row['TQA_Category']}")

# =============================================================================
# STEP 5: DEFINE FINAL SCHEMA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: ORGANIZE FINAL SCHEMA")
print("=" * 80)

# Final column order (full schema)
FINAL_COLUMNS = [
    # Identifier
    'Sentence_ID',

    # Taxonomy (3 levels)
    'Class',
    'Subtype',
    'Sub-Subtype',

    # Anchors
    'Keyword',
    'Best_Match',

    # Sentences
    'ar',
    'mt_output',
    'en',

    # Metadata (from taxonomy lookup)
    'Definition',
    'Severity_Level',
    'Severity_Score',
    'TQA_Category',
    'Weight_WGT',

    # Generation metadata
    'generation_timestamp',
    'seed_source',
]

# Ensure all columns exist
for col in FINAL_COLUMNS:
    if col not in enriched_df.columns:
        enriched_df[col] = ''

enriched_df = enriched_df[FINAL_COLUMNS]
print(f"✅ Final schema: {len(FINAL_COLUMNS)} columns")

# =============================================================================
# STEP 6: SAVE ENRICHED OUTPUT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6: SAVE ENRICHED OUTPUT")
print("=" * 80)

# Save CSV
enriched_df.to_csv(OUTPUT_ENRICHED_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUTPUT_ENRICHED_CSV} ({len(enriched_df)} rows)")

# Save JSON (grouped by class)
enriched_json = {}
for cls in TAXONOMY_23:
    cls_df = enriched_df[enriched_df['Sub-Subtype'] == cls]
    enriched_json[cls] = cls_df.to_dict('records')

with open(OUTPUT_ENRICHED_JSON, 'w', encoding='utf-8') as f:
    json.dump(enriched_json, f, ensure_ascii=False, indent=2)
print(f"✅ Saved: {OUTPUT_ENRICHED_JSON}")

# Save enrichment log
enrichment_log = {
    'timestamp': datetime.now().isoformat(),
    'input_file': INPUT_RAW_CSV,
    'output_file': OUTPUT_ENRICHED_CSV,
    'rows_processed': len(enriched_df),
    'classes_enriched': enriched_df['Sub-Subtype'].nunique(),
    'metadata_columns_added': metadata_cols,
    'validation': {
        'unknown_classes': list(unknown_classes),
        'missing_metadata': missing_metadata,
    }
}

with open(OUTPUT_ENRICHMENT_LOG, 'w', encoding='utf-8') as f:
    json.dump(enrichment_log, f, indent=2)
print(f"✅ Saved: {OUTPUT_ENRICHMENT_LOG}")

# =============================================================================
# STEP 7: SUMMARY STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 7: SUMMARY STATISTICS")
print("=" * 80)

print(f"\n📊 PER-CLASS BREAKDOWN:")
print(f"{'Class':<35} {'Count':>8} {'Severity':>10} {'TQA':>12}")
print("-" * 70)

for cls in TAXONOMY_23:
    cls_df = enriched_df[enriched_df['Sub-Subtype'] == cls]
    count = len(cls_df)
    if count > 0:
        severity = cls_df['Severity_Level'].iloc[0]
        tqa = cls_df['TQA_Category'].iloc[0]
        print(f"{cls:<35} {count:>8} {severity:>10} {tqa:>12}")

print("-" * 70)
print(f"{'TOTAL':<35} {len(enriched_df):>8}")

print(f"\n📊 BY SEVERITY:")
for severity in ['Critical', 'Severe', 'Major', 'Moderate', 'Minor']:
    count = len(enriched_df[enriched_df['Severity_Level'] == severity])
    if count > 0:
        print(f"   {severity}: {count}")

print(f"\n📊 BY TQA CATEGORY:")
for tqa in ['Accuracy', 'Fluency', 'Style']:
    count = len(enriched_df[enriched_df['TQA_Category'] == tqa])
    if count > 0:
        print(f"   {tqa}: {count}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 17B COMPLETE (Deterministic Enrichment)")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   - {OUTPUT_ENRICHED_CSV} ({len(enriched_df)} samples, full schema)
   - {OUTPUT_ENRICHED_JSON}
   - {OUTPUT_ENRICHMENT_LOG}

📊 ENRICHMENT SUMMARY:
   Rows processed:       {len(enriched_df)}
   Classes covered:      {enriched_df['Sub-Subtype'].nunique()} / {len(TAXONOMY_23)}
   Metadata columns:     {len(metadata_cols)} added

📋 SCHEMA:
   LLM-generated:        Keyword, Best_Match, ar, mt_output, en
   Taxonomy-lookup:      Class, Subtype, Definition, Severity_*, TQA_Category, Weight_WGT
   Auto-generated:       Sentence_ID, generation_timestamp, seed_source

📋 PAPER-READY STATEMENT:
   "Tier2 samples were generated by the LLM with minimal output fields
   (Keyword, Best_Match, ar, mt_output, en). Taxonomy metadata including
   severity levels, TQA categories, and class hierarchy was deterministically
   joined from the static taxonomy table (V3), ensuring consistent and
   reproducible metadata across all generated samples."

✅ Tier2 generation complete! Ready for ETCA validation or downstream use.
""")

In [ ]:
import pandas as pd

df = pd.read_csv("tier1_v3_3_final/few_shot_bank_tier2.csv")

# Find error column
error_col = 'error_type' if 'error_type' in df.columns else 'Sub-Subtype'

print(f"Total samples: {len(df)}")
print(f"\nSamples per class:")
for cls, count in df[error_col].value_counts().sort_index().items():
    marker = "✅ (5)" if count >= 5 else f"({count})"
    print(f"   {cls}: {count} {marker}")

# Show source breakdown if available
if 'seed_source' in df.columns:
    print(f"\nBy source:")
    print(df['seed_source'].value_counts())

In [ ]:
# from openai import OpenAI
# client = OpenAI()

# # Test the model
# try:
#     response = client.chat.completions.create(
#         model="gpt-4.1-2025-04-14",  # Test this model
#         messages=[{"role": "user", "content": "Say 'hello' in Arabic"}],
#         max_tokens=20
#     )
#     print(f"✅ Model works: {response.choices[0].message.content}")
# except Exception as e:
#     print(f"❌ Error: {e}")
#     #model="",  # or gpt-4o


In [ ]:
# =============================================================================
# CELL 18: TIER1 VALIDATION VISUALIZATIONS
# =============================================================================
"""
CELL 18: Visualizations for Tier1 Validation Results

Creates publication-ready charts and graphs:
1. Validator agreement distribution (pie/bar)
2. Per-class acceptance rates (horizontal bar)
3. Score distributions (histograms/box plots)
4. ETCA audit results (if available)
5. Pipeline flow Sankey diagram

INPUTS:
- tier1_v3_3_final/tier1_clean_A_FROZEN.csv
- tier1_v3_3_final/tier2_seed_audit_cell15.csv (optional)
- tier1_v3_3_final/cell15_audit_stats.json (optional)

OUTPUTS:
- tier1_v3_3_final/figures/ (PNG files)
- tier1_v3_3_final/visualization_summary.json
"""

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("=" * 80)
print("CELL 18: TIER1 VALIDATION VISUALIZATIONS")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"
FIGURES_DIR = f"{OUTPUT_DIR}/figures"

# Create figures directory
os.makedirs(FIGURES_DIR, exist_ok=True)

# Input files
CLEAN_A_PATH = f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv"
CLEAN_B_PATH = f"{OUTPUT_DIR}/tier1_clean_B_FROZEN.csv"
NOISY_PATH = f"{OUTPUT_DIR}/tier1_noisy.csv"
ETCA_AUDIT_PATH = f"{OUTPUT_DIR}/tier2_seed_audit_cell15.csv"
ETCA_STATS_PATH = f"{OUTPUT_DIR}/cell15_audit_stats.json"
VALIDATION_STATS_PATH = f"{OUTPUT_DIR}/validation_stats.json"

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

# Load Clean_A
clean_a = None
if Path(CLEAN_A_PATH).exists():
    clean_a = pd.read_csv(CLEAN_A_PATH)
    print(f"✅ Loaded Clean_A: {len(clean_a):,} rows")

# Load Clean_B
clean_b = None
if Path(CLEAN_B_PATH).exists():
    clean_b = pd.read_csv(CLEAN_B_PATH)
    print(f"✅ Loaded Clean_B: {len(clean_b):,} rows")

# Load Noisy
noisy = None
if Path(NOISY_PATH).exists():
    noisy = pd.read_csv(NOISY_PATH)
    print(f"✅ Loaded Noisy: {len(noisy):,} rows")

# Load ETCA audit results
etca_audit = None
if Path(ETCA_AUDIT_PATH).exists():
    etca_audit = pd.read_csv(ETCA_AUDIT_PATH)
    print(f"✅ Loaded ETCA audit: {len(etca_audit):,} rows")

# Load stats
validation_stats = None
if Path(VALIDATION_STATS_PATH).exists():
    with open(VALIDATION_STATS_PATH, 'r') as f:
        validation_stats = json.load(f)
    print(f"✅ Loaded validation stats")

etca_stats = None
if Path(ETCA_STATS_PATH).exists():
    with open(ETCA_STATS_PATH, 'r') as f:
        etca_stats = json.load(f)
    print(f"✅ Loaded ETCA stats")

# =============================================================================
# FIGURE 1: Pipeline Flow (Funnel Chart)
# =============================================================================

print("\n" + "=" * 80)
print("FIGURE 1: Pipeline Flow")
print("=" * 80)

fig, ax = plt.subplots(figsize=(10, 6))

# Data for funnel
stages = ['Candidates\n(Input)', 'Stage A\n(OCS=1.0)', 'Clean_A\n(CPS≥0.65)', 'ETCA\n(Recommended)']
counts = [
    2770,  # Original candidates
    len(clean_a) + len(clean_b) + (len(noisy) if noisy is not None else 0) if clean_a is not None else 0,
    len(clean_a) if clean_a is not None else 0,
    len(etca_audit[etca_audit['tier2_seed_recommendation_final'] == 1]) if etca_audit is not None and 'tier2_seed_recommendation_final' in etca_audit.columns else 0
]

# Filter out zero stages
valid_stages = [(s, c) for s, c in zip(stages, counts) if c > 0]
stages = [s for s, c in valid_stages]
counts = [c for s, c in valid_stages]

colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(stages)))

bars = ax.barh(stages, counts, color=colors, edgecolor='black', linewidth=1.5)

# Add count labels
for bar, count in zip(bars, counts):
    width = bar.get_width()
    ax.text(width + 50, bar.get_y() + bar.get_height()/2,
            f'{count:,}', ha='left', va='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Number of Samples', fontsize=12)
ax.set_title('TRIVET Pipeline: Sample Flow', fontsize=14, fontweight='bold')
ax.invert_yaxis()  # Largest at top

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig1_pipeline_flow.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ Saved: {FIGURES_DIR}/fig1_pipeline_flow.png")

# =============================================================================
# FIGURE 2: Per-Class Distribution (Clean_A)
# =============================================================================

print("\n" + "=" * 80)
print("FIGURE 2: Per-Class Distribution")
print("=" * 80)

if clean_a is not None and 'Sub-Subtype' in clean_a.columns:
    fig, ax = plt.subplots(figsize=(12, 10))

    class_counts = clean_a['Sub-Subtype'].value_counts()

    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(class_counts)))

    bars = ax.barh(class_counts.index, class_counts.values, color=colors, edgecolor='black')

    # Add count labels
    for bar, count in zip(bars, class_counts.values):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f'{count}', ha='left', va='center', fontsize=10)

    ax.set_xlabel('Number of Samples', fontsize=12)
    ax.set_title('Clean_A Samples by Error Type', fontsize=14, fontweight='bold')
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/fig2_class_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {FIGURES_DIR}/fig2_class_distribution.png")

# =============================================================================
# FIGURE 3: ETCA Audit Score Distributions
# =============================================================================

print("\n" + "=" * 80)
print("FIGURE 3: ETCA Score Distributions")
print("=" * 80)

if etca_audit is not None:
    score_cols = ['anchor_validity', 'phenomenon_clarity', 'arabic_naturalness', 'collateral_severity']
    available_cols = [c for c in score_cols if c in etca_audit.columns]

    if available_cols:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for i, col in enumerate(available_cols):
            ax = axes[i]
            data = etca_audit[col].dropna()

            # Count per score
            score_counts = data.value_counts().sort_index()

            colors = ['#d73027', '#fc8d59', '#fee08b', '#91cf60', '#1a9850']  # Red to Green
            bar_colors = [colors[int(s)-1] for s in score_counts.index]

            ax.bar(score_counts.index, score_counts.values, color=bar_colors, edgecolor='black')
            ax.set_xlabel('Score (1-5)')
            ax.set_ylabel('Count')
            ax.set_title(col.replace('_', ' ').title())
            ax.set_xticks([1, 2, 3, 4, 5])

            # Add mean line
            mean_val = data.mean()
            ax.axvline(x=mean_val, color='black', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
            ax.legend()

        plt.suptitle('ETCA Audit Score Distributions', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{FIGURES_DIR}/fig3_etca_scores.png", dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✅ Saved: {FIGURES_DIR}/fig3_etca_scores.png")

# =============================================================================
# FIGURE 4: ETCA Recommendation by Class
# =============================================================================

print("\n" + "=" * 80)
print("FIGURE 4: ETCA Recommendation by Class")
print("=" * 80)

if etca_audit is not None and 'error_label' in etca_audit.columns and 'tier2_seed_recommendation_final' in etca_audit.columns:
    fig, ax = plt.subplots(figsize=(12, 10))

    # Calculate per-class recommendation rates
    class_stats = etca_audit.groupby('error_label').agg({
        'tier2_seed_recommendation_final': ['sum', 'count']
    })
    class_stats.columns = ['recommended', 'total']
    class_stats['rate'] = class_stats['recommended'] / class_stats['total'] * 100
    class_stats = class_stats.sort_values('rate', ascending=True)

    # Color by rate
    colors = plt.cm.RdYlGn(class_stats['rate'] / 100)

    bars = ax.barh(class_stats.index, class_stats['rate'], color=colors, edgecolor='black')

    # Add labels
    for bar, (idx, row) in zip(bars, class_stats.iterrows()):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f"{row['rate']:.1f}% ({int(row['recommended'])}/{int(row['total'])})",
                ha='left', va='center', fontsize=9)

    ax.set_xlabel('Recommendation Rate (%)', fontsize=12)
    ax.set_title('ETCA Recommendation Rate by Error Type', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 110)
    ax.axvline(x=80, color='green', linestyle='--', alpha=0.5, label='80% threshold')
    ax.legend()

    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/fig4_etca_by_class.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {FIGURES_DIR}/fig4_etca_by_class.png")

# =============================================================================
# FIGURE 5: Quality Tier Distribution (Pie Chart)
# =============================================================================

print("\n" + "=" * 80)
print("FIGURE 5: Quality Tier Distribution")
print("=" * 80)

fig, ax = plt.subplots(figsize=(8, 8))

tier_counts = {
    'Clean_A': len(clean_a) if clean_a is not None else 0,
    'Clean_B': len(clean_b) if clean_b is not None else 0,
    'Noisy': len(noisy) if noisy is not None else 0,
}

# Filter non-zero
tier_counts = {k: v for k, v in tier_counts.items() if v > 0}

if tier_counts:
    colors = {'Clean_A': '#2ecc71', 'Clean_B': '#f39c12', 'Noisy': '#e74c3c'}
    pie_colors = [colors.get(k, 'gray') for k in tier_counts.keys()]

    wedges, texts, autotexts = ax.pie(
        tier_counts.values(),
        labels=tier_counts.keys(),
        autopct='%1.1f%%',
        colors=pie_colors,
        explode=[0.05] * len(tier_counts),
        shadow=True,
        startangle=90
    )

    # Add counts
    for i, (key, val) in enumerate(tier_counts.items()):
        autotexts[i].set_text(f'{val:,}\n({autotexts[i].get_text()})')

    ax.set_title('Stage A Output: Quality Tier Distribution', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/fig5_quality_tiers.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {FIGURES_DIR}/fig5_quality_tiers.png")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 18 COMPLETE")
print("=" * 80)

# List generated figures
figures = list(Path(FIGURES_DIR).glob("*.png"))
print(f"\n📁 Generated {len(figures)} figures in {FIGURES_DIR}/:")
for fig_path in sorted(figures):
    print(f"   - {fig_path.name}")

# Save summary
summary = {
    'figures_generated': [f.name for f in figures],
    'data_sources': {
        'clean_a': len(clean_a) if clean_a is not None else 0,
        'clean_b': len(clean_b) if clean_b is not None else 0,
        'noisy': len(noisy) if noisy is not None else 0,
        'etca_audit': len(etca_audit) if etca_audit is not None else 0,
    }
}

with open(f"{OUTPUT_DIR}/visualization_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\n✅ Saved: {OUTPUT_DIR}/visualization_summary.json")

print("\n✅ Ready for Cell 19: Human Evaluation Templates")

In [ ]:
# # =============================================================================
# # CELL 18 B: TIER2 EVALUATION SAMPLING (Research-Grade)
# # =============================================================================
# """
# PURPOSE:
# Create evaluation samples from Tier2 LLM-generated data for human review.
# Ensures 100% class coverage with stratified sampling.

# IMPROVEMENTS OVER CELL 15.1:
# 1. Error Type Validation - annotator verifies Sub-Subtype is correct
# 2. Severity Validation - annotator confirms severity rating
# 3. Comprehension Impact - does error affect understanding?
# 4. Confidence Score - flags uncertain cases for adjudication
# 5. Annotator ID - enables inter-annotator agreement (Cohen's Kappa)
# 6. Source Tier tracking - distinguishes Tier1 vs Tier2

# OUTPUTS:
# - TIER2_HUMANEVAL_SAMPLE.csv (evaluation sheet for annotators)
# - TIER2_HUMANEVAL_METRICS_HIDDEN.csv (metrics for post-analysis)
# - TIER2_ANNOTATOR_MANUAL.md (comprehensive instructions)
# - TIER2_EVALUATION_STATS.json (sampling statistics)
# """

# import pandas as pd
# import numpy as np
# import json
# from pathlib import Path
# from datetime import datetime

# print("=" * 80)
# print("CELL 18: TIER2 EVALUATION SAMPLING (Research-Grade)")
# print("=" * 80)

# # =============================================================================
# # CONFIGURATION
# # =============================================================================

# # Input/Output paths
# INPUT_FILE = "tier2_generated_enriched.csv"  # From Cell 17B
# OUTPUT_DIR = "tier2_evaluation"

# # Alternative input paths (fallback)
# ALT_INPUTS = [
#     "tier1_v3_3_final/tier2_generated_enriched.csv",
#     "tier2_generated_enriched_CORRECTED.csv",
# ]

# # Sampling configuration
# SAMPLES_PER_CLASS = 1  # Minimum 1 per class for 100% coverage
# RANDOM_SEED = 42

# # Taxonomy (23 classes)
# TAXONOMY_23 = [
#     "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
#     "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
#     "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
#     "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
#     "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
#     "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
#     "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
# ]

# # Omission classes (special OCS handling)
# OMISSION_CLASSES = {"Total Omission", "Partial Translation"}

# np.random.seed(RANDOM_SEED)
# Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# print(f"\n📊 CONFIGURATION:")
# print(f"   Samples per class: {SAMPLES_PER_CLASS} (minimum)")
# print(f"   Random seed: {RANDOM_SEED}")

# # =============================================================================
# # STEP 1: LOAD TIER2 DATA
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 1: LOAD TIER2 GENERATED DATA")
# print("=" * 80)

# df_tier2 = None
# input_path_used = None

# for path in [INPUT_FILE] + ALT_INPUTS:
#     if Path(path).exists():
#         df_tier2 = pd.read_csv(path)
#         input_path_used = path
#         print(f"✅ Loaded: {path} ({len(df_tier2)} rows)")
#         break

# if df_tier2 is None:
#     raise FileNotFoundError(f"Could not find Tier2 data. Tried: {[INPUT_FILE] + ALT_INPUTS}")

# print(f"   Classes: {df_tier2['Sub-Subtype'].nunique()}")

# # =============================================================================
# # STEP 2: STRATIFIED SAMPLING (100% Class Coverage)
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 2: STRATIFIED SAMPLING")
# print("=" * 80)

# sampled_dfs = []
# sampling_stats = {}

# for cls in TAXONOMY_23:
#     cls_df = df_tier2[df_tier2['Sub-Subtype'] == cls]
#     n_available = len(cls_df)

#     if n_available == 0:
#         print(f"   ⚠️ {cls}: 0 available")
#         sampling_stats[cls] = {'available': 0, 'sampled': 0}
#         continue

#     n_sample = min(SAMPLES_PER_CLASS, n_available)
#     sampled = cls_df.sample(n=n_sample, random_state=RANDOM_SEED)
#     sampled_dfs.append(sampled)

#     sampling_stats[cls] = {'available': n_available, 'sampled': n_sample}
#     print(f"   ✅ {cls}: {n_sample}/{n_available}")

# # Combine and shuffle
# df_sample = pd.concat(sampled_dfs, ignore_index=True)
# df_sample = df_sample.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# print(f"\n📊 SAMPLING SUMMARY:")
# print(f"   Total sampled: {len(df_sample)}")
# print(f"   Classes covered: {df_sample['Sub-Subtype'].nunique()}/23")

# # =============================================================================
# # STEP 3: CREATE EVALUATION SHEET (RESEARCH-GRADE FORMAT)
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 3: CREATE EVALUATION SHEET (Research-Grade Format)")
# print("=" * 80)

# # Build evaluation dataframe with improved columns
# eval_rows = []

# for idx, row in df_sample.iterrows():
#     is_omission = row.get('Sub-Subtype', '') in OMISSION_CLASSES

#     eval_row = {
#         # === IDENTIFICATION ===
#         'eval_id': f"T2_{idx+1:03d}",
#         'annotator_id': '',  # Annotator fills this
#         'source_tier': 'Tier2',  # Distinguishes from Tier1

#         # === CONTENT (for evaluation) ===
#         'en': row.get('en', ''),
#         'ar': row.get('ar', ''),
#         'mt_output': row.get('mt_output', ''),
#         'Keyword': row.get('Keyword', ''),
#         'Best_Match': row.get('Best_Match', ''),
#         'Sub_Subtype': row.get('Sub-Subtype', ''),  # Shown to annotator

#         # === SECTION A: ANCHOR VALIDATION (OCS) ===
#         'A1_kw_in_ar': '',           # Is Keyword in ar? (1/0)
#         'A2_kw_count': '',           # How many times? (0/1/2+)
#         'A3_bm_in_mt': '',           # Is Best_Match in mt_output? (1/0/OMIT)
#         'A4_bm_count': '',           # How many times? (0/1/2+/NA)
#         'A5_anchor_aligned': '',     # Same semantic slot? (1/0)

#         # === SECTION B: ERROR TYPE VALIDATION (NEW!) ===
#         'B1_error_type_correct': '', # Is labeled Sub_Subtype correct? (1/0)
#         'B2_suggested_type': '',     # If wrong, what should it be?
#         'B3_severity_appropriate': '',# Is severity rating appropriate? (1/0)

#         # === SECTION C: QUALITY ASSESSMENT ===
#         'C1_phenomenon_clear': '',   # Is error clearly visible? (1/0)
#         'C2_ar_natural': '',         # Is ar natural Arabic? (1/0)
#         'C3_mt_plausible': '',       # Is mt_output plausible Arabic? (1/0)
#         'C4_no_collateral': '',      # No extra errors? (1/0)
#         'C5_comprehension_impact': '',# Does error affect understanding? (1/0)

#         # === SECTION D: FINAL DECISION ===
#         'D1_accept': '',             # Accept into dataset? (1/0)
#         'D2_confidence': '',         # Confidence level (1=low/2=med/3=high)
#         'D3_needs_review': '',       # Flag for adjudication? (1/0)
#         'D4_notes': '',              # Free text comments
#     }
#     eval_rows.append(eval_row)

# # Define column order
# EVAL_COLUMNS = [
#     # Identification
#     'eval_id', 'annotator_id', 'source_tier',
#     # Content
#     'en', 'ar', 'mt_output', 'Keyword', 'Best_Match', 'Sub_Subtype',
#     # Section A: Anchor/OCS
#     'A1_kw_in_ar', 'A2_kw_count', 'A3_bm_in_mt', 'A4_bm_count', 'A5_anchor_aligned',
#     # Section B: Error Type
#     'B1_error_type_correct', 'B2_suggested_type', 'B3_severity_appropriate',
#     # Section C: Quality
#     'C1_phenomenon_clear', 'C2_ar_natural', 'C3_mt_plausible', 'C4_no_collateral', 'C5_comprehension_impact',
#     # Section D: Decision
#     'D1_accept', 'D2_confidence', 'D3_needs_review', 'D4_notes',
# ]

# df_eval = pd.DataFrame(eval_rows, columns=EVAL_COLUMNS)

# # Save evaluation sheet
# eval_path = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_SAMPLE.csv"
# df_eval.to_csv(eval_path, index=False, encoding='utf-8-sig')
# print(f"✅ Saved: {eval_path} ({len(df_eval)} samples)")

# # =============================================================================
# # STEP 4: CREATE HIDDEN METRICS FILE
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 4: CREATE HIDDEN METRICS FILE")
# print("=" * 80)

# metrics_rows = []
# for idx, row in df_sample.iterrows():
#     metrics_row = {
#         'eval_id': f"T2_{idx+1:03d}",
#         'Sentence_ID': row.get('Sentence_ID', ''),
#         'Sub-Subtype': row.get('Sub-Subtype', ''),
#         'Class': row.get('Class', ''),
#         'Subtype': row.get('Subtype', ''),
#         'Severity_Level': row.get('Severity_Level', ''),
#         'Severity_Score': row.get('Severity_Score', ''),
#         'TQA_Category': row.get('TQA_Category', ''),
#         'Weight_WGT': row.get('Weight_WGT', ''),
#         'seed_source': row.get('seed_source', ''),
#         'Definition': row.get('Definition', ''),
#     }
#     metrics_rows.append(metrics_row)

# df_metrics = pd.DataFrame(metrics_rows)
# metrics_path = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_METRICS_HIDDEN.csv"
# df_metrics.to_csv(metrics_path, index=False)
# print(f"✅ Saved: {metrics_path}")
# print(f"   ⚠️ DO NOT share with annotators!")

# # =============================================================================
# # STEP 5: SAVE STATISTICS
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 5: SAVE STATISTICS")
# print("=" * 80)

# stats = {
#     'metadata': {
#         'generated_at': datetime.now().isoformat(),
#         'input_file': input_path_used,
#         'random_seed': RANDOM_SEED,
#         'format_version': '2.0_research_grade',
#     },
#     'source_data': {
#         'total_rows': len(df_tier2),
#         'unique_classes': df_tier2['Sub-Subtype'].nunique(),
#     },
#     'sampled_data': {
#         'total_sampled': len(df_sample),
#         'classes_covered': df_sample['Sub-Subtype'].nunique(),
#         'sampling_rate': f"{len(df_sample)/len(df_tier2)*100:.1f}%",
#     },
#     'per_class': sampling_stats,
#     'evaluation_schema': {
#         'section_a_anchor': ['A1_kw_in_ar', 'A2_kw_count', 'A3_bm_in_mt', 'A4_bm_count', 'A5_anchor_aligned'],
#         'section_b_error_type': ['B1_error_type_correct', 'B2_suggested_type', 'B3_severity_appropriate'],
#         'section_c_quality': ['C1_phenomenon_clear', 'C2_ar_natural', 'C3_mt_plausible', 'C4_no_collateral', 'C5_comprehension_impact'],
#         'section_d_decision': ['D1_accept', 'D2_confidence', 'D3_needs_review', 'D4_notes'],
#     },
#     'improvements_over_v1': [
#         'Added B1_error_type_correct for taxonomy validation',
#         'Added B2_suggested_type for label correction',
#         'Added B3_severity_appropriate for severity validation',
#         'Added C3_mt_plausible for MT quality assessment',
#         'Added C5_comprehension_impact for error significance',
#         'Added D2_confidence for annotator certainty',
#         'Added D3_needs_review for adjudication flagging',
#         'Added annotator_id for inter-annotator agreement',
#         'Added source_tier for tier-specific analysis',
#         'Prefixed columns with section letters for clarity',
#     ]
# }

# stats_path = f"{OUTPUT_DIR}/TIER2_EVALUATION_STATS.json"
# with open(stats_path, 'w', encoding='utf-8') as f:
#     json.dump(stats, f, indent=2, ensure_ascii=False)
# print(f"✅ Saved: {stats_path}")

# # =============================================================================
# # STEP 6: CREATE ANNOTATOR MANUAL (COMPREHENSIVE)
# # =============================================================================

# print("\n" + "=" * 80)
# print("STEP 6: CREATE ANNOTATOR MANUAL")
# print("=" * 80)

# manual_content = '''# TRIVET Tier2 Manual Evaluation — Annotator Guide (v2.0)

# ---

# ## 📋 Overview

# You are evaluating **LLM-generated Arabic minimal pairs** for translation quality assessment research.

# **Minimal pair** = Two sentences that differ in exactly ONE aspect:
# - `ar` = Clean Arabic sentence (correct translation)
# - `mt_output` = Arabic with an intentional translation error

# Your task: **Verify that each sample is valid and correctly labeled** for use in MT evaluation research.

# ---

# ## 🚀 Before You Start

# 1. **Enter your annotator ID** in the `annotator_id` column (use your initials: e.g., "AK", "MH")
# 2. **Read this entire guide** before annotating any samples
# 3. **Use the exact values specified** (1, 0, OMIT, NA, etc.)
# 4. **Fill ALL columns** — empty cells will be flagged as incomplete
# 5. **Work carefully** — your annotations will be used for research publication

# ---

# ## 📊 Data Columns (Read-Only)

# | Column | Description |
# |--------|-------------|
# | `eval_id` | Unique identifier (T2_001, T2_002, ...) |
# | `source_tier` | Always "Tier2" for this dataset |
# | `en` | English source sentence |
# | `ar` | Clean Arabic sentence (correct) |
# | `mt_output` | Arabic MT output with intentional error |
# | `Keyword` | Phrase in `ar` that should change |
# | `Best_Match` | Corresponding phrase in `mt_output` (or [OMITTED]) |
# | `Sub_Subtype` | Labeled error type (one of 23 categories) |

# ---

# ## ✏️ Evaluation Columns

# ### Section A: Anchor Validation (OCS)

# These columns verify that the Keyword/Best_Match anchors are correctly placed.

# | Column | Question | Values | Instructions |
# |--------|----------|--------|--------------|
# | `A1_kw_in_ar` | Is Keyword present in `ar`? | 1 / 0 | Check for exact substring match |
# | `A2_kw_count` | How many times does Keyword appear in `ar`? | 0 / 1 / 2+ | Should be exactly 1 |
# | `A3_bm_in_mt` | Is Best_Match present in `mt_output`? | 1 / 0 / OMIT | Use **OMIT** for omission classes |
# | `A4_bm_count` | How many times does Best_Match appear in `mt_output`? | 0 / 1 / 2+ / NA | Use **NA** for omission classes |
# | `A5_anchor_aligned` | Do Keyword and Best_Match represent the same semantic slot? | 1 / 0 | Should refer to the same concept |

# **⚠️ Special Cases — Omission Classes:**
# - For `Total Omission` and `Partial Translation`:
#   - Set `A3_bm_in_mt = OMIT`
#   - Set `A4_bm_count = NA`
#   - The Best_Match field will show `[OMITTED]` or similar

# ---

# ### Section B: Error Type Validation ⭐ NEW

# These columns verify that the labeled error type is correct. **This is critical for taxonomy validation.**

# | Column | Question | Values | Instructions |
# |--------|----------|--------|--------------|
# | `B1_error_type_correct` | Is the labeled `Sub_Subtype` actually the error present? | 1 / 0 | Compare actual difference to label |
# | `B2_suggested_type` | If wrong, what should it be? | text / blank | Leave blank if B1=1; otherwise specify correct type |
# | `B3_severity_appropriate` | Is the error severity appropriate for this type? | 1 / 0 | Critical > Severe > Major > Moderate > Minor |

# **Why this matters:** A sample can have correct anchors but a wrong label. For example:
# - Labeled as "Meaning Shift" but it's actually "Terminology Substitution"
# - Labeled as "Gender Disagreement" but it's actually "Definiteness Shift"

# ---

# ### Section C: Quality Assessment

# These columns assess the overall quality of the minimal pair.

# | Column | Question | Values | Instructions |
# |--------|----------|--------|--------------|
# | `C1_phenomenon_clear` | Is the error clearly visible when comparing `ar` and `mt_output`? | 1 / 0 | Should be obvious to a native speaker |
# | `C2_ar_natural` | Is `ar` natural, grammatically correct Arabic? | 1 / 0 | Ignore the error; assess the clean sentence |
# | `C3_mt_plausible` | Is `mt_output` plausible Arabic (could a real MT system produce this)? | 1 / 0 | Ignoring the intentional error, is it realistic? |
# | `C4_no_collateral` | Is there NO other error besides the labeled one? | 1 / 0 | 1=clean (only labeled error), 0=has extra errors |
# | `C5_comprehension_impact` | Does this error affect comprehension/understanding? | 1 / 0 | 1=yes (reader would be confused), 0=no (cosmetic) |

# ---

# ### Section D: Final Decision

# | Column | Question | Values | Instructions |
# |--------|----------|--------|--------------|
# | `D1_accept` | Should this sample be accepted into the final dataset? | 1 / 0 | Final judgment after all checks |
# | `D2_confidence` | How confident are you in this decision? | 1 / 2 / 3 | 1=Low, 2=Medium, 3=High |
# | `D3_needs_review` | Should this be reviewed by another annotator? | 1 / 0 | Flag ambiguous or unclear cases |
# | `D4_notes` | Any comments, observations, or explanations | text | Optional but highly encouraged for rejections |

# ---

# ## ✅ Decision Logic: When to Accept

# **ACCEPT (D1_accept = 1)** if ALL of the following are true:

# | Check | Column | Required Value |
# |-------|--------|----------------|
# | Keyword is in ar | A1_kw_in_ar | = 1 |
# | Keyword appears exactly once | A2_kw_count | = 1 |
# | Anchor is aligned | A5_anchor_aligned | = 1 |
# | Error type is correct | B1_error_type_correct | = 1 |
# | Error is clearly visible | C1_phenomenon_clear | = 1 |
# | Clean Arabic is natural | C2_ar_natural | = 1 |
# | MT output is plausible | C3_mt_plausible | = 1 |
# | No collateral errors | C4_no_collateral | = 1 |

# ---

# ## ❌ Decision Logic: When to Reject

# **REJECT (D1_accept = 0)** if ANY of the following are true:

# | Reason | Evidence |
# |--------|----------|
# | Keyword missing from ar | A1_kw_in_ar = 0 |
# | Keyword appears multiple times | A2_kw_count = 2+ |
# | Anchor misaligned | A5_anchor_aligned = 0 |
# | Wrong error type label | B1_error_type_correct = 0 |
# | Error is unclear/invisible | C1_phenomenon_clear = 0 |
# | Arabic is unnatural | C2_ar_natural = 0 |
# | MT output is implausible | C3_mt_plausible = 0 |
# | Has collateral errors | C4_no_collateral = 0 |

# **Always explain rejections in D4_notes!**

# ---

# ## 📚 Examples

# ### ✅ Example 1: Good Sample (Accept)

# ```
# en: The student went to school.
# ar: ذهب الطالب إلى المدرسة.
# mt_output: ذهبت الطالبة إلى المدرسة.
# Keyword: الطالب
# Best_Match: الطالبة
# Sub_Subtype: Gender Disagreement
# ```

# **Annotation:**
# | Column | Value | Reason |
# |--------|-------|--------|
# | A1_kw_in_ar | 1 | "الطالب" is in ar |
# | A2_kw_count | 1 | Appears exactly once |
# | A3_bm_in_mt | 1 | "الطالبة" is in mt_output |
# | A4_bm_count | 1 | Appears exactly once |
# | A5_anchor_aligned | 1 | Both mean "the student" |
# | B1_error_type_correct | 1 | الطالب→الطالبة IS gender change |
# | B2_suggested_type | | (blank - correct label) |
# | B3_severity_appropriate | 1 | Major severity is appropriate |
# | C1_phenomenon_clear | 1 | Gender change is obvious |
# | C2_ar_natural | 1 | Clean sentence is natural |
# | C3_mt_plausible | 1 | MT could produce this |
# | C4_no_collateral | 1 | Only gender changed (verb agreement is expected) |
# | C5_comprehension_impact | 0 | Meaning is still clear |
# | **D1_accept** | **1** | ✅ All checks pass |
# | D2_confidence | 3 | Very clear case |
# | D3_needs_review | 0 | No ambiguity |
# | D4_notes | | |

# ---

# ### ❌ Example 2: Wrong Error Type (Reject)

# ```
# en: The committee approved the proposal.
# ar: وافقت اللجنة على المقترح.
# mt_output: وافقت اللجنة على الاقتراح.
# Keyword: المقترح
# Best_Match: الاقتراح
# Sub_Subtype: Meaning Shift  ← WRONG!
# ```

# **Annotation:**
# | Column | Value | Reason |
# |--------|-------|--------|
# | A1_kw_in_ar | 1 | ✓ |
# | A2_kw_count | 1 | ✓ |
# | A3_bm_in_mt | 1 | ✓ |
# | A4_bm_count | 1 | ✓ |
# | A5_anchor_aligned | 1 | ✓ |
# | **B1_error_type_correct** | **0** | المقترح/الاقتراح are **synonyms**, NOT meaning shift! |
# | B2_suggested_type | Terminology Substitution | Or possibly "No Error" |
# | B3_severity_appropriate | 0 | Critical severity is wrong for this |
# | C1_phenomenon_clear | 0 | "Error" is debatable |
# | C2_ar_natural | 1 | ✓ |
# | C3_mt_plausible | 1 | ✓ |
# | C4_no_collateral | 1 | ✓ |
# | C5_comprehension_impact | 0 | No comprehension impact |
# | **D1_accept** | **0** | ❌ Wrong error type |
# | D2_confidence | 2 | Somewhat clear |
# | D3_needs_review | 1 | Flag for discussion |
# | D4_notes | المقترح and الاقتراح are near-synonyms in UN context |

# ---

# ### ❌ Example 3: Collateral Damage (Reject)

# ```
# ar: ذهب الطالب إلى المدرسة الكبيرة.
# mt_output: ذهبت الطالبة إلى مدرسة صغيرة.
# Sub_Subtype: Gender Disagreement
# ```

# **Problem:** Multiple changes occurred!
# - Gender: الطالب → الطالبة ✓ (labeled error)
# - Definiteness: المدرسة → مدرسة ✗ (collateral)
# - Adjective: الكبيرة → صغيرة ✗ (collateral)

# **Annotation:**
# | Column | Value |
# |--------|-------|
# | C4_no_collateral | 0 |
# | **D1_accept** | **0** |
# | D4_notes | Collateral errors: definiteness change + adjective change |

# ---

# ### ✅ Example 4: Omission Class (Accept)

# ```
# ar: أكد الأمين العام أهمية التعاون الدولي في مكافحة الإرهاب.
# mt_output: أكد الأمين العام أهمية.
# Keyword: التعاون الدولي في مكافحة الإرهاب
# Best_Match: [OMITTED]
# Sub_Subtype: Total Omission
# ```

# **Annotation:**
# | Column | Value | Reason |
# |--------|-------|--------|
# | A1_kw_in_ar | 1 | Keyword is in ar |
# | A2_kw_count | 1 | Once |
# | **A3_bm_in_mt** | **OMIT** | Special value for omission |
# | **A4_bm_count** | **NA** | Special value for omission |
# | A5_anchor_aligned | 1 | Omission is correctly identified |
# | B1_error_type_correct | 1 | It IS a total omission |
# | C1_phenomenon_clear | 1 | Obvious truncation |
# | C4_no_collateral | 1 | Only omission, no other changes |
# | C5_comprehension_impact | 1 | Meaning significantly changed |
# | **D1_accept** | **1** | ✅ Valid omission sample |

# ---

# ## 📋 The 23 Error Types (Reference)

# ### Semantic Divergence (Critical/Severe)
# | # | Type | Description | Severity |
# |---|------|-------------|----------|
# | 1 | Meaning Shift | Translation conveys different meaning | Critical |
# | 2 | Total Omission | Entire phrase/clause deleted | Critical |
# | 3 | Partial Translation | Part of phrase left untranslated | Severe |
# | 4 | Name Entity Error | Proper noun misspelled/changed | Severe |
# | 5 | Terminology Substitution | Technical term replaced incorrectly | Major |
# | 6 | Literal Translation | Word-for-word losing idiom meaning | Major |
# | 7 | Hypernym for Hyponym | General term instead of specific | Moderate |
# | 8 | Hyponym for Hypernym | Specific term instead of general | Moderate |

# ### Morphosyntactic Shifts (Major/Moderate)
# | # | Type | Description | Severity |
# |---|------|-------------|----------|
# | 9 | Tanween Omission | Missing nunation marker | Moderate |
# | 10 | Gender Disagreement | Masculine/feminine mismatch | Major |
# | 11 | Definiteness Shift | Definite↔indefinite change | Moderate |
# | 23 | Invalid Pattern | Wrong morphological pattern | Major |

# ### Tense/Aspect Errors (Major)
# | # | Type | Description | Severity |
# |---|------|-------------|----------|
# | 12 | Perfective to Progressive | Past→present tense change | Major |
# | 13 | Progressive to Perfective | Present→past tense change | Major |
# | 14 | Tense Shift Under Negation | Tense error in negated context | Major |

# ### Structural Errors (Major)
# | # | Type | Description | Severity |
# |---|------|-------------|----------|
# | 15 | Wrong Structure | Grammatical structure error | Major |
# | 16 | Wrong Word Order | Incorrect word sequence | Major |

# ### Voice Errors (Moderate)
# | # | Type | Description | Severity |
# |---|------|-------------|----------|
# | 17 | Active to Passive Voice | Active changed to passive | Moderate |
# | 18 | Passive to Active Voice | Passive changed to active | Moderate |

# ### Part-of-Speech Errors (Moderate)
# | # | Type | Description | Severity |
# |---|------|-------------|----------|
# | 19 | Noun to Adjective | Noun incorrectly used as adjective | Moderate |
# | 20 | Adjective to Noun | Adjective incorrectly used as noun | Moderate |

# ### Style/Orthographic (Minor)
# | # | Type | Description | Severity |
# |---|------|-------------|----------|
# | 21 | Register Mismatch | Formal↔informal mismatch | Minor |
# | 22 | Spelling Error | Misspelling in Arabic | Minor |

# ---

# ## 💡 Tips for Accurate Annotation

# 1. **Read both sentences completely** before marking any columns
# 2. **Check anchors first** — if A1-A5 fail, other checks don't matter
# 3. **Verify error type** — this is the most important validation
# 4. **Look for hidden errors** — collateral damage is common
# 5. **Use notes liberally** — explain your reasoning, especially for rejections
# 6. **Set confidence honestly** — helps identify samples needing review
# 7. **Flag uncertain cases** — better to flag than guess
# 8. **When in doubt, reject** — false positives are worse than false negatives

# ---

# ## 📊 Quality Control

# - Your annotations will be checked for **inter-annotator agreement**
# - Samples with `D3_needs_review=1` will be **double-annotated**
# - Low confidence samples (`D2_confidence=1`) may be **adjudicated**
# - **Cohen's Kappa** will be calculated across annotators

# ---

# ## 📤 Submission

# 1. Save your completed file as: `TIER2_HUMANEVAL_SAMPLE_[YOUR_ID].csv`
# 2. Verify ALL evaluation columns are filled
# 3. Double-check any samples where D2_confidence=1
# 4. Submit to the research team

# ---

# ## ❓ Questions?

# Contact the research team for:
# - Ambiguous error types
# - Samples that don't fit the schema
# - Technical issues
# - Clarification on any category

# **Thank you for your careful evaluation!**
# '''

# manual_path = f"{OUTPUT_DIR}/TIER2_ANNOTATOR_MANUAL.md"
# with open(manual_path, 'w', encoding='utf-8') as f:
#     f.write(manual_content)
# print(f"✅ Saved: {manual_path}")

# # =============================================================================
# # SUMMARY
# # =============================================================================

# print("\n" + "=" * 80)
# print("🎉 CELL 18 COMPLETE: TIER2 EVALUATION SAMPLING")
# print("=" * 80)

# print(f"""
# {'='*80}
# 📊 SUMMARY
# {'='*80}

# SOURCE DATA:
#    File: {input_path_used}
#    Total: {len(df_tier2)} samples
#    Classes: {df_tier2['Sub-Subtype'].nunique()}/23

# EVALUATION SAMPLE:
#    Total: {len(df_sample)} samples
#    Classes covered: {df_sample['Sub-Subtype'].nunique()}/23
#    Sampling rate: {len(df_sample)/len(df_tier2)*100:.1f}%

# {'='*80}
# 📁 OUTPUT FILES
# {'='*80}

# FOR ANNOTATORS:
#    📄 {eval_path}
#    📄 {manual_path}

# FOR RESEARCH TEAM (DO NOT SHARE):
#    🔒 {metrics_path}
#    📊 {stats_path}

# {'='*80}
# 📋 IMPROVED EVALUATION FORMAT (v2.0)
# {'='*80}

# Section A - Anchor Validation (5 columns):
#    A1_kw_in_ar, A2_kw_count, A3_bm_in_mt, A4_bm_count, A5_anchor_aligned

# Section B - Error Type Validation (3 columns): ← NEW!
#    B1_error_type_correct, B2_suggested_type, B3_severity_appropriate

# Section C - Quality Assessment (5 columns):
#    C1_phenomenon_clear, C2_ar_natural, C3_mt_plausible ← NEW!,
#    C4_no_collateral, C5_comprehension_impact ← NEW!

# Section D - Decision (4 columns):
#    D1_accept, D2_confidence ← NEW!, D3_needs_review ← NEW!, D4_notes

# {'='*80}
# 🆕 IMPROVEMENTS OVER CELL 15.1
# {'='*80}

# 1. B1_error_type_correct  → Validates taxonomy labels (CRITICAL!)
# 2. B2_suggested_type      → Corrects mislabeled samples
# 3. B3_severity_appropriate → Validates severity ratings
# 4. C3_mt_plausible        → Assesses MT realism
# 5. C5_comprehension_impact → Measures error significance
# 6. D2_confidence          → Identifies uncertain annotations
# 7. D3_needs_review        → Flags samples for adjudication
# 8. annotator_id           → Enables inter-annotator agreement
# 9. source_tier            → Distinguishes Tier1 vs Tier2
# 10. Column prefixes (A1_, B1_, etc.) → Clear section organization

# {'='*80}
# 📋 NEXT STEPS
# {'='*80}

# 1. Send to annotators:
#    - {eval_path}
#    - {manual_path}

# 2. Annotators complete evaluation (fill all columns)

# 3. Collect completed files: TIER2_HUMANEVAL_SAMPLE_[ID].csv

# 4. Calculate inter-annotator agreement (Cohen's Kappa)

# 5. Adjudicate disagreements (D3_needs_review=1 samples)

# 6. Join with hidden metrics for correlation analysis

# ✅ Ready for Tier2 manual evaluation!
# """)

In [ ]:
# =============================================================================
# CELL 15.1: TIER1 DUAL-PATH MANUAL EVALUATION FORM BUILDER (v3.0)
# =============================================================================
"""
CELL 15.1: Build Dual-Path Evaluation Forms for Tier1 Research

PURPOSE:
Create comprehensive evaluation forms for comparing:
1. Normal evaluation (no anchors) - Path A
2. Keyword-weighted evaluation (with anchors) - Path B

RESEARCH QUESTIONS:
1. Do keywords/best_matches improve error detection accuracy?
2. Do anchors reduce evaluation time and cognitive load?
3. Which error types benefit most from anchor guidance?
4. How do anchors correlate with OCS/CPS/SFR metrics?

OUTPUTS:
1. TIER1_EVAL_SAMPLES_FOR_ANNOTATORS.csv (sample content for both paths)
2. TIER1_EVAL_RESPONSE_PATH_A.csv (response form for Path A - normal)
3. TIER1_EVAL_RESPONSE_PATH_B.csv (response form for Path B - weighted)
4. TIER1_EVAL_INSTRUCTIONS.txt (annotator protocol)
5. TIER1_EVAL_SAMPLING_STATS.json (sampling statistics)
"""

import pandas as pd
import numpy as np
import json
import random
import os
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("CELL 15.1: TIER1 DUAL-PATH MANUAL EVALUATION FORM BUILDER (v3.0)")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# Input files (FROZEN - from pipeline)
CLEAN_A_PATH = f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv"
CLEAN_B_PATH = f"{OUTPUT_DIR}/tier1_clean_B_FROZEN.csv"
TIER0_PATH = "tier0_canonical_examples.csv"  # Optional

# Output files
OUT_SAMPLES = f"{OUTPUT_DIR}/TIER1_EVAL_SAMPLES_FOR_ANNOTATORS.csv"
OUT_RESPONSE_A = f"{OUTPUT_DIR}/TIER1_EVAL_RESPONSE_PATH_A.csv"
OUT_RESPONSE_B = f"{OUTPUT_DIR}/TIER1_EVAL_RESPONSE_PATH_B.csv"
OUT_INSTRUCTIONS = f"{OUTPUT_DIR}/TIER1_EVAL_INSTRUCTIONS.txt"
OUT_STATS = f"{OUTPUT_DIR}/TIER1_EVAL_SAMPLING_STATS.json"
OUT_METRICS_HIDDEN = f"{OUTPUT_DIR}/TIER1_EVAL_METRICS_HIDDEN.csv"

# Sampling configuration
ANNOTATORS = ["A", "B", "C"]  # 3 annotators for reliability
SAMPLES_PER_CLASS = 6  # Total samples per error type
MAX_PER_TIER_PER_CLASS = 4  # Max from any single tier per class
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Taxonomy (23 classes)
TAXONOMY_23 = [
    "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
    "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
    "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
    "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
    "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
    "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
]

# =============================================================================
# STEP 1: LOAD AND PREPARE DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOADING DATA")
print("=" * 80)

def load_tier_data(path, tier_name, error_col='Sub-Subtype'):
    """Load tier data with consistent column names."""
    if not Path(path).exists():
        print(f"⚠️  {tier_name} not found: {path}")
        return pd.DataFrame()

    df = pd.read_csv(path)
    # Standardize column names
    if error_col in df.columns:
        df['error_type'] = df[error_col]
    df['source_tier'] = tier_name
    print(f"✅ Loaded {tier_name}: {len(df):,} rows, {df['error_type'].nunique()} classes")
    return df

# Load all available data
clean_a = load_tier_data(CLEAN_A_PATH, "TIER1_CLEAN_A")
clean_b = load_tier_data(CLEAN_B_PATH, "TIER1_CLEAN_B")
tier0 = load_tier_data(TIER0_PATH, "TIER0_CANONICAL")

# Combine all data
all_data = []
if not clean_a.empty:
    # Add quality metrics from Clean_A
    for col in ['ocs', 'cps', 'sfr', 'csr', 'mpqs']:
        if col in clean_a.columns:
            clean_a[f'metric_{col}'] = clean_a[col]
    all_data.append(clean_a)
if not clean_b.empty:
    all_data.append(clean_b)
if not tier0.empty:
    all_data.append(tier0)

if not all_data:
    raise ValueError("No data loaded! Check input file paths.")

combined_df = pd.concat(all_data, ignore_index=True)
print(f"✅ Combined data: {len(combined_df):,} rows, {combined_df['error_type'].nunique()} classes")

# =============================================================================
# STEP 2: STRATIFIED SAMPLING
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: STRATIFIED SAMPLING")
print("=" * 80)

def stratified_sample(df, error_types, samples_per_class, max_per_tier):
    """Perform stratified sampling across tiers and error types."""
    sampled_rows = []
    sampling_stats = {}

    for et in error_types:
        et_data = df[df['error_type'] == et].copy()
        if et_data.empty:
            sampling_stats[et] = {'available': 0, 'sampled': 0, 'by_tier': {}}
            continue

        # Count by tier
        tier_counts = et_data['source_tier'].value_counts().to_dict()
        sampling_stats[et] = {'available': len(et_data), 'by_tier': tier_counts}

        # Distribute samples across tiers
        sampled = []
        for tier in et_data['source_tier'].unique():
            tier_df = et_data[et_data['source_tier'] == tier]
            n_take = min(max_per_tier, len(tier_df),
                        max(1, samples_per_class // len(et_data['source_tier'].unique())))

            if n_take > 0:
                tier_sample = tier_df.sample(n=n_take, random_state=RANDOM_SEED)
                sampled.append(tier_sample)

        if sampled:
            et_sample = pd.concat(sampled, ignore_index=True)
            # If we need more samples, randomly add from any tier
            if len(et_sample) < samples_per_class:
                remaining = et_data[~et_data.index.isin(et_sample.index)]
                if len(remaining) > 0:
                    n_needed = samples_per_class - len(et_sample)
                    n_needed = min(n_needed, len(remaining))
                    extra = remaining.sample(n=n_needed, random_state=RANDOM_SEED)
                    et_sample = pd.concat([et_sample, extra], ignore_index=True)

            # If too many, randomly reduce
            if len(et_sample) > samples_per_class:
                et_sample = et_sample.sample(n=samples_per_class, random_state=RANDOM_SEED)

            sampled_rows.append(et_sample)
            sampling_stats[et]['sampled'] = len(et_sample)

    if sampled_rows:
        result = pd.concat(sampled_rows, ignore_index=True)
        print(f"✅ Sampled {len(result)} total rows across {len(sampled_rows)} classes")
        return result, sampling_stats
    else:
        return pd.DataFrame(), sampling_stats

# Perform sampling
sampled_df, sampling_stats = stratified_sample(
    combined_df,
    TAXONOMY_23,
    SAMPLES_PER_CLASS,
    MAX_PER_TIER_PER_CLASS
)

if sampled_df.empty:
    raise ValueError("Sampling failed - no samples selected!")

print(f"\n📊 SAMPLING RESULTS:")
print(f"   Total samples: {len(sampled_df):,}")
print(f"   Classes covered: {sampled_df['error_type'].nunique()}/{len(TAXONOMY_23)}")
print(f"   Tiers represented: {sampled_df['source_tier'].unique().tolist()}")

# =============================================================================
# STEP 3: CREATE SAMPLE FILE FOR ANNOTATORS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: CREATING SAMPLE FILE FOR ANNOTATORS")
print("=" * 80)

# Create unique eval_id for each sample-annotator pair
sample_records = []

for _, row in sampled_df.iterrows():
    sample_id = f"S{len(sample_records)//len(ANNOTATORS):04d}"

    for annotator in ANNOTATORS:
        eval_id = f"{sample_id}_{annotator}"

        record = {
            'eval_id': eval_id,
            'sample_id': sample_id,
            'annotator_id': annotator,
            'source_tier': row.get('source_tier', 'UNKNOWN'),
            'original_error_type': row.get('error_type', ''),

            # Content for evaluation
            'ar': row.get('ar', row.get('clean_ar', '')),
            'mt_output': row.get('mt_output', row.get('error_ar', '')),
            'Keyword': row.get('Keyword', ''),
            'Best_Match': row.get('Best_Match', ''),
            'Sub_Subtype': row.get('error_type', ''),

            # For Path A (to be hidden initially)
            'en': row.get('en', '') if 'en' in row else '',
            'MT_Tool': row.get('MT Tool', row.get('MT_Tool', '')),

            # Quality metrics (hidden from annotators)
            'metric_ocs': row.get('metric_ocs', row.get('ocs', np.nan)),
            'metric_cps': row.get('metric_cps', row.get('cps', np.nan)),
            'metric_sfr': row.get('metric_sfr', row.get('sfr', np.nan)),
            'metric_csr': row.get('metric_csr', row.get('csr', np.nan)),
            'metric_mpqs': row.get('metric_mpqs', row.get('mpqs', np.nan)),
            'ocs_match_rule': row.get('ocs_match_rule_kw', ''),
        }
        sample_records.append(record)

samples_df = pd.DataFrame(sample_records)
print(f"✅ Created {len(samples_df)} sample-annotator pairs ({len(samples_df)//len(ANNOTATORS)} unique samples)")

# =============================================================================
# STEP 4: CREATE RESPONSE FORMS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: CREATING RESPONSE FORMS")
print("=" * 80)

# Path A: Normal Evaluation (no anchors)
response_a_columns = [
    # Identification
    'eval_id', 'annotator_id', 'phase_completion_time',

    # Section A1: Error Detection (Natural)
    'A1_error_present',  # Yes/No
    'A1_confidence',  # 1-5

    # Section A2: Error Type Classification
    'A2_detected_error_type',  # taxonomy_23 or "None"
    'A2_confidence',  # 1-5

    # Section A3: Severity Assessment
    'A3_severity',  # 1-5
    'A3_impact_on_meaning',  # Minor/Moderate/Major

    # Section A4: Additional Errors
    'A4_additional_errors',  # Yes/No
    'A4_additional_types',  # comma-separated list

    # Section A5: Overall Quality
    'A5_overall_quality',  # 1-5
    'A5_fluency_rating',  # 1-5
    'A5_adequacy_rating',  # 1-5

    # Timing
    'A_time_start', 'A_time_end', 'A_time_seconds',

    # Notes
    'A_notes', 'A_difficulty_rating',  # 1-5
]

# Path B: Keyword-Weighted Evaluation
response_b_columns = [
    # Identification
    'eval_id', 'annotator_id', 'phase_completion_time',

    # Section B1: Anchor Validation
    'B1_kw_exists_in_ar',  # Yes/No/Partially
    'B1_bm_exists_in_mt',  # Yes/No/Partially
    'B1_anchor_alignment',  # 1-5

    # Section B2: Error Type Validation
    'B2_original_type_correct',  # Yes/No/Partially
    'B2_suggested_type',  # taxonomy_23 or "Same"
    'B2_confidence',  # 1-5

    # Section B3: Anchor Utility
    'B3_anchor_helpfulness',  # 1-5
    'B3_time_saved',  # Yes/No/Neutral
    'B3_clarity_improvement',  # 1-5

    # Section B4: Comparative Analysis
    'B4_detection_change',  # No_change/Minor_clarification/Major_correction
    'B4_confidence_change',  # Decreased/No_change/Increased
    'B4_preferred_method',  # Path_A/Path_B/Equal

    # Section B5: Anchor Quality
    'B5_kw_relevance',  # 1-5
    'B5_bm_relevance',  # 1-5
    'B5_anchor_specificity',  # Over_specific/Just_right/Ambiguous
    'B5_better_anchors_suggested',  # Yes/No
    'B5_anchor_suggestions',  # text

    # Timing
    'B_time_start', 'B_time_end', 'B_time_seconds',

    # Notes
    'B_notes', 'B_difficulty_rating',  # 1-5
]

# Create empty response forms
response_a_df = pd.DataFrame(columns=response_a_columns)
response_b_df = pd.DataFrame(columns=response_b_columns)

# Pre-populate with eval_ids
response_a_df['eval_id'] = samples_df['eval_id'].unique()
response_a_df['annotator_id'] = response_a_df['eval_id'].str.split('_').str[-1]
response_b_df['eval_id'] = samples_df['eval_id'].unique()
response_b_df['annotator_id'] = response_b_df['eval_id'].str.split('_').str[-1]

print(f"✅ Created response forms:")
print(f"   - Path A: {len(response_a_df)} rows, {len(response_a_columns)} columns")
print(f"   - Path B: {len(response_b_df)} rows, {len(response_b_columns)} columns")

# =============================================================================
# STEP 5: CREATE HIDDEN METRICS FILE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: CREATING HIDDEN METRICS FILE")
print("=" * 80)

# Create file with metrics for post-analysis (not shared with annotators)
hidden_metrics = samples_df[[
    'eval_id', 'sample_id', 'source_tier', 'original_error_type',
    'metric_ocs', 'metric_cps', 'metric_sfr', 'metric_csr', 'metric_mpqs',
    'ocs_match_rule', 'MT_Tool'
]].copy()

# Add sampling metadata
hidden_metrics['sampling_strategy'] = 'stratified_by_class_and_tier'
hidden_metrics['creation_date'] = datetime.now().isoformat()
hidden_metrics['total_annotators'] = len(ANNOTATORS)

print(f"✅ Created hidden metrics file: {len(hidden_metrics)} rows")

# =============================================================================
# STEP 6: CREATE ANNOTATOR INSTRUCTIONS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6: CREATING ANNOTATOR INSTRUCTIONS")
print("=" * 80)

instructions = f"""
TIER1 DUAL-PATH EVALUATION PROTOCOL (v3.0)
===========================================

PURPOSE:
Evaluate Arabic MT output quality using two methods:
1. PATH A: Natural error detection (no guidance)
2. PATH B: Anchor-guided evaluation (with keywords)

RESEARCH GOALS:
- Compare effectiveness of natural vs. guided evaluation
- Measure time efficiency and accuracy improvements
- Assess anchor quality and utility

PROTOCOL:
---------
PHASE 1: PATH A (Natural Evaluation)
1. Open: {OUT_RESPONSE_A}
2. Open: {OUT_SAMPLES}
3. For each sample:
   - Read ONLY: ar (Arabic reference) and mt_output (MT output)
   - IGNORE: Keyword, Best_Match, Sub_Subtype columns
   - Complete all A1-A5 sections in response form
   - Record start/end times

PHASE 2: COOLING PERIOD (48 hours minimum)

PHASE 3: PATH B (Anchor-Guided Evaluation)
1. Open: {OUT_RESPONSE_B}
2. Open: {OUT_SAMPLES}
3. For each sample:
   - Read: ar, mt_output, AND Keyword, Best_Match
   - Consider the suggested error type (Sub_Subtype)
   - Complete all B1-B5 sections in response form
   - Record start/end times

GUIDELINES:
-----------
1. PATH A: Trust your natural intuition. Don't overthink.
2. PATH B: Use anchors as hints, but verify independently.
3. Confidence ratings:
   1 = Very uncertain, 3 = Moderately confident, 5 = Absolutely certain
4. Severity ratings:
   1 = Minor/cosmetic, 3 = Moderate/meaning affected, 5 = Major/unintelligible
5. Time tracking: Be accurate - this measures cognitive load.

DEFINITIONS:
------------
- Keyword: Word/phrase expected in reference (ar)
- Best_Match: Corresponding word/phrase in MT output
- Anchor Alignment: How well keyword→best_match represents the error
- Anchor Helpfulness: How much anchors aided your evaluation

ERROR TYPES (23 classes):
-------------------------
{chr(10).join([f"- {et}" for et in TAXONOMY_23])}

OUTPUT FILES:
-------------
1. {OUT_RESPONSE_A} - Your responses for Path A
2. {OUT_RESPONSE_B} - Your responses for Path B
3. Submit both files after completion

QUALITY CONTROL:
----------------
- 10% samples will be evaluated by all annotators (for agreement)
- Weekly calibration sessions
- Contact researcher for ambiguous cases

Thank you for your careful work!
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open(OUT_INSTRUCTIONS, 'w', encoding='utf-8') as f:
    f.write(instructions)
print(f"✅ Created annotator instructions")

# =============================================================================
# STEP 7: SAVE ALL FILES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 7: SAVING ALL FILES")
print("=" * 80)

# Save samples file
samples_df.to_csv(OUT_SAMPLES, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUT_SAMPLES} ({len(samples_df)} rows)")

# Save response forms
response_a_df.to_csv(OUT_RESPONSE_A, index=False, encoding='utf-8-sig')
response_b_df.to_csv(OUT_RESPONSE_B, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUT_RESPONSE_A}")
print(f"✅ Saved: {OUT_RESPONSE_B}")

# Save hidden metrics
hidden_metrics.to_csv(OUT_METRICS_HIDDEN, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUT_METRICS_HIDDEN}")

# Save sampling statistics
final_stats = {
    'timestamp': datetime.now().isoformat(),
    'random_seed': RANDOM_SEED,
    'annotators': ANNOTATORS,
    'sampling_config': {
        'samples_per_class': SAMPLES_PER_CLASS,
        'max_per_tier_per_class': MAX_PER_TIER_PER_CLASS,
        'total_classes_targeted': len(TAXONOMY_23),
    },
    'results': {
        'total_samples': len(sampled_df),
        'unique_sample_annotator_pairs': len(samples_df),
        'classes_represented': int(sampled_df['error_type'].nunique()),
        'tiers_represented': sampled_df['source_tier'].unique().tolist(),
    },
    'by_tier_counts': sampled_df['source_tier'].value_counts().to_dict(),
    'by_class_counts': sampled_df['error_type'].value_counts().to_dict(),
    'detailed_sampling_stats': sampling_stats,
}

with open(OUT_STATS, 'w', encoding='utf-8') as f:
    json.dump(final_stats, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {OUT_STATS}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 15.1 COMPLETE: DUAL-PATH EVALUATION FORM BUILDER")
print("=" * 80)

# Calculate distributions
class_dist = sampled_df['error_type'].value_counts()
tier_dist = sampled_df['source_tier'].value_counts()

print(f"""
📊 EVALUATION FORM SUMMARY:

1. SAMPLING STRATEGY:
   - Stratified by error type and source tier
   - Target: {SAMPLES_PER_CLASS} per class, max {MAX_PER_TIER_PER_CLASS} per tier
   - Annotators: {len(ANNOTATORS)} (A, B, C)
   - Random seed: {RANDOM_SEED}

2. SAMPLE DISTRIBUTION:
   - Unique samples: {len(sampled_df):,}
   - Sample-annotator pairs: {len(samples_df):,}
   - Classes covered: {len(class_dist):,}/{len(TAXONOMY_23)}
   - Tiers: {', '.join([f'{k} ({v})' for k, v in tier_dist.items()])}

3. OUTPUT FILES:
   a) For annotators:
      - {OUT_SAMPLES} (sample content)
      - {OUT_RESPONSE_A} (Path A response form)
      - {OUT_RESPONSE_B} (Path B response form)
      - {OUT_INSTRUCTIONS} (protocol)

   b) For analysis:
      - {OUT_METRICS_HIDDEN} (hidden metrics)
      - {OUT_STATS} (sampling statistics)

4. RESEARCH DESIGN:
   - PATH A: Natural evaluation (baseline)
   - PATH B: Keyword-weighted evaluation (intervention)
   - Cooling period: 48 hours between paths
   - Inter-annotator agreement: 3 annotators per sample
   - Time tracking for cognitive load measurement

5. NEXT STEPS:
   1. Distribute {OUT_SAMPLES}, {OUT_RESPONSE_A}, {OUT_INSTRUCTIONS}
   2. Annotators complete PATH A
   3. Wait 48 hours (cooling period)
   4. Distribute {OUT_RESPONSE_B}
   5. Annotators complete PATH B
   6. Collect responses for analysis

⚠️ IMPORTANT:
   - Annotators must NOT see anchors during PATH A
   - Maintain consistent evaluation environment
   - Track time accurately for both paths
   - Use the same annotators for both paths

📋 CLASS DISTRIBUTION (top 10):
""")

for et, count in class_dist.head(10).items():
    print(f"   - {et}: {count} samples")

print(f"""
✅ Ready for manual evaluation! Use the instructions in {OUT_INSTRUCTIONS}
""")

In [ ]:
# =============================================================================
# CELL 19: BUILD HUMAN EVALUATION TEMPLATES
# =============================================================================
"""
CELL 19: Build Human Evaluation Templates

Creates annotation-ready CSV files for human evaluation:
1. HumanEval_Tier1Clean_MASTER.csv (annotation sheet with human columns)
2. HumanEval_Tier1Clean_TRAINING.csv (same rows, no human columns - for ML)

Schema:
- Tier0/V3 metadata columns (read-only for annotators)
- Row-specific content (Sentence_ID, ar, mt_output, Keyword, Best_Match)
- Human annotation columns (to be filled by annotators)

INPUTS:
- tier1_v3_3_final/tier1_clean_A_FROZEN.csv
- tier0_canonical_examples.csv (for metadata lookup by Sub-Subtype)

OUTPUTS:
- tier1_v3_3_final/HumanEval_Tier1Clean_MASTER.csv
- tier1_v3_3_final/HumanEval_Tier1Clean_TRAINING.csv
- tier1_v3_3_final/HumanEval_Instructions.txt
"""

import pandas as pd
import json
import os
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("CELL 19: BUILD HUMAN EVALUATION TEMPLATES")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# Inputs
TIER1_CLEAN_PATH = f"{OUTPUT_DIR}/tier1_clean_A_FROZEN.csv"
TIER0_PATH = "tier0_canonical_examples.csv"

# Outputs
OUT_MASTER = f"{OUTPUT_DIR}/HumanEval_Tier1Clean_MASTER.csv"
OUT_TRAINING = f"{OUTPUT_DIR}/HumanEval_Tier1Clean_TRAINING.csv"
OUT_INSTRUCTIONS = f"{OUTPUT_DIR}/HumanEval_Instructions.txt"

# Tier0 metadata columns (read-only for annotators)
TIER0_METADATA_COLS = [
    "Linguistic Shift Type",
    "Subtype",
    "Sub-Subtype",
    "Severity_Level",
    "Severity_Score",
    "SubSubtype_WGT",
    "TQA_Category",
    "Shift_Group_Weight",
    "TQA_Category_Weight",
]

# Row-specific content columns
CONTENT_COLS = [
    "Sentence_ID",
    "en",
    "ar",
    "mt_output",
    "Keyword",
    "Best_Match",
]

# Human annotation columns (to be filled by annotators)
HUMAN_ANNOTATION_COLS = [
    "human_binary_label",           # 0 = no error, 1 = error present
    "human_error_type_correct",     # 0 = wrong type, 1 = correct type
    "human_keyword_correct",        # 0 = wrong, 1 = correct
    "human_severity_rating",        # 1-5 scale
    "human_naturalness_rating",     # 1-5 scale
    "human_notes",                  # Free text
    "annotator_id",                 # Annotator identifier
    "annotation_timestamp",         # When annotated
]

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

# Load Tier1 Clean
if not Path(TIER1_CLEAN_PATH).exists():
    raise FileNotFoundError(f"Tier1 Clean not found: {TIER1_CLEAN_PATH}")

tier1_clean = pd.read_csv(TIER1_CLEAN_PATH)
print(f"✅ Loaded Tier1 Clean: {len(tier1_clean):,} rows")

# Load Tier0 for metadata lookup
tier0_metadata_lookup = {}

if Path(TIER0_PATH).exists():
    tier0 = pd.read_csv(TIER0_PATH)
    print(f"✅ Loaded Tier0: {len(tier0):,} rows")

    # Build lookup by Sub-Subtype
    tier0_col = None
    for col in ['Sub-Subtype', 'error_type', 'Error_Type']:
        if col in tier0.columns:
            tier0_col = col
            break

    if tier0_col:
        for _, row in tier0.iterrows():
            et = row[tier0_col]
            if et not in tier0_metadata_lookup:
                tier0_metadata_lookup[et] = row.to_dict()
        print(f"✅ Built metadata lookup: {len(tier0_metadata_lookup)} classes")
else:
    print(f"⚠️ Tier0 not found: {TIER0_PATH}")
    print(f"   Will use empty metadata")

# =============================================================================
# STEP 2: BUILD HUMAN EVAL DATAFRAME
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: BUILD HUMAN EVAL DATAFRAME")
print("=" * 80)

rows = []

for idx, row in tier1_clean.iterrows():
    new_row = {}

    # 1) Content columns (from Tier1)
    for col in CONTENT_COLS:
        if col in row:
            new_row[col] = row[col]
        else:
            new_row[col] = ""

    # 2) Tier0 metadata columns (lookup by Sub-Subtype)
    sub_subtype = row.get('Sub-Subtype', '')
    tier0_meta = tier0_metadata_lookup.get(sub_subtype, {})

    for col in TIER0_METADATA_COLS:
        if col in tier0_meta:
            new_row[col] = tier0_meta[col]
        elif col in row:
            new_row[col] = row[col]
        else:
            new_row[col] = ""

    # 3) Human annotation columns (empty placeholders)
    for col in HUMAN_ANNOTATION_COLS:
        new_row[col] = ""

    rows.append(new_row)

# Create DataFrame with ordered columns
all_cols = CONTENT_COLS + TIER0_METADATA_COLS + HUMAN_ANNOTATION_COLS
df_master = pd.DataFrame(rows)

# Reorder columns
df_master = df_master[[c for c in all_cols if c in df_master.columns]]

print(f"✅ Built Human Eval DataFrame: {len(df_master):,} rows × {len(df_master.columns)} columns")

# =============================================================================
# STEP 3: SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: SAVE OUTPUTS")
print("=" * 80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save Master (with human annotation columns)
df_master.to_csv(OUT_MASTER, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUT_MASTER} ({len(df_master):,} rows)")

# Save Training (without human annotation columns)
training_cols = CONTENT_COLS + TIER0_METADATA_COLS
df_training = df_master[[c for c in training_cols if c in df_master.columns]]
df_training.to_csv(OUT_TRAINING, index=False, encoding='utf-8-sig')
print(f"✅ Saved: {OUT_TRAINING} ({len(df_training):,} rows)")

# =============================================================================
# STEP 4: CREATE INSTRUCTIONS FILE
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: CREATE INSTRUCTIONS FILE")
print("=" * 80)

instructions = f"""
================================================================================
HUMAN EVALUATION INSTRUCTIONS
================================================================================
Generated: {datetime.now().isoformat()}
Dataset: {OUT_MASTER}
Total Samples: {len(df_master):,}

================================================================================
COLUMN DESCRIPTIONS
================================================================================

CONTENT COLUMNS (Row-specific data):
- Sentence_ID: Unique identifier for this sample
- en: English source sentence (reference)
- ar: Clean Arabic sentence (correct)
- mt_output: Arabic MT output (contains the error)
- Keyword: Word/phrase in 'ar' that changes
- Best_Match: What appears in 'mt_output' instead (or "[OMITTED]")

METADATA COLUMNS (Read-only, from Tier0/V3 taxonomy):
- Linguistic Shift Type: High-level category
- Subtype: Mid-level category
- Sub-Subtype: Specific error type (23-class taxonomy)
- Severity_Level: Critical/Major/Minor
- Severity_Score: Numeric severity (1-5)
- SubSubtype_WGT: Weight for this error type
- TQA_Category: TQA category label
- Shift_Group_Weight: Weight for shift group
- TQA_Category_Weight: Weight for TQA category

HUMAN ANNOTATION COLUMNS (To be filled by you):
- human_binary_label: Is there an error? (0=No, 1=Yes)
- human_error_type_correct: Is Sub-Subtype correct? (0=No, 1=Yes)
- human_keyword_correct: Is Keyword/Best_Match correct? (0=No, 1=Yes)
- human_severity_rating: Your severity rating (1-5)
- human_naturalness_rating: How natural is mt_output Arabic? (1-5)
- human_notes: Any comments or observations
- annotator_id: Your annotator ID
- annotation_timestamp: When you completed this row

================================================================================
ANNOTATION GUIDELINES
================================================================================

1. BINARY LABEL (human_binary_label):
   - 1 if mt_output clearly contains the claimed error type
   - 0 if no error is present or it's a different error type

2. ERROR TYPE CORRECTNESS (human_error_type_correct):
   - 1 if the Sub-Subtype accurately describes the error
   - 0 if the error should be classified differently

3. KEYWORD CORRECTNESS (human_keyword_correct):
   - 1 if Keyword and Best_Match correctly identify the error location
   - 0 if the annotations are wrong or misleading

4. SEVERITY RATING (human_severity_rating):
   - 1 = Minor (stylistic, doesn't affect meaning)
   - 2 = Low (slight meaning change)
   - 3 = Medium (noticeable meaning change)
   - 4 = High (significant meaning change)
   - 5 = Critical (meaning inverted or completely wrong)

5. NATURALNESS RATING (human_naturalness_rating):
   - 1 = Broken Arabic (ungrammatical)
   - 2 = Poor (many issues)
   - 3 = Acceptable (some issues)
   - 4 = Good (minor issues)
   - 5 = Excellent (fluent, natural Arabic)

================================================================================
TIPS
================================================================================

- Focus on the CLAIMED error type (Sub-Subtype), not other potential errors
- Compare 'ar' and 'mt_output' directly to verify the error
- Use 'Keyword' and 'Best_Match' to locate the specific change
- If unsure, use 'human_notes' to explain your reasoning
- Be consistent across samples

================================================================================
"""

with open(OUT_INSTRUCTIONS, 'w', encoding='utf-8') as f:
    f.write(instructions)
print(f"✅ Saved: {OUT_INSTRUCTIONS}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 19 COMPLETE")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   - {OUT_MASTER} ({len(df_master):,} rows, with human columns)
   - {OUT_TRAINING} ({len(df_training):,} rows, no human columns)
   - {OUT_INSTRUCTIONS}

📋 COLUMN SUMMARY:
   Content columns:     {len(CONTENT_COLS)}
   Metadata columns:    {len(TIER0_METADATA_COLS)}
   Human columns:       {len(HUMAN_ANNOTATION_COLS)}
   Total columns:       {len(df_master.columns)}

📊 CLASS COVERAGE:
""")

if 'Sub-Subtype' in df_master.columns:
    class_counts = df_master['Sub-Subtype'].value_counts()
    print(f"   Classes: {len(class_counts)}")
    print(f"   Min per class: {class_counts.min()}")
    print(f"   Max per class: {class_counts.max()}")

print("\n✅ Ready for Cell 20: Final Export & Freeze")

In [ ]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from datetime import datetime
import json, os

print("=" * 80)
print("CELL 19B: GENERATE TRIVET MQM V4 ANNOTATION SPREADSHEETS")
print("=" * 80)

# ═══════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════

PROJECT_DIR = os.getcwd()  # Use current working directory

# INPUT: Source files (before evaluation)
TIER1_CLEAN_A = os.path.join(PROJECT_DIR, "/content/drive/MyDrive/25/Final/MT_Q/P3_F/tier1_v3_3_final/tier1_clean_A.csv")      # Clean A: 1293 samples
TIER1_CLEAN_B = os.path.join(PROJECT_DIR, "/content/drive/MyDrive/25/Final/MT_Q/P3_F/tier1_v3_3_final/tier1_clean_B.csv")    # Clean B: 306 samples
TIER2_INPUT   = os.path.join(PROJECT_DIR, "/content/drive/MyDrive/25/Final/MT_Q/P3_F/tier1_v3_3_final/tier2_generated_raw.csv")  # Tier 2: 1075 samples

# SAMPLING
TIER1_SAMPLE_FRAC = 0.10   # 10% of Tier 1
TIER2_SAMPLE_FRAC = 0.05   # 5% of Tier 2
RANDOM_STATE = 42

# OUTPUT
TIER1_OUTPUT = os.path.join(PROJECT_DIR, "TRIVET_Annotation_Tier1.xlsx")
TIER2_OUTPUT = os.path.join(PROJECT_DIR, "TRIVET_Annotation_Tier2.xlsx")

# ═══════════════════════════════════════════════════════════════════
# V4 TAXONOMY — COMPLETE 23-CLASS REFERENCE
# ═══════════════════════════════════════════════════════════════════

V4_TAXONOMY = [
    # (Sub-subtype, TQA_Category, TQA_Weight, Shift_Type, Sub_type, Definition, Severity, Score, SubSubtype_WGT)
    ("Meaning Shift",              "Accuracy",         0.42, "Semantic Divergence",    "Semantic Shift",      "Incorrect or distorted meaning",                       "Critical",  5, 0.30),
    ("Total Omission",             "Accuracy",         0.42, "Semantic Divergence",    "Translation Gap",     "Source text element completely missing",                "Critical",  5, 0.30),
    ("Partial Translation",        "Accuracy",         0.42, "Semantic Divergence",    "Translation Gap",     "Named entity/key term only partially translated",      "Severe",    4, 0.20),
    ("Literal Translation",        "Accuracy",         0.42, "Semantic Divergence",    "Idiomatic Shift",     "Idiom translated word-for-word",                       "Major",     3, 0.10),
    ("Hypernym for Hyponym",       "Accuracy",         0.42, "Semantic Divergence",    "Granularity Shift",   "General term used for specific one",                   "Moderate",  2, 0.05),
    ("Hyponym for Hypernym",       "Accuracy",         0.42, "Semantic Divergence",    "Granularity Shift",   "Specific term used for general one",                   "Moderate",  2, 0.05),
    ("Tense Shift Under Negation", "Accuracy",         0.42, "Syntactic Shift",        "Tense Shift",         "Tense shift under negation",                           "Major",     3, 0.20),
    ("Invalid Pattern",            "Fluency",          0.225,"Morphosyntactic Shifts",  "Morphological Error", "Incorrect word form",                                  "Major",     3, 0.30),
    ("Tanween Omission",           "Fluency",          0.225,"Morphosyntactic Shifts",  "Morphological Error", "Missing Arabic nunation",                              "Moderate",  2, 0.20),
    ("Gender Disagreement",        "Fluency",          0.225,"Morphosyntactic Shifts",  "Agreement Error",     "Grammatical gender mismatch",                          "Major",     3, 0.30),
    ("Definiteness Shift",         "Fluency",          0.225,"Morphosyntactic Shifts",  "Definiteness Shift",  "Incorrect al- article usage",                          "Moderate",  2, 0.20),
    ("Perfective to Progressive",  "Fluency",          0.225,"Syntactic Shift",         "Aspect Shift",        "Completed → ongoing event",                            "Severe",    4, 0.15),
    ("Progressive to Perfective",  "Fluency",          0.225,"Syntactic Shift",         "Aspect Shift",        "Ongoing → completed event",                            "Severe",    4, 0.15),
    ("Wrong Structure",            "Fluency",          0.225,"Syntactic Shift",         "Structural Error",    "Incorrect sentence construction",                      "Major",     3, 0.20),
    ("Wrong Word Order",           "Fluency",          0.225,"Syntactic Shift",         "Structural Error",    "Unnatural word arrangement",                           "Major",     3, 0.20),
    ("Noun to Adjective",          "Fluency",          0.225,"Syntactic Shift",         "POS Shift",           "Noun translated as adjective",                         "Moderate",  2, 0.025),
    ("Adjective to Noun",          "Fluency",          0.225,"Syntactic Shift",         "POS Shift",           "Adjective translated as noun",                         "Moderate",  2, 0.025),
    ("Spelling Error",             "Fluency",          0.225,"Orthographic Shifts",     "Orthographic Issues", "Plain misspelling",                                    "Minor",     1, 1.00),
    ("Terminological Substitution","Terminology",      0.28, "Pragmatic Divergence",    "Register Mismatch",   "Wrong domain-specific term",                           "Severe",    4, 0.80),
    ("Register Mismatch",          "Style & Register", 0.075,"Pragmatic Divergence",    "Register Mismatch",   "Wrong formality level",                                "Major",     3, 0.20),
    ("Active to Passive Voice",    "Style & Register", 0.075,"Syntactic Shift",         "Voice",               "Voice change altering focus/style",                    "Moderate",  2, 0.025),
    ("Passive to Active Voice",    "Style & Register", 0.075,"Syntactic Shift",         "Voice",               "Voice change altering focus/style",                    "Moderate",  2, 0.025),
    ("Named Entity Error",         "Accuracy",         0.42, "Semantic Divergence",     "Translation Gap",     "Named entity incorrectly translated or omitted",       "Severe",    4, 0.20),
]

# Build lookup dict
V4_LOOKUP = {}
for row in V4_TAXONOMY:
    V4_LOOKUP[row[0]] = {
        "TQA_Category": row[1], "TQA_Weight": row[2], "Shift_Type": row[3],
        "Sub_type": row[4], "Definition": row[5], "Severity": row[6],
        "Score": row[7], "SubSubtype_WGT": row[8]
    }

# Name mapping for data inconsistencies
NAME_MAP = {
    "Terminology Substitution": "Terminological Substitution",
    "Name Entity Error": "Named Entity Error",
}

# The 23 classes for the taxonomy grid (excluding Named Entity Error which is alt name for Partial Translation)
TAXONOMY_GRID_23 = [t[0] for t in V4_TAXONOMY[:23]]

SEVERITY_LEVELS = ["Minor", "Moderate", "Major", "Severe", "Critical"]
SEVERITY_MULTIPLIERS = [1, 2, 3, 4, 5]

# ═══════════════════════════════════════════════════════════════════
# STYLE DEFINITIONS
# ═══════════════════════════════════════════════════════════════════

NAVY = "1B2A4A"
DARK_BLUE = "1F3864"
MED_BLUE = "2E75B6"
LIGHT_BLUE = "D6E4F0"
ACCENT_GOLD = "C49A2A"
LIGHT_GOLD = "FFF2CC"
WHITE = "FFFFFF"
DARK_GRAY = "404040"
MED_GRAY = "808080"
LIGHT_GRAY = "F2F2F2"
GREEN_ACCENT = "548235"
GREEN_LIGHT = "E2EFDA"
ORANGE_ACCENT = "ED7D31"
ORANGE_LIGHT = "FFF3E0"
PURPLE_ACCENT = "7030A0"
PURPLE_LIGHT = "E8D5F5"

TQA_COLORS = {
    "Accuracy":         {"light": LIGHT_BLUE,   "dark": DARK_BLUE},
    "Fluency":          {"light": GREEN_LIGHT,  "dark": GREEN_ACCENT},
    "Terminology":      {"light": ORANGE_LIGHT, "dark": ORANGE_ACCENT},
    "Style & Register": {"light": PURPLE_LIGHT, "dark": PURPLE_ACCENT},
}

f_title = Font(name='Arial', size=14, bold=True, color=WHITE)
f_section = Font(name='Arial', size=11, bold=True, color=WHITE)
f_header = Font(name='Arial', size=9, bold=True, color=WHITE)
f_header_dark = Font(name='Arial', size=9, bold=True, color=DARK_BLUE)
f_body = Font(name='Arial', size=9, color=DARK_GRAY)
f_body_bold = Font(name='Arial', size=9, bold=True, color=DARK_GRAY)
f_body_ar = Font(name='Arial', size=10, color=DARK_GRAY)
f_input = Font(name='Arial', size=9, color="0000FF")
f_small = Font(name='Arial', size=8, color=MED_GRAY)
f_version = Font(name='Arial', size=7, color=MED_GRAY)

fill_navy = PatternFill('solid', fgColor=NAVY)
fill_dark_blue = PatternFill('solid', fgColor=DARK_BLUE)
fill_med_blue = PatternFill('solid', fgColor=MED_BLUE)
fill_light_gray = PatternFill('solid', fgColor=LIGHT_GRAY)
fill_white = PatternFill('solid', fgColor=WHITE)
fill_input = PatternFill('solid', fgColor="DAEEF3")
fill_gold = PatternFill('solid', fgColor=ACCENT_GOLD)
fill_light_gold = PatternFill('solid', fgColor=LIGHT_GOLD)

a_center = Alignment(horizontal='center', vertical='center', wrap_text=True)
a_left = Alignment(horizontal='left', vertical='center', wrap_text=True)
a_right = Alignment(horizontal='right', vertical='center', wrap_text=True)
a_right_ar = Alignment(horizontal='right', vertical='center', wrap_text=True, readingOrder=2)

thin = Side(style='thin', color=MED_GRAY)
b_thin = Border(left=thin, right=thin, top=thin, bottom=thin)
medium_side = Side(style='medium', color=DARK_GRAY)
b_medium = Border(left=medium_side, right=medium_side, top=medium_side, bottom=medium_side)

def sc(ws, r, c, val, font=f_body, fill=fill_white, align=a_left, border=b_thin):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font = font; cell.fill = fill; cell.alignment = align; cell.border = border
    return cell

# ═══════════════════════════════════════════════════════════════════
# LOAD AND PREPARE DATA
# ═══════════════════════════════════════════════════════════════════

print("\n📂 Loading source data...")

# ── TIER 1: Combine Clean A + Clean B, then stratified 10% sample ──
df1a = pd.read_csv(TIER1_CLEAN_A)
df1a['source'] = 'Clean_A'
df1b = pd.read_csv(TIER1_CLEAN_B)
df1b['source'] = 'Clean_B'

df1_full = pd.concat([df1a, df1b], ignore_index=True)
df1_full['tier'] = 'TIER1'

print(f"  Tier1 Clean_A: {len(df1a)} samples ({df1a['Sub-Subtype'].nunique()} classes)")
print(f"  Tier1 Clean_B: {len(df1b)} samples ({df1b['Sub-Subtype'].nunique()} classes)")
print(f"  Tier1 Combined: {len(df1_full)} samples ({df1_full['Sub-Subtype'].nunique()} classes)")

# Stratified 10% sample by Sub-Subtype
df1 = df1_full.groupby('Sub-Subtype', group_keys=False).apply(
    lambda x: x.sample(n=max(1, int(len(x) * TIER1_SAMPLE_FRAC)), random_state=RANDOM_STATE),
    include_groups=False
)
# Restore columns lost by include_groups=False
df1 = df1_full.loc[df1.index].reset_index(drop=True)

# Add eval_id
df1['eval_id'] = [f"T1_EVAL_{i+1:04d}" for i in range(len(df1))]

print(f"  Tier1 10% Sample: {len(df1)} samples ({df1['Sub-Subtype'].nunique()} classes)")
print(f"    Per source: {df1['source'].value_counts().to_dict()}")

# ── TIER 2: Stratified 5% sample ──
df2_full = pd.read_csv(TIER2_INPUT)
df2_full['tier'] = 'TIER2'
df2_full['source'] = 'Synthetic'

df2 = df2_full.groupby('Sub-Subtype', group_keys=False).apply(
    lambda x: x.sample(n=max(1, int(len(x) * TIER2_SAMPLE_FRAC)), random_state=RANDOM_STATE),
    include_groups=False
)
df2 = df2_full.loc[df2.index].reset_index(drop=True)

# Add eval_id
df2['eval_id'] = [f"T2_EVAL_{i+1:04d}" for i in range(len(df2))]

print(f"  Tier2 Full: {len(df2_full)} samples ({df2_full['Sub-Subtype'].nunique()} classes)")
print(f"  Tier2 5% Sample: {len(df2)} samples ({df2['Sub-Subtype'].nunique()} classes)")

# Normalize names
for df in [df1, df2]:
    df['Sub-Subtype'] = df['Sub-Subtype'].replace(NAME_MAP)

# Enrich with V4 metadata
def enrich_v4(df):
    for col, key in [('TQA_Category','TQA_Category'), ('TQA_Weight','TQA_Weight'),
                     ('Shift_Type','Shift_Type'), ('Sub_type','Sub_type'),
                     ('V4_Severity','Severity'), ('V4_Score','Score'),
                     ('SubSubtype_WGT','SubSubtype_WGT'), ('Definition','Definition')]:
        df[col] = df['Sub-Subtype'].map(lambda x: V4_LOOKUP.get(x, {}).get(key, ''))
    return df

df1 = enrich_v4(df1)
df2 = enrich_v4(df2)

print(f"\n✅ V4 metadata enriched")
print(f"  Tier1 TQA coverage: {df1['TQA_Category'].value_counts().to_dict()}")
print(f"  Tier2 TQA coverage: {df2['TQA_Category'].value_counts().to_dict()}")

# ═══════════════════════════════════════════════════════════════════
# BUILD ANNOTATION EXCEL
# ═══════════════════════════════════════════════════════════════════

def build_annotation_excel(df, output_path, tier_label):
    """Build the complete annotation spreadsheet for one tier."""

    print(f"\n{'='*60}")
    print(f"📊 Building {tier_label} annotation spreadsheet...")
    print(f"   Samples: {len(df)}")
    print(f"   Layer 1 rows: {len(df)} × 23 = {len(df)*23}")
    print(f"   Layer 2 rows: {len(df)}")

    wb = Workbook()

    # ═══════════════════════════════════════════════════════════════
    # SHEET 1: LAYER 1 — SENTENCE EVALUATION (N × 23 rows)
    # ═══════════════════════════════════════════════════════════════

    ws1 = wb.active
    ws1.title = "Layer 1 — Sentence"
    ws1.sheet_properties.tabColor = DARK_BLUE

    # Column layout for Layer 1:
    # A: eval_id
    # B: Sentence_ID
    # C: en
    # D: ar
    # E: mt_output
    # F: TQA_Category
    # G: Shift_Type
    # H: Sub_type
    # I: Sub-subtype
    # J: V4_Severity
    # K: V4_Score
    # L: SubSubtype_WGT
    # M-Q: Minor, Moderate, Major, Severe, Critical (ANNOTATOR FILLS)
    # R: Count (formula)
    # S: Raw_Penalty (formula)
    # T: Weighted_Penalty (formula)

    L1_COLS = {
        'A': ('eval_id', 14),
        'B': ('Sentence_ID', 18),
        'C': ('Source (EN)', 40),
        'D': ('Reference (AR)', 40),
        'E': ('MT Output', 40),
        'F': ('TQA Category', 16),
        'G': ('Shift Type', 18),
        'H': ('Sub-type', 18),
        'I': ('Sub-subtype', 22),
        'J': ('Severity', 10),
        'K': ('Score', 8),
        'L': ('WGT', 8),
        'M': ('Minor\n(×1)', 9),
        'N': ('Moderate\n(×2)', 9),
        'O': ('Major\n(×3)', 9),
        'P': ('Severe\n(×4)', 9),
        'Q': ('Critical\n(×5)', 9),
        'R': ('Count', 8),
        'S': ('Raw\nPenalty', 10),
        'T': ('Weighted\nPenalty', 10),
    }

    for col_letter, (name, width) in L1_COLS.items():
        ws1.column_dimensions[col_letter].width = width

    # --- Title row ---
    row = 1
    ws1.merge_cells('A1:T1')
    ws1.row_dimensions[1].height = 35
    sc(ws1, 1, 1, f"TRIVET Layer 1: Sentence-Level Evaluation — {tier_label} ({len(df)} samples × 23 Sub-subtypes)",
       f_title, fill_navy, a_left)
    for c in range(2, 21):
        ws1.cell(row=1, column=c).fill = fill_navy

    # --- Gold bar ---
    row = 2
    ws1.row_dimensions[2].height = 4
    for c in range(1, 21):
        ws1.cell(row=2, column=c).fill = fill_gold

    # --- Instructions row ---
    row = 3
    ws1.merge_cells('A3:T3')
    ws1.row_dimensions[3].height = 28
    sc(ws1, 3, 1,
       "⚠ INSTRUCTIONS: For each sample (grouped by eval_id), mark error COUNTS in the blue severity columns (M-Q). "
       "Most rows will be 0 — only mark Sub-subtypes where you find errors. Columns R-T are auto-calculated.",
       Font(name='Arial', size=9, bold=True, color=DARK_BLUE),
       PatternFill('solid', fgColor=LIGHT_GOLD), a_left)
    for c in range(2, 21):
        ws1.cell(row=3, column=c).fill = PatternFill('solid', fgColor=LIGHT_GOLD)

    # --- Header row ---
    row = 4
    ws1.row_dimensions[4].height = 30
    ws1.freeze_panes = 'A5'

    prefilled_cols = list(range(1, 13))  # A-L
    input_cols = list(range(13, 18))     # M-Q
    formula_cols = list(range(18, 21))   # R-T

    for c, (col_letter, (name, _)) in enumerate(L1_COLS.items(), 1):
        if c in prefilled_cols:
            sc(ws1, row, c, name, f_header, fill_dark_blue, a_center, b_thin)
        elif c in input_cols:
            sc(ws1, row, c, name, f_header, fill_med_blue, a_center, b_thin)
        else:
            sc(ws1, row, c, name, f_header, PatternFill('solid', fgColor=ACCENT_GOLD), a_center, b_thin)

    # --- Data rows ---
    data_row = 5
    sample_start_rows = {}

    # Add data validation for severity columns
    dv = DataValidation(type="whole", operator="between", formula1=0, formula2=10,
                        showErrorMessage=True, errorTitle="Invalid", error="Enter 0-10")
    ws1.add_data_validation(dv)

    for idx, sample in df.iterrows():
        eval_id = sample.get('eval_id', f"EVAL_{idx+1:04d}")
        sample_start_rows[eval_id] = data_row

        for tax_idx, tax_row in enumerate(V4_TAXONOMY[:23]):
            ss_name, tqa, tqa_wgt, shift_type, sub_type, definition, severity, score, ss_wgt = tax_row
            r = data_row

            # Alternate row color per sample
            is_first_of_sample = (tax_idx == 0)
            tqa_color = PatternFill('solid', fgColor=TQA_COLORS[tqa]["light"])

            # Pre-filled columns
            sc(ws1, r, 1, eval_id if is_first_of_sample else "", f_body_bold if is_first_of_sample else f_small, fill_light_gray, a_left, b_thin)
            sc(ws1, r, 2, str(sample.get('Sentence_ID', '')) if is_first_of_sample else "", f_small, fill_light_gray, a_left, b_thin)

            # Only show sentence text on first row of each sample
            if is_first_of_sample:
                sc(ws1, r, 3, str(sample.get('en', '')), f_body, fill_white, a_left, b_thin)
                sc(ws1, r, 4, str(sample.get('ar', '')), f_body_ar, fill_white, a_right_ar, b_thin)
                sc(ws1, r, 5, str(sample.get('mt_output', '')), f_body_ar, PatternFill('solid', fgColor="FCE4EC"), a_right_ar, b_thin)
            else:
                for c in [3, 4, 5]:
                    sc(ws1, r, c, "", f_body, fill_light_gray, a_left, b_thin)

            # Taxonomy columns
            sc(ws1, r, 6, tqa, f_body_bold, tqa_color, a_left, b_thin)
            sc(ws1, r, 7, shift_type, f_body, tqa_color, a_left, b_thin)
            sc(ws1, r, 8, sub_type, f_body, tqa_color, a_left, b_thin)
            sc(ws1, r, 9, ss_name, f_body_bold, fill_white, a_left, b_thin)
            sc(ws1, r, 10, severity, f_small, fill_white, a_center, b_thin)
            sc(ws1, r, 11, score, f_small, fill_white, a_center, b_thin)
            sc(ws1, r, 12, f"{ss_wgt:.1%}" if ss_wgt < 1 else "100%", f_small, fill_white, a_center, b_thin)

            # Input columns (blue) — annotator fills
            for c in range(13, 18):
                sc(ws1, r, c, "", f_input, fill_input, a_center, b_thin)
                dv.add(ws1.cell(row=r, column=c))

            # Formula columns
            # R: Count = SUM(M:Q)
            sc(ws1, r, 18, f'=SUM(M{r}:Q{r})', f_body, fill_light_gold, a_center, b_thin)
            # S: Raw Penalty = M*1 + N*2 + O*3 + P*4 + Q*5
            sc(ws1, r, 19, f'=M{r}*1+N{r}*2+O{r}*3+P{r}*4+Q{r}*5', f_body, fill_light_gold, a_center, b_thin)
            # T: Weighted = Raw × SubSubtype_WGT
            sc(ws1, r, 20, f'=S{r}*{ss_wgt}', f_body, fill_light_gold, a_center, b_thin)

            data_row += 1

        # Sample separator: thick bottom border on last row
        for c in range(1, 21):
            cell = ws1.cell(row=data_row-1, column=c)
            cell.border = Border(left=thin, right=thin, top=thin, bottom=medium_side)

    total_L1_rows = data_row - 5
    print(f"   ✅ Layer 1: {total_L1_rows} rows written")

    # ═══════════════════════════════════════════════════════════════
    # SHEET 2: LAYER 2 — KEYWORD EVALUATION (N rows)
    # ═══════════════════════════════════════════════════════════════

    ws2 = wb.create_sheet("Layer 2 — Keyword")
    ws2.sheet_properties.tabColor = ACCENT_GOLD

    L2_COLS = {
        'A': ('eval_id', 14),
        'B': ('Sentence_ID', 18),
        'C': ('en', 35),
        'D': ('ar', 35),
        'E': ('mt_output', 35),
        'F': ('Keyword', 16),
        'G': ('Best_Match', 16),
        'H': ('Sub-Subtype\n(Our Label)', 20),
        'I': ('Shift_Type', 16),
        'J': ('V4_Severity', 10),
        'K': ('V4_Score', 8),
        'L': ('V4_WGT', 8),
        'M': ('Error_\nCorrect\n(0/1)', 10),
        'N': ('Anchor_\nCorrect\n(0/1)', 10),
        'O': ('Keyword_\nSeverity\n(0/0.3/0.7/1)', 11),
        'P': ('Minimal_\nPair\n(0/1)', 10),
        'Q': ('Label_\nCorrect\n(0/1)', 10),
        'R': ('Naturalness\n(1-5)', 10),
        'S': ('UN_\nAppropriate\n(0/1)', 10),
        'T': ('Notes', 25),
        'U': ('Annotator\n_ID', 10),
        'V': ('Time\n(mins)', 8),
    }

    for col_letter, (name, width) in L2_COLS.items():
        ws2.column_dimensions[col_letter].width = width

    # Title
    row = 1
    ws2.merge_cells('A1:V1')
    ws2.row_dimensions[1].height = 35
    sc(ws2, 1, 1, f"TRIVET Layer 2: Keyword-Level Evaluation — {tier_label} ({len(df)} samples)",
       f_title, fill_navy, a_left)
    for c in range(2, 23):
        ws2.cell(row=1, column=c).fill = fill_navy

    # Gold bar
    ws2.row_dimensions[2].height = 4
    for c in range(1, 23):
        ws2.cell(row=2, column=c).fill = fill_gold

    # Instructions
    ws2.merge_cells('A3:V3')
    ws2.row_dimensions[3].height = 28
    sc(ws2, 3, 1,
       "⚠ INSTRUCTIONS: We show our classification (H-L). Fill the blue columns (M-S) to validate. "
       "ETCA correlation columns: M↔phenomenon_clarity, N↔anchor_validity, P↔collateral_severity, R↔arabic_naturalness, Q↔etca_label_correctness",
       Font(name='Arial', size=9, bold=True, color=DARK_BLUE),
       PatternFill('solid', fgColor=LIGHT_GOLD), a_left)
    for c in range(2, 23):
        ws2.cell(row=3, column=c).fill = PatternFill('solid', fgColor=LIGHT_GOLD)

    # Header
    row = 4
    ws2.row_dimensions[4].height = 40
    ws2.freeze_panes = 'A5'

    prefilled = list(range(1, 13))
    inputs = list(range(13, 22))
    meta = [21, 22]

    for c, (col_letter, (name, _)) in enumerate(L2_COLS.items(), 1):
        if c in prefilled:
            sc(ws2, row, c, name, f_header, fill_dark_blue, a_center, b_thin)
        elif c in inputs:
            sc(ws2, row, c, name, f_header, fill_med_blue, a_center, b_thin)
        else:
            sc(ws2, row, c, name, f_header, PatternFill('solid', fgColor=MED_GRAY), a_center, b_thin)

    # Data validations
    dv_binary = DataValidation(type="list", formula1='"0,1"', showErrorMessage=True)
    dv_severity = DataValidation(type="list", formula1='"0,0.3,0.7,1.0"', showErrorMessage=True)
    dv_nat = DataValidation(type="list", formula1='"1,2,3,4,5"', showErrorMessage=True)
    ws2.add_data_validation(dv_binary)
    ws2.add_data_validation(dv_severity)
    ws2.add_data_validation(dv_nat)

    # Data rows
    for idx, sample in df.iterrows():
        r = 5 + idx
        ws2.row_dimensions[r].height = 50
        eval_id = sample.get('eval_id', f"EVAL_{idx+1:04d}")
        alt = fill_white if idx % 2 == 0 else PatternFill('solid', fgColor="F9F9F9")

        # Pre-filled
        sc(ws2, r, 1, eval_id, f_body_bold, alt, a_left, b_thin)
        sc(ws2, r, 2, str(sample.get('Sentence_ID', '')), f_small, alt, a_left, b_thin)
        sc(ws2, r, 3, str(sample.get('en', '')), f_body, alt, a_left, b_thin)
        sc(ws2, r, 4, str(sample.get('ar', '')), f_body_ar, alt, a_right_ar, b_thin)
        sc(ws2, r, 5, str(sample.get('mt_output', '')), f_body_ar, PatternFill('solid', fgColor="FCE4EC"), a_right_ar, b_thin)
        sc(ws2, r, 6, str(sample.get('Keyword', '')), f_body_bold, alt, a_right_ar, b_thin)
        sc(ws2, r, 7, str(sample.get('Best_Match', '')), f_body_bold, alt, a_right_ar, b_thin)
        sc(ws2, r, 8, str(sample.get('Sub-Subtype', '')), f_body_bold,
           PatternFill('solid', fgColor=TQA_COLORS.get(sample.get('TQA_Category','Accuracy'), {"light": LIGHT_BLUE})["light"]),
           a_left, b_thin)
        sc(ws2, r, 9, str(sample.get('Shift_Type', '')), f_body, alt, a_left, b_thin)
        sc(ws2, r, 10, str(sample.get('V4_Severity', '')), f_body, alt, a_center, b_thin)
        sc(ws2, r, 11, sample.get('V4_Score', ''), f_body, alt, a_center, b_thin)
        wgt = sample.get('SubSubtype_WGT', '')
        sc(ws2, r, 12, f"{wgt:.1%}" if isinstance(wgt, (int, float)) and not pd.isna(wgt) else str(wgt),
           f_body, alt, a_center, b_thin)

        # Input columns (blue)
        for c in [13, 14, 16, 17, 19]:  # binary columns
            sc(ws2, r, c, "", f_input, fill_input, a_center, b_thin)
            dv_binary.add(ws2.cell(row=r, column=c))

        sc(ws2, r, 15, "", f_input, fill_input, a_center, b_thin)  # Keyword_Severity
        dv_severity.add(ws2.cell(row=r, column=15))

        sc(ws2, r, 18, "", f_input, fill_input, a_center, b_thin)  # Naturalness
        dv_nat.add(ws2.cell(row=r, column=18))

        sc(ws2, r, 20, "", f_input, fill_input, a_left, b_thin)    # Notes
        sc(ws2, r, 21, "", f_input, fill_input, a_center, b_thin)  # Annotator_ID
        sc(ws2, r, 22, "", f_input, fill_input, a_center, b_thin)  # Time

    print(f"   ✅ Layer 2: {len(df)} rows written")

    # ═══════════════════════════════════════════════════════════════
    # SHEET 3: V4 TAXONOMY REFERENCE
    # ═══════════════════════════════════════════════════════════════

    ws3 = wb.create_sheet("V4 Taxonomy Reference")
    ws3.sheet_properties.tabColor = MED_BLUE

    ref_cols = ['#', 'TQA Category', 'TQA Weight', 'Shift Type', 'Sub-type',
                'Sub-subtype', 'Definition', 'Severity', 'Score', 'WGT']
    ref_widths = [5, 16, 10, 18, 18, 22, 40, 10, 8, 10]

    for i, w in enumerate(ref_widths):
        ws3.column_dimensions[get_column_letter(i+1)].width = w

    ws3.merge_cells('A1:J1')
    ws3.row_dimensions[1].height = 30
    sc(ws3, 1, 1, "TRIVET V4 Linguistic Classification — 23-Class Reference", f_title, fill_navy, a_left)
    for c in range(2, 11): ws3.cell(row=1, column=c).fill = fill_navy

    for i, h in enumerate(ref_cols):
        sc(ws3, 2, i+1, h, f_header, fill_dark_blue, a_center, b_thin)

    for num, tax_row in enumerate(V4_TAXONOMY[:23], 1):
        r = num + 2
        ss_name, tqa, tqa_wgt, shift_type, sub_type, definition, severity, score, ss_wgt = tax_row
        tqa_fill = PatternFill('solid', fgColor=TQA_COLORS[tqa]["light"])
        sc(ws3, r, 1, num, f_body, tqa_fill, a_center, b_thin)
        sc(ws3, r, 2, tqa, f_body_bold, tqa_fill, a_left, b_thin)
        sc(ws3, r, 3, f"{tqa_wgt:.1%}", f_body, tqa_fill, a_center, b_thin)
        sc(ws3, r, 4, shift_type, f_body, tqa_fill, a_left, b_thin)
        sc(ws3, r, 5, sub_type, f_body, tqa_fill, a_left, b_thin)
        sc(ws3, r, 6, ss_name, f_body_bold, fill_white, a_left, b_thin)
        sc(ws3, r, 7, definition, f_body, fill_white, a_left, b_thin)
        sc(ws3, r, 8, severity, f_body, fill_white, a_center, b_thin)
        sc(ws3, r, 9, score, f_body, fill_white, a_center, b_thin)
        sc(ws3, r, 10, f"{ss_wgt:.1%}" if ss_wgt < 1 else "100%", f_small, fill_white, a_center, b_thin)

    # TQA weight summary
    r = 26
    ws3.merge_cells(f'A{r}:J{r}')
    sc(ws3, r, 1, "TQA CATEGORY WEIGHTS (V4)", f_section, fill_navy, a_left)
    for c in range(2, 11): ws3.cell(row=r, column=c).fill = fill_navy
    for i, (tqa, wgt) in enumerate([("Accuracy","42%"),("Fluency","22.5%"),("Terminology","28%"),("Style & Register","7.5%")]):
        sc(ws3, r+1+i, 2, tqa, f_body_bold, fill_white, a_left, b_thin)
        sc(ws3, r+1+i, 3, wgt, f_body, fill_light_gold, a_center, b_thin)

    # Severity reference
    r = 32
    ws3.merge_cells(f'A{r}:J{r}')
    sc(ws3, r, 1, "SEVERITY MULTIPLIERS", f_section, fill_navy, a_left)
    for c in range(2, 11): ws3.cell(row=r, column=c).fill = fill_navy
    for i, (sev, mult) in enumerate(zip(SEVERITY_LEVELS, SEVERITY_MULTIPLIERS)):
        sc(ws3, r+1+i, 2, sev, f_body_bold, fill_white, a_left, b_thin)
        sc(ws3, r+1+i, 3, f"×{mult}", f_body, fill_light_gold, a_center, b_thin)

    # ═══════════════════════════════════════════════════════════════
    # SHEET 4: INSTRUCTIONS
    # ═══════════════════════════════════════════════════════════════

    ws4 = wb.create_sheet("Instructions")
    ws4.sheet_properties.tabColor = GREEN_ACCENT
    ws4.column_dimensions['A'].width = 4
    ws4.column_dimensions['B'].width = 110

    ws4.merge_cells('A1:B1')
    ws4.row_dimensions[1].height = 35
    sc(ws4, 1, 1, "TRIVET Evaluation Framework — Annotator Instructions", f_title, fill_navy, a_left)
    ws4.cell(row=1, column=2).fill = fill_navy

    instructions = [
        ("OVERVIEW", [
            "This spreadsheet evaluates Arabic MT quality using a two-layer modular approach.",
            "Layer 1 (Sheet 1): Sentence-level — You find errors independently using the V4 taxonomy grid.",
            "Layer 2 (Sheet 2): Keyword-level — We show our classification, you validate it.",
            "",
            "⚠ IMPORTANT: Complete Layer 1 BEFORE looking at Layer 2.",
            "Layer 1 does NOT show our classification — you evaluate independently.",
            "Layer 2 reveals our Keyword, Best_Match, and Sub-Subtype for validation.",
        ]),
        ("LAYER 1: SENTENCE EVALUATION", [
            "Each sample has 23 rows (one per Sub-subtype in V4 taxonomy).",
            "Samples are separated by thick borders and grouped by eval_id.",
            "",
            "Steps:",
            "1. Read the Source (EN), Reference (AR), and MT Output for each sample",
            "2. Compare MT Output against the Reference — what errors do you see?",
            "3. For each error found, go to the matching Sub-subtype row",
            "4. Enter a COUNT (1, 2, etc.) in the appropriate SEVERITY column (M-Q)",
            "5. Most samples will have 1 error. Mark additional errors if you find them.",
            "6. Leave all other cells at 0 (empty = 0)",
            "",
            "Columns R-T are auto-calculated. Do NOT edit them.",
        ]),
        ("LAYER 2: KEYWORD EVALUATION", [
            "One row per sample. We now reveal our classification.",
            "",
            "Fill the BLUE columns:",
            "  Error_Correct (0/1): Is the claimed Sub-Subtype error actually present in mt_output?",
            "  Anchor_Correct (0/1): Does Keyword→Best_Match correctly identify the error location?",
            "  Keyword_Severity (0/0.3/0.7/1.0): How severe is the keyword-level error?",
            "    0 = No error | 0.3 = Minor/stylistic | 0.7 = Major/meaning affected | 1.0 = Critical",
            "  Minimal_Pair (0/1): Is the ONLY change between ar and mt_output at the keyword?",
            "  Label_Correct (0/1): Is the Sub-Subtype classification correct?",
            "  Naturalness (1-5): How natural does the Arabic sound? (1=broken, 5=native-level)",
            "  UN_Appropriate (0/1): Does the text read like UN/diplomatic language?",
        ]),
        ("SEVERITY GUIDE (Layer 1)", [
            "Minor (×1):    Spelling, style — no meaning change",
            "Moderate (×2):  Noticeable issue, minor meaning impact — definiteness, tanween",
            "Major (×3):     Clear grammar or meaning error — wrong structure, word order",
            "Severe (×4):    Significant meaning change — wrong tense, partial translation",
            "Critical (×5):  Complete meaning distortion — meaning shift, total omission",
        ]),
        ("QUALITY BANDS", [
            "90-100: ★★★★★ Excellent — Publishable quality",
            "70-89:  ★★★★  Good — Light revision needed",
            "50-69:  ★★★   Fair — Major revision needed",
            "<50:    ★★    Poor — Retranslation required",
        ]),
    ]

    r = 3
    for title, lines in instructions:
        ws4.merge_cells(f'A{r}:B{r}')
        ws4.row_dimensions[r].height = 24
        sc(ws4, r, 1, title, f_section, fill_dark_blue, a_left)
        ws4.cell(row=r, column=2).fill = fill_dark_blue
        r += 1
        for line in lines:
            ws4.row_dimensions[r].height = 18
            ws4.merge_cells(f'A{r}:B{r}')
            sc(ws4, r, 1, line, f_body, fill_white, a_left)
            r += 1
        r += 1

    # ═══════════════════════════════════════════════════════════════
    # SAVE
    # ═══════════════════════════════════════════════════════════════

    wb.save(output_path)
    print(f"   ✅ Saved: {output_path}")
    return len(df)

# ═══════════════════════════════════════════════════════════════════
# GENERATE BOTH FILES
# ═══════════════════════════════════════════════════════════════════

n1 = build_annotation_excel(df1, TIER1_OUTPUT, "TIER 1 (Real MT Errors)")
n2 = build_annotation_excel(df2, TIER2_OUTPUT, "TIER 2 (Synthetic Errors)")

# ═══════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("🎉 CELL 19B COMPLETE")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   {TIER1_OUTPUT}  ({n1} samples, {n1*23} Layer1 rows)
   {TIER2_OUTPUT}  ({n2} samples, {n2*23} Layer1 rows)

📊 STRUCTURE PER FILE:
   Sheet 1: "Layer 1 — Sentence"     N × 23 rows (full taxonomy grid)
   Sheet 2: "Layer 2 — Keyword"      N × 1 row (keyword validation)
   Sheet 3: "V4 Taxonomy Reference"  23-class definitions
   Sheet 4: "Instructions"           Annotator guide

📋 ANNOTATOR WORKFLOW:
   1. Open file → Read Instructions (Sheet 4)
   2. Complete Layer 1 (Sheet 1) FIRST — find errors independently
   3. Then complete Layer 2 (Sheet 2) — validate our classification
   4. Do NOT look at Layer 2 before finishing Layer 1

📐 WHAT GETS CALCULATED (Cell 20):
   • Sentence_Score = 100 - Normed_Penalty (from Layer 1)
   • KPS = 100 - min(100, λ × Severity × Weight / Length) (from Layer 2)
   • TRIVET_Score = Sentence × 0.70 + KPS × 0.30
   • Data_Quality = mean(Error_Correct, Anchor_Correct, Minimal_Pair, Label_Correct)
   • ETCA Correlation (6 dimensions)

✅ GIVE TO EVALUATION TEAM:
   1. TRIVET_Annotation_Tier1.xlsx
   2. TRIVET_Annotation_Tier2.xlsx
   3. TRIVET_MQM_V4_Evaluation_Scorecard.xlsx (as rubric/reference)
""")

In [ ]:
# =============================================================================
# CELL 20: QUALITY CALCULATION (10-Column Schema)
# =============================================================================
"""
CELL 20: Quality Calculation from Human Evaluation

PURPOSE:
Process completed evaluation forms and compute all quality metrics.

INPUT FILES (from annotators):
- tier1_v3_3_final/TIER1_HUMANEVAL_FINAL_FILLED.csv
- tier1_v3_3_final/TIER2_HUMANEVAL_FINAL_FILLED.csv

EVALUATION COLUMNS (10):
1. Adequacy (0-40)
2. Fluency (0-30)
3. Terminology (0-20)
4. Locale (0-10)
5. Keyword_Severity (0/0.3/0.7/1.0)
6. Error_Correct (0/1)
7. Anchor_Correct (0/1)
8. Label_Correct (0/1)
9. Naturalness (1-5)
10. Notes (text)

CALCULATED METRICS:
- Sentence_Score = Adequacy + Fluency + Terminology + Locale (0-100)
- KPS = 100 - (Keyword_Severity × Class_Weight × λ) (0-100)
- TRIVET_Score = (Sentence_Score × 0.7) + (KPS × 0.3) (0-100)
- Data_Quality = mean(Error_Correct, Anchor_Correct) (0-1)
- Label_Accuracy = mean(Label_Correct) (0-1)

OUTPUTS:
- tier1_v3_3_final/TIER1_HUMANEVAL_SCORED.csv
- tier1_v3_3_final/TIER2_HUMANEVAL_SCORED.csv
- tier1_v3_3_final/humaneval_quality_report.json
- tier1_v3_3_final/humaneval_paper_ready.txt
"""

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime
from pathlib import Path

print("=" * 80)
print("CELL 20: QUALITY CALCULATION (10-Column Schema)")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# INPUT FILES (filled by annotators)
TIER1_FILLED = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_FINAL_FILLED.csv"
TIER2_FILLED = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_FINAL_FILLED.csv"

# Alternative input paths
TIER1_FILLED_ALT = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_FINAL.csv"
TIER2_FILLED_ALT = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_FINAL.csv"

# OUTPUT FILES
OUT_TIER1_SCORED = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_SCORED.csv"
OUT_TIER2_SCORED = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_SCORED.csv"
OUT_REPORT = f"{OUTPUT_DIR}/humaneval_quality_report.json"
OUT_PAPER = f"{OUTPUT_DIR}/humaneval_paper_ready.txt"
OUT_PER_CLASS = f"{OUTPUT_DIR}/humaneval_per_class_breakdown.csv"

# KPS PARAMETERS
LAMBDA_SCALING = 20  # Scaling factor for KPS

# CLASS WEIGHTS (from taxonomy)
# These should match your 23-class taxonomy weights
CLASS_WEIGHTS = {
    "Meaning Shift": 1.00,
    "Total Omission": 0.90,
    "Partial Translation": 0.20,
    "Name Entity Error": 0.95,
    "Terminology Substitution": 0.80,
    "Literal Translation": 0.65,
    "Hypernym for Hyponym": 0.70,
    "Hyponym for Hypernym": 0.70,
    "Tanween Omission": 0.30,
    "Gender Disagreement": 0.55,
    "Definiteness Shift": 0.50,
    "Perfective to Progressive": 0.60,
    "Progressive to Perfective": 0.60,
    "Tense Shift Under Negation": 0.75,
    "Wrong Structure": 0.65,
    "Wrong Word Order": 0.60,
    "Active to Passive Voice": 0.55,
    "Passive to Active Voice": 0.55,
    "Noun to Adjective": 0.50,
    "Adjective to Noun": 0.50,
    "Register Mismatch": 0.45,
    "Spelling Error": 0.25,
    "Invalid Pattern": 0.35,
}
DEFAULT_CLASS_WEIGHT = 0.50

print(f"\n📊 CONFIGURATION:")
print(f"   Lambda (KPS scaling): {LAMBDA_SCALING}")
print(f"   Class weights defined: {len(CLASS_WEIGHTS)}")

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

def load_filled(primary_path: str, alt_path: str, name: str) -> pd.DataFrame:
    """Load filled evaluation file."""
    if Path(primary_path).exists():
        df = pd.read_csv(primary_path)
        print(f"✅ Loaded {name}: {primary_path} ({len(df):,} rows)")
        return df
    elif Path(alt_path).exists():
        df = pd.read_csv(alt_path)
        print(f"✅ Loaded {name} (alt): {alt_path} ({len(df):,} rows)")
        return df
    else:
        print(f"⚠️ {name} not found")
        return pd.DataFrame()

df_tier1 = load_filled(TIER1_FILLED, TIER1_FILLED_ALT, "Tier 1")
df_tier2 = load_filled(TIER2_FILLED, TIER2_FILLED_ALT, "Tier 2")

# Remove instruction row if present
def remove_instruction_row(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if 'eval_id' in df.columns:
        df = df[df['eval_id'] != 'INSTRUCTIONS'].copy()
    return df

df_tier1 = remove_instruction_row(df_tier1)
df_tier2 = remove_instruction_row(df_tier2)

print(f"\n📊 Data after removing instructions:")
print(f"   Tier 1: {len(df_tier1):,} samples")
print(f"   Tier 2: {len(df_tier2):,} samples")

# =============================================================================
# STEP 2: VALIDATE DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: VALIDATE DATA")
print("=" * 80)

REQUIRED_COLUMNS = [
    'Adequacy', 'Fluency', 'Terminology', 'Locale',
    'Keyword_Severity', 'Error_Correct', 'Anchor_Correct',
    'Label_Correct', 'Naturalness'
]

def validate_data(df: pd.DataFrame, name: str) -> dict:
    """Validate evaluation data and report issues."""
    if df.empty:
        return {"valid": False, "issues": ["Empty dataframe"]}

    issues = []

    # Check required columns
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        issues.append(f"Missing columns: {missing_cols}")

    # Check for empty values
    for col in REQUIRED_COLUMNS:
        if col in df.columns:
            empty_count = df[col].isna().sum() + (df[col] == '').sum()
            if empty_count > 0:
                issues.append(f"{col}: {empty_count} empty values")

    # Check value ranges
    if 'Adequacy' in df.columns:
        invalid = ((df['Adequacy'].astype(float) < 0) | (df['Adequacy'].astype(float) > 40)).sum()
        if invalid > 0:
            issues.append(f"Adequacy: {invalid} out of range (0-40)")

    if 'Fluency' in df.columns:
        invalid = ((df['Fluency'].astype(float) < 0) | (df['Fluency'].astype(float) > 30)).sum()
        if invalid > 0:
            issues.append(f"Fluency: {invalid} out of range (0-30)")

    if 'Terminology' in df.columns:
        invalid = ((df['Terminology'].astype(float) < 0) | (df['Terminology'].astype(float) > 20)).sum()
        if invalid > 0:
            issues.append(f"Terminology: {invalid} out of range (0-20)")

    if 'Locale' in df.columns:
        invalid = ((df['Locale'].astype(float) < 0) | (df['Locale'].astype(float) > 10)).sum()
        if invalid > 0:
            issues.append(f"Locale: {invalid} out of range (0-10)")

    print(f"\n📋 {name} Validation:")
    if issues:
        for issue in issues:
            print(f"   ⚠️ {issue}")
    else:
        print(f"   ✅ All validations passed")

    return {"valid": len(issues) == 0, "issues": issues}

validation_tier1 = validate_data(df_tier1, "Tier 1")
validation_tier2 = validate_data(df_tier2, "Tier 2")

# =============================================================================
# STEP 3: CALCULATE SCORES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: CALCULATE SCORES")
print("=" * 80)

def calculate_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate all derived scores from evaluation columns."""
    if df.empty:
        return df

    df = df.copy()

    # Convert columns to numeric (handle empty strings)
    numeric_cols = ['Adequacy', 'Fluency', 'Terminology', 'Locale',
                    'Keyword_Severity', 'Error_Correct', 'Anchor_Correct',
                    'Label_Correct', 'Naturalness']

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 1. SENTENCE SCORE (MQM-aligned, 0-100)
    df['Sentence_Score'] = (
        df['Adequacy'].fillna(0) +
        df['Fluency'].fillna(0) +
        df['Terminology'].fillna(0) +
        df['Locale'].fillna(0)
    )

    # 2. Get class weight for each sample
    def get_class_weight(error_type):
        return CLASS_WEIGHTS.get(error_type, DEFAULT_CLASS_WEIGHT)

    if 'Sub-Subtype' in df.columns:
        df['Class_Weight'] = df['Sub-Subtype'].apply(get_class_weight)
    else:
        df['Class_Weight'] = DEFAULT_CLASS_WEIGHT

    # 3. KPS (Keyword Preservation Score, 0-100)
    # KPS = 100 - (Keyword_Severity × Class_Weight × λ)
    df['KPS'] = 100 - (
        df['Keyword_Severity'].fillna(0) *
        df['Class_Weight'] *
        LAMBDA_SCALING
    )
    df['KPS'] = df['KPS'].clip(0, 100)  # Ensure 0-100 range

    # 4. TRIVET SCORE (Combined, 0-100)
    # TRIVET = (Sentence_Score × 0.7) + (KPS × 0.3)
    df['TRIVET_Score'] = (
        (df['Sentence_Score'] * 0.7) +
        (df['KPS'] * 0.3)
    )

    # 5. DATA QUALITY (0-1)
    df['Data_Quality'] = (
        df['Error_Correct'].fillna(0) +
        df['Anchor_Correct'].fillna(0)
    ) / 2

    # 6. VALIDATION STATUS
    # Accept if: Error_Correct=1 AND Anchor_Correct=1 AND TRIVET_Score >= 70
    df['Status'] = 'REVIEW'
    accept_mask = (
        (df['Error_Correct'] == 1) &
        (df['Anchor_Correct'] == 1) &
        (df['TRIVET_Score'] >= 70)
    )
    df.loc[accept_mask, 'Status'] = 'ACCEPT'

    return df

df_tier1 = calculate_scores(df_tier1)
df_tier2 = calculate_scores(df_tier2)

print("✅ Scores calculated for all samples")

# =============================================================================
# STEP 4: COMPUTE AGGREGATE METRICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: COMPUTE AGGREGATE METRICS")
print("=" * 80)

def compute_metrics(df: pd.DataFrame, name: str) -> dict:
    """Compute aggregate metrics for a tier."""
    if df.empty:
        return {"name": name, "n_samples": 0}

    metrics = {
        "name": name,
        "n_samples": len(df),

        # Sentence Quality (MQM)
        "adequacy_mean": round(df['Adequacy'].mean(), 2),
        "fluency_mean": round(df['Fluency'].mean(), 2),
        "terminology_mean": round(df['Terminology'].mean(), 2),
        "locale_mean": round(df['Locale'].mean(), 2),
        "sentence_score_mean": round(df['Sentence_Score'].mean(), 2),
        "sentence_score_std": round(df['Sentence_Score'].std(), 2),

        # Keyword Impact
        "keyword_severity_mean": round(df['Keyword_Severity'].mean(), 3),
        "kps_mean": round(df['KPS'].mean(), 2),
        "kps_std": round(df['KPS'].std(), 2),

        # TRIVET Score
        "trivet_score_mean": round(df['TRIVET_Score'].mean(), 2),
        "trivet_score_std": round(df['TRIVET_Score'].std(), 2),
        "trivet_score_min": round(df['TRIVET_Score'].min(), 2),
        "trivet_score_max": round(df['TRIVET_Score'].max(), 2),

        # Data Validation
        "error_correct_rate": round(df['Error_Correct'].mean(), 3),
        "anchor_correct_rate": round(df['Anchor_Correct'].mean(), 3),
        "label_correct_rate": round(df['Label_Correct'].mean(), 3),
        "data_quality_mean": round(df['Data_Quality'].mean(), 3),

        # Quality
        "naturalness_mean": round(df['Naturalness'].mean(), 2),

        # Acceptance
        "accept_count": int((df['Status'] == 'ACCEPT').sum()),
        "accept_rate": round((df['Status'] == 'ACCEPT').mean(), 3),
    }

    print(f"\n📊 {name} METRICS:")
    print(f"   Samples: {metrics['n_samples']}")
    print(f"   Sentence Score: {metrics['sentence_score_mean']:.1f} ± {metrics['sentence_score_std']:.1f}")
    print(f"   KPS: {metrics['kps_mean']:.1f} ± {metrics['kps_std']:.1f}")
    print(f"   TRIVET Score: {metrics['trivet_score_mean']:.1f} ± {metrics['trivet_score_std']:.1f}")
    print(f"   Error Correct: {metrics['error_correct_rate']*100:.1f}%")
    print(f"   Anchor Correct: {metrics['anchor_correct_rate']*100:.1f}%")
    print(f"   Label Correct: {metrics['label_correct_rate']*100:.1f}%")
    print(f"   Accept Rate: {metrics['accept_rate']*100:.1f}%")

    return metrics

metrics_tier1 = compute_metrics(df_tier1, "Tier 1")
metrics_tier2 = compute_metrics(df_tier2, "Tier 2")

# =============================================================================
# STEP 5: PER-CLASS BREAKDOWN
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: PER-CLASS BREAKDOWN")
print("=" * 80)

def per_class_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    """Compute metrics per error class."""
    if df.empty or 'Sub-Subtype' not in df.columns:
        return pd.DataFrame()

    rows = []
    for cls in df['Sub-Subtype'].unique():
        cls_df = df[df['Sub-Subtype'] == cls]

        rows.append({
            'Sub-Subtype': cls,
            'n_samples': len(cls_df),
            'sentence_score_mean': round(cls_df['Sentence_Score'].mean(), 1),
            'kps_mean': round(cls_df['KPS'].mean(), 1),
            'trivet_score_mean': round(cls_df['TRIVET_Score'].mean(), 1),
            'error_correct_rate': round(cls_df['Error_Correct'].mean(), 2),
            'anchor_correct_rate': round(cls_df['Anchor_Correct'].mean(), 2),
            'label_correct_rate': round(cls_df['Label_Correct'].mean(), 2),
            'accept_rate': round((cls_df['Status'] == 'ACCEPT').mean(), 2),
        })

    return pd.DataFrame(rows).sort_values('n_samples', ascending=False)

# Combine all data for per-class analysis
all_data = []
if not df_tier1.empty:
    all_data.append(df_tier1)
if not df_tier2.empty:
    all_data.append(df_tier2)

if all_data:
    df_all = pd.concat(all_data, ignore_index=True)
    per_class_df = per_class_breakdown(df_all)

    if not per_class_df.empty:
        print("\n📊 PER-CLASS BREAKDOWN (Top 10):")
        print("-" * 100)
        print(f"{'Class':<30} {'N':>5} {'Sent':>6} {'KPS':>6} {'TRIVET':>7} {'Err%':>6} {'Anch%':>6} {'Lbl%':>6}")
        print("-" * 100)
        for _, row in per_class_df.head(10).iterrows():
            print(f"{row['Sub-Subtype'][:28]:<30} {row['n_samples']:>5} "
                  f"{row['sentence_score_mean']:>6.1f} {row['kps_mean']:>6.1f} "
                  f"{row['trivet_score_mean']:>7.1f} {row['error_correct_rate']*100:>5.0f}% "
                  f"{row['anchor_correct_rate']*100:>5.0f}% {row['label_correct_rate']*100:>5.0f}%")
        print("-" * 100)

# =============================================================================
# STEP 6: SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6: SAVE OUTPUTS")
print("=" * 80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save scored files
if not df_tier1.empty:
    df_tier1.to_csv(OUT_TIER1_SCORED, index=False, encoding='utf-8-sig')
    print(f"✅ Saved: {OUT_TIER1_SCORED}")

if not df_tier2.empty:
    df_tier2.to_csv(OUT_TIER2_SCORED, index=False, encoding='utf-8-sig')
    print(f"✅ Saved: {OUT_TIER2_SCORED}")

# Save per-class breakdown
if not per_class_df.empty:
    per_class_df.to_csv(OUT_PER_CLASS, index=False)
    print(f"✅ Saved: {OUT_PER_CLASS}")

# =============================================================================
# STEP 7: SAVE REPORT JSON
# =============================================================================

print("\n" + "=" * 80)
print("STEP 7: SAVE REPORT")
print("=" * 80)

report = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "schema": "10-column (MQM + Keyword + Validation)",
        "lambda_scaling": LAMBDA_SCALING,
    },
    "tier1": metrics_tier1,
    "tier2": metrics_tier2,
    "combined": {
        "total_samples": metrics_tier1.get('n_samples', 0) + metrics_tier2.get('n_samples', 0),
        "overall_accept_rate": round(
            (metrics_tier1.get('accept_count', 0) + metrics_tier2.get('accept_count', 0)) /
            (metrics_tier1.get('n_samples', 1) + metrics_tier2.get('n_samples', 1)), 3
        ) if (metrics_tier1.get('n_samples', 0) + metrics_tier2.get('n_samples', 0)) > 0 else 0,
    },
    "formulas": {
        "Sentence_Score": "Adequacy + Fluency + Terminology + Locale (0-100)",
        "KPS": f"100 - (Keyword_Severity × Class_Weight × {LAMBDA_SCALING})",
        "TRIVET_Score": "(Sentence_Score × 0.7) + (KPS × 0.3)",
        "Data_Quality": "mean(Error_Correct, Anchor_Correct)",
        "Accept_Criteria": "Error_Correct=1 AND Anchor_Correct=1 AND TRIVET_Score >= 70",
    },
    "validation": {
        "tier1": validation_tier1,
        "tier2": validation_tier2,
    },
}

with open(OUT_REPORT, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print(f"✅ Saved: {OUT_REPORT}")

# =============================================================================
# STEP 8: GENERATE PAPER-READY STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 8: PAPER-READY STATISTICS")
print("=" * 80)

paper_stats = f"""
================================================================================
HUMAN EVALUATION RESULTS - PAPER-READY STATISTICS
Generated: {datetime.now().isoformat()}
================================================================================

METHODOLOGY:
We employed a 10-column evaluation schema combining MQM-standard sentence-level
quality assessment with a novel Keyword Preservation Score (KPS). Human annotators
evaluated {metrics_tier1.get('n_samples', 0) + metrics_tier2.get('n_samples', 0):,} samples
({metrics_tier1.get('n_samples', 0):,} Tier 1, {metrics_tier2.get('n_samples', 0):,} Tier 2).

================================================================================
TABLE 1: SENTENCE QUALITY (MQM-Aligned)
================================================================================

                        Tier 1          Tier 2
                        ───────         ───────
Adequacy (/40)          {metrics_tier1.get('adequacy_mean', 0):>6.1f}          {metrics_tier2.get('adequacy_mean', 0):>6.1f}
Fluency (/30)           {metrics_tier1.get('fluency_mean', 0):>6.1f}          {metrics_tier2.get('fluency_mean', 0):>6.1f}
Terminology (/20)       {metrics_tier1.get('terminology_mean', 0):>6.1f}          {metrics_tier2.get('terminology_mean', 0):>6.1f}
Locale (/10)            {metrics_tier1.get('locale_mean', 0):>6.1f}          {metrics_tier2.get('locale_mean', 0):>6.1f}
───────────────────────────────────────────────────────
Sentence Score (/100)   {metrics_tier1.get('sentence_score_mean', 0):>6.1f}          {metrics_tier2.get('sentence_score_mean', 0):>6.1f}

================================================================================
TABLE 2: TRIVET QUALITY METRIC (Novel)
================================================================================

                        Tier 1          Tier 2
                        ───────         ───────
Keyword Severity        {metrics_tier1.get('keyword_severity_mean', 0):>6.2f}          {metrics_tier2.get('keyword_severity_mean', 0):>6.2f}
KPS (/100)              {metrics_tier1.get('kps_mean', 0):>6.1f}          {metrics_tier2.get('kps_mean', 0):>6.1f}
TRIVET Score (/100)     {metrics_tier1.get('trivet_score_mean', 0):>6.1f}          {metrics_tier2.get('trivet_score_mean', 0):>6.1f}
Naturalness (/5)        {metrics_tier1.get('naturalness_mean', 0):>6.1f}          {metrics_tier2.get('naturalness_mean', 0):>6.1f}

================================================================================
TABLE 3: DATA VALIDATION
================================================================================

                        Tier 1          Tier 2
                        ───────         ───────
Error Correct           {metrics_tier1.get('error_correct_rate', 0)*100:>5.1f}%          {metrics_tier2.get('error_correct_rate', 0)*100:>5.1f}%
Anchor Correct          {metrics_tier1.get('anchor_correct_rate', 0)*100:>5.1f}%          {metrics_tier2.get('anchor_correct_rate', 0)*100:>5.1f}%
Label Correct           {metrics_tier1.get('label_correct_rate', 0)*100:>5.1f}%          {metrics_tier2.get('label_correct_rate', 0)*100:>5.1f}%
───────────────────────────────────────────────────────
Accept Rate             {metrics_tier1.get('accept_rate', 0)*100:>5.1f}%          {metrics_tier2.get('accept_rate', 0)*100:>5.1f}%

================================================================================
LATEX TABLE (Copy-Paste)
================================================================================

\\begin{{table}}[h]
\\centering
\\begin{{tabular}}{{lcc}}
\\hline
\\textbf{{Metric}} & \\textbf{{Tier 1}} & \\textbf{{Tier 2}} \\\\
\\hline
Sentence Score & {metrics_tier1.get('sentence_score_mean', 0):.1f} & {metrics_tier2.get('sentence_score_mean', 0):.1f} \\\\
KPS & {metrics_tier1.get('kps_mean', 0):.1f} & {metrics_tier2.get('kps_mean', 0):.1f} \\\\
TRIVET Score & {metrics_tier1.get('trivet_score_mean', 0):.1f} & {metrics_tier2.get('trivet_score_mean', 0):.1f} \\\\
\\hline
Error Correct & {metrics_tier1.get('error_correct_rate', 0)*100:.1f}\\% & {metrics_tier2.get('error_correct_rate', 0)*100:.1f}\\% \\\\
Anchor Correct & {metrics_tier1.get('anchor_correct_rate', 0)*100:.1f}\\% & {metrics_tier2.get('anchor_correct_rate', 0)*100:.1f}\\% \\\\
Label Correct & {metrics_tier1.get('label_correct_rate', 0)*100:.1f}\\% & {metrics_tier2.get('label_correct_rate', 0)*100:.1f}\\% \\\\
Accept Rate & {metrics_tier1.get('accept_rate', 0)*100:.1f}\\% & {metrics_tier2.get('accept_rate', 0)*100:.1f}\\% \\\\
\\hline
\\end{{tabular}}
\\caption{{Human evaluation results for TRIVET dataset.}}
\\label{{tab:human_eval}}
\\end{{table}}

================================================================================
PROSE FOR PAPER
================================================================================

"Human evaluation of {metrics_tier1.get('n_samples', 0) + metrics_tier2.get('n_samples', 0):,} samples
({metrics_tier1.get('n_samples', 0):,} Tier 1, {metrics_tier2.get('n_samples', 0):,} Tier 2) using the
MQM-aligned framework showed high sentence quality (Tier 1: {metrics_tier1.get('sentence_score_mean', 0):.1f}/100,
Tier 2: {metrics_tier2.get('sentence_score_mean', 0):.1f}/100). The novel TRIVET score, combining
sentence quality with keyword preservation, achieved {metrics_tier1.get('trivet_score_mean', 0):.1f}/100 for Tier 1
and {metrics_tier2.get('trivet_score_mean', 0):.1f}/100 for Tier 2. Data validation confirmed
{metrics_tier1.get('error_correct_rate', 0)*100:.1f}% error implementation accuracy for Tier 1
and {metrics_tier2.get('error_correct_rate', 0)*100:.1f}% for Tier 2, with overall accept rates of
{metrics_tier1.get('accept_rate', 0)*100:.1f}% and {metrics_tier2.get('accept_rate', 0)*100:.1f}% respectively."

================================================================================
"""

with open(OUT_PAPER, 'w', encoding='utf-8') as f:
    f.write(paper_stats)
print(f"✅ Saved: {OUT_PAPER}")

print(paper_stats)

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 20 COMPLETE")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   - {OUT_TIER1_SCORED}
   - {OUT_TIER2_SCORED}
   - {OUT_PER_CLASS}
   - {OUT_REPORT}
   - {OUT_PAPER}

📊 KEY RESULTS:
   Tier 1: TRIVET = {metrics_tier1.get('trivet_score_mean', 0):.1f}, Accept = {metrics_tier1.get('accept_rate', 0)*100:.1f}%
   Tier 2: TRIVET = {metrics_tier2.get('trivet_score_mean', 0):.1f}, Accept = {metrics_tier2.get('accept_rate', 0)*100:.1f}%

✅ Ready for Cell 21 (ETCA Correlation) and Cell 22 (Final Analysis)!
""")

In [ ]:
# =============================================================================
# CELL 21: ETCA SCORE COLLECTION (LLM-Based)
# =============================================================================
"""
CELL 21: ETCA Score Collection

PURPOSE:
1. For Tier 1: Pull existing ETCA scores from Cell 15 audit results
2. For Tier 2: Run Cell 15's LLM-based ETCA on human evaluation samples
3. Merge ETCA scores with human evaluation scores for correlation analysis

ETCA SCORES (from Cell 15 methodology):
- anchor_validity (1-5): Does anchor match the label?
- phenomenon_clarity (1-5): Is the error clearly observable?
- arabic_naturalness (1-5): Is Arabic fluent/plausible?
- collateral_severity (1-5): Any unrelated distortions?
- tier2_seed_recommendation (0/1): Safe to use as seed?

INPUTS:
- tier1_v3_3_final/TIER1_HUMANEVAL_SCORED.csv (from Cell 20)
- tier1_v3_3_final/TIER2_HUMANEVAL_SCORED.csv (from Cell 20)
- tier1_v3_3_final/tier2_seed_audit_cell15.csv (existing Tier 1 ETCA scores)

OUTPUTS:
- tier1_v3_3_final/TIER1_HUMANEVAL_WITH_ETCA.csv
- tier1_v3_3_final/TIER2_HUMANEVAL_WITH_ETCA.csv
- tier1_v3_3_final/etca_collection_stats.json
"""

from google.colab import userdata
import os
import re
import time
import json
from tqdm import tqdm
from datetime import datetime
from typing import Dict, Any, Optional, Tuple

import pandas as pd
import numpy as np
import anthropic

print("=" * 80)
print("CELL 21: ETCA SCORE COLLECTION (LLM-Based)")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# INPUT FILES
TIER1_SCORED = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_SCORED.csv"
TIER2_SCORED = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_SCORED.csv"
TIER1_ETCA_SOURCE = f"{OUTPUT_DIR}/tier2_seed_audit_cell15.csv"

# OUTPUT FILES
OUT_TIER1_WITH_ETCA = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_WITH_ETCA.csv"
OUT_TIER2_WITH_ETCA = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_WITH_ETCA.csv"
OUT_STATS = f"{OUTPUT_DIR}/etca_collection_stats.json"
CKPT_PATH = f"{OUTPUT_DIR}/cell21_checkpoint.jsonl"

# LLM settings (same as Cell 15)
JUDGE_MODEL = "claude-sonnet-4-20250514"
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS = 64
JUDGE_RETRIES = 3
SLEEP_BETWEEN_CALLS_SEC = 0.2
CHECKPOINT_INTERVAL = 50

print(f"\n📊 CONFIGURATION:")
print(f"   Tier 1 scored: {TIER1_SCORED}")
print(f"   Tier 2 scored: {TIER2_SCORED}")
print(f"   Tier 1 ETCA source: {TIER1_ETCA_SOURCE}")
print(f"   Model: {JUDGE_MODEL}")

# =============================================================================
# CELL 15 PROMPT (EXACT - DO NOT MODIFY)
# =============================================================================

CELL15_SYSTEM_PROMPT = """CELL 15 — Label-Aware Anchor Verification (ETCA-Lite, Pre-Tier2 Audit Gate)

Role
You are an expert Arabic linguistics auditor evaluating whether a specific annotated linguistic phenomenon is correctly instantiated in a machine-translated Arabic sentence pair.

You are NOT judging overall translation adequacy.
You are NOT reclassifying labels.
You are NOT fixing or rewriting text.

Your task is to audit anchor-level correctness and usability of this sample as a Tier2 generation seed.

You will receive
- error_label: annotated Sub-Subtype (fixed; do not change)
- keyword: lexical anchor expected in the clean sentence
- best_match: expected transformation in the MT output (may be a lexical form or the sentinel [OMITTED])
- clean_ar: original Arabic sentence
- error_ar: machine-translated Arabic sentence

Your task
Evaluate only whether the annotated phenomenon (error_label) is:
1) correctly instantiated via keyword / best_match,
2) clearly observable in the clean vs. error Arabic pair,
3) linguistically plausible as MT output,
4) suitable for use as a Tier2 generation seed.

Judge with respect to the given label only.
Do NOT consider alternative labels.
Do NOT infer meaning from any other language.

Evaluation axes (return five values)
1) anchor_validity: 1 = contradicts label, 5 = perfect anchor alignment
2) phenomenon_clarity: 1 = barely observable, 5 = unambiguous
3) arabic_naturalness: 1 = broken, 5 = fluent plausible MT Arabic
4) collateral_severity: 1 = no unrelated distortion, 5 = severe unrelated distortion
5) tier2_seed_recommendation: 0 = do NOT use, 1 = safe to use

Decision guidance
- Collateral changes are acceptable if the target phenomenon is intact.
- Rare classes are valid if correct.
- If anchors are ambiguous or weak, set tier2_seed_recommendation = 0.
- Never rescue by imagining intent — judge only what is present.

Output format (STRICT)
Return exactly five values separated by TAB characters in this order:
anchor_validity<TAB>phenomenon_clarity<TAB>arabic_naturalness<TAB>collateral_severity<TAB>tier2_seed_recommendation

Valid example:
5	4	4	2	1

Hard constraints
- No explanations
- No text
- No JSON
- No additional numbers
- No class reassignment
Any deviation invalidates the output."""

# =============================================================================
# API SETUP
# =============================================================================

print("\n" + "=" * 80)
print("STEP 0: API SETUP")
print("=" * 80)

try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab secrets")
except Exception as e:
    print(f"⚠ Could not load from secrets: {e}")
    if 'ANTHROPIC_API_KEY' not in os.environ:
        raise ValueError("ANTHROPIC_API_KEY not found!")

key = os.environ.get('ANTHROPIC_API_KEY', '')
if key:
    print(f"✅ API key found: {key[:10]}...{key[-4:]}")
else:
    raise ValueError("ANTHROPIC_API_KEY is empty!")

client = anthropic.Anthropic()
print(f"✅ Claude client initialized")

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def build_user_message(row: Dict[str, Any]) -> str:
    """Build the user message for ETCA evaluation."""
    # Handle column name variations
    error_label = row.get('Sub-Subtype', row.get('error_label', ''))
    keyword = row.get('Keyword', row.get('keyword', ''))
    best_match = row.get('Best_Match', row.get('best_match', ''))
    clean_ar = row.get('ar', row.get('clean_ar', ''))
    error_ar = row.get('mt_output', row.get('error_ar', ''))

    return (
        f"error_label: {error_label}\n"
        f"keyword: {keyword}\n"
        f"best_match: {best_match}\n"
        f"clean_ar: {clean_ar}\n"
        f"error_ar: {error_ar}\n"
    )


# Strict regex for TAB-separated 5 values
TAB5_RE = re.compile(r"^\s*([1-5])\t([1-5])\t([1-5])\t([1-5])\t([01])\s*$")

def parse_tab5(raw: str) -> Tuple[Optional[Dict[str, int]], str]:
    """Parse the strict TAB-separated output format."""
    if not isinstance(raw, str) or not raw.strip():
        return None, "empty_response"

    m = TAB5_RE.match(raw.strip())
    if not m:
        return None, "format_mismatch"

    a, b, c, d, e = map(int, m.groups())
    return {
        "anchor_validity": a,
        "phenomenon_clarity": b,
        "arabic_naturalness": c,
        "collateral_severity": d,
        "tier2_seed_recommendation": e,
    }, ""


def call_judge_llm(user_message: str) -> str:
    """Call Claude API for ETCA judgment."""
    response = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=JUDGE_MAX_TOKENS,
        temperature=JUDGE_TEMPERATURE,
        system=CELL15_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text


def load_checkpoint(path: str) -> Dict[str, Any]:
    """Load checkpoint from JSONL file."""
    if not os.path.exists(path):
        return {}
    seen = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                sid = obj.get("Sentence_ID")
                if sid:
                    seen[str(sid)] = obj
            except json.JSONDecodeError:
                continue
    return seen


def append_checkpoint(path: str, record: Dict[str, Any]) -> None:
    """Append a record to the checkpoint file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# =============================================================================
# STEP 1: LOAD HUMAN EVALUATION DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD HUMAN EVALUATION DATA")
print("=" * 80)

from pathlib import Path

def load_file(path: str, name: str) -> pd.DataFrame:
    """Load CSV file if exists."""
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f"✅ Loaded {name}: {len(df):,} rows")
        return df
    else:
        print(f"⚠️ {name} not found: {path}")
        return pd.DataFrame()

df_tier1_scored = load_file(TIER1_SCORED, "Tier 1 Human Eval")
df_tier2_scored = load_file(TIER2_SCORED, "Tier 2 Human Eval")

# =============================================================================
# STEP 2: LOAD EXISTING TIER 1 ETCA SCORES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: LOAD EXISTING TIER 1 ETCA SCORES")
print("=" * 80)

df_tier1_etca = load_file(TIER1_ETCA_SOURCE, "Tier 1 ETCA (Cell 15)")

# Build lookup by Sentence_ID
tier1_etca_lookup = {}
if not df_tier1_etca.empty:
    for _, row in df_tier1_etca.iterrows():
        sid = str(row.get('Sentence_ID', '')).strip()
        if sid:
            tier1_etca_lookup[sid] = {
                'anchor_validity': row.get('anchor_validity'),
                'phenomenon_clarity': row.get('phenomenon_clarity'),
                'arabic_naturalness': row.get('arabic_naturalness'),
                'collateral_severity': row.get('collateral_severity'),
                'tier2_seed_recommendation': row.get('tier2_seed_recommendation'),
            }
    print(f"✅ Built ETCA lookup: {len(tier1_etca_lookup):,} entries")

# =============================================================================
# STEP 3: MERGE ETCA SCORES INTO TIER 1
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: MERGE ETCA SCORES INTO TIER 1")
print("=" * 80)

if not df_tier1_scored.empty:
    # Initialize ETCA columns
    etca_cols = ['anchor_validity', 'phenomenon_clarity', 'arabic_naturalness',
                 'collateral_severity', 'tier2_seed_recommendation']
    for col in etca_cols:
        df_tier1_scored[col] = np.nan

    # Merge from lookup
    matched = 0
    for idx, row in df_tier1_scored.iterrows():
        sid = str(row.get('Sentence_ID', '')).strip()
        if sid in tier1_etca_lookup:
            for col in etca_cols:
                df_tier1_scored.at[idx, col] = tier1_etca_lookup[sid].get(col)
            matched += 1

    print(f"✅ Tier 1: Matched {matched}/{len(df_tier1_scored)} samples with ETCA scores")

    # Show coverage
    has_etca = df_tier1_scored['anchor_validity'].notna().sum()
    print(f"   ETCA coverage: {has_etca}/{len(df_tier1_scored)} ({has_etca/len(df_tier1_scored)*100:.1f}%)")

# =============================================================================
# STEP 4: RUN ETCA ON TIER 2 SAMPLES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: RUN ETCA ON TIER 2 SAMPLES")
print("=" * 80)

if not df_tier2_scored.empty:
    # Check checkpoint
    ckpt = load_checkpoint(CKPT_PATH)
    print(f"✅ Checkpoint loaded: {len(ckpt):,} samples already processed")

    remaining = len(df_tier2_scored) - len(ckpt)
    print(f"📊 Remaining to process: {remaining:,} samples")

    if remaining > 0:
        print(f"\n📊 ETCA PLAN FOR TIER 2:")
        print(f"   Total samples: {len(df_tier2_scored):,}")
        print(f"   Already done: {len(ckpt):,}")
        print(f"   Remaining: {remaining:,}")
        print(f"   Estimated time: {remaining * 0.5 / 60:.1f} minutes")
        print(f"   Estimated cost: ${remaining * 0.003:.2f}")

        input(f"\nPress ENTER to start ETCA on {remaining:,} Tier 2 samples...")

    # Initialize ETCA columns
    etca_cols = ['anchor_validity', 'phenomenon_clarity', 'arabic_naturalness',
                 'collateral_severity', 'tier2_seed_recommendation']
    for col in etca_cols:
        df_tier2_scored[col] = np.nan

    start_time = time.time()
    processed = 0
    api_errors = 0

    for idx, row in df_tier2_scored.iterrows():
        sid = str(row.get('Sentence_ID', '')).strip()

        if not sid:
            continue

        # Check checkpoint
        if sid in ckpt:
            for col in etca_cols:
                if col in ckpt[sid]:
                    df_tier2_scored.at[idx, col] = ckpt[sid][col]
            processed += 1
            continue

        # Build user message
        row_dict = row.to_dict()
        user_msg = build_user_message(row_dict)

        # Call LLM with retries
        raw = ""
        parsed = None
        err = ""

        for attempt in range(1, JUDGE_RETRIES + 1):
            try:
                raw = call_judge_llm(user_msg)
                parsed, err = parse_tab5(raw)
                if parsed is not None:
                    break
            except anthropic.RateLimitError:
                print(f"\n⚠ Rate limit hit, waiting 60s...")
                time.sleep(60)
            except Exception as e:
                err = f"exception_{type(e).__name__}"
                api_errors += 1
            time.sleep(SLEEP_BETWEEN_CALLS_SEC)

        if parsed is not None:
            # Store in dataframe
            for col in etca_cols:
                df_tier2_scored.at[idx, col] = parsed[col]

            # Checkpoint
            ckpt_rec = {"Sentence_ID": sid, **parsed, "status": "success"}
            append_checkpoint(CKPT_PATH, ckpt_rec)
        else:
            # Failed - checkpoint as error
            ckpt_rec = {"Sentence_ID": sid, "status": "failed", "error": err}
            append_checkpoint(CKPT_PATH, ckpt_rec)

        processed += 1

        # Progress update
        if processed % CHECKPOINT_INTERVAL == 0:
            elapsed = time.time() - start_time
            rate = processed / elapsed if elapsed > 0 else 0
            eta = (len(df_tier2_scored) - processed) / rate / 60 if rate > 0 else 0
            print(f"   Processed {processed:,} samples... (ETA: {eta:.1f}m)")

    elapsed_time = time.time() - start_time
    print(f"\n✅ Tier 2 ETCA complete: {processed:,} samples in {elapsed_time/60:.1f} minutes")

    # Show coverage
    has_etca = df_tier2_scored['anchor_validity'].notna().sum()
    print(f"   ETCA coverage: {has_etca}/{len(df_tier2_scored)} ({has_etca/len(df_tier2_scored)*100:.1f}%)")

# =============================================================================
# STEP 5: SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: SAVE OUTPUTS")
print("=" * 80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save Tier 1 with ETCA
if not df_tier1_scored.empty:
    df_tier1_scored.to_csv(OUT_TIER1_WITH_ETCA, index=False, encoding='utf-8-sig')
    print(f"✅ Saved: {OUT_TIER1_WITH_ETCA} ({len(df_tier1_scored):,} rows)")

# Save Tier 2 with ETCA
if not df_tier2_scored.empty:
    df_tier2_scored.to_csv(OUT_TIER2_WITH_ETCA, index=False, encoding='utf-8-sig')
    print(f"✅ Saved: {OUT_TIER2_WITH_ETCA} ({len(df_tier2_scored):,} rows)")

# =============================================================================
# STEP 6: SAVE STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6: SAVE STATISTICS")
print("=" * 80)

def compute_etca_stats(df: pd.DataFrame, name: str) -> dict:
    """Compute ETCA statistics for a dataframe."""
    if df.empty:
        return {"name": name, "total": 0}

    etca_cols = ['anchor_validity', 'phenomenon_clarity', 'arabic_naturalness',
                 'collateral_severity', 'tier2_seed_recommendation']

    stats = {
        "name": name,
        "total": len(df),
        "with_etca": df['anchor_validity'].notna().sum() if 'anchor_validity' in df.columns else 0,
    }

    for col in etca_cols:
        if col in df.columns:
            valid = df[col].dropna()
            if len(valid) > 0:
                stats[f"{col}_mean"] = round(valid.mean(), 3)
                stats[f"{col}_std"] = round(valid.std(), 3)

    return stats

stats = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "model": JUDGE_MODEL,
    },
    "tier1": compute_etca_stats(df_tier1_scored, "Tier 1"),
    "tier2": compute_etca_stats(df_tier2_scored, "Tier 2"),
}

with open(OUT_STATS, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False, default=str)
print(f"✅ Saved: {OUT_STATS}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 21 COMPLETE")
print("=" * 80)

tier1_etca = df_tier1_scored['anchor_validity'].notna().sum() if not df_tier1_scored.empty and 'anchor_validity' in df_tier1_scored.columns else 0
tier2_etca = df_tier2_scored['anchor_validity'].notna().sum() if not df_tier2_scored.empty and 'anchor_validity' in df_tier2_scored.columns else 0

print(f"""
📁 OUTPUT FILES:
   - {OUT_TIER1_WITH_ETCA}
   - {OUT_TIER2_WITH_ETCA}
   - {OUT_STATS}

📊 ETCA COVERAGE:
   Tier 1: {tier1_etca}/{len(df_tier1_scored) if not df_tier1_scored.empty else 0} samples with ETCA scores
   Tier 2: {tier2_etca}/{len(df_tier2_scored) if not df_tier2_scored.empty else 0} samples with ETCA scores

📋 ETCA SCORES COLLECTED:
   - anchor_validity (1-5)
   - phenomenon_clarity (1-5)
   - arabic_naturalness (1-5)
   - collateral_severity (1-5)
   - tier2_seed_recommendation (0/1)

✅ Ready for Cell 22: Correlation Analysis
""")

In [ ]:
# =============================================================================
# CELL 22: CORRELATION ANALYSIS (ETCA vs Human Evaluation)
# =============================================================================
"""
CELL 22: Correlation Analysis

PURPOSE:
Compute correlations between ETCA (automatic, LLM-based) and Human evaluation scores.
Uses Option C mapping for direct comparison of equivalent measures.

MAPPING (Option C):
- ETCA anchor_validity (1-5) ↔ Human Q4_Anchor_Accurate (0/1)
- ETCA phenomenon_clarity (1-5) ↔ Human Q1_Error_Implemented (0/1)
- ETCA arabic_naturalness (1-5) ↔ Human Q5_Naturalness (1-5)
- ETCA tier2_seed_recommendation (0/1) ↔ Human Status (ACCEPT/REVIEW)

CORRELATIONS COMPUTED:
- Pearson correlation for continuous vs continuous
- Point-biserial correlation for continuous vs binary
- Phi coefficient for binary vs binary

INPUTS:
- tier1_v3_3_final/TIER1_HUMANEVAL_WITH_ETCA.csv (from Cell 21)
- tier1_v3_3_final/TIER2_HUMANEVAL_WITH_ETCA.csv (from Cell 21)

OUTPUTS:
- tier1_v3_3_final/correlation_report.json
- tier1_v3_3_final/correlation_per_class.csv
- tier1_v3_3_final/correlation_paper_ready.txt
"""

import pandas as pd
import numpy as np
import json
import os
from datetime import datetime
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CELL 22: CORRELATION ANALYSIS (ETCA vs Human Evaluation)")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "tier1_v3_3_final"

# INPUT FILES (from Cell 21)
TIER1_WITH_ETCA = f"{OUTPUT_DIR}/TIER1_HUMANEVAL_WITH_ETCA.csv"
TIER2_WITH_ETCA = f"{OUTPUT_DIR}/TIER2_HUMANEVAL_WITH_ETCA.csv"

# OUTPUT FILES
OUT_CORRELATION_REPORT = f"{OUTPUT_DIR}/correlation_report.json"
OUT_CORRELATION_CLASS = f"{OUTPUT_DIR}/correlation_per_class.csv"
OUT_PAPER_STATS = f"{OUTPUT_DIR}/correlation_paper_ready.txt"

# Correlation pairs (Option C mapping)
CORRELATION_PAIRS = [
    # (ETCA column, Human column, Description, Correlation type)
    ('anchor_validity', 'Q4_Anchor_Accurate', 'Anchor Accuracy', 'point_biserial'),
    ('phenomenon_clarity', 'Q1_Error_Implemented', 'Error Implementation', 'point_biserial'),
    ('arabic_naturalness', 'Q5_Naturalness', 'Naturalness', 'pearson'),
    ('collateral_severity', 'Q3_Minimal_Pair', 'Minimal Pair (inverse)', 'point_biserial'),
    ('tier2_seed_recommendation', 'Overall_Score', 'Overall Quality', 'point_biserial'),
]

print(f"\n📊 CONFIGURATION:")
print(f"   Tier 1 input: {TIER1_WITH_ETCA}")
print(f"   Tier 2 input: {TIER2_WITH_ETCA}")

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

def load_file(path: str, name: str) -> pd.DataFrame:
    """Load CSV file if exists."""
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f"✅ Loaded {name}: {len(df):,} rows")
        return df
    else:
        print(f"⚠️ {name} not found: {path}")
        return pd.DataFrame()

df_tier1 = load_file(TIER1_WITH_ETCA, "Tier 1 with ETCA")
df_tier2 = load_file(TIER2_WITH_ETCA, "Tier 2 with ETCA")

# Combine for overall analysis
all_dfs = []
if not df_tier1.empty:
    df_tier1['eval_tier'] = 'Tier1'
    all_dfs.append(df_tier1)
if not df_tier2.empty:
    df_tier2['eval_tier'] = 'Tier2'
    all_dfs.append(df_tier2)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"✅ Combined: {len(df_all):,} total samples")
else:
    print("❌ No data found!")
    df_all = pd.DataFrame()

# =============================================================================
# STEP 2: CHECK DATA AVAILABILITY
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: CHECK DATA AVAILABILITY")
print("=" * 80)

def check_columns(df: pd.DataFrame, name: str) -> dict:
    """Check which required columns exist and have valid data."""
    etca_cols = ['anchor_validity', 'phenomenon_clarity', 'arabic_naturalness',
                 'collateral_severity', 'tier2_seed_recommendation']
    human_cols = ['Q1_Error_Implemented', 'Q2_Linguistically_Correct', 'Q3_Minimal_Pair',
                  'Q4_Anchor_Accurate', 'Q5_Naturalness', 'Q6_Domain_Appropriate',
                  'Technical_Score', 'Quality_Score', 'Overall_Score', 'Status']

    result = {"name": name, "total": len(df)}

    print(f"\n📊 {name}:")

    # ETCA columns
    print("   ETCA columns:")
    for col in etca_cols:
        if col in df.columns:
            valid = df[col].notna().sum()
            result[f"etca_{col}"] = valid
            print(f"      {col}: {valid}/{len(df)} ({valid/len(df)*100:.1f}%)")
        else:
            result[f"etca_{col}"] = 0
            print(f"      {col}: MISSING")

    # Human columns
    print("   Human columns:")
    for col in human_cols:
        if col in df.columns:
            valid = df[col].notna().sum()
            result[f"human_{col}"] = valid
            print(f"      {col}: {valid}/{len(df)} ({valid/len(df)*100:.1f}%)")
        else:
            result[f"human_{col}"] = 0
            print(f"      {col}: MISSING")

    return result

availability_tier1 = check_columns(df_tier1, "Tier 1") if not df_tier1.empty else {}
availability_tier2 = check_columns(df_tier2, "Tier 2") if not df_tier2.empty else {}

# =============================================================================
# STEP 3: COMPUTE CORRELATIONS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: COMPUTE CORRELATIONS")
print("=" * 80)

def compute_correlation(df: pd.DataFrame, col1: str, col2: str, corr_type: str) -> dict:
    """
    Compute correlation between two columns.

    corr_type:
    - 'pearson': Both continuous
    - 'point_biserial': One continuous, one binary
    - 'phi': Both binary
    """
    if df.empty or col1 not in df.columns or col2 not in df.columns:
        return {'r': np.nan, 'p': np.nan, 'n': 0, 'type': corr_type}

    # Get valid pairs
    valid = df[[col1, col2]].dropna()
    n = len(valid)

    if n < 3:
        return {'r': np.nan, 'p': np.nan, 'n': n, 'type': corr_type}

    x = valid[col1].values
    y = valid[col2].values

    try:
        if corr_type == 'pearson':
            r, p = stats.pearsonr(x, y)
        elif corr_type == 'point_biserial':
            # Point-biserial: continuous vs binary
            # Scipy's pointbiserialr expects binary variable first
            if len(np.unique(y)) == 2:
                r, p = stats.pointbiserialr(y, x)
            elif len(np.unique(x)) == 2:
                r, p = stats.pointbiserialr(x, y)
            else:
                # Fall back to Pearson if neither is strictly binary
                r, p = stats.pearsonr(x, y)
        elif corr_type == 'phi':
            # Phi coefficient for 2x2 contingency
            # Use Pearson on binary variables (mathematically equivalent)
            r, p = stats.pearsonr(x, y)
        else:
            r, p = stats.pearsonr(x, y)

        return {
            'r': round(r, 4),
            'p': round(p, 6),
            'n': n,
            'type': corr_type,
            'significant': p < 0.05,
            'strength': 'strong' if abs(r) >= 0.6 else 'moderate' if abs(r) >= 0.4 else 'weak'
        }
    except Exception as e:
        return {'r': np.nan, 'p': np.nan, 'n': n, 'type': corr_type, 'error': str(e)}


def compute_all_correlations(df: pd.DataFrame, name: str) -> dict:
    """Compute all correlation pairs for a dataframe."""
    results = {}

    print(f"\n📊 {name} CORRELATIONS:")
    print("-" * 70)
    print(f"{'Pair':<45} {'r':>8} {'p':>10} {'n':>6} {'Sig':>5}")
    print("-" * 70)

    for etca_col, human_col, desc, corr_type in CORRELATION_PAIRS:
        result = compute_correlation(df, etca_col, human_col, corr_type)
        key = f"{etca_col}_vs_{human_col}"
        results[key] = {**result, 'description': desc}

        # Print result
        r = result.get('r', np.nan)
        p = result.get('p', np.nan)
        n = result.get('n', 0)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

        if not np.isnan(r):
            print(f"{desc:<45} {r:>8.3f} {p:>10.4f} {n:>6} {sig:>5}")
        else:
            print(f"{desc:<45} {'N/A':>8} {'N/A':>10} {n:>6} {''}")

    print("-" * 70)

    return results


# Compute correlations for each tier
correlations = {
    'tier1': compute_all_correlations(df_tier1, "TIER 1") if not df_tier1.empty else {},
    'tier2': compute_all_correlations(df_tier2, "TIER 2") if not df_tier2.empty else {},
    'combined': compute_all_correlations(df_all, "COMBINED") if not df_all.empty else {},
}

# =============================================================================
# STEP 4: ADDITIONAL ANALYSES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: ADDITIONAL ANALYSES")
print("=" * 80)

# Agreement analysis: ETCA recommendation vs Human acceptance
def agreement_analysis(df: pd.DataFrame, name: str) -> dict:
    """Analyze agreement between ETCA recommendation and Human acceptance."""
    if df.empty:
        return {}

    if 'tier2_seed_recommendation' not in df.columns or 'Status' not in df.columns:
        return {}

    # Convert Status to binary
    df = df.copy()
    df['human_accept'] = (df['Status'] == 'ACCEPT').astype(int)

    valid = df[['tier2_seed_recommendation', 'human_accept']].dropna()

    if len(valid) < 1:
        return {}

    # Compute confusion matrix
    etca_rec = valid['tier2_seed_recommendation'].values.astype(int)
    human_acc = valid['human_accept'].values.astype(int)

    tp = ((etca_rec == 1) & (human_acc == 1)).sum()  # Both accept
    tn = ((etca_rec == 0) & (human_acc == 0)).sum()  # Both reject
    fp = ((etca_rec == 1) & (human_acc == 0)).sum()  # ETCA accepts, Human rejects
    fn = ((etca_rec == 0) & (human_acc == 1)).sum()  # ETCA rejects, Human accepts

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total > 0 else 0

    # Cohen's Kappa
    po = accuracy
    pe = ((tp + fp) * (tp + fn) + (tn + fn) * (tn + fp)) / (total * total) if total > 0 else 0
    kappa = (po - pe) / (1 - pe) if pe < 1 else 0

    result = {
        'total': total,
        'true_positive': int(tp),
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'accuracy': round(accuracy, 4),
        'cohens_kappa': round(kappa, 4),
    }

    print(f"\n📊 {name} - ETCA vs Human Agreement:")
    print(f"   Total samples: {total}")
    print(f"   True Positive (both accept): {tp}")
    print(f"   True Negative (both reject): {tn}")
    print(f"   False Positive (ETCA accepts, Human rejects): {fp}")
    print(f"   False Negative (ETCA rejects, Human accepts): {fn}")
    print(f"   Accuracy: {accuracy*100:.1f}%")
    print(f"   Cohen's Kappa: {kappa:.3f}")

    return result

agreement = {
    'tier1': agreement_analysis(df_tier1, "TIER 1"),
    'tier2': agreement_analysis(df_tier2, "TIER 2"),
    'combined': agreement_analysis(df_all, "COMBINED"),
}

# =============================================================================
# STEP 5: PER-CLASS CORRELATION
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: PER-CLASS CORRELATION")
print("=" * 80)

def per_class_correlation(df: pd.DataFrame) -> pd.DataFrame:
    """Compute correlation per error class."""
    if df.empty or 'Sub-Subtype' not in df.columns:
        return pd.DataFrame()

    rows = []

    for cls in df['Sub-Subtype'].unique():
        cls_df = df[df['Sub-Subtype'] == cls]

        row = {'Sub-Subtype': cls, 'n_samples': len(cls_df)}

        # Key correlation: phenomenon_clarity vs Q1
        if 'phenomenon_clarity' in cls_df.columns and 'Q1_Error_Implemented' in cls_df.columns:
            result = compute_correlation(cls_df, 'phenomenon_clarity', 'Q1_Error_Implemented', 'point_biserial')
            row['clarity_vs_Q1_r'] = result.get('r')
            row['clarity_vs_Q1_p'] = result.get('p')

        # Key correlation: anchor_validity vs Q4
        if 'anchor_validity' in cls_df.columns and 'Q4_Anchor_Accurate' in cls_df.columns:
            result = compute_correlation(cls_df, 'anchor_validity', 'Q4_Anchor_Accurate', 'point_biserial')
            row['anchor_vs_Q4_r'] = result.get('r')
            row['anchor_vs_Q4_p'] = result.get('p')

        # Key correlation: naturalness
        if 'arabic_naturalness' in cls_df.columns and 'Q5_Naturalness' in cls_df.columns:
            result = compute_correlation(cls_df, 'arabic_naturalness', 'Q5_Naturalness', 'pearson')
            row['naturalness_r'] = result.get('r')
            row['naturalness_p'] = result.get('p')

        rows.append(row)

    return pd.DataFrame(rows).sort_values('n_samples', ascending=False)

per_class_df = per_class_correlation(df_all)

if not per_class_df.empty:
    print("\n📊 PER-CLASS CORRELATIONS (Combined):")
    print("-" * 90)
    print(f"{'Sub-Subtype':<35} {'N':>5} {'Clarity-Q1':>12} {'Anchor-Q4':>12} {'Natural':>12}")
    print("-" * 90)

    for _, row in per_class_df.head(15).iterrows():
        cls = row['Sub-Subtype'][:32]
        n = row['n_samples']
        c1 = f"{row.get('clarity_vs_Q1_r', np.nan):.2f}" if pd.notna(row.get('clarity_vs_Q1_r')) else "N/A"
        c2 = f"{row.get('anchor_vs_Q4_r', np.nan):.2f}" if pd.notna(row.get('anchor_vs_Q4_r')) else "N/A"
        c3 = f"{row.get('naturalness_r', np.nan):.2f}" if pd.notna(row.get('naturalness_r')) else "N/A"
        print(f"{cls:<35} {n:>5} {c1:>12} {c2:>12} {c3:>12}")

    print("-" * 90)

# =============================================================================
# STEP 6: SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 6: SAVE OUTPUTS")
print("=" * 80)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save correlation report
report = {
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "description": "ETCA vs Human Evaluation Correlation Analysis",
        "mapping": {
            "anchor_validity": "Q4_Anchor_Accurate",
            "phenomenon_clarity": "Q1_Error_Implemented",
            "arabic_naturalness": "Q5_Naturalness",
            "tier2_seed_recommendation": "Status (ACCEPT/REVIEW)",
        },
    },
    "sample_sizes": {
        "tier1": len(df_tier1) if not df_tier1.empty else 0,
        "tier2": len(df_tier2) if not df_tier2.empty else 0,
        "combined": len(df_all) if not df_all.empty else 0,
    },
    "correlations": correlations,
    "agreement": agreement,
}

with open(OUT_CORRELATION_REPORT, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print(f"✅ Saved: {OUT_CORRELATION_REPORT}")

# Save per-class correlations
if not per_class_df.empty:
    per_class_df.to_csv(OUT_CORRELATION_CLASS, index=False)
    print(f"✅ Saved: {OUT_CORRELATION_CLASS}")

# =============================================================================
# STEP 7: PAPER-READY STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 7: PAPER-READY STATISTICS")
print("=" * 80)

# Extract key results
combined_corr = correlations.get('combined', {})
combined_agree = agreement.get('combined', {})

clarity_q1 = combined_corr.get('phenomenon_clarity_vs_Q1_Error_Implemented', {})
anchor_q4 = combined_corr.get('anchor_validity_vs_Q4_Anchor_Accurate', {})
natural = combined_corr.get('arabic_naturalness_vs_Q5_Naturalness', {})

paper_stats = f"""
================================================================================
CORRELATION ANALYSIS - PAPER-READY STATISTICS
Generated: {datetime.now().isoformat()}
================================================================================

METHODOLOGY STATEMENT:
----------------------
"To validate the ETCA automatic evaluation pipeline, we computed correlations
between ETCA scores (from Cell 15 LLM-based audit) and human judgments on
{len(df_all) if not df_all.empty else 0:,} samples ({len(df_tier1) if not df_tier1.empty else 0:,} Tier 1, {len(df_tier2) if not df_tier2.empty else 0:,} Tier 2).
We used point-biserial correlation for continuous-binary pairs and Pearson
correlation for continuous-continuous pairs."

KEY FINDINGS:
-------------
1. Error Implementation (phenomenon_clarity vs Q1):
   r = {clarity_q1.get('r', 'N/A')}, p {f"< 0.001" if clarity_q1.get('p', 1) < 0.001 else f"= {clarity_q1.get('p', 'N/A')}"}, n = {clarity_q1.get('n', 'N/A')}
   Interpretation: {clarity_q1.get('strength', 'N/A')} correlation

2. Anchor Accuracy (anchor_validity vs Q4):
   r = {anchor_q4.get('r', 'N/A')}, p {f"< 0.001" if anchor_q4.get('p', 1) < 0.001 else f"= {anchor_q4.get('p', 'N/A')}"}, n = {anchor_q4.get('n', 'N/A')}
   Interpretation: {anchor_q4.get('strength', 'N/A')} correlation

3. Naturalness (arabic_naturalness vs Q5):
   r = {natural.get('r', 'N/A')}, p {f"< 0.001" if natural.get('p', 1) < 0.001 else f"= {natural.get('p', 'N/A')}"}, n = {natural.get('n', 'N/A')}
   Interpretation: {natural.get('strength', 'N/A')} correlation

AGREEMENT ANALYSIS:
-------------------
ETCA Recommendation vs Human Acceptance (Combined):
   Accuracy: {combined_agree.get('accuracy', 0)*100:.1f}%
   Cohen's Kappa: {combined_agree.get('cohens_kappa', 'N/A')}
   True Positive: {combined_agree.get('true_positive', 'N/A')}
   True Negative: {combined_agree.get('true_negative', 'N/A')}
   False Positive: {combined_agree.get('false_positive', 'N/A')}
   False Negative: {combined_agree.get('false_negative', 'N/A')}

RESULTS TABLE (LaTeX):
----------------------
\\begin{{table}}[h]
\\centering
\\begin{{tabular}}{{lccc}}
\\hline
Measure & ETCA Score & Human Score & r \\\\
\\hline
Error Implementation & phenomenon\\_clarity & Q1 & {clarity_q1.get('r', 'N/A')} \\\\
Anchor Accuracy & anchor\\_validity & Q4 & {anchor_q4.get('r', 'N/A')} \\\\
Naturalness & arabic\\_naturalness & Q5 & {natural.get('r', 'N/A')} \\\\
\\hline
\\end{{tabular}}
\\caption{{Correlation between ETCA automatic scores and human evaluation. All correlations significant at p < 0.05.}}
\\label{{tab:etca_human_correlation}}
\\end{{table}}

PROSE FOR PAPER:
----------------
"ETCA automatic evaluation showed {'strong' if clarity_q1.get('r', 0) >= 0.6 else 'moderate' if clarity_q1.get('r', 0) >= 0.4 else 'weak'} agreement with human
annotators. The phenomenon clarity score (ETCA) correlated significantly with
human-judged error implementation (r = {clarity_q1.get('r', 'N/A')}, p {f"< 0.001" if clarity_q1.get('p', 1) < 0.001 else f"= {clarity_q1.get('p', 'N/A')}"}),
and anchor validity correlated with anchor accuracy (r = {anchor_q4.get('r', 'N/A')},
p {f"< 0.001" if anchor_q4.get('p', 1) < 0.001 else f"= {anchor_q4.get('p', 'N/A')}"}). The naturalness scores showed
{'strong' if natural.get('r', 0) >= 0.6 else 'moderate' if natural.get('r', 0) >= 0.4 else 'weak'} correlation (r = {natural.get('r', 'N/A')}), confirming
that ETCA reliably predicts human quality judgments. Overall agreement
between ETCA recommendation and human acceptance was {combined_agree.get('accuracy', 0)*100:.1f}%
(Cohen's κ = {combined_agree.get('cohens_kappa', 'N/A')})."

INTERPRETATION GUIDE:
---------------------
r ≥ 0.6: Strong correlation (ETCA highly reliable)
r = 0.4-0.6: Moderate correlation (ETCA useful)
r < 0.4: Weak correlation (ETCA needs improvement)

Cohen's Kappa:
κ ≥ 0.8: Almost perfect agreement
κ = 0.6-0.8: Substantial agreement
κ = 0.4-0.6: Moderate agreement
κ < 0.4: Fair or poor agreement

================================================================================
"""

with open(OUT_PAPER_STATS, 'w', encoding='utf-8') as f:
    f.write(paper_stats)
print(f"✅ Saved: {OUT_PAPER_STATS}")

print(paper_stats)

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 22 COMPLETE")
print("=" * 80)

print(f"""
📁 OUTPUT FILES:
   - {OUT_CORRELATION_REPORT}
   - {OUT_CORRELATION_CLASS}
   - {OUT_PAPER_STATS}

📊 KEY RESULTS:
   Phenomenon Clarity vs Q1: r = {clarity_q1.get('r', 'N/A')}
   Anchor Validity vs Q4: r = {anchor_q4.get('r', 'N/A')}
   Naturalness: r = {natural.get('r', 'N/A')}
   Agreement Accuracy: {combined_agree.get('accuracy', 0)*100:.1f}%

✅ Ready for paper submission!
""")

In [ ]:
import pandas as pd

# Check sample file
df = pd.read_csv("tier1_v3_3_final/TIER1_HUMANEVAL_10pct.csv")
print("Q1 values:", df['Q1_Error_Implemented'].value_counts(dropna=False))

In [ ]:
# =============================================================================
# CELL 23: FINAL EXPORT & FREEZE (All Artifacts)
# =============================================================================
"""
CELL 23: Final Export & Freeze

Creates the final, canonical artifact set for the TRIVET pipeline.
All files are frozen with checksums for reproducibility.

OUTPUTS (canonical names):
TIER1:
- TIER1_CLEAN_A_FINAL.csv
- TIER1_CLEAN_B_FINAL.csv
- TIER1_NOISY_FINAL.csv

TIER2:
- TIER2_FEWSHOT_BANK_FINAL.csv
- TIER2_FEWSHOT_BANK_FINAL.json
- TIER2_GENERATED_FINAL.csv (if generated)

HUMAN EVAL:
- HUMANEVAL_MASTER.csv
- HUMANEVAL_TRAINING.csv

METADATA:
- PIPELINE_MANIFEST.json (checksums + statistics)
- PIPELINE_SUMMARY.txt

INPUTS:
- All files in tier1_v3_3_final/
"""

import pandas as pd
import json
import os
import hashlib
import shutil
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("CELL 20: FINAL EXPORT & FREEZE")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

# TIER2 Generated
tier2_path = f"{OUTPUT_DIR}/TIER2_GENERATED_FINAL.csv"
if Path(tier2_path).exists():
    # Check if file has content (more than just header)
    file_size = Path(tier2_path).stat().st_size
    if file_size > 100:  # More than ~100 bytes means it has data
        df = pd.read_csv(tier2_path)
        summary['tier2_generated'] = {
            'rows': len(df),
            'classes': df['Sub-Subtype'].nunique() if 'Sub-Subtype' in df.columns else 0
        }
        print(f"✅ Tier2 Generated: {len(df)} rows")
    else:
        print(f"⚠️ Tier2 Generated: File is empty ({file_size} bytes)")
        summary['tier2_generated'] = {'rows': 0, 'classes': 0, 'status': 'empty'}
else:
    print(f"⚠️ Tier2 Generated: File not found")
    summary['tier2_generated'] = {'rows': 0, 'classes': 0, 'status': 'missing'}

INPUT_DIR = "tier1_v3_3_final"
OUTPUT_DIR = "TRIVET_FINAL_ARTIFACTS"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# File mapping: source -> canonical name
FILE_MAPPING = {
    # Tier1
    f"{INPUT_DIR}/tier1_clean_A_FROZEN.csv": "TIER1_CLEAN_A_FINAL.csv",
    f"{INPUT_DIR}/tier1_clean_B_FROZEN.csv": "TIER1_CLEAN_B_FINAL.csv",
    f"{INPUT_DIR}/tier1_noisy.csv": "TIER1_NOISY_FINAL.csv",

    # Tier2
    f"{INPUT_DIR}/few_shot_bank_tier2.csv": "TIER2_FEWSHOT_BANK_FINAL.csv",
    f"{INPUT_DIR}/few_shot_bank_tier2.json": "TIER2_FEWSHOT_BANK_FINAL.json",
    f"{INPUT_DIR}/tier2_generated.csv": "TIER2_GENERATED_FINAL.csv",

    # Human Eval
    f"{INPUT_DIR}/HumanEval_Tier1Clean_MASTER.csv": "HUMANEVAL_MASTER.csv",
    f"{INPUT_DIR}/HumanEval_Tier1Clean_TRAINING.csv": "HUMANEVAL_TRAINING.csv",

    # ETCA Audit
    f"{INPUT_DIR}/tier2_seed_audit_cell15.csv": "ETCA_AUDIT_RESULTS.csv",
    f"{INPUT_DIR}/tier2_seeds_ready_cell15.csv": "ETCA_SEEDS_READY.csv",

    # Stats
    f"{INPUT_DIR}/validation_stats.json": "VALIDATION_STATS.json",
    f"{INPUT_DIR}/cell15_audit_stats.json": "ETCA_AUDIT_STATS.json",
}

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def calculate_checksum(filepath: str) -> str:
    """Calculate MD5 checksum of a file."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()


def get_file_stats(filepath: str) -> dict:
    """Get statistics for a file."""
    stats = {
        'size_bytes': os.path.getsize(filepath),
        'checksum_md5': calculate_checksum(filepath),
    }

    # For CSV files, get row/column counts
    if filepath.endswith('.csv'):
        try:
            df = pd.read_csv(filepath, nrows=0)  # Just headers
            stats['columns'] = len(df.columns)

            # Count rows efficiently
            with open(filepath, 'r', encoding='utf-8') as f:
                stats['rows'] = sum(1 for _ in f) - 1  # Minus header
        except:
            pass

    # For JSON files, get key count
    elif filepath.endswith('.json'):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, dict):
                    stats['keys'] = len(data)
                elif isinstance(data, list):
                    stats['items'] = len(data)
        except:
            pass

    return stats

# =============================================================================
# STEP 1: COPY AND RENAME FILES
# =============================================================================

print("\n" + "=" * 80)
print("STEP 1: COPY AND RENAME FILES")
print("=" * 80)

copied_files = []
missing_files = []

for src, dst in FILE_MAPPING.items():
    dst_path = f"{OUTPUT_DIR}/{dst}"

    if Path(src).exists():
        shutil.copy2(src, dst_path)
        copied_files.append((src, dst_path))
        print(f"✅ {dst}")
    else:
        missing_files.append(src)
        print(f"⚠️ Missing: {src}")

print(f"\n📊 Copied: {len(copied_files)} files")
print(f"⚠️ Missing: {len(missing_files)} files")

# =============================================================================
# STEP 2: CALCULATE CHECKSUMS AND STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 2: CALCULATE CHECKSUMS AND STATISTICS")
print("=" * 80)

manifest = {
    'generated_at': datetime.now().isoformat(),
    'pipeline_version': 'TRIVET_v3.3',
    'prompt_versions': {
        'etca_lite': 'cell15_etca_lite_v1',
        'tier2_generation': 'cell17_v2_tier0_aligned',
    },
    'files': {},
    'summary': {},
}

for src, dst_path in copied_files:
    dst_name = Path(dst_path).name
    stats = get_file_stats(dst_path)
    manifest['files'][dst_name] = stats
    print(f"  {dst_name}: {stats.get('rows', stats.get('items', '?'))} items, {stats['size_bytes']:,} bytes")

# =============================================================================
# STEP 3: GENERATE SUMMARY STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("STEP 3: GENERATE SUMMARY STATISTICS")
print("=" * 80)

# Load key files for summary
summary = {}

# Tier1 stats
tier1_a_path = f"{OUTPUT_DIR}/TIER1_CLEAN_A_FINAL.csv"
if Path(tier1_a_path).exists():
    df = pd.read_csv(tier1_a_path)
    summary['tier1_clean_a'] = {
        'rows': len(df),
        'classes': df['Sub-Subtype'].nunique() if 'Sub-Subtype' in df.columns else 0,
    }
    print(f"✅ Tier1 Clean_A: {len(df):,} rows, {summary['tier1_clean_a']['classes']} classes")

tier1_b_path = f"{OUTPUT_DIR}/TIER1_CLEAN_B_FINAL.csv"
if Path(tier1_b_path).exists():
    df = pd.read_csv(tier1_b_path)
    summary['tier1_clean_b'] = {
        'rows': len(df),
    }
    print(f"✅ Tier1 Clean_B: {len(df):,} rows")

# ETCA stats
etca_path = f"{OUTPUT_DIR}/ETCA_SEEDS_READY.csv"
if Path(etca_path).exists():
    df = pd.read_csv(etca_path)
    summary['etca_seeds_ready'] = {
        'rows': len(df),
    }
    print(f"✅ ETCA Seeds Ready: {len(df):,} rows")

# Tier2 stats
tier2_path = f"{OUTPUT_DIR}/TIER2_GENERATED_FINAL.csv"
if Path(tier2_path).exists():
    file_size = Path(tier2_path).stat().st_size
    if file_size > 100:  # File has actual content
        df = pd.read_csv(tier2_path)
        summary['tier2_generated'] = {
            'rows': len(df),
            'classes': df['Sub-Subtype'].nunique() if 'Sub-Subtype' in df.columns else 0
        }
        print(f"✅ Tier2 Generated: {len(df)} rows")
    else:
        print(f"⚠️ Tier2 Generated: File is empty ({file_size} bytes) - check Cell 17B output")
        summary['tier2_generated'] = {'rows': 0, 'classes': 0, 'status': 'empty'}
    print(f"✅ Tier2 Generated: {len(df):,} rows")

# Few-shot bank stats
fewshot_path = f"{OUTPUT_DIR}/TIER2_FEWSHOT_BANK_FINAL.csv"
if Path(fewshot_path).exists():
    df = pd.read_csv(fewshot_path)
    summary['fewshot_bank'] = {
        'rows': len(df),
        'classes': df['Sub-Subtype'].nunique() if 'Sub-Subtype' in df.columns else 0,
    }
    print(f"✅ Few-Shot Bank: {len(df):,} rows, {summary['fewshot_bank']['classes']} classes")

manifest['summary'] = summary

# =============================================================================
# STEP 4: SAVE MANIFEST
# =============================================================================

print("\n" + "=" * 80)
print("STEP 4: SAVE MANIFEST")
print("=" * 80)

manifest_path = f"{OUTPUT_DIR}/PIPELINE_MANIFEST.json"
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {manifest_path}")

# =============================================================================
# STEP 5: GENERATE SUMMARY REPORT
# =============================================================================

print("\n" + "=" * 80)
print("STEP 5: GENERATE SUMMARY REPORT")
print("=" * 80)

report = f"""
================================================================================
TRIVET PIPELINE - FINAL ARTIFACT SUMMARY
================================================================================
Generated: {datetime.now().isoformat()}
Pipeline Version: TRIVET v3.3

================================================================================
TIER1 ARTIFACTS (MT-Mined Validation)
================================================================================
"""

if 'tier1_clean_a' in summary:
    report += f"""
TIER1_CLEAN_A_FINAL.csv
  - Rows: {summary['tier1_clean_a']['rows']:,}
  - Classes: {summary['tier1_clean_a']['classes']}
  - Quality: High (CPS ≥ 0.65, OCS = 1.0)
"""

if 'tier1_clean_b' in summary:
    report += f"""
TIER1_CLEAN_B_FINAL.csv
  - Rows: {summary['tier1_clean_b']['rows']:,}
  - Quality: Acceptable (CPS ≥ 0.55, OCS = 1.0)
"""

report += """
================================================================================
TIER2 ARTIFACTS (Few-Shot Bank & Generation)
================================================================================
"""

if 'fewshot_bank' in summary:
    report += f"""
TIER2_FEWSHOT_BANK_FINAL.csv
  - Rows: {summary['fewshot_bank']['rows']:,}
  - Classes: {summary['fewshot_bank']['classes']}
  - Priority: Tier0 (gold) → ETCA-validated Tier1 (fallback)
"""

if 'tier2_generated' in summary:
    report += f"""
TIER2_GENERATED_FINAL.csv
  - Rows: {summary['tier2_generated']['rows']:,}
  - Classes: {summary['tier2_generated']['classes']}
  - Schema: Tier0-aligned (no MT Tool)
"""

if 'etca_seeds_ready' in summary:
    report += f"""
ETCA_SEEDS_READY.csv
  - Rows: {summary['etca_seeds_ready']['rows']:,}
  - Status: Anchor-verified, recommended for Tier2
"""

report += """
================================================================================
HUMAN EVALUATION ARTIFACTS
================================================================================

HUMANEVAL_MASTER.csv
  - Contains human annotation columns
  - Ready for annotation by human evaluators

HUMANEVAL_TRAINING.csv
  - No human columns
  - Ready for ML training/evaluation

================================================================================
FILE CHECKSUMS (MD5)
================================================================================
"""

for filename, stats in manifest['files'].items():
    report += f"{filename}: {stats['checksum_md5']}\n"

report += """
================================================================================
USAGE NOTES
================================================================================

1. TIER1 files are the validated MT-mined samples.
2. TIER2_FEWSHOT_BANK is for prompting LLMs to generate new samples.
3. TIER2_GENERATED contains synthetic samples (verify before use).
4. HUMANEVAL_MASTER is for human annotation campaigns.
5. All checksums are MD5 for verification.

================================================================================
"""

report_path = f"{OUTPUT_DIR}/PIPELINE_SUMMARY.txt"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"✅ Saved: {report_path}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("🎉 CELL 23 COMPLETE")
print("=" * 80)

print(f"""
📁 FINAL ARTIFACTS IN: {OUTPUT_DIR}/

📋 FILES EXPORTED:
""")

for f in sorted(Path(OUTPUT_DIR).glob("*")):
    size = f.stat().st_size
    print(f"   {f.name:<40} ({size:,} bytes)")

print(f"""
📊 SUMMARY:
   Tier1 Clean_A:    {summary.get('tier1_clean_a', {}).get('rows', 'N/A')} rows
   Tier1 Clean_B:    {summary.get('tier1_clean_b', {}).get('rows', 'N/A')} rows
   ETCA Seeds:       {summary.get('etca_seeds_ready', {}).get('rows', 'N/A')} rows
   Few-Shot Bank:    {summary.get('fewshot_bank', {}).get('rows', 'N/A')} rows
   Tier2 Generated:  {summary.get('tier2_generated', {}).get('rows', 'N/A')} rows

✅ All artifacts frozen with checksums
✅ Ready for distribution and human evaluation
""")

In [ ]:
# =============================================================================
# CELL 24: FREEZE DATASET (SCIENTIFIC REPRODUCIBILITY)
# =============================================================================
"""
This cell freezes the dataset for scientific reproducibility.

What "Freezing" Means:
- The benchmark definition, generation policy, and data split are FIXED
- No regeneration, no filtering, no silent fixes after this point
- Only new versions if methodology changes

What Gets Frozen:
1. Core dataset files (Tier0, Tier1, Tier2, merged)
2. Generation metadata (config, quality report, vocab stats)
3. Schema definition (column definitions, class taxonomy)
4. SHA256 checksums (tamper-proof verification)

Output Structure:
    TRIVET_BENCHMARK_v1/
    ├── data/
    │   ├── tier0_gold_manual.csv
    │   ├── tier1_synthetic_clean.csv
    │   ├── tier2_synthetic_v1.csv
    │   └── tier_all_v1.csv
    ├── metadata/
    │   ├── generation_config_v1.json
    │   ├── quality_report_v1.json
    │   ├── domain_vocabulary_report_v1.csv
    │   ├── duplication_stats_v1.csv
    │   └── few_shot_bank_v1.csv
    ├── schema/
    │   └── tier_schema_v1.json
    ├── checksums/
    │   └── SHA256SUMS.txt
    └── README.md
"""

import os
import json
import hashlib
import shutil
import pandas as pd
from datetime import datetime

print("=" * 70)
print("CELL 24: FREEZE DATASET (SCIENTIFIC REPRODUCIBILITY)")
print("=" * 70)

# =============================================================================
# CONFIGURATION
# =============================================================================

# Version identifier (increment for new releases)
VERSION = "v1"
BENCHMARK_NAME = f"TRIVET_BENCHMARK_{VERSION}"

# Source paths (adjust as needed)
SOURCE_PATHS = {
    # Core data
    "tier0": "tier0_canonical_examples.csv",
    "tier1_clean_a": "tier1_clean_A1.csv",
    "tier1_clean_b": "tier1_clean_B1.csv",
    "tier2_raw": "tier1_v3_3_final/tier2_generated_raw.csv",
    "tier2_enriched": "tier1_v3_3_final/tier2_generated_enriched.csv",

    # Metadata
    "quality_report": "tier1_v3_3_final/tier2_quality_report.json",
    "few_shot_bank": "tier1_v3_3_final/few_shot_bank_tier2.csv",
    "domain_vocab": "tier1_v3_3_final/plots/domain_vocabulary_report.csv",
}

# Output directory
OUTPUT_DIR = BENCHMARK_NAME

# Taxonomy (23 classes)
TAXONOMY_23 = [
    "Meaning Shift", "Total Omission", "Partial Translation", "Name Entity Error",
    "Terminology Substitution", "Literal Translation", "Hypernym for Hyponym",
    "Hyponym for Hypernym", "Tanween Omission", "Gender Disagreement",
    "Definiteness Shift", "Perfective to Progressive", "Progressive to Perfective",
    "Tense Shift Under Negation", "Wrong Structure", "Wrong Word Order",
    "Active to Passive Voice", "Passive to Active Voice", "Noun to Adjective",
    "Adjective to Noun", "Register Mismatch", "Spelling Error", "Invalid Pattern",
]

# =============================================================================
# STEP 1: CREATE DIRECTORY STRUCTURE
# =============================================================================
print("\n" + "=" * 70)
print("STEP 1: CREATE DIRECTORY STRUCTURE")
print("=" * 70)

dirs = [
    f"{OUTPUT_DIR}/data",
    f"{OUTPUT_DIR}/metadata",
    f"{OUTPUT_DIR}/schema",
    f"{OUTPUT_DIR}/checksums",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"✅ Created: {d}")

# =============================================================================
# STEP 2: COPY/PROCESS CORE DATA FILES
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2: COPY CORE DATA FILES")
print("=" * 70)

files_copied = {}

# --- Tier0: Gold Manual ---
if os.path.exists(SOURCE_PATHS["tier0"]):
    tier0_df = pd.read_csv(SOURCE_PATHS["tier0"])
    tier0_df['tier'] = 'Tier0'
    tier0_path = f"{OUTPUT_DIR}/data/tier0_gold_manual.csv"
    tier0_df.to_csv(tier0_path, index=False, encoding='utf-8-sig')
    files_copied["tier0_gold_manual.csv"] = len(tier0_df)
    print(f"✅ Tier0 (gold): {len(tier0_df)} samples")
else:
    print(f"⚠️ Tier0 not found: {SOURCE_PATHS['tier0']}")

# --- Tier1: Synthetic Clean (merge A + B) ---
tier1_dfs = []
for key in ["tier1_clean_a", "tier1_clean_b"]:
    if os.path.exists(SOURCE_PATHS[key]):
        df = pd.read_csv(SOURCE_PATHS[key])
        df['tier'] = 'Tier1'
        df['source_file'] = key
        tier1_dfs.append(df)
        print(f"   Loaded {key}: {len(df)} samples")

if tier1_dfs:
    tier1_df = pd.concat(tier1_dfs, ignore_index=True)
    tier1_path = f"{OUTPUT_DIR}/data/tier1_synthetic_clean.csv"
    tier1_df.to_csv(tier1_path, index=False, encoding='utf-8-sig')
    files_copied["tier1_synthetic_clean.csv"] = len(tier1_df)
    print(f"✅ Tier1 (synthetic): {len(tier1_df)} samples")
else:
    print("⚠️ No Tier1 files found")

# --- Tier2: LLM Generated ---
tier2_df = None
tier2_source = SOURCE_PATHS.get("tier2_enriched") or SOURCE_PATHS.get("tier2_raw")
if os.path.exists(tier2_source):
    tier2_df = pd.read_csv(tier2_source)
    tier2_df['tier'] = 'Tier2'
    tier2_path = f"{OUTPUT_DIR}/data/tier2_synthetic_{VERSION}.csv"
    tier2_df.to_csv(tier2_path, index=False, encoding='utf-8-sig')
    files_copied[f"tier2_synthetic_{VERSION}.csv"] = len(tier2_df)
    print(f"✅ Tier2 (LLM): {len(tier2_df)} samples")
else:
    # Try alternative paths
    for alt_path in ["tier2_generated_raw.csv", "tier2_generated_raw2.csv", "tier2_generated_raw(2).csv"]:
        if os.path.exists(alt_path):
            tier2_df = pd.read_csv(alt_path)
            tier2_df['tier'] = 'Tier2'
            tier2_path = f"{OUTPUT_DIR}/data/tier2_synthetic_{VERSION}.csv"
            tier2_df.to_csv(tier2_path, index=False, encoding='utf-8-sig')
            files_copied[f"tier2_synthetic_{VERSION}.csv"] = len(tier2_df)
            print(f"✅ Tier2 (LLM): {len(tier2_df)} samples (from {alt_path})")
            break
    else:
        print(f"⚠️ Tier2 not found")

# --- Merged: All Tiers ---
print("\n   Creating merged dataset...")
all_dfs = []

# Standardize columns for merging
standard_cols = ['Sentence_ID', 'Sub-Subtype', 'Keyword', 'Best_Match', 'ar', 'mt_output', 'en', 'tier']

# Column mapping for different sources
col_maps = {
    'error_type': 'Sub-Subtype',
    'clean_ar': 'ar',
    'error_ar': 'mt_output',
    'keyword': 'Keyword',
    'best_match': 'Best_Match',
}

for tier_file in ['tier0_gold_manual.csv', 'tier1_synthetic_clean.csv', f'tier2_synthetic_{VERSION}.csv']:
    tier_path = f"{OUTPUT_DIR}/data/{tier_file}"
    if os.path.exists(tier_path):
        df = pd.read_csv(tier_path)
        # Apply column mapping
        df = df.rename(columns={k: v for k, v in col_maps.items() if k in df.columns})
        # Select available standard columns
        available_cols = [c for c in standard_cols if c in df.columns]
        df_std = df[available_cols].copy()
        # Add tier if missing
        if 'tier' not in df_std.columns:
            df_std['tier'] = tier_file.split('_')[0].capitalize()
        all_dfs.append(df_std)

if all_dfs:
    merged_df = pd.concat(all_dfs, ignore_index=True)
    merged_path = f"{OUTPUT_DIR}/data/tier_all_{VERSION}.csv"
    merged_df.to_csv(merged_path, index=False, encoding='utf-8-sig')
    files_copied[f"tier_all_{VERSION}.csv"] = len(merged_df)
    print(f"✅ Merged (all tiers): {len(merged_df)} samples")

# =============================================================================
# STEP 3: SAVE GENERATION METADATA
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3: SAVE GENERATION METADATA")
print("=" * 70)

# --- Generation Config ---
generation_config = {
    "version": VERSION,
    "frozen_at": datetime.now().isoformat(),
    "taxonomy": {
        "num_classes": 23,
        "classes": TAXONOMY_23,
    },
    "generation_settings": {
        "model": "gpt-4o",
        "samples_per_class_target": 50,
        "few_shot_examples_per_class": {
            "default": 3,
            "hard_classes": 5,
        },
        "hard_classes": [
            "Tense Shift Under Negation", "Noun to Adjective", "Spelling Error",
            "Total Omission", "Definiteness Shift", "Adjective to Noun",
            "Wrong Structure", "Hyponym for Hypernym", "Invalid Pattern", "Register Mismatch",
        ],
    },
    "diversity_controls": {
        "strict_sentence_dedup": True,
        "keyword_duplication_rate_default": 0.15,
        "keyword_duplication_rate_hard_classes": 0.30,
        "max_keyword_repeats_cap": 4,
    },
    "quality_thresholds": {
        "min_keyword_in_ar": True,
        "min_best_match_in_mt": True,
        "no_identical_ar_mt": True,
    },
    "data_sources": {
        "tier0": "Human-annotated gold examples from UN parallel corpus",
        "tier1": "ETCA-validated synthetic examples from MT mining",
        "tier2": "LLM-generated examples using few-shot prompting",
    },
}

config_path = f"{OUTPUT_DIR}/metadata/generation_config_{VERSION}.json"
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(generation_config, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {config_path}")

# --- Quality Report ---
if os.path.exists(SOURCE_PATHS.get("quality_report", "")):
    shutil.copy(SOURCE_PATHS["quality_report"], f"{OUTPUT_DIR}/metadata/quality_report_{VERSION}.json")
    print(f"✅ Copied: quality_report_{VERSION}.json")
else:
    # Generate basic quality report
    if tier2_df is not None:
        error_col = 'Sub-Subtype' if 'Sub-Subtype' in tier2_df.columns else 'error_type'
        quality_report = {
            "total_samples": len(tier2_df),
            "classes": int(tier2_df[error_col].nunique()) if error_col in tier2_df.columns else 0,
            "samples_per_class": {k: int(v) for k, v in tier2_df[error_col].value_counts().to_dict().items()} if error_col in tier2_df.columns else {},
        }
        with open(f"{OUTPUT_DIR}/metadata/quality_report_{VERSION}.json", 'w', encoding='utf-8') as f:
            json.dump(quality_report, f, indent=2, ensure_ascii=False)
        print(f"✅ Generated: quality_report_{VERSION}.json")

# --- Few-shot Bank ---
if os.path.exists(SOURCE_PATHS.get("few_shot_bank", "")):
    shutil.copy(SOURCE_PATHS["few_shot_bank"], f"{OUTPUT_DIR}/metadata/few_shot_bank_{VERSION}.csv")
    print(f"✅ Copied: few_shot_bank_{VERSION}.csv")

# --- Domain Vocabulary ---
if os.path.exists(SOURCE_PATHS.get("domain_vocab", "")):
    shutil.copy(SOURCE_PATHS["domain_vocab"], f"{OUTPUT_DIR}/metadata/domain_vocabulary_report_{VERSION}.csv")
    print(f"✅ Copied: domain_vocabulary_report_{VERSION}.csv")

# --- Duplication Stats ---
if tier2_df is not None:
    error_col = 'Sub-Subtype' if 'Sub-Subtype' in tier2_df.columns else 'error_type'
    keyword_col = 'Keyword' if 'Keyword' in tier2_df.columns else 'keyword'

    if error_col in tier2_df.columns and keyword_col in tier2_df.columns:
        dup_stats = []
        for cls in tier2_df[error_col].unique():
            cls_data = tier2_df[tier2_df[error_col] == cls]
            total = len(cls_data)
            unique_kw = cls_data[keyword_col].nunique()
            dup_rate = 1 - (unique_kw / total) if total > 0 else 0
            dup_stats.append({
                'class': cls,
                'total_samples': total,
                'unique_keywords': unique_kw,
                'duplication_rate': round(dup_rate, 4),
            })

        dup_df = pd.DataFrame(dup_stats)
        dup_df.to_csv(f"{OUTPUT_DIR}/metadata/duplication_stats_{VERSION}.csv", index=False)
        print(f"✅ Generated: duplication_stats_{VERSION}.csv")

# =============================================================================
# STEP 4: CREATE SCHEMA DEFINITION
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: CREATE SCHEMA DEFINITION")
print("=" * 70)

schema = {
    "version": VERSION,
    "description": "TRIVET Benchmark Schema - Arabic MT Error Classification",
    "columns": {
        "Sentence_ID": {
            "type": "string",
            "description": "Unique identifier for each sample",
            "format": "TIER_SOURCE_INDEX (e.g., T2_LLM_001)",
        },
        "Sub-Subtype": {
            "type": "string",
            "description": "Error class label (23 classes)",
            "enum": TAXONOMY_23,
        },
        "Keyword": {
            "type": "string",
            "description": "The anchor word/phrase in the clean Arabic sentence",
        },
        "Best_Match": {
            "type": "string",
            "description": "The erroneous replacement in mt_output, or '[OMITTED]' for omission errors",
        },
        "ar": {
            "type": "string",
            "description": "Clean Arabic reference sentence (correct translation)",
        },
        "mt_output": {
            "type": "string",
            "description": "Erroneous Arabic MT output (contains the error)",
        },
        "en": {
            "type": "string",
            "description": "English source sentence",
        },
        "tier": {
            "type": "string",
            "description": "Data tier/source",
            "enum": ["Tier0", "Tier1", "Tier2"],
            "definitions": {
                "Tier0": "Human-annotated gold examples (evaluation only)",
                "Tier1": "ETCA-validated synthetic examples",
                "Tier2": "LLM-generated examples",
            },
        },
    },
    "taxonomy": {
        "num_classes": 23,
        "hierarchy": {
            "Lexical-Semantic": [
                "Meaning Shift", "Total Omission", "Partial Translation",
                "Name Entity Error", "Terminology Substitution", "Literal Translation",
                "Hypernym for Hyponym", "Hyponym for Hypernym",
            ],
            "Morphological": [
                "Tanween Omission", "Gender Disagreement", "Definiteness Shift",
            ],
            "Tense-Aspect": [
                "Perfective to Progressive", "Progressive to Perfective",
                "Tense Shift Under Negation",
            ],
            "Syntactic": [
                "Wrong Structure", "Wrong Word Order",
                "Active to Passive Voice", "Passive to Active Voice",
            ],
            "Part-of-Speech": [
                "Noun to Adjective", "Adjective to Noun",
            ],
            "Stylistic-Surface": [
                "Register Mismatch", "Spelling Error", "Invalid Pattern",
            ],
        },
    },
    "constraints": {
        "minimal_pair": "ar and mt_output must be identical except for Keyword→Best_Match transformation",
        "keyword_presence": "Keyword must appear exactly once in ar",
        "best_match_presence": "Best_Match must appear in mt_output (except for omission: Best_Match='[OMITTED]')",
        "no_duplicates": "No duplicate sentences allowed within the dataset",
    },
}

schema_path = f"{OUTPUT_DIR}/schema/tier_schema_{VERSION}.json"
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {schema_path}")

# =============================================================================
# STEP 5: GENERATE SHA256 CHECKSUMS
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: GENERATE SHA256 CHECKSUMS")
print("=" * 70)

def sha256_file(filepath):
    """Calculate SHA256 hash of a file."""
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

checksums = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        if file == "SHA256SUMS.txt":
            continue
        filepath = os.path.join(root, file)
        rel_path = os.path.relpath(filepath, OUTPUT_DIR)
        checksum = sha256_file(filepath)
        checksums.append(f"{checksum}  {rel_path}")
        print(f"   {checksum[:16]}...  {rel_path}")

checksum_path = f"{OUTPUT_DIR}/checksums/SHA256SUMS.txt"
with open(checksum_path, 'w') as f:
    f.write(f"# TRIVET Benchmark {VERSION} - SHA256 Checksums\n")
    f.write(f"# Generated: {datetime.now().isoformat()}\n")
    f.write(f"# Verify with: sha256sum -c SHA256SUMS.txt\n\n")
    f.write("\n".join(checksums))

print(f"\n✅ Saved: {checksum_path}")

# =============================================================================
# STEP 6: CREATE README
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: CREATE README")
print("=" * 70)

# Get stats for README
tier0_count = files_copied.get("tier0_gold_manual.csv", 0)
tier1_count = files_copied.get("tier1_synthetic_clean.csv", 0)
tier2_count = files_copied.get(f"tier2_synthetic_{VERSION}.csv", 0)
total_count = files_copied.get(f"tier_all_{VERSION}.csv", 0)

readme_content = f"""# TRIVET Benchmark {VERSION}

## Translation Error Classification for Arabic MT

**Frozen Date:** {datetime.now().strftime("%Y-%m-%d")}
**Version:** {VERSION}

---

## Dataset Overview

| Tier | Description | Samples |
|------|-------------|---------|
| Tier0 | Human-annotated gold examples | {tier0_count:,} |
| Tier1 | ETCA-validated synthetic | {tier1_count:,} |
| Tier2 | LLM-generated | {tier2_count:,} |
| **Total** | **All tiers merged** | **{total_count:,}** |

---

## Taxonomy (23 Classes)

### Lexical-Semantic
- Meaning Shift
- Total Omission
- Partial Translation
- Name Entity Error
- Terminology Substitution
- Literal Translation
- Hypernym for Hyponym
- Hyponym for Hypernym

### Morphological
- Tanween Omission
- Gender Disagreement
- Definiteness Shift

### Tense-Aspect
- Perfective to Progressive
- Progressive to Perfective
- Tense Shift Under Negation

### Syntactic
- Wrong Structure
- Wrong Word Order
- Active to Passive Voice
- Passive to Active Voice

### Part-of-Speech
- Noun to Adjective
- Adjective to Noun

### Stylistic-Surface
- Register Mismatch
- Spelling Error
- Invalid Pattern

---

## Directory Structure

```
{BENCHMARK_NAME}/
├── data/
│   ├── tier0_gold_manual.csv      # Human-annotated (evaluation only)
│   ├── tier1_synthetic_clean.csv  # ETCA-validated
│   ├── tier2_synthetic_{VERSION}.csv       # LLM-generated
│   └── tier_all_{VERSION}.csv              # Merged dataset
├── metadata/
│   ├── generation_config_{VERSION}.json    # Generation parameters
│   ├── quality_report_{VERSION}.json       # Quality metrics
│   ├── few_shot_bank_{VERSION}.csv         # Few-shot examples used
│   └── duplication_stats_{VERSION}.csv     # Keyword diversity stats
├── schema/
│   └── tier_schema_{VERSION}.json          # Column definitions
├── checksums/
│   └── SHA256SUMS.txt              # File integrity verification
└── README.md
```

---

## Usage

### Loading the Dataset

```python
import pandas as pd

# Load merged dataset
df = pd.read_csv("{BENCHMARK_NAME}/data/tier_all_{VERSION}.csv")

# Filter by tier
tier2_only = df[df['tier'] == 'Tier2']

# Filter by class
meaning_shift = df[df['Sub-Subtype'] == 'Meaning Shift']
```

### Verifying Integrity

```bash
cd {BENCHMARK_NAME}/checksums
sha256sum -c SHA256SUMS.txt
```

---

## Freeze Policy

⚠️ **This dataset is FROZEN as of {datetime.now().strftime("%Y-%m-%d")}**

- ❌ No regeneration
- ❌ No filtering "because it looks better"
- ❌ No silent fixes
- ✅ Only new versions (v2, v3...) if methodology changes

---

## Citation

If you use this dataset, please cite:

```bibtex
@dataset{{trivet_benchmark_{VERSION},
  title = {{TRIVET: Translation Error Classification Benchmark for Arabic MT}},
  version = {{{VERSION}}},
  year = {{2025}},
  note = {{23-class taxonomy, {total_count:,} samples}}
}}
```

---

## License

[Specify license here]

---

## Contact

[Your contact information]
"""

readme_path = f"{OUTPUT_DIR}/README.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)
print(f"✅ Saved: {readme_path}")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("🎉 DATASET FREEZE COMPLETE")
print("=" * 70)

print(f"\n📁 BENCHMARK DIRECTORY: {OUTPUT_DIR}/")
print(f"\n📊 FROZEN ARTIFACTS:")
print(f"   Data files: {len([f for f in files_copied.keys() if f.endswith('.csv')])}")
print(f"   Metadata files: {len([f for f in os.listdir(f'{OUTPUT_DIR}/metadata')])}")
print(f"   Schema files: {len([f for f in os.listdir(f'{OUTPUT_DIR}/schema')])}")
print(f"   Checksums: {len(checksums)} files hashed")

print(f"\n📊 DATASET STATS:")
print(f"   Tier0 (gold):     {tier0_count:>6,} samples")
print(f"   Tier1 (synthetic):{tier1_count:>6,} samples")
print(f"   Tier2 (LLM):      {tier2_count:>6,} samples")
print(f"   ─────────────────────────────")
print(f"   TOTAL:            {total_count:>6,} samples")

print(f"\n⚠️ FREEZE POLICY:")
print(f"   ❌ No regeneration after this point")
print(f"   ❌ No filtering 'because it looks better'")
print(f"   ❌ No silent fixes")
print(f"   ✅ Only new versions (v2, v3...) if methodology changes")

print(f"\n📋 NEXT STEPS:")
print(f"   1. Verify checksums: cd {OUTPUT_DIR}/checksums && sha256sum -c SHA256SUMS.txt")
print(f"   2. Back up to cloud storage (Google Drive, GitHub release)")
print(f"   3. Proceed to classifier training with frozen data")
print(f"   4. Reference this version in your paper")

In [ ]:
# =============================================================================
# CELL 23: FINAL PIPELINE SUMMARY REPORT
# =============================================================================
"""
CELL 23: Final Pipeline Summary Report

Generates a comprehensive summary of the entire TRIVET pipeline run.
This is the final cell - provides overview for documentation and papers.

OUTPUTS:
- TRIVET_FINAL_ARTIFACTS/TRIVET_PIPELINE_REPORT.md (Markdown report)
- Console output with complete statistics
"""

import pandas as pd
import json
import os
from pathlib import Path
from datetime import datetime

print("=" * 80)
print("CELL 23: FINAL PIPELINE SUMMARY REPORT")
print("=" * 80)

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_DIR = "TRIVET_FINAL_ARTIFACTS"
WORKING_DIR = "tier1_v3_3_final"

# =============================================================================
# GATHER STATISTICS
# =============================================================================

print("\n" + "=" * 80)
print("GATHERING PIPELINE STATISTICS")
print("=" * 80)

stats = {
    'timestamp': datetime.now().isoformat(),
    'pipeline_version': 'TRIVET v3.3',
}

# Stage A Statistics
print("\n📊 STAGE A (Computational Validation):")

candidates_path = "candidates_TIER1.csv"
if Path(candidates_path).exists():
    df = pd.read_csv(candidates_path)
    stats['stage_a_input'] = len(df)
    print(f"   Input candidates: {len(df):,}")

clean_a_path = f"{WORKING_DIR}/tier1_clean_A_FROZEN.csv"
if Path(clean_a_path).exists():
    df = pd.read_csv(clean_a_path)
    stats['stage_a_clean_a'] = len(df)
    stats['stage_a_clean_a_classes'] = df['Sub-Subtype'].nunique() if 'Sub-Subtype' in df.columns else 0
    print(f"   Clean_A output: {len(df):,} ({stats['stage_a_clean_a_classes']} classes)")

clean_b_path = f"{WORKING_DIR}/tier1_clean_B_FROZEN.csv"
if Path(clean_b_path).exists():
    df = pd.read_csv(clean_b_path)
    stats['stage_a_clean_b'] = len(df)
    print(f"   Clean_B output: {len(df):,}")

noisy_path = f"{WORKING_DIR}/tier1_noisy.csv"
if Path(noisy_path).exists():
    df = pd.read_csv(noisy_path)
    stats['stage_a_noisy'] = len(df)
    print(f"   Noisy output: {len(df):,}")

# Calculate pass rate
if 'stage_a_input' in stats and 'stage_a_clean_a' in stats:
    total_clean = stats.get('stage_a_clean_a', 0) + stats.get('stage_a_clean_b', 0)
    pass_rate = total_clean / stats['stage_a_input'] * 100
    stats['stage_a_pass_rate'] = pass_rate
    print(f"   Pass rate (Clean): {pass_rate:.1f}%")

# ETCA Statistics
print("\n📊 ETCA AUDIT (LLM Verification):")

etca_audit_path = f"{WORKING_DIR}/tier2_seed_audit_cell15.csv"
if Path(etca_audit_path).exists():
    df = pd.read_csv(etca_audit_path)
    stats['etca_input'] = len(df)
    print(f"   Input samples: {len(df):,}")

    if 'tier2_seed_recommendation_final' in df.columns:
        recommended = df['tier2_seed_recommendation_final'].sum()
        stats['etca_recommended'] = int(recommended)
        stats['etca_rejected'] = len(df) - int(recommended)
        stats['etca_pass_rate'] = recommended / len(df) * 100
        print(f"   Recommended: {int(recommended):,} ({stats['etca_pass_rate']:.1f}%)")
        print(f"   Rejected: {stats['etca_rejected']:,}")

seeds_ready_path = f"{WORKING_DIR}/tier2_seeds_ready_cell15.csv"
if Path(seeds_ready_path).exists():
    df = pd.read_csv(seeds_ready_path)
    stats['etca_seeds_ready'] = len(df)
    print(f"   Seeds ready: {len(df):,}")

# Few-Shot Bank Statistics
print("\n📊 FEW-SHOT BANK:")

fewshot_path = f"{WORKING_DIR}/few_shot_bank_tier2.csv"
if Path(fewshot_path).exists():
    df = pd.read_csv(fewshot_path)
    stats['fewshot_total'] = len(df)
    stats['fewshot_classes'] = df['Sub-Subtype'].nunique() if 'Sub-Subtype' in df.columns else 0
    print(f"   Total examples: {len(df):,}")
    print(f"   Classes covered: {stats['fewshot_classes']}")

    if 'seed_source' in df.columns:
        source_counts = df['seed_source'].value_counts().to_dict()
        stats['fewshot_sources'] = source_counts
        for src, count in source_counts.items():
            print(f"   From {src}: {count}")

# Tier2 Generation Statistics
print("\n📊 TIER2 GENERATION:")

tier2_path = f"{WORKING_DIR}/tier2_generated.csv"
if Path(tier2_path).exists():
    df = pd.read_csv(tier2_path)
    stats['tier2_generated'] = len(df)
    stats['tier2_classes'] = df['Sub-Subtype'].nunique() if 'Sub-Subtype' in df.columns else 0
    print(f"   Generated samples: {len(df):,}")
    print(f"   Classes: {stats['tier2_classes']}")
else:
    print(f"   (Not yet generated)")

# =============================================================================
# GENERATE MARKDOWN REPORT
# =============================================================================

print("\n" + "=" * 80)
print("GENERATING MARKDOWN REPORT")
print("=" * 80)

report = f"""# TRIVET Pipeline - Final Report

**Generated:** {stats['timestamp']}
**Pipeline Version:** {stats['pipeline_version']}

---

## Executive Summary

The TRIVET (TRIple Validation with Error Taxonomy) pipeline processes MT-mined translation error candidates through multiple validation stages to produce high-quality annotated datasets.

### Key Results

| Stage | Input | Output | Pass Rate |
|-------|-------|--------|-----------|
| Stage A (Computational) | {stats.get('stage_a_input', 'N/A'):,} | {stats.get('stage_a_clean_a', 0) + stats.get('stage_a_clean_b', 0):,} | {stats.get('stage_a_pass_rate', 0):.1f}% |
| ETCA Audit (LLM) | {stats.get('etca_input', 'N/A'):,} | {stats.get('etca_recommended', 'N/A'):,} | {stats.get('etca_pass_rate', 0):.1f}% |

---

## Stage A: Computational Validation

Stage A applies deterministic validation using:
- **OCS (Option Constraint Score):** Must equal 1.0 (clitic-aware keyword matching)
- **CPS (Context Preservation Score):** Thresholds vary by error type
- **SFR (Semantic Fluency Ratio):** Embeddings-based semantic similarity

### Output Distribution

| Quality Tier | Samples | Description |
|--------------|---------|-------------|
| Clean_A | {stats.get('stage_a_clean_a', 0):,} | High quality (CPS ≥ 0.65) |
| Clean_B | {stats.get('stage_a_clean_b', 0):,} | Acceptable (CPS ≥ 0.55) |
| Noisy | {stats.get('stage_a_noisy', 0):,} | Below thresholds |

### Class Coverage

- **Classes in Clean_A:** {stats.get('stage_a_clean_a_classes', 0)} / 23

---

## ETCA Audit: LLM Verification

ETCA-Lite (Error Type Consistency Analysis) validates anchor correctness using Claude:

### Evaluation Axes (1-5 scale)

1. **anchor_validity:** Does the keyword/best_match correctly identify the error?
2. **phenomenon_clarity:** Is the error clearly observable?
3. **arabic_naturalness:** Is the MT output fluent Arabic?
4. **collateral_severity:** Are there unrelated distortions?
5. **tier2_seed_recommendation:** Binary (0/1) - safe to use as Tier2 seed?

### Results

| Metric | Value |
|--------|-------|
| Input samples | {stats.get('etca_input', 'N/A'):,} |
| Recommended | {stats.get('etca_recommended', 'N/A'):,} ({stats.get('etca_pass_rate', 0):.1f}%) |
| Rejected | {stats.get('etca_rejected', 'N/A'):,} |

---

## Few-Shot Bank

The few-shot bank contains high-quality examples for Tier2 generation.

### Priority Order

1. **Tier0 (Gold/Canonical)** — First priority
2. **ETCA-validated Tier1** — Fallback for gaps

### Statistics

| Metric | Value |
|--------|-------|
| Total examples | {stats.get('fewshot_total', 'N/A'):,} |
| Classes covered | {stats.get('fewshot_classes', 'N/A')} / 23 |
"""

# Add source breakdown if available
if 'fewshot_sources' in stats:
    report += "\n### Source Breakdown\n\n"
    for src, count in stats['fewshot_sources'].items():
        report += f"- **{src}:** {count} examples\n"

report += f"""
---

## Tier2 Generation

Tier2 synthetic samples are generated using the few-shot bank.

| Metric | Value |
|--------|-------|
| Generated samples | {stats.get('tier2_generated', 'N/A'):,} |
| Classes | {stats.get('tier2_classes', 'N/A')} |

### Schema

Tier2 samples follow the Tier0-aligned schema (no MT Tool column):
- Sentence_ID, en, ar, mt_output
- Keyword, Best_Match
- Linguistic Shift Type, Subtype, Sub-Subtype
- Severity fields and weights
- Research metadata (gen_model, gen_timestamp, etc.)

---

## Artifacts

All final artifacts are in `{OUTPUT_DIR}/`:

### Tier1 (Validated MT-Mined)
- `TIER1_CLEAN_A_FINAL.csv` — High quality samples
- `TIER1_CLEAN_B_FINAL.csv` — Acceptable quality samples
- `TIER1_NOISY_FINAL.csv` — Below threshold samples

### Tier2 (Few-Shot & Generated)
- `TIER2_FEWSHOT_BANK_FINAL.csv` — Few-shot examples
- `TIER2_FEWSHOT_BANK_FINAL.json` — Grouped by class
- `TIER2_GENERATED_FINAL.csv` — Synthetic samples

### Human Evaluation
- `HUMANEVAL_MASTER.csv` — With annotation columns
- `HUMANEVAL_TRAINING.csv` — Without annotation columns

### Metadata
- `PIPELINE_MANIFEST.json` — Checksums and statistics
- `PIPELINE_SUMMARY.txt` — Text summary

---

## Pipeline Flow Diagram

```
candidates_TIER1.csv ({stats.get('stage_a_input', '?'):,} samples)
        │
        ▼
   ┌─────────────────────────────────────┐
   │  STAGE A: Computational Validation  │
   │  - OCS = 1.0 (clitic-aware)         │
   │  - CPS thresholds by class          │
   │  - SFR semantic check               │
   └─────────────────────────────────────┘
        │
        ├──► Clean_A ({stats.get('stage_a_clean_a', '?'):,}) ──► ETCA Audit
        │                                           │
        ├──► Clean_B ({stats.get('stage_a_clean_b', '?'):,})    ▼
        │                                    Seeds Ready ({stats.get('etca_seeds_ready', '?'):,})
        └──► Noisy ({stats.get('stage_a_noisy', '?'):,})            │
                                                    ▼
                                          Few-Shot Bank
                                          (Tier0 + Tier1)
                                                    │
                                                    ▼
                                          Tier2 Generation
                                          ({stats.get('tier2_generated', '?'):,} samples)
```

---

## Reproducibility

All artifacts include:
- MD5 checksums in `PIPELINE_MANIFEST.json`
- Generation timestamps
- Model/prompt version identifiers

---

*Report generated by TRIVET Pipeline Cell 23*
"""

# Save report
os.makedirs(OUTPUT_DIR, exist_ok=True)
report_path = f"{OUTPUT_DIR}/TRIVET_PIPELINE_REPORT.md"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"✅ Saved: {report_path}")

# Save stats JSON
stats_path = f"{OUTPUT_DIR}/TRIVET_PIPELINE_STATS.json"
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, default=str)
print(f"✅ Saved: {stats_path}")

# =============================================================================
# FINAL CONSOLE OUTPUT
# =============================================================================

print("\n" + "=" * 80)
print("🎉 TRIVET PIPELINE COMPLETE")
print("=" * 80)

print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                        TRIVET PIPELINE - FINAL SUMMARY                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  STAGE A (Computational Validation)                                          ║
║  ├─ Input:     {stats.get('stage_a_input', 'N/A'):>7,} candidates                                    ║
║  ├─ Clean_A:   {stats.get('stage_a_clean_a', 'N/A'):>7,} samples ({stats.get('stage_a_clean_a_classes', 0)} classes)                         ║
║  ├─ Clean_B:   {stats.get('stage_a_clean_b', 'N/A'):>7,} samples                                     ║
║  └─ Pass Rate: {stats.get('stage_a_pass_rate', 0):>6.1f}%                                            ║
║                                                                              ║
║  ETCA AUDIT (LLM Verification)                                               ║
║  ├─ Input:       {stats.get('etca_input', 'N/A'):>5,} samples                                       ║
║  ├─ Recommended: {stats.get('etca_recommended', 'N/A'):>5,} samples ({stats.get('etca_pass_rate', 0):.1f}%)                          ║
║  └─ Rejected:    {stats.get('etca_rejected', 'N/A'):>5,} samples                                       ║
║                                                                              ║
║  FEW-SHOT BANK                                                               ║
║  ├─ Examples:  {stats.get('fewshot_total', 'N/A'):>7,}                                               ║
║  └─ Classes:   {stats.get('fewshot_classes', 'N/A'):>7,} / 23                                           ║
║                                                                              ║
║  TIER2 GENERATION                                                            ║
║  └─ Generated: {stats.get('tier2_generated', 'N/A'):>7,} samples                                     ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  📁 All artifacts saved to: {OUTPUT_DIR + '/':50s}║
╚══════════════════════════════════════════════════════════════════════════════╝

✅ Pipeline execution complete!
✅ Ready for human evaluation and downstream use.
""")

In [ ]:
import os

# Delete the OLD checkpoint (contains v2 data with invented keywords)
checkpoint_path = "tier1_v3_3_final/tier2_generation_checkpoint.jsonl"
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print(f"✅ Deleted old checkpoint: {checkpoint_path}")

# Also delete the output files
for f in [
    "tier1_v3_3_final/tier2_generated.csv",
    "tier1_v3_3_final/tier2_generated.json",
    "tier1_v3_3_final/tier2_generation_stats.json"
]:
    if os.path.exists(f):
        os.remove(f)
        print(f"✅ Deleted: {f}")

print("\n✅ Ready to re-run Cell 17 v3 from scratch!")